# Stage 2가 채널 오류에 대한 강건성을 높였는가? — v32

**재학습 없이** 완료된 원본 Stage 1(v12, 100epoch)과 그 자손인 Stage 2(v25 wide1375)를 고정해 비교합니다. v31과 **같은 32장·36개 전력 조건**을 사용합니다. 새 holdout이 아니라 기존 영상으로 원인을 점검하는 실험입니다. 전력축 이동·곡선 맞추기·오류율에 따른 이미지 선별은 없습니다.

## 실행

이 파일 하나를 **Colab GPU**에서 위부터 **모두 실행**하세요. H100/A100 등 기존 GPU를 사용하면 됩니다. Torch 재설치, BPG 빌드, LDPC, Drive API는 사용하지 않습니다.

Drive에 필요한 입력은 다음뿐입니다. NAS로 옮겼다면 원래 경로로 복원하세요.

- `FSO_data/GEE_test.zip`
- `FSO_data/rebuild_v12/rebuild_v12_run1/stage1_final.pth` — 완료된 100epoch 원본입니다. `stage1_epoch005`나 Stage 2 파일로 대체하면 안 됩니다.
- `FSO_data/stage2_error_range_to1_v25/extension_.../` 안의 `stage2_error_range_to1_summary.json`, `configuration.json`, `resume/stage2_resume_a.pth` — 기존 wide1375 가중치입니다. 정확한 summary SHA로 폴더를 자동 탐색합니다.

v27 BPG 파일, v28–v31 결과, 학습용 ImageNet/GEE ZIP은 필요 없습니다. 입력이 다르면 다른 가중치를 고르거나 처음부터 학습하지 않고 **누락된 원본 경로를 알려줍니다.** 모델은 순서대로 하나씩 GPU에 올립니다. 재개 체크포인트의 Adam은 추론에 복원하지 않습니다.

로컬 디스크 **10GiB**, Drive **0.5GiB** 정도 여유를 두세요. 처음에는 두 가중치(합계 약4.1GiB)의 로컬 복사·검증과 LPIPS 가중치 다운로드가 필요합니다. 그 뒤 **2,304회 채널 복원 + 무채널64회**만 실행합니다. 총 걸리는 시간은 전송과 GPU에 따라 달라지며, 조건별 진행률과 남은 추론 시간을 출력합니다.

새 결과만 `FSO_data/stage_effect_v32/paired_.../`에 저장합니다. 원본·휴지통·체크포인트는 수정/삭제하지 않습니다. 중단되면 위부터 다시 실행하면 됩니다. 완료한 **모델×전력 조건**은 검증 후 건너뜁니다. Torch/GPU가 달라지면 비교 결과를 섞지 않고 새 결과 폴더를 사용합니다.

## 무엇을 비교하나

맑음16QAM 14–42dBm, 눈QPSK 24–54dBm의 v31 전력 그리드를 유지합니다. **36dBm 초과는 시뮬레이터 진단일 뿐 장비 운용 권장값이나 논문 원래 측정점이 아닙니다.** 채널·수신기·14-bit 경로는 v31과 같습니다.

각 모델은 자기 encoder·codebook·decoder를 그대로 씁니다. 동일 영상/조건에는 **동일 seed·프레임 길이·채널 실현**을 사용하고 채널 gain/RNG 해시도 확인합니다. 두 모델의 토큰/파형은 달라질 수 있으므로 같은 BER·토큰 오류·실제 프레임 광전력이라고 가정하지 않습니다. 16QAM에서 토큰별 심볼 에너지가 달라지는 기존 동작을 숨기거나 정규화하지 않고 실제 Es/N0도 기록합니다.

기본 지표는 Alex LPIPS, 보조는 VGG LPIPS입니다. 작은 값이 좋은 점수입니다.

1. **원래 LPIPS:** 최종 복원 성능 자체를 비교합니다.
2. **채널 때문에 추가된 품질 저하:** 모델별 `채널 LPIPS − 그 모델의 무채널 LPIPS`를 같은 영상에서 계산합니다.
3. **Stage 2의 추가 저하 차이:** `(S2 채널−S2 무채널) − (S1 채널−S1 무채널)`입니다. **음수이면 Stage 2가 덜 악화된 것**입니다. 음수 값을 0으로 자르지 않습니다.

이미지 단위 paired bootstrap95% 구간을 표시합니다. 36개 전력을 36배의 독립 표본으로 세지 않으며, 이미지당 채널실현1회의 제한은 남습니다. 같은 dBm/명목 단위심볼 SNR에서 비교하고, 사후에 한 모델의 토큰 오류율에 맞춰 이미지나 전력을 다시 고르지 않습니다.

**이 비교가 말해주는 것은 두 checkpoint 전체의 차이입니다.** Stage 2에는 추가 업데이트와 encoder/codebook/decoder 변화가 함께 있으므로, 개선이 나오더라도 잡음 주입만의 인과 효과나 논문의 수치 재현 성공을 입증하지는 않습니다. 그러한 분리가 필요할 때만 같은 Stage 1에서 동일 업데이트 수의 무잡음/잡음 추가학습 대조를 고려합니다. 자동 재학습이나 모델 선택은 없습니다.

고정 모델을 채널 품질별로 평가해 graceful degradation을 확인하는 방식은 [DeepJSCC §IV](https://arxiv.org/abs/1809.01733)를 참고했습니다. 이 노트북의 paired clean-subtracted 대조·32장·전력 확장은 독립 진단 설계이며 [FSO-SC 원 논문](https://doi.org/10.1109/TCOMM.2025.3562356)의 공개되지 않은 설정을 복원했다는 주장이 아닙니다.

완료되면 **`stage_effect_v32_results.zip`** 하나를 보내주세요. 모델별 곡선, 무채널 대비 저하, paired 차이와 오류율 그래프, 이미지별 기록이 포함됩니다.


In [ ]:
%pip -q install lpips==0.1.4 matplotlib


In [ ]:
COMPACT_FINAL_INPUT_LOCK_V26={'author_original_test_set_or_receiver_claimed': False,
 'base_seed': 20261107,
 'bootstrap_samples': 2000,
 'bpg': {'bit_depth': 8,
         'build_recipe': 'libbpg098-cmake3283-v2',
         'chroma_format': '420',
         'cmake_version': '3.28.3',
         'color_space': 'ycbcr',
         'encoder': 'x265',
         'release': '0.9.8',
         'search_mode': 'binary_checked',
         'source_sha256': 'c0788e23bdf1a7d36cb4424ccb2fae4c7789ac94949563c4ad0e2569d3bf0095',
         'speed': 4},
 'chunk_size': 25,
 'clean_methods': ['fso_sc', 'bpg'],
 'clean_reconstructions': 400,
 'conditions': [{'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'clear16',
                 'launch_power_dbm': 30,
                 'modulation': '16QAM',
                 'pointing_error_urad': 20,
                 'scattered_power_ratio': 0.95,
                 'weather': 'clear',
                 'zenith_angle_deg': 30},
                {'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'clear16',
                 'launch_power_dbm': 33,
                 'modulation': '16QAM',
                 'pointing_error_urad': 20,
                 'scattered_power_ratio': 0.95,
                 'weather': 'clear',
                 'zenith_angle_deg': 30},
                {'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'clear16',
                 'launch_power_dbm': 36,
                 'modulation': '16QAM',
                 'pointing_error_urad': 20,
                 'scattered_power_ratio': 0.95,
                 'weather': 'clear',
                 'zenith_angle_deg': 30},
                {'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'snowqpsk',
                 'launch_power_dbm': 32,
                 'modulation': 'QPSK',
                 'pointing_error_urad': 15,
                 'scattered_power_ratio': 0.95,
                 'weather': 'snow',
                 'zenith_angle_deg': 0},
                {'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'snowqpsk',
                 'launch_power_dbm': 34,
                 'modulation': 'QPSK',
                 'pointing_error_urad': 15,
                 'scattered_power_ratio': 0.95,
                 'weather': 'snow',
                 'zenith_angle_deg': 0},
                {'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'snowqpsk',
                 'launch_power_dbm': 36,
                 'modulation': 'QPSK',
                 'pointing_error_urad': 15,
                 'scattered_power_ratio': 0.95,
                 'weather': 'snow',
                 'zenith_angle_deg': 0}],
 'data_manifest_json_sha256': '140d5919305f3592d144ed4a1a8f8d53dcd19e170b212416591b454c0ea4e7e2',
 'evaluation_count': 200,
 'expected_source_stage2_updates': 1375,
 'final_evaluation_selects_model': False,
 'geographic_independence_claimed': False,
 'ldpc': {'k': 1536,
          'llr_clip': 30,
          'max_iterations': 50,
          'n': 2048,
          'normalization': 0.8,
          'rate': 0.75,
          'seed': 20261028},
 'methods': ['fso_sc', 'bpg', 'bpg_ldpc'],
 'optimizer_updates': 0,
 'physical_reconstructions': 3600,
 'primary_lpips': 'alex_v0.1',
 'primary_model_selection': 'fixed v25 wide1375 selected before new evaluation by prior development3 '
                            'mean VGG; no test-dependent checkpoint switch',
 'profile': 'coherent_receiver_extension_v6',
 'rate_matching_scope': 'dataset mean source bitrate, not per-image constant rate or equal channel-use '
                        'budget',
 'rate_tolerance_bpp': 0.0001,
 'secondary_lpips': 'vgg_v0.1',
 'source_bpp_target': 0.21875,
 'source_branch': None,
 'source_branch_discovery': 'unique immediate extension_* child with exact pinned raw summary SHA256; '
                            'verify child name from common configuration',
 'source_checkpoint': {'bytes': 3250750999,
                       'configuration_file': 'configuration.json',
                       'configuration_sha256': 'bb0e546ded8dc5e6483d3f5f6d0b0a18dec6758ce1a8a535130d19e571fe590d',
                       'file': 'resume/stage2_resume_a.pth',
                       'model_state_sha256': 'c383d97f27ccbf12620a629b7666a7b63d62d8c2ca78f70d757938ee309f2f09',
                       'sha256': 'c4443bd14c0e3292e4a456a715e5de27a42855097a46244d93fc573fb656f121',
                       'updates': 1375},
 'source_code_sha256': '4a2f1764f6c00002335070dff6bd21def8d59b9557cc17775e69ae3eb3ea2324',
 'source_files_modified': False,
 'source_root_name': 'stage2_error_range_to1_v25',
 'source_runtime': {'cuda': '13.0',
                    'device': 'cuda',
                    'gpu': 'NVIDIA RTX PRO 6000 Blackwell Server Edition',
                    'torch': '2.11.0+cu130'},
 'source_summary_bytes': 5049075,
 'source_summary_file': 'stage2_error_range_to1_summary.json',
 'source_summary_file_sha256': 'd8d68f3129c91ea5e71f60c2ee64e700e8c0dab044dabe6ec5c729ac0aa3e9d1',
 'target_mean_bpg_bytes': 1792,
 'total_reconstructions': 4000,
 'training': False,
 'training_inventory_sha256': '04768543c975e0627eceb5615024145dde6fc1f4c636cfa614587a55fed3b0b3',
 'version': 'fixed_wide1375_compact_final_inputs_v26_1'}
COMPACT_FINAL_DATA_LOCK_V26={'all_file_sha256': ['0066845163f38a25828bfa1e29516b0a472661a09893d66fdf611694ad8f8284',
                     '0084f8c888def7ca8e5150edd21a7248b1feb0ba3ef9e1d0b3c988f46501f818',
                     '009bc6b1df70faa550bb399c021f4f7e9fa664eeab43d7c364d281aa28421946',
                     '00aee9401eab0787c80bed346259bd59328a10d561dcfa3ba3bb9c3c9cc3b95d',
                     '00c9a055214eedc9d0ef7dbf675b2a5d6c5dd1f936637b9052affd5f1b37ca0f',
                     '0127d7350d9710721527acfacfed0492dc9bacb0cee58108f994bee8cf03fec3',
                     '014aeb0ef7d76df016065061c6c381bcbbbd6fa8f4e287cc6fb400d6aa0cce4d',
                     '017f1ca9cf29c2ba58943ff61f089f2fbde10ca6b7327ad05283f4b2ab12f20a',
                     '02cfc775aafdb1fdd0382189d3cb813f2c890a1fb6e176d907fda86ab905a684',
                     '02f2301e2a66ba274c929367147510d2429dab0907f78acb13ced9e28ef2f1bb',
                     '030c5987e036da1be5245fbc9385db5a7cdc09f2743d5ea25478c503e0cb68f1',
                     '031288bf4dc636a13973e5523dc48f03df93d0d8ae0da94a4101cb226bf23c42',
                     '0318784a8740a6f2c4539e77cdcc938168574309e179b3a3ea56c77336802023',
                     '031dba86bd4ffa1a1d51960b39da70c5ce50b3edf183b70dda9d3218f905187b',
                     '036bca3d81f26090a944f59a858a923d5975c5c5176d5e010ab25c5a7715d3f0',
                     '0371dd8e6d255c728c45103c8adc87fbf93770559b1b2534f4fc848d1a58b9ea',
                     '03b71f226ea44462a1ff05875a802b60fe89c41a952f879979956d32b8cc57c6',
                     '03e2e9232c3afe0a759861d9ec32a74c194bd6d27997aec69db31be536f4a322',
                     '03fbbc01f1bcbdfa658955e88e044dda30d316ba3075bcac14bab0ffd6df2d61',
                     '040b69056ff6a04a13d1d6bb91c0c479efe2ac4f8fcb60efb0a83975496e750c',
                     '0428ba35ff505fa6cedc57e046a455ade950dcd487cdfee7a94743c494a68d2d',
                     '04af976817fd65aa924e2147b45d6c97c8134b18113053f41cce8c07566440e0',
                     '04dff21d7d781ff8ea2ed80f6da1a165e7797f9a398b4fd930bb935d84fae8d8',
                     '04ead216e16cdb9c6176d5c581c10dbcdf3a01a98871533e0ca504ade94860c3',
                     '04eefd04b4ea48004fa383bcbf35aa751dedaa95efab605c77674021bc794f71',
                     '058d5c353be680ab65f033a8962772325d98cd96eafb59f6e70c077576f91ed4',
                     '05b5ec1e4f49623780d4dd121309ac91579e6aa5325b9f337aea24ea143648bc',
                     '05c79664ff9424c0b3c30f9ec2ab084687d490fd6d0f4948b2e51176f923acab',
                     '05e6ba245ddd2e59a3e021823ef935ebf49f6a273dcc3f2679902f74fa31940b',
                     '05fa9cbc11ca5d438f4a7fd8f2f263f85f0f014f034dcc099a711dee58022289',
                     '0679c8b0537a146003135e865121b10b489c18b661562795074a4c46987f2edd',
                     '074a0104e1e1f72a5e989f7ffbb8331981538a168f494e2a75e9b18aee71013b',
                     '075cf56c692f03d94602f7174152164e29f6fcc3e213fe20b86394d29c044af2',
                     '07b5f1ea62e221c7dd148a47a9c62d11ac460bef909ca3a737b5c395221fb146',
                     '07cd670dd70670e173fd703dc00c298cdf81cd60cbe5bb1884e4c281a89df151',
                     '07fcee7db0fa51cb125bf301e6eba10b263abaaad08b4949d695f088b52b1e77',
                     '087a8df98839fd9ccf3cd63df3817899c260d6429424e093386cef1c4c65cb3a',
                     '09143054231e9df0f34551bc20ea40775fed5b60e64f6f9c7292ec04e772ca59',
                     '093a0130893752bf4785ac38e567c9d799a365483a376c158f3a2561fe6170c6',
                     '0999f9393820e8917812ddd25413ba91f6f152fbd1220acb379a7a5474810d24',
                     '0a91dcc15c839e62816387cb8262b74a81f9143cc5baec723099d777c9469768',
                     '0a9239f28d97f43c0ab7e9a16e5ef50731094a71f9408b194519b5a616809780',
                     '0b3376578021a541194d1f0c26c97e16b508d706e0fdc0e53b2ef7bb989aaf5f',
                     '0b34c3f737905d058400a3b299b7853c27e4cbf302bfb4f2fcaa09047cadfac8',
                     '0b7dffb63da06365e02956ee0c0520dc5ff8812e339270410d5b9364f1a9eb6b',
                     '0b9cdd6af7411c4095d9840e3ee8ed559daa6dae5beebc93c16cc51d15c63506',
                     '0ba797f0b2beb834faf4dc70cd0cf0d872817f03b8f49b14bf536e32eeacf785',
                     '0bbf9f9de7802db5ce31a6fa718abcd6ba30789ff1500b5dd01b7e9da4768321',
                     '0bdb354e03f3f76aefbf2e28dc71e5eb60e8a8af095dd57c7e2a0320cc83e6df',
                     '0cb426ae32adfdc3288ea3c4a9a10b2c123ab3d4664a32b02f052e86f1bbc4c8',
                     '0d34e9157d178298911b0240717d824589011ebc35a8909da1de3ee1a9897a94',
                     '0df6c0ecf3cb3f92cfdd8940843309292f5f8fd9d60dec27cf56b196d43827ec',
                     '0e2ddf287e7584f51552daf7d41f4d85e9cd406da782015f5031a95cd95871a0',
                     '0e408be1135a55441fb76763bd9a2aa751b847f4a0f64a9959d8a7e3700eacc7',
                     '0e5b291376a58a933f15e5ca8adbd8987ec600c138d1308bab9c147416162e19',
                     '0e8ea29d040315ac021abec592b9f033b759adb5bb98570d1d01e46539d1b62f',
                     '0e9f754a8e12b5e0f56d4fe577ec1be800f8f84bc196db18144a989daac35552',
                     '0f328a45a3cf2bfb5a4cf60e13ef3dd6324a6af2b4b15a03e68dd864136c7984',
                     '0f38a11d51b8b18807a2ff0238896414bb4334639ca0acf36695fd55ffd80b4e',
                     '0f9fd78f1a6a56b61c0a7357fc27f8ddbb4fcd69346d8459981485b4389b165f',
                     '0fa3fae44c1eb015fe9adc1c7dd2da907acce30ef080449e5b1d53b71a9d843b',
                     '0fdda50d89fde702209df478033b4da09af4650288ecb4d364ff8af666dfea37',
                     '0fe27584cc2a358f288b63532fbe7a68d2596fd9b8aecd58467f0a982e7d0cfb',
                     '105fcab86f503a97dfef2f8f9a3a2cca2641306714df060f553ada3acd96c8dd',
                     '11305e2605ce0d223099b48600bc36d20a6706314f69c859b893bfdc25d05431',
                     '11cd45d44877e9aaa9c41984f0d1651c7321b5d7049ee0d2c21cb7d3582729cb',
                     '12383bd875b6a9639e103cfbe7e31d835157db1d1dd971bedf34c70546031a42',
                     '12535b48c89b2d7a339e1fa11c4edb84ec232f5d565200a06e5dd42413a70dbd',
                     '1258d9ee5de9d744f782f81570c18b0a9e8d8c407169977d57b98d70a4d246cb',
                     '1269e5f1b5b64c8442881cf6aae6380d387eced7312ec5e89d677466967ee3d2',
                     '12bfc81c2d82674c39ca0e07a1ab4075cd53e1e45059574e0ffa7db800dd1319',
                     '134b0136fbc951cc55e75a08a9335a524835aecbed8acf91332a655fc1af8c97',
                     '136955a3177da1c55d7623487904f0d0fa75568f1f4f4cb10bb833e99d816063',
                     '136d362b7f3df7ad18a07b0778c9478e0efee3157baabcae0afea5da412fc9d6',
                     '13bc2e029468235b81cde631b0475fa3405cb3ae076088d58937d66bdf3ac3cf',
                     '13c0f19424fae5c97500c90cff9e11ea5ee76a3a8623ea0e26f315f4d8e6cbc3',
                     '13cf06d91aecfbbf8ce84553c0cee40fb0a2f8a428aa78434b5371780686a56d',
                     '13fbe49abe6eacc07a41fee4cf891a280ce0ea3b02097e8f51942e855673af0b',
                     '13fce8aedce5fab9133488366c17473423687df9ccd5682f038ca815b008c817',
                     '1403729748e4eb03c9a7df679357e3788438e5b8ab6d49e351f6d993cd4e1319',
                     '14078b172fc8858b0b638c731fca5ac0bd40a18001005529f22c0525e3af18ff',
                     '141fe37ce0253cb31db8fd9b604c4c9673f5b7de9aea16d7c00c4758aa6f35e6',
                     '1422861477e119b080e3e93293b9f45cd0ce20067e9beceb1bac42ddb083428b',
                     '142f21c6e71616efbd5dc9ea64b99c6d0f07bb9c18a871a95b216715b8e1c827',
                     '1490bfe18e8954e3094927b5f38c66768ca55e7bbaa947db2b96ad94d28071e1',
                     '14b62856e863f705d95d59c8fc30cab9e9a2a3e07be3c5ea248fe841e0088444',
                     '14d704549005045ccdef7464bad5df4f81f8e3313d92fafad26d551d1b2e02e2',
                     '151bd74973256bf122a0260db34c96e4e102f20e3bc1fd0a6c93d049c7654d97',
                     '153b6bad06d0fde37a8defebaebada3a0e10926b7fceb967b944cdd1783bb603',
                     '15747090059d549da3c39e1f08c4928adca0378b0a00559922556cd3d0e64b45',
                     '1628c2c04b97b49be78116becb52a0c881654de00604da82928aa7b4dae90013',
                     '167b3d90ebbffc9837a5cee597b72b178c524652090642a58c0286621737dbb9',
                     '16877e5a83f8b18b70b3a9e5c01ba954cbac334efad1b5f5a6c49ff69777e96c',
                     '168fa335f79d34804812bdc3379df3ca3ecdbb470530caed5dfa028250ba4e02',
                     '16a21a2f6f98374ea4c2bad76692ceaa1c493f7e3ef1e494ec1059e6b17617c3',
                     '16b6565dc760a3f5ff57369247163d9c7dbfc5b7bb9446d8c65e480caaaa9f73',
                     '176260b73073aae1dcb99f05af28555b4682c100e063d4d4c45aebc6bb209a48',
                     '17836231febd88d85f1fe707ed8aff7c801fc3059e1579d9460c0322d7340b8d',
                     '17d71d00032256f7aa022b26da0cb06267b554a692893213b315feacc39d6ac5',
                     '181bd43cadd95ab231952fc078cacfe2384fe1125f877b4370147dd2685813df',
                     '1820bdb61280be77c3e6eb491953711815de08e201c8f6b2d6a9b65ce2b2c1f6',
                     '1832519d80c865562a7f448d63b365f9289a8d63764c5c08766cf9886f9d7087',
                     '18b6af8db9f87bb9fbeedc355792c117a12f04eeacc3e51e9e239917e6e7e7ae',
                     '18cb13aebb80e32029fe77bceb4a67a124f779e24fa87231288a59a30873b731',
                     '18d63182f56c67e61fc78dc3179ad89ab7e443d6c95f431721202fe0129de547',
                     '18e95729761855bd29115435ea4aa3720418e954fab04dc1996a423317d5ba75',
                     '191224e97d30f7ea88221396753fe3b179640d179f1248addbbd5e1ecafcdd75',
                     '1932b91f7bbe4cde3fe574328ab25ee7c7c92f8a1c8889ade9a5bcce8bb4a6fa',
                     '19ac7efc4f73f3596bcc209c2aa678c292aa11d5bc6366e835e2044c8002bb78',
                     '19c4310b6e09bc4b22082dfb82e88d33c8fd53daea76c08bbf32c099c93e22e2',
                     '1a012e6a3fa504fbf634665a7c8294daef728a218d6d8e5b3a51da49498dc017',
                     '1a4b3599f15372469347e6a3ba262d663076626d90890c31417d50e36a856149',
                     '1b8930c4e710f3484d1b6bbde884a4c24f83cd8b77a694a59716e849ee506841',
                     '1b902ece7dd95344dc5b01077ecce2550ab8b0535d5a0b885ae4d2db86964f3d',
                     '1bee85ca8055cdc8125538e865caa59bcece6aa90a149758d4bdca3ae855835e',
                     '1bf9feae32fe5ce0a380a3d37b4754ce1b18b63d05d6bb931854519ea1a661b6',
                     '1caaa04d1678b772bf5bdeee0982880657cdc8741609a1dd7beb4a67d0212e55',
                     '1d36b5bb3a75f95e0c2d624ba5acf329117b2a331f7f83624b0a819966a5e605',
                     '1d65cbf7f778fc51185fd3d4a883e44d60ae3df2d0333b55e31d90664e9889fa',
                     '1e0a49b6a81176ac9096f259f96258490442029f298a8dd6943c4b09076ddc7c',
                     '1ed1c79090c93e3f3ba4503b11b19d8f4a027c9ddcb4394fc3b6859584ea80f8',
                     '1f2b852bde28f71a1e9334e223dc81d717addd893e7870a5295fc78d33df212b',
                     '1f736a38f947b4863750c3508765ee1ab1d55d48398c6104f29e862a8f62c4d0',
                     '1fe265ca1362508807b461ed6103e4b815323de60f70cd2bfa11fd4640e0d808',
                     '20aaedee6d4486a09ff247ad24d80a562f0be2a338e35bdf8f82740d1255b0cf',
                     '20f7ff1137e25b4238efe36281032f8e219e48ff2bec837c02d6941c83b2c46c',
                     '20ff7ce26d56fdbb1f57239cc7f13d9c7187a2fe76d8d41fc4040cc67b793228',
                     '210cdd5b6d4be8d597547d69766af2e6e5d5087a3a9e4bfcaaaafc116451152b',
                     '217be8d51db955e31e325acf1f7c1e4e6032cce31647353d113b931128ffd3b9',
                     '217c563dc3fddfb462f4abbb7ccd872ded097413301f0fb166c180ffaf10680c',
                     '21977530aa79b9e0e499784b1827e6b75877f1f30b3debd2419377a9fccbeefb',
                     '21a1ca36770cac71ae26295310db18513925786dbf6fdc9c8e4ed8f99dd50154',
                     '21bff1f8c093cc63fead0b7638ffc9940ae579d8362b986fe77cf6835abf8c67',
                     '21ea5e7d5d4f6a52ae5b1775eb83bde484d36cd1c1cf08fd4cd6cdc01688cd03',
                     '22790101b93cb7b9e5f4136c6ac0ad559271f5d3130e1e488e863ce1ad5ec16e',
                     '228c1a35ffdf65cb9a368379ed028cbe98e0968d4a827522c6d7ccf09f574eb0',
                     '2295f88b3afc6eb315197029301c0611399638894e36d518de076c7c95112cf2',
                     '22b050d12e7582bd0b6b6a39bb5cdfc3b5b23ce3b376b88d5871877ec6b4b5d0',
                     '233bca9c796ea4e12411240ae6efaedb6187ccb4257cd2faeb825d758f72f7a3',
                     '234b820997661167ff7a92381ef0c3a8876b900b6890a24a54b748f290a4a2d5',
                     '2364459e6ed9584b7eb6513c29c655ea7db9debd65722a5f441e86b6f9000b72',
                     '23cb62fa2e1dc3c9dfd92c1190e7cdfa51889b067045a01de9db28b379171d85',
                     '240c1357b7891c548e77f9069bc09e03799c79e085eba7d24cfceb00387475e3',
                     '241f921d4cd85481e077b18605fb2d2b1ae4d52f954c86a8a743e603af0b75ec',
                     '2420281d08d9fd14717298d71717302f35a9299601607e41b548b37011e86c9f',
                     '242f9140b8650ed6065382d89b4df19bbc281fb8388f7e76b096daa17c6efe01',
                     '24cd687a287375d73d5cf90864e7c0314a180fe81af899cce5fe86ddb7f26d64',
                     '24f79f0a5e3b82171a9702b32f698668549ec806f5df4eae2f2eec402caed843',
                     '24f9acbe34a04732264fb9105f54cc565760bbdab686faff514336de0fe0cb3c',
                     '2591258305dd07328766dcffb7232b23ae78a88a61d9a878ed913105cd21c3e8',
                     '25f4bdb171cb66d609c3b41e5af6d9fe6c4b43a20478fa8817e4204ab33634e6',
                     '260c6068d56fddfecdaa367fbf870801776df37aaa2d954a6a0ddab092769a0c',
                     '263a7ec3ee646b32ca436cae9a4baeddeeeb3cff8b98c67aac39a8b099a6f7b9',
                     '26b9c2917759aa9d90241b26b7fc1de4e64ad52b1a6307a5ba33095d054e19c2',
                     '26c32b6b61f258765c735ccaa8f3377fd4aeffd8ab6c4d9eb1ba72ac34b7b73d',
                     '26fcc503337caf416882e5de6b7762129877c4926d72927db93c6dd3b3aecd6e',
                     '2732d8b5234671375ad29100adbf7bf36ca52500697e384398b5ed24d9eabdd4',
                     '2783792192d71135c704a7d48f4335e8c3b43673df6f77673edb3561afec80e7',
                     '279cc38806a4e37f44a799845db932d4839b34e7b14b933146bcff349661d78f',
                     '27a780b7a4eb0ca137c56aa2261ae63b279279ca452565fa8cbced8a40636ee6',
                     '27b8b43eae7602fda0940a6c698b675fdadf8d201f56146dcfcf32c5e7bcde59',
                     '27eb5e3c07fcc322a18a8a5a63fd03877e16b86c0516e6fe4cd656da5e280c05',
                     '27fd2d7e1e17bef7a17b67a751f6e48dd6d58c870df93fe72935a67224a9cf8f',
                     '2840069ca725acd3e097f15bd532375ddef326880f3683f2054f118e70ef98a4',
                     '284fab2961066535c65cb962f88aa86b463048395b305e1793c4849dd5603156',
                     '2909154fe14311910e85e24c5abfe8fd9523d819071f3f6556f2d910ca1b0f90',
                     '298dcc57143531d79d9dd82730cbe360c46766a4f573dcac764e947382556a7b',
                     '29f63c30a424c41864cf024bff7badecc8a25f141f50236c7eaca44c851d148f',
                     '2a3a2a6b2a5b321c19baf4de1a0431816c9e1299e1a67aa3bf1f2fccb447a6ad',
                     '2a40eac16287efd2e163305ea3a4b517ab642c23b229cc77ffdeb6f600e4e544',
                     '2a4f80a0d5d95539103c5c67136ecde9c3f1e62fe65fdcede56ef22628e8d091',
                     '2a64cd8728cbc2950dbcd72f61e3c88025e4892157c106c5bf1ae4d43bb7335e',
                     '2b19544397470829dab6f7a402fe4196b2654d8140465715e2a7e23a1b225a37',
                     '2b85d6901691431bde7b2cf2e1de969ffccdedacd63a1b2cd1047a7ebc69e10a',
                     '2c1c7e342534fd04b325eb242db82ea8d9cd8c58e996676330905cb2a45a59e7',
                     '2c353cb2f84bdc3b46be2025a09349b66886bd7a6e21764dafabf2ef73cabb36',
                     '2cb2bf160f187ff6be6a300886bbae7d855388251fde410f3ea962f8dea672ca',
                     '2ce23fdc68c43b25b9169ab09fa124885e5e4e0dc2c90cf875032a5950b5d8e8',
                     '2d28a2897efa0b74c752b4287e2facee710509c8e840aa849620a1a2b3b37104',
                     '2d90ad29da4cc1144a890a98c668f275336330d6a7b6ec3aebe8ea0b71b0c26c',
                     '2da7cc2b46d1630701efddaafeb0d17618649382f2e2adc6ceca325d208dab98',
                     '2dadda6e349ab535222e8937b573e325b662300c7dd7ac77782ab0ecf93b8525',
                     '2e63938b2fcd265175ae05748f4e266ac8dc01bd2a38294bd42d515f3a21ac85',
                     '2e8151a6bb8aeec62a83a29bdb93303bb7265cb28ef5deb9fc2bb6f933e4801e',
                     '2ef5e210eadc3638aecb896f3c3e906c284585ff0a1d1476b94ad3333feaf314',
                     '2f94a3f3b360d0847336e6167d47d7d3f81d8d36b34c970dcd36befccead0eee',
                     '2fad80f5c24928cfd53dce72ff07573f693291059baf3d78a37b139b195b8796',
                     '2fb3107b17ebb27324d93eaea065b85dc2f4538c305e8f1045df56fb583f7909',
                     '2fe7c85b683d6b09700bfa1d1e2ab63e696e53621b1255dbffbb41789ad50a18',
                     '3030f8580b36db8eecb0fbbc889a0ea673a3acad45836dd991f98e20e9e11517',
                     '3084c85b7fa890e77560bc72e757fadae863d9d32bfa30b3cb28cd74188ebcd9',
                     '30c131076a387144a12eea1f98f9a4fca2f5af812c0e4a57d7325adff5fb0c49',
                     '31bcd149f49c97e3fdac5b661e2b6b3cec17a132623ee307c36a92b3d39228db',
                     '31d7041f2573fe087dd00a9faa0f6c372fb958bea8b15a956af218977e933f4d',
                     '31f704c359ea51a9f5f599c7f331d0656792ba4f3fd6417cd35bd970d19c608b',
                     '321eabc5858dd5f95463c798407ba16a9a26459943269cdf0228ba025f20825f',
                     '3220a60a4764025feb88510610f61f0ba228041e66a56f3dc4a332ce1b9a884b',
                     '322bb75b4f5a05b0eba7ba80b796474b81e5f47951d18342e2b9223b137a2c46',
                     '3238bd3fa558879d98a3512ba61046f2f34272ae9319d7ca12a2fca67de1aedd',
                     '32955221a7888fae9aa1ba5a7bab8d6ff03e326ae972c3594cf4e49a1988334b',
                     '32aaab923cda5b9aa647da5818096ed7c6f2c9f857bf5c04f6238428fcd9bf4d',
                     '32b668e9e325e12ef684de614ce85d30fd2b2dd3496fa6a196f3f70270a65728',
                     '32cbf015d1710168320a9866696e2746807f2793b602a3328d18583cb8ba45ec',
                     '32dc68b3f5dbaf39eaa39b34639d7e34099b8cce9adc7ff75cd16be78db66878',
                     '3327f2a690c915ced7bcca0d4f81597dc24daa4a7a2c454826c46fb70695ab39',
                     '332c392379f7a2307826bc57368d861dae2c596475f3193d861ca586cf3ad873',
                     '339b927f038ba1c46dff3a7497ba07609578b8a85ee910819aebe439770f57dd',
                     '33aa26f99805b56707861c3c359796c18fc6f1a8b434cbe8dcb6e9c02901e00e',
                     '33b4869594fcfa5d1e19313cf9d9e79493a06d4e072464a4670c62c3b401d701',
                     '33fd7f7a26c5859a578948cc837d6b02038d763558c4c5c205dac8eca16ffe3a',
                     '34e30aa6fed35f75372c32706537f0c8053da8706567a99639742d36e23b297c',
                     '3558079ab6c1f10699922b3dae791721a79dea43128312de14bcc85f46d9165b',
                     '3567e09c58476c5c793bfa9db08e16833c0efcb31a723b11dede08b2e383420d',
                     '358ad67805609e1377ee55dd68ab5389b3659855b7f06c6274f3f54f8a7c6d4e',
                     '358bc297fcb7bc33ac812029f1b6917c9e25618300ed46b7b93c68eab5ac2b34',
                     '35eff67e26cac12bc12c75cdaee75d76b3dddb27ea3b4a0eb259b300addf3633',
                     '35f5b161b03998a66c750105c9c9d01a326c076f7f41ef5ec730ba4a6e9e1404',
                     '3635ff890769e4f426eb238fc1097748d7915d0ddadc0d01fe3e124dc6b36718',
                     '363c6a14c10015b31f753994b513daa3dde497fe485b75fb90ab2695f49d873f',
                     '36559d2545c16bb7eca9ee449b893c48596d2114b7b000a1819e16d4035e826e',
                     '367126122831aa7424a9498c641885faf8461456fd6e86b1d8b915d920401d81',
                     '3684e33c3883dab27bbab61eaec02a028c332b10416c3668810ff80593273227',
                     '36930bfca54fb42fa28a0a8939bdcb6181bfda3da5295f8e6d30d8b658ee8079',
                     '36984c7cebec9af59bdd0ced39e22c29d5208c02346001388601a989adfbd098',
                     '36df2fa4a13411acf35dbfae062a15ca58c58331e52a20019fffb17c3ebbe8a7',
                     '36e5a076cf2ea05749a06cb2bcc26e4fdf39cc250f8139d40b8182d03c7148f4',
                     '37604d0b16ed45d50ad73e92f69292b29ab552895940e6344fd0c27848841bde',
                     '37a34ab5c0eec0ca861e91e4a042b51fea62c0b2dfc0f3081719cfea57be04cb',
                     '37e7e20d527ee6051d49773937c09beb8d178c20de96786ec44494927e9e5190',
                     '38285e3424177b69ec49d8db865a0c83452012dd4a00b46e9dda6f406836110c',
                     '383f1af4bb2eaa49811f0027d7ec0f89db9b340aef36616367c692b5c81d5f24',
                     '384e0cda59102c2e30428d0327ed6d9de19a1ed8a5634423fc8d1456d950f920',
                     '38ef6a56173fd38bd4129130a9632bc3767d5369a2d486900f8568e43eb8d9f4',
                     '38f327e85514eb45fef5ae3eb06117ab2e16df93ec4a8b85e475d3e04265c69b',
                     '3938bbb08b82fe0378d1f55530f94e5645fb61ee09867340aefe5dc258f0070e',
                     '3983a97cb60d76efcdbb1b037e8b03a770cba61d413d2dfc58fd1017207d8635',
                     '39e1dba8068d2af620de27421472afbe76be97fdad46683b25745fc6deb883a8',
                     '39e989e179963e8cf061abc0bd32c30579dc0d77eb7ec363356284915c5d294e',
                     '3a4cde911a3bec8da83cd0b644ee297788d5b85a0af9f2eff2a5cfcd779c333e',
                     '3a52e5716470b7ee0a695cb522e1d38858bba011de37e93a3f31bac90874f3e3',
                     '3af527a45ed23d571fb6a8e24ab8eb5699621194310f012da20118a900e7a7f7',
                     '3b2c492954627e8bc310bc32893d825da555b631c34c5acaf633ca62e8d5a401',
                     '3b8b929104cf13efab270b7e56ba0f38183857955095ec84191495eac11c740f',
                     '3b96f65812c35e52be5d10261363d221385700c1cbd9e1cae02bca2eb358b771',
                     '3c68b109aa59c24b0bee0a32a25f67e6764731d6db59e604a29401e5f67076ce',
                     '3c892fa76bb770337f86ac817941f6623c04663e4e7bace7f0b8c3ced70f5d66',
                     '3c9c672675554754ee7385dfb80303f8fcf0915e80aafe56495bd3505ecfbe5c',
                     '3cbfaa6d4563c235b0907c2375dd44f3b96d591099def6946e870d66a2a86547',
                     '3d356c05b4ae18471dbbe483d7558246322d4f1ad06616b748239e9a257f68d8',
                     '3d5cf0e29b74cbd7119cdb100fb760769be2c0201990dbe0894abec1bb58212e',
                     '3dcd17e02aae26c95a49837a391966b0b650d44d575c751e3a47bb1c73d3c23e',
                     '3dd1e77c4355d1ab77b029fbeb02d4112853e312cc78cba125ae02e54ca49f5d',
                     '3e516e19ca3370762eb87c89f0eeb9a70979b5185aa3640609a97d09cc668753',
                     '3efa128f5ede36086fcc5ba77fc91e380b6da1e0c6f90c2f2362f7740c0ce5d4',
                     '3f103a95be2a938f16ec55e5a2a5fec46f3eb34b76fd10536e5fdefbe99e7813',
                     '3fc5066e006c9978ee0fe225c223a63ac4958acdaefba6d763cd6d3a7284f3c7',
                     '3fc9e0e2a63e532e2d55563c09ca19d2797bdb207f32d65419cbe5a19ed4d050',
                     '3fca60957977d0994804c0158199755aff91032f2c9c91db55154255ba36eb8a',
                     '3fdcbc48a62c2fcff67e436a6bdf7aae0a40d277694895c7e80144ba41a8ebe4',
                     '403d86d395da009dfea68201bb656e7af6cec9c56ca6f5a145ae5722e62c8698',
                     '40d9ebc0983e831d0f8780cac97d2dd788a681bc11f56f85d429311dceee5902',
                     '413f07cfa1eb10f6413ea8d977b392d05de767bf4bda512038954b9a004a13a0',
                     '4178ffc9cb3279e8269a677822af23c47e97ae3ae88873aad0cb5978f31ad19d',
                     '417d998a8270f1ef1050f140355d995126b567a3f02dba0186b6b56725ac8005',
                     '41bb5562b8edca1028ff34db563dd1d43445aeed5b9e1d080bbe63b0c4e657e9',
                     '41bd481208205366912cd9a36cfa1cc5c0407552b88c38d846b579692f7af208',
                     '41ff58423114103524a85024eb29549a416af37b154de6966992f43847ca0564',
                     '422945d0c129ed7803b523c91cde540f976da7618f4e2bc59aec187d0989dc53',
                     '4300d25abee91559787f8d70f087ffcc87e63b55c9f7f475947eef2ca7d0ec7f',
                     '4333f1987986561eafb75a8d6c91ab2d7d142b5d5305263cba0b571eca1b2eb4',
                     '43d5a5f4f30c04de152b912b9e50128ec2ff837ec4c0b340aee15e025a1eb905',
                     '43f460a6393a2b675e86d0b308cacb5fba50add169ed477976e7b66084aa7a43',
                     '44507d44cce9d30e3a58ed0a140ce9ebedc22a44c0ab9eff825f1253a638a0f2',
                     '445ecfb409bc55ea2edbf185a3301d54bb48978c552d66da81ab5f654ab3ba26',
                     '446896b6489600ee23701379b84232fca38edc64e8c4cc90efeb579006be6662',
                     '4477fcdd39a8d7496347489d19943f5e3034a19fdb1fbe7efc4ac80f967a71f8',
                     '44a5ccab459559e6c665ce80951d0e5a679d5c67950682ec8c4519836553b0df',
                     '455a8e3207deaab6af6caa2a85c4577aaebd1b494f6e5d2fb47ce1f7471a0347',
                     '455fbff1af91c14371e3ae707558819cf97c55f50e08f1273ff17c3e8352bb55',
                     '45b78077660897b40a0f02bd1166111058fc7a0c885f090a64562b5c95af7791',
                     '45de925223f0a62875e752bb4c4bdaddb13b60c7fdc7c2846c83130bfef27eb1',
                     '45fb36f975631ab002fef0cf6cbb878ecc1ee949eba5519c8dd53f70aec6cbdf',
                     '46533741c06ed67856bbaf606dcd67357979e06c9434991d8cbc88aa5ede851d',
                     '4723537bb00acb02744e4438846c85f14005d8673c66a842d30f8cc4a3a187df',
                     '47551a2489eb09ad2da883c6adec5e4fc2f6f12dc47d376d17f6dc7cc6fef285',
                     '4755d79b5742075aafe9136853b3a45c18e2c74de1a2e037d6a292604c17b154',
                     '47731bdc412904e4d597b0a40365380d6a69c373dd3cc8924bd1a73ce357d635',
                     '47b2eb51a1f560d0be0833c6d040a97d523054c16c052d0c6d0d8500f7be5507',
                     '47d78f81622fab2b16724c81c6cf0c578e2fb2796f232bd4eb24e594373559ca',
                     '48097204f0e1e608f4050b14ee5ed523e1fbd7f42e64144f3733b27634b2928c',
                     '481d4d1b796c64e8d9447a128154774a0e68d3168d06751c942704ef18e2b9e7',
                     '48229a8b14150adae365f0cb009bab911d9771dbbbd7bf959ea7c6403e8dd78e',
                     '48644c96046d9bfb85294d1df49ac65c615515b59c6ea7af721b2f0b2a88933a',
                     '486b18e3bd5b423384fe51027e0cdb1384e827ae1045c8a5b33626ab98d4e25c',
                     '4909caa91644f73e0c459679143722235a1aef1fc58621cce427861966c6a28d',
                     '494177b028681bf52c409f064078f954db33534a72e96370c1a7bfa11429cc39',
                     '49551daa261549f2f64482ccfb65b32fbb79e139be7a7f19cdcd12bb579df5d8',
                     '499c8b5c58b4cce859361df6854f179566072c4426c8d9ed020749ffe7f9b50f',
                     '49b38edf9a960824ab73270077ed5e9d52aea300ffe454336d35daf0dd90f812',
                     '49d7aed962c2ed37ab2c75bc32a946b70f07ac1d8fe56a91c15e1d4822dadf34',
                     '49fef414fb763998c606ba88cb83cb653a769244fcf0d4bd2026076045c14ecb',
                     '4a5666f982cdc7b359ba16bb64b8a2c57bc334d5cce141179377324d4e5a9611',
                     '4a6a2e6fc197b3f61e187b6c420399f456bd99d9ec4f7e678b4098792310ca0d',
                     '4af55d03853a2c14ed992d81f6db525d502f7d22aa21905879ac1f1ef1f12d52',
                     '4b656b4f43b3de43565b38e1f5a013193ae8fbceb330505aa30f46e0c7d266aa',
                     '4b6fb1c47826d61e76839279441979db0a33fad6d071fc5d360a0a9651c0d5ac',
                     '4bb7d85a488846ac1a91d8a95193831d8ab2d01b65939d6b50e439198302fb04',
                     '4bdfe57004a8cd7dae812253b057800ef3f265e0ea92ece98514856c15bdefaf',
                     '4bf6c554fa4928d8da6dc7a784cb7e6f275ebec4de98b15ddf02fa5d0a5dfb2d',
                     '4c0dc526b6db4afeace727eba01436f121fa40d20c366e48efbf98f46f50fda0',
                     '4c722fa5afb35046c62ddb85504502d535cbed2fecdd9fbc1ee6b464a11a8d0a',
                     '4c9cdbe77c5df3682c148b2f6d82f53d8273a815a8ee173d5ca7967b88706bc1',
                     '4d135c5e1622a00434e847ceda99c2a093a2127712ad3ab51594e18616bdc798',
                     '4d8dcb6103b5cde63d643b86d5232a092c0adb5ac64d5805cecacfa4301867a9',
                     '4da39f2289e89580e18ece044b3df3ea9694b8cccf0583aa6f8d2f7384b71f10',
                     '4dfbdd4d41adb1016f919671481acc0416531d65f119bc8f85743db5ff3cc616',
                     '4e2e74c6905213cfdbc9062f1318493c03bae0f8b383a6776cfa281d6e837afd',
                     '4e6784537fbf5e264f16ddd041cecc2bb9bd1a14af2717010d0823015b6e0a6f',
                     '4e77140028023d0c72512da6344a5a434a7123659aca077267d9f361fcd0d253',
                     '4e88410983856bf3a09e91b936db2ced9d720d09259f120d6f10011917b94fbd',
                     '4fc4725e453bb817aa0189d37f12123078c7608a04462d210c0c1c815be89dbe',
                     '501a19a8f83296edd3e291d899153d4af2b8d4307219ac31a257dd6e70c9eefb',
                     '502ae35ae6960632b52a750cd3dea6f181b75dfae008428db818483125274d2c',
                     '50c36b1dc6fe948923c1653b715c4806ebc86f061b2ab48f0e95c42eec544b94',
                     '51272a4151c4e994491cdc6c2ec65d64bf3dd2d70e4c223b578531635eb11042',
                     '513a05cf9fffd22ef77fff7686a9bf0054c06ed46ea506fde62763b867415e20',
                     '5144ab1e87c86a75492c7a6495c1bfef5778980a3803ca2caf00f3d8fc410f28',
                     '51d5b11aca2a022fceeb7f40f853cca843acb0927dbcd1451525a6d048c6a52d',
                     '521642452f15e3c142bd07fbcd015e9cb141fd29f947b584fcff64f9987e07fc',
                     '5229daa45017552fd32669dcf4a96e45dff97671f5adcff6aa2e46f7d0140112',
                     '523335490b51187974cf36b6ca64c91830fb4a7672609a4f671dd667fef1617a',
                     '5250c06edc7da6c627c5621e9c08ec7c3218efc99c3383b07de989438de3e393',
                     '526845f850b649ead4487cc741e0949e96fd81ba36f4da9810e93caff4e9aceb',
                     '52faea885ae054ee7266959cd930816ebf9fffa581e34729d05dc2394a64508c',
                     '531977dcd74ea6ec96f68e5c0496bdbb882ef80187bf6922542ebedfe30406b2',
                     '531c76caf1ac8c69f1d4d07a8e17a86c20f4e45977dd676d9c267f3a1b448e74',
                     '536ea5cfa9c5e967fc3134241720847df974adf0d33ba3a7607fa577f2fe7ad3',
                     '539db8a0a48e494ad3c116ca192447326b78da107c79f18308a17f70240a2ba3',
                     '53ae5f55519bab8de2d01e6979fbe0db650be9cfc718bca921091d316cb0f941',
                     '53beecbd7f9a7eee5aef8a60144d417cf8ab350c8bf9399cc13bb61733b6cc65',
                     '53cd7c148f0c8e7aef05de9686449ccb178fd4a1f695de2cbd0784fe4cb311d5',
                     '5416f31fef1d2f4b52d001216e07fbe78ba73b88251ae698689f9a6b218f34b9',
                     '54679d0adb8a82eac4afde1ab7fdcde2308e1156254d8fa040d6e5236cfb6e3c',
                     '54792b4b4041509792612a46a3a9e3c61cf7db058cfd3ce54df9bfba958658b1',
                     '548287a4813fa0fdbe20d67578484f075ef37bd1dd6f2a8385076cf7a12e13b8',
                     '54890c5a60d81109680fe246b525c5fb10a5ea0040489d86c48312b2d2fec3b5',
                     '54bebe36f250ef417216594b1cef00bb5594aee3ea36f3df54c0f18e540a8ada',
                     '553338da2701b87521a32ba805028a46d66ee0c6c6ad4e85c50df9cf50e167b6',
                     '55a7deba4bce9f2a249ce09a0b49b4447f748be5e30b9f343913c7a1a40bce1b',
                     '55dcb7e2220bfa7fc6500252b8db0113bd2ef50b15d43a583f032d87825dd791',
                     '5647ff3c18cadc44baa80528bf6aaa790f05a1b0d67d96ac4fd99112c5c42238',
                     '56f89f771ebb863c0de8cc3016e931e76ca427b59161e2bd32e1b96806c99e4a',
                     '5715abe0581dba527e32f73cc3a6d92c67ba69448b31636135501012da89841b',
                     '575549bba4ada4a16aa82123728211408b480b8fde752bd071af9eec053eb75e',
                     '57c7f9893706b98cb1b263db5da76f6b7336380684efcfe9a167b77384cf5c6b',
                     '58008f06339e96301472f4146cec2558e25e79df4d0e0aa8511d08a9fe5d409e',
                     '583a382cc04cc2c8b0db2a06c108994f8b5d0f14162db5b9630da75f6764fbf7',
                     '589ea0e14d751b5ecefc3ed53d7685f72fcbf1caa3ad6b616129d3797702235e',
                     '58d4d35e699fe59f3d5e0d147fb2702d37c4c02a817c3d5c99d5fb54986c0504',
                     '5958f91d8bb273bbf211d7ab1511d885afee981e219b4943eae1ecfab3370b17',
                     '59f6c867d4f120303824620f8b9378b410dec603c8ac6150aec9ded24192bf43',
                     '5a2ff3c6aa24a2fa0eb0a25d0c74bb392e387d2cce1a0c4b2cf125fe2448446e',
                     '5a540eca79a0d67917f57f3e0d7207e0dd1f60c0f25d3394711a4d0ac9ede1dd',
                     '5a6a952153dd90401ec40eceda54ab9c37709676eaa21f8ad83503a7b24b1951',
                     '5a9e0fcf1ebb9a309c67c2bdf48fb2c3b62c75184aa11097788ec0f5efd1b846',
                     '5aa2d1269adab9c2c36d2787c1f357f5d8161b8ed87a0698847ccce315b09664',
                     '5ae385ed224de1f7172cd56c30ab55188becbe0a9bb7dcbaa8b0a247dd73845c',
                     '5ae768884ba8c293738e481645b30e939930a6494ed00f2df6c1b2a7140148de',
                     '5aedb3e6a7e8c0c6a06f67ed854ab11b02884ff38448ba4088ef87b4fe289339',
                     '5b44f9500cbf14d62e713a24adf813b92bb206d1eb842943aaf90a655037d7ef',
                     '5b6b164ca7c58b4ad9be13b016aa2b66a64cfb5f27ec24498a73d2927b869aa5',
                     '5bb51e4ddda8bf153b6623460c5029d355d7ccd835a9ce1b63e46f29bc27dae0',
                     '5c120f923a95c00b5a9c1f6328bbc9cda0850ab7cd9fe68c35b60bbe51ece0f3',
                     '5c223b60f1086863b605479c580f3ef29ee22bfbb7c76c22691184d173f21fc5',
                     '5c79464863cac894c5ad13d984d6cdb6834b72c8437b9de4b9a3e4072f3fea15',
                     '5c954a4f79e4aac172708c08351b58752e66b90742925f7bdcad2ef696141266',
                     '5ca636821c1d4914eddf26a49f8592a83ca6d1630f096e44ff4f2b8352aefb02',
                     '5cb541536ef41d7d011429679a0628423e27df7b4c2f1d921506333e354bb463',
                     '5cd3b15957e86d85be27e14c05c35b8b1d3a0aa9646995404117f630b18a758e',
                     '5d542cf81e85c032a23824fb06a5954b8471f77bb7ea92fdef962002e4413a37',
                     '5d83c7a55d1684799a32354258b06da03937744a04ef843d47c73fae116151ca',
                     '5debd9bd1bcf027bd3df503b3fb73a68924fbd5b3d7c55852f379c06cb61f238',
                     '5e30e31cd36bfe003c71b8d0e7675c5dc82c4f70f637963a4540c744f8466c64',
                     '5e525649f0e748cb0c6e6f62d6d338e884cfc1baef6d5df5d228b542df2d43a7',
                     '5ebce6409062d0d09c88ecd8b8873a68e3bf252d4d43f896a8bb2ff06b767a80',
                     '5f09033528e4da13c2ebefd8dafc8d749293d6a12e1021a6c2ff78aa0e54d157',
                     '5f77be7c9b614908c147be524eb802b464eefb1fd8b08bca03f286e1842d057b',
                     '5fa9c23441645312c7101423c3d670854475f2261e725a81b8372340446fbdd5',
                     '5fb1e2ff524adfae2f878fa8ec1653e6cec86112020c2c1d2dc69993c163429e',
                     '5fd5ea59567a4d929551fc89ff66d786cb34673d350f52207d7d0e0ce5ea4e2f',
                     '600043039e4262ce95e81c82a970c426f132bc413e97c0c6b73afc992e08c52c',
                     '6005ccb5612b23aba5df83fb1481e96647f15c6e34d11eb26225ee342ecca069',
                     '6012e6be5921ea56820771f7c5f69cb83ed7f76cab3ddb45e0d271b6e98ee613',
                     '602b7b7180d02c143d7d1364110a2e96113a6dfeb70d6c2300e114afa004f47a',
                     '605195b4df6d5c9b8694a313337bd3b8a2ef4759c1813a638bf967c728f7f3d7',
                     '6055a941f23c0adfabdf76e54581c320ad8f2315d69bee5521d5416ecaef7a6c',
                     '609e504207c753ddd5e1cd970d651576b96a41ede0fc6366711a76a8bfb7fb82',
                     '61357eca6b6068fb3a47597031319d933d97893076c86d3c9ab346d913e1b600',
                     '61616312b79ce0a6ebea0c1edb9d2dec7cd4727165e8f7d2529393983d401447',
                     '61622e5cf391c002c83dd24ac4d38737dbead13edc921b8618172f3b95db4ecf',
                     '618ad1b14bdfeb156d401375001b811948ffd21943d644b8b4fbac28801eb12f',
                     '6197f56fd48512c651dc9d17532a2f08afb691c1b0902e02212680d496f73683',
                     '61bf83cb9f40ecf7cd3ebd724c9d61f14b9305306a9e8914639ea703486524b1',
                     '6212fdd2df4135dff2a4f1e8952041ef98143dd4dbb0974f0fb172a21ca4adc5',
                     '625023052b7d4ec21f9449e9e2983590976dca8ad4040b42961775bdd94b7e13',
                     '627051b23a3628de3cc5a104f482849437d719a42bdebcd59ee548faef94d0ec',
                     '62a61125f006927196d35fcaab47bba5b4ba3ffa6a023edc8270a632efe35acf',
                     '630da7ce1bfde039240eb9b37b0f3546ac9d2a24eb724f33dd7cad0eacd0fab3',
                     '6318d44a2ce4e3378a20f1713cd47237c71b194f4235196ba6ea5a978402fb0d',
                     '633242f139c4c894ba04a49308ba5a94a23e0ad0b634924a5827405eb5436c8c',
                     '634ee8ab28a8b1ddfa650054230cdcc97c970f8c51c45c719c34b0605964200e',
                     '63ebb29d693bb899b4f8271559b08766c525fd555519f0d20d70df319dee630a',
                     '63f7c384b9e293cb12ce7740866bdcaae6c710930fbd81a521e7b9512ae30717',
                     '643fd6108a4f055165d18e3233cb086084f8116955fcdf8564b51638779da849',
                     '6445ae0879de1d6f17abc5b64fe0e21bdf2a5cba055267e518565c2f3b33b1b5',
                     '64585202cc972d21ab8dd0f603b76f9833f981f29b98908bc36a3bdf5dd77d51',
                     '648b75001ebf872b35bbee5d498a94ec5d35c0086b8e03981ecb45f3b1f0c257',
                     '6494be55906ce443273f21d6829cb2edbf38a73933f0b5299c1d1e0c5a43f887',
                     '65223c7f321b6ceeeefc982049cd0fe96bd41df4f93c8125c9b29f72956e9159',
                     '6546982643a9a818a288b099c5f87e0c7f16a18597b104da060239b405a0313a',
                     '657965471d6d3d059a76badf497c3d0ee3613564b9698c4326b25b7e8536f4ac',
                     '66886428bcd081bb7db43e6220dcbdd0c02cd99015c8a102cdceaaf7b8bc2236',
                     '669b967662ebb763669d3d3d1486b36b269a8bf9b8805dc21481138eb4ec5505',
                     '669f272262a785fb651d25fac88a0e461ae2669f6ea3fe3619172f9ac78b4e4a',
                     '66a658465049aa95d0105690b8d79d9eb5f556bb97aa58e4fb0aab3c264b094a',
                     '66df0f9e8befe6c405699906669c03bfc1c28d3a7e74a1fc5af1431c6ab4bdaa',
                     '671226cea6820bb36a38267e9eea5591ae1a7f1c351b57a59d8cb326af7701c5',
                     '675954a180056d28e080a78fcbf0efb49e619e17dcf56f22369051a2331470b3',
                     '6773700bf36ca39556ce968a73811531ed9fd27db54b72b959a1a33505f653b9',
                     '678ef7d21f94b80531d7e67eac5bc0c63f7387cfc8fd0f32f4aa3bf0a060e854',
                     '67bd27e0e0202fc0f7c911f71233c63c2191df086d0c5fe99cec8cea9e1ae46f',
                     '680a0673b02d0e1157d321f8597d09419f1d4ddcae44d515eb310b718621fdb5',
                     '682d8e92bc195d1b0ac55206279b85330f612a07de0cea46913ef1dcb1a0a1fd',
                     '6875b7f3b4696412011b80ccac0ae7b1afafab19e70a195bce1186d49e021cda',
                     '6878cbee5f711d1b6b67cba9f3258f126e269d62e795e6a43e7641e0683f903b',
                     '68d030b51a5c51f42c8914531a75672882ac1efe2628123eb262086f7b0445fd',
                     '68e43b527592abd4fdc0e35f901f25019f9a7562097128632a3fb85805cb4f14',
                     '68fea67ec0719386b07199fa625761fde0926e75a6b84fe1cc84455c2d03a5f9',
                     '69024f7e56c62dc4feca94ef84032e39ecd935a91ce75216a0bfb1e8b2e1ef99',
                     '694a9d3ef39ee9a9a8c5e37d3ce3721f435e8265547752b713bb4e03fd23eb65',
                     '697212be1d00f0ebd4d251fd66e9aca0b2a2e56db4db8b38de5e066deddfedc5',
                     '697c90155f4b75a5236cb78246c65d7a39a4e7fc13080dbc0d44919b99551857',
                     '69876366726703ea021b734a84489bb4163a4bfedfb5659aabc2d1717ea001db',
                     '6a68787b742f537eb20b088a5c9834bb03c28d0eb03ff671f9b8ff3f07395388',
                     '6b379aecea6514f6a76fd867ed5812830569fd228f9779b6726999cc3d3f3ecd',
                     '6b3b65b93269d46972a31f9a62d88575f7215c594fbb51c9701bfcd3f9781b28',
                     '6ba25a25b0fc8f2e0b8e61004ff7d594801a845e78ea3733024d85ee8bc9fb41',
                     '6bec79e8a2b94767be0e74dde2ce246946e2690b924dbd820379d68969230261',
                     '6bee3eb2eb0c7ffacfb67b4d18df51f272c59285c3866e9c14eaf518a7dd8040',
                     '6c6b71c394797b942e87cd7037a729547e0baa1a83bf9bed818f70200f1d78b9',
                     '6ce7d2bad658bb48845625e87136ad74422072322416f430039151fc498f2b43',
                     '6d1f0862f5e340d61854b140619859d5941cf9835788f3e71711c839ead556b4',
                     '6de599b17713fe4664ce8d3e6b861a63f8f399f87a9f69b46a187735c14f68a5',
                     '6de91cbf7e2fbd7d913674223dce53d96ff951141c497677aed9634a5905c927',
                     '6e13000b4639a55e40ee13717b9539b8802b01a18a09e68d2601a15f7e7c9f87',
                     '6e21f864e12c8895ab11f41e37cd2da110e47f550373ac99c17cf75045cbecd3',
                     '6f27f7aba151e830cf48e2f5df8031b3810729e419fa618b6b30ba3355fa8ff3',
                     '6f49f0f05d2df15e70a03ca6caf35d98655beb6d7fdcf036160958749f25abc4',
                     '6f7dd32e5457545710a9da309ec1a3813c9c2d8ad955686d57c8d314cf296bf9',
                     '6fe56567be2ca329e75234c985ee95430839817b2030c9918025dc917983f7ef',
                     '70337223dcf51900995cb0732bba08085688b72295e5e792b224f7779e834fb1',
                     '706d2aac56c2dca935746f7d5de572d200a7b1138c1525b42007b8a1539f3fb1',
                     '70e047dcf9cb8e47d91ffa0a8c05f42eb4a54de0526cfaa07178f81713ceeb69',
                     '711429f02f197c6aa9d1e405152cd99a5c2bc2942d231ddda6fdc11239643ae7',
                     '7114d2751df79fb97a951b37f7d7aa25983dbbc758d566b955075fd207dbbd59',
                     '71284f9ad2bf5619ed99d70f89839a85092eb67bb45c9e8da52eebf958ee49e3',
                     '71372c599c85aafa4e9714e33f875aef48f00da63640b1bc387c77ce30b09a9a',
                     '71a53233501116dec0b7a2306a681dcaf83343ad24f418a056fb594006cfa173',
                     '71bcdc5e7cf534d0de602e8052147b1a88ffc40b622904caca87102bcf8be2b0',
                     '71c504fc93d8b39a24c79f5ca6c1792f38dd34145cc075bd7d2a3c1bc6196ad2',
                     '72e4dcb218e48f09df4eab58412389c259a85fe6fc1b649f0b6f4d8c92faad81',
                     '73867ebc8c1becc179bcecce843fb21f6a9dfa1f5580101c0119552b736af699',
                     '748a12289c1e5c39d6ddf297aad473c94c5471d52bfcc26b925b86431e4d9f50',
                     '749a2c0ae14ffac85a74ae6ad8f518175790c4ca1d997895ffa9b863ccb62dbb',
                     '74e2ae59d0c07c953be1be0036dc681324bb7b350dd43db9ae09049e358003d9',
                     '74f2e2b84e80926b1d1cfa12820a74b4a2734f14779890559f8966c5a9d2f2a0',
                     '7521219c4dc87eb3caba5341f1a0ee2d23721e51d2fbbd72b3eb7bfb00fd4f07',
                     '7524c3ae8661a7c7d361b8149056d31acb09e9e31ec5fc2f5a4e62889552e3ef',
                     '75300d527a44073bdc241b9f87a402e5389fcaad05ea3ff5af191da24e5aea4f',
                     '75b884dd33d394d344c21ee8e7b66421b43ea8cfa0982cbb76f78685428cd9f2',
                     '75fe4b476e2d596cbf623bda69ed58bc09434d0168c67776fdf2059a52098e1e',
                     '762bee91ab3300e45c3e67c791694f5e9e40dd48676458697ecdc45d09a14569',
                     '76cd5408e575953f7162c493f1c2b30773be0d4ac843bedeb60267391603d23b',
                     '76ebba41843d5e5007eca1207ba93950938ac0d2c82163bcd6bbee0bdb55e861',
                     '77157d41ddece5b386573af5f5d82113cc4e1bd5827411144ab912eb5241ac29',
                     '77cde4cee01ec4f412069aa2ae6ca3d50154e6a95587596652f4d002d83325d7',
                     '77d458a52bc14e015e74fc2534a8c38520284435cf5663aae1e5355b99547ce6',
                     '77df4b1761723477ed9c65124f6843f97c960bfe5c80b69443724f78b5d85870',
                     '78174f9ee66a7fc9366eccf25520a7e251e173fd5ca1d744bf1f5ad6e2c0bc56',
                     '7819eb8cc1453988f305a6d84103f8a3dd366916aa8414e63b9b84a7dfa5c4a0',
                     '78fd175ad214e57673ca88fcfab5619a1f0efc40b60433e8d8eece1594bdb0e6',
                     '792506db3de2185698771bcba1ab498bc7ec04e1d6d40406167cadccdd378770',
                     '79294e97c363afb031381a69c8f86c1742c842a55ec108730efb354078f53122',
                     '7930d7d0cc620352ee89a5f549c07e06ebdd0bc199e53684333b198c8a51e188',
                     '79c1d30e3334e1762680107597bde0fa5eaa791998fd674cceffa8c20bfe5a1b',
                     '79d2ee709fe90a80392c3ea347b245853a2df6885b5a08e12695e95a5b1fa5ae',
                     '79f7031a4e43a6c097ac52e3cd3c2f2e0902c7f4992259092c855dc591712ea1',
                     '7a2391d2a2a38dbef609e990a824b07332bff012adb280dce711c019b0636fa9',
                     '7a66bf7c0560ddc5d003b0869c6b7e8ef4bb06029a3cb0c258820a2390522a41',
                     '7a9010cfa49379a1e234839532f5aad546561373dc5dbb60d0231c7f361fe837',
                     '7a935559ec33c860f6fc4eb3d7fb5f1488cefc60d5412f377562826633a7bd00',
                     '7abe3f31882f40b4e38c9c2f6468b7dc696a7687d97ed7599646b293d5b781b8',
                     '7adfb0914d1916012dab439ecf342839835f41a68eaaf1c4517afa9d7847b626',
                     '7aefcb954cfe40eb083fb2f35e0b334ef45b43c2ffffc25c825a0b6a3df1d839',
                     '7b85f86b38759a1c0c4716a2755340fc2bf6f8d60d4da5cd5a75a9a4d20d21fa',
                     '7b93f1f472b90c5000cf999220b99e9dbd8ecb9db806851f4f3a002a3c79c6fa',
                     '7bcdd0f5faf5011f4a12267f0968b33edd53a8a6b496eb9896cf41e73155c450',
                     '7c5c847563d9e772388777c232d50716a3685e50a702138f9244def7a90635eb',
                     '7d1c26659b5abc30b4461304e0d55a45938a0202fb4dda162642ff6cdbc7583b',
                     '7d21b69701d4c9da4cb0876448f089b3529af2c70e63c52aba7bf368e8b450cf',
                     '7d33bf62ab19e16fd55c54a0860216fd57f4b39944133b40d9be0eacd635cd9c',
                     '7d33d0329a2b9d2c2784176df8e36680b9ccd16896c9677e9cc96307c4e7c5b0',
                     '7dc3bb17b14ee6737329517be715a9a8cf175b5e169be2d6ab7e35f77fbcc6b1',
                     '7dcf4a149ee83236fc6467851c1a992b4107d833c99d7977324bc24ed989c927',
                     '7e0bdb6d99d8328a7ade0036ab35392de3b7753e3c87ad4656db59e23b1f76e7',
                     '7e51e3fc640024c1a77502b6b8564be1d4305b67590b4f60f0f135963787a155',
                     '7e65c5ab2140d1a28103fba10c2bc53e9da2cf7b43afb0a61e0c651992bd5e25',
                     '7e74b61f3e6334fb3c0b359f71c9cd53e1982b50cd5427a94f36940db9de8fb2',
                     '7e82b6c309cfbf1f791bd1f19623d5e6862eec2bcc44c5b44f3c2520da1ffd80',
                     '7ec5f47b35338bb532fa1fc0cf6893691bf8c712af1b55b942a9dc2225b6369a',
                     '7f804b7022765c0ec7233fda4241ce784305afcd23e365c12d23398b94012545',
                     '7f85d9d3836a7943bf98317d347614821d15c01a9030113f7de5e60439c0dc89',
                     '7fbed8b563cdee3197fd7b380bd5150452b5c3dbe674c395b4325943fa873a49',
                     '7fefa921a6f14bca5cc222a3036cdeedb95a1b7058f18752969289ec2648f4ed',
                     '801ed1c5358365b330eebb20cb4f7480fe83a96cb240544509db85b37410deaf',
                     '8020a0d1ff778a8e8ea8385188891bad0841a590bbe445d2176dc2616d44748c',
                     '809e23b1b836ad9341ae10ba31858b25035fa32b4733e7a1ec1b9dce55cfca4b',
                     '809ec82680cb5fbcbad008d83a971f58d48e4327e2642d41cf7cbf69bbbde9a6',
                     '80ac13bb6a2f846365ca4d0359de753ba0946695565c8e980cd675f0c07d89fd',
                     '812e1c5a661fd1120ac325cafcc5da65932555f2816dd45d578545ff0c8ae616',
                     '81fea856794e207094db96aad6c8201d44dd885d7c03d8388f045bc650456a3f',
                     '820006f4ed92e36119f0469ea3311e2739f34c88500eb583a9c37737c9f4fb45',
                     '82045496f034b46fe151140490787b0795564edcfa0b4b6f7027f63726901e21',
                     '824072b3ce8c6abef26bbb045d3c9abca759b0a8b1e8e8cdcaeb64a4e403ef34',
                     '82aa3ccdf29aeaa6f8b3a1255532f8f58c550f0f343615d0ba8d0cd6b762267a',
                     '82ba018e345e09ec588090eaa6d5030da344d6393ecbc0b091d7d6c01027403d',
                     '832d52d7219035a23a80c986cd41c2961c1be6aa7f83300e928d9a7435d5f893',
                     '83ce0bb096a862358a45d465ecddeebc1249cf4a97fa29072123cc94f9bcc59a',
                     '83e5a8ed400d291c17f48e8b432b8633aaa0de3c3b482808552a7928ac3c6a62',
                     '83f8436a8ad19dc5f504b38407177ce9a761b8550b2dd78f6394676221c2bca8',
                     '840db84c75cbeca1f2147634e8f410d96a758627824cf5a9ea4570135cd6d1e4',
                     '8411ef2357129651282c83c5658e1fd6ed5af85edebe985abc56f41f48eac25e',
                     '8462f8dba78b9dd70cb78df9732b695992863d00f429897644f6af1d8228de27',
                     '84f7da83fd4a19d839b38b02fe9cfc5e4e7a232e8799794b3013cf0bde1a7066',
                     '855bbb0ca119676f9c9135fa70d90547f606e5ae78faa5b0d8a463097f4b9ce6',
                     '85d25a5325a781e3e0d276e47a4cfbced0e428e63ffcda50f2654ab2be699c3d',
                     '861d81fd660255594cf9a736c87ca68c82ae34f45607f925da9c6967c658c231',
                     '867884ca840bb4cffc94cb14784a59146ec17d21445fcbd912a76dcd56a288d9',
                     '86c50af95f34be370076127e6a447c555f45ae8581ffa5ef42ce70dcec44700e',
                     '871e2cb50a8aa7427d0875e8f46f600864787978cbb5ffb9199f523fb96f8f7a',
                     '877827498290281af14ea2667c83ea0d1ba796a853aff5e101d03448c6f2240a',
                     '878c11cfb20e4561c89e6ce8d973d17d7b02857f0e3481b1cff39bd954579a11',
                     '87c9fa95545393b913efee2667d28f4fe46e23a08315602be8a4f3052dd1a729',
                     '881e5969b0cc5ae398f845580772401efe205756f3a17d63549f23ff5c6a0a6d',
                     '8865258e4cb393d1cd57ea265936df75da7fc46aa91e6ba16bb21e0d9d16a6a4',
                     '88aafa92a657bbf38e233d76a6bbbf312c43dfc68f8db36b6fab1a9516cd9c42',
                     '88b8d4589f35fffc2553ee7229badc66a0bfec0ca0bfdae2431fa50116ea798c',
                     '8952b58cabd16eaec4df0db56be1ea2c8222755e5a3fef7789d983e95ef696fd',
                     '89dfbfd337cebf268d49b75bc5513607be26f76058f709fb9bfc0fe5b419882a',
                     '89f8d84b36432628b137c6455f17c425d49bc5b8b274fedc2103b668f7be7a04',
                     '8a0543c1de72f5538c3a5e5a056e0139fa703eac01b1b27528bd9ca1879b77d0',
                     '8a090e223e0ea8b594b339816bdb6205b4c46c401be9a5b937f72ae965652299',
                     '8a8d20199886a3ef7714620594ed1f0261d99693616842b6ce4f1d71b845be10',
                     '8ba078f1f9220437c84953f3491b2fa59b7caf3bb6f4fff2861e67669ed00bc3',
                     '8bd404d2fc31b447ce91a7fe116316806b05e729f6150b04243d221c5a136588',
                     '8ce56387c6c068ddbd1694f74c3c9032a83892f615f7fa94205e70246c4317c4',
                     '8d0575e83fe369ccd672f27ad889a9dfdc4f517dfb0ece25e4c291d5724732b9',
                     '8d3be63c4da550a3913178f9b2de4f073b8391bab78aac1e5a02a8c656749814',
                     '8d47b475097e840bf895f5ecaf19c05df56d90d47b21cd858f090e1c677ff29e',
                     '8d7c37d7b3c2ae9536a79dc73c978546bdae97f5050531989354ec29f2a7dd59',
                     '8e55b99db889e80fb254f328283da5a53874c787ef4e4b6ad20876f5e65a06e4',
                     '8e5f776a6caa856cae5010234a899971f74549dad7df58cbe7d1d6afe6b803b8',
                     '8e9d5716cdb8b34a4e88bbc0bfd89a7239b26dc28a5677cf1bc892ae513b9a85',
                     '8ec11e81552d3696e99982476a898b20ef63825e77e3cf2a7c5cc0d8411a381f',
                     '8ef0899200ac11910cabe7a7db0f903fcbcae68a353d3fed04811df414f726ce',
                     '8ef55ac94c3d2e418592b37bbab5dbb9fa0e085988d949da289e8361334be06d',
                     '8f0f4324cced8cdf87b4aceff242c7ae78bdaca735f1f3ae2281eb6080caa1d9',
                     '8f363f31f3dff7f5311bc296aa677441c61a4f7635472aa841c1198027037e7d',
                     '8f3bc84cd99e1e485f567795b569421cc839176a422105d12f123c63c8d99a05',
                     '8fa574972578872d1b818a67c4dfa1fd585e66f8c605023fb0d81075abd81339',
                     '8faadf42f61d007e7fd5bf2305426a5499c94ab3ec5b9ddd08c23e8c7baff408',
                     '900deae74c7de1a6635f99bc11278ddd09d98078b7df8dd0839103d51ab0ad97',
                     '907e694caeb5cfa5a2dd1553a6e847b757d6a20a013435afad338ab45619e885',
                     '90e839ef0d50dba6337d776d6a08d6c9686912415b32cebf861e772ecf9555e4',
                     '90f28dcd3b0705cde9bf71db1eb36d1471be78d7b5c64a61c77db2c9c006b5cf',
                     '9169d847d0daea32a05a772d12b309e793ccc1c91f82fd483cd49a5a7591bc5b',
                     '923016553382e157b61c598820d60ea4e1f86839523968396be794ac69231567',
                     '927cc8e6a81091f01c534c53c04dac2ea17b398ff4b99d7726c996e21024d3fb',
                     '929fdc642069bea33341b3c2460e3ceda1622a031bb9b02a3f1570bb1a5a6c1c',
                     '92e0727f52430f0f457dcd108ca3ae6df86e38f153d3281f2f9d7c3c0d635805',
                     '948c52fac43e824a898f1b322ef815d1c85ae5d2e84001d078585ad0ce53347c',
                     '94b957fe02c8b2166040635a490c0571e47f45a2e664e67c7d9f6c8228d960c0',
                     '94c9fe03acf5799bea7c605273cdb6a0b772711f17ba369ed9ed25643539c400',
                     '94d6437634b1311bec99174fad65520de1a385543eed9feb7dd67016f18d035a',
                     '94d75d14c434f35a4cd69c76973a093a02287c2ad01ce8a01f0d6052a04e19a3',
                     '94e36539ff24293128ff394f1455a1c2cf8af129424a2b13080c5a4c569d3677',
                     '94e57d886c69e577b9eedf1858a8533477d597b6e11738542cd28d701bc6c005',
                     '9505898d4d8813394c9b6cb7c0b240b5ef0bf45436a4ee83c89743b990031805',
                     '9569d188333c72f87939e4fdb401a94af7aec89a8878209ef113a17b4b9481f2',
                     '95dcf2b60941b5910ab09a008eddc61fad56dfb03a59b3a7ab1b6f845cd0fe03',
                     '95ee20d1b5189d5b4233925435218569f908e3053c94848d5d0d651b1bdbe83b',
                     '96c1db9596316e1c7a4f0d7000782a35c749292993a6eda9c180eb0ba65ee73a',
                     '972a679c850c5ccb2ef2a5217a1692b5371693d086485a07bc0a63a84f983927',
                     '9746e86839c203f87344ae97c67204004eb99c35a04074a2b3a76e1c62600798',
                     '975d180742931dafbc745cd64bd338be1514f8dbebbeb2137401db4166a5a08c',
                     '976a330772d02cedab2f0d183f7f793f93117ab62e2f6efd7c25eee1634a4265',
                     '9792b8709c6ac19d37bcb83aaca038f7c34b171f0cc3c0ad907179d70eab5fd2',
                     '97a2b95067c1c51e4bfe50d4a19eaaa18758f3e5b92080b19eac2d429dfd4d51',
                     '97de593258d3376b43cb3f34896d98c025e4529517e9b8914a7d5bb88c74c5fc',
                     '982bbfff8cdc53dd939dd43265f4c5c8787ba2dc52b5e6ec37ad757c69684554',
                     '9847193e99e3e4a8d5df9fb8dfdbb7db903630ffbf651fa6662b986f1ba4a16f',
                     '986f5977972ec1fc0879556c5cbe75e1b871c7f5aff819996003d6c8552d53bc',
                     '98aa4599a2a4076d7aa41d7659e257d6a62bbff69cce3464b0ebae45af5322da',
                     '98f7731a4426d8c50ad3f9203f1e1d3bbf643aed9564e29139f29f9f0dcab26e',
                     '99c3b84ddf987d4ca7d4034bc82b45cdc386bb7b59b757ff8d136d71ec118509',
                     '9a1820f5de0cefd881355d8ac5e8d4578f6db2e204430132e97b539d222a8320',
                     '9a4aee6a95b54619f7c8541a6497e69133bde39d782f18aa6b119ecdd6da0ffb',
                     '9b2370aabd801d45f73bda922cb6939173181898894f95a0dd8309a94a265896',
                     '9b3bace51edbb59fcc110966337df057830f08393d2d72d26ccbea69a1eb4817',
                     '9b6c5d785c00afe6d77ca81ad052d8d55bf034dbd734093d4b72aa2d3e4d2afa',
                     '9c72c13f66fad520b50e438ff8b0d221ba935e367ff7022db24a16d9e2ce8022',
                     '9c94b4e851eb03ba13d6f5c6a9630261b7eb45bd4d817592f7d13cac529831ae',
                     '9c9d4d1735d5a581186801df1412bfc768ac65976a3cffa85127640f7d9245a4',
                     '9cbabc345948633cb689c7409dea95d1416fd82141725b327d701d2e5e2fdf4d',
                     '9d25f09c5077033f0c33abcef3b585c5c54d25c86251ffaeb36ca377a7d9350f',
                     '9d7258d88bad246c31aae1fe27311016399fed9409d188ef8ec6b5971d130e8f',
                     '9db4679cbd14bac82a2270d094c18046b9f7dabd07802b493e7c25a220aead90',
                     '9df229ed3b72c6f9ca679843b977eedc9fd4a0c7d9cba087242950483e98266f',
                     '9e13fc095ac630e20c2148357a9042d615fbd3c8d9a986a70dcf85d193f95825',
                     '9ea81f44498d04af4cd2b84af078e2ca6aa42bc2a0e6939d229ce5a96d21f49c',
                     '9ef8d947eb2796927fb9fa8926b44df99a1888cb92b40060f78ce6f532bac088',
                     '9f670bfb650dc6add106d49afb2b8fd2256c1a887649f444d7c108c170cbc216',
                     '9f74a9537358bd01beedb7ae65a5abeb9ba74c06d1cd036aa70256e3b7793029',
                     '9f879f6d8aa9e5eb62cb3af349cfe8fc15e8eb9fc6262cb8a377688c597dd242',
                     '9fa4dad55e35af304fdde434747cb718a18ceb3617257d1d936f0d09c4df5f41',
                     '9fc869fdc7abf58c463486c7316e769bd7afd60be59fd089f7241b6a8cdf090a',
                     '9fcadd64f6c0f9c3cdebf1030814a9f3d88e40971a5665f39b9fbd20be82002b',
                     'a076cb646f301211bc6b986c59a3f6506b6c42e1e6f645d8aa335b1a6c9bbafb',
                     'a08b6a901fca192459f24e64173f78247a955a404f6b54d706ff8c71a4bb0660',
                     'a090bebaa5187803f002adb0aa189ee9ce5c6c434a95b80f8b62d4eb072224fa',
                     'a11786337ff4f59ec22cb8b9292df7220c20debc689ec52ff0328b218d041939',
                     'a124ded59b97ffe042097a735b80b1a92fc568753bcb1d09226ae34a306234ed',
                     'a1f8788a02599524a177b2452e712c201246453fa44f49ca2697150860566234',
                     'a20efed7426e1a5e2ac2337fce00bcbdd8259d7b3444df7c7f14368f8579c69a',
                     'a232c63616c5d104835a81c72c440af4a185f1929da16c473bd7c4f07be9bb12',
                     'a233dad262372d2384c51036e6be069d8187616dc31d6de51ddc4387d30600d4',
                     'a236495c4bccc421c257fa520149f2041e6759494e36eb779d62c1fa19292e9a',
                     'a27e8db63340cb177a85d996eba4e9fe23764fb05fb9687133ee99910a0ea34a',
                     'a294a4e5c0be525728ed82c78d07e4433f4fbfb03f01a520154bc4025da14be6',
                     'a2d63e41d1c12c1cf46f2bd3aa5efde29710e69a512122e7c9f7c25ddebd4cd8',
                     'a31bcc6af1b7dc739114ef1d8037be760b99a6fec1c93efd3fb594bc06f99358',
                     'a33dd56b32848a246dff2bc7f46da43a29c49ef8a7a669c4bfd942a192bd4aba',
                     'a35875fa300436200c4fe9b56687c01feb06f1ce504fc6d3977013643a925f37',
                     'a399dc2f02c8d4432dc14604458e3ab591b6205a8bad07c965efc84ffb7384a6',
                     'a3bd77ef4e109265d159ac450fd96b2b2bd3630fafdb57fe57fa57a0519637bf',
                     'a3d52830978adc38964ede3026a65893f5d05d1169115ee550cd04823048c830',
                     'a3db4522f6a2f4ab697c3d108848d708113fc8ef5114ae564d5fb214dc304d5f',
                     'a409ed9ff99ba4e15aeaa32db6bfb186a72204a376899033f62a1ba270355940',
                     'a488e5780fb8bcd2676617c5fe2a78df84cbed7fe997c3ae13a81baf0b29bb65',
                     'a496df64d6db9c001644f1f3e41a1e45f22969effd772168f2feee9fd6c2bb6e',
                     'a4c2056540a85324deb93c58a008d6332ab9ce8a5a1b657ef975f2446a450518',
                     'a4faa72f0286e0f7e7ebc4d7c43cf6e553eb0414eb83ac5618d0c1b22a0037e0',
                     'a52c26a9be9cd50a57d2516e4c157704a2a80ccd21337649d4b638870c9f09bb',
                     'a539eb0160f433664ffb9ac52dc0f9ff61b21518dce0fe4d5d4b20c495a8f586',
                     'a54662435c1d76fa7e591b18593a38fb653a4f807158a3fcf72625a4e4679f7a',
                     'a55621a66ebff5ef62a96a300b1167c7331c2a9fad28396cdba67b9cf9d73455',
                     'a5e6bf45b685bb32c0b426e3b5360c1ed7645b067dc2973d0de258910cc23095',
                     'a6348e3bcd5456b8b69cd5e09b133653a211fe0634aacb3634a5f815d270e589',
                     'a6914c89e4c89af82929f0126a26d08e2297f650a8ceec97e0cf000a9fb65d4a',
                     'a696c02a38cce507ecad933f06a47a1a65c26b2c3c1c6452a0a403737f958a53',
                     'a6b40cde31c8955c49a6d3c167c6fd82b1a4b277b6033f933c3a8a2bd8fe3107',
                     'a6b632bcdc8e1b77a219c6464867b4fb3ae55d453cd0fa6244e29d9a54bb80a3',
                     'a6c119963b0d00f60e8b58e2beda4573aafba98f9660d001710a6021123d471e',
                     'a750a1bb9d6934dba40c0060946b195fce0d66209f03846afc550f334f154929',
                     'a7a207334af3649a356e6a9b2a05c05300eef762aa73627e9773d329704051f4',
                     'a7b5aab179ed65fc4b0ebe2ebb28a31a8535f93fa3e1f899dd3c6f81053b8deb',
                     'a7c50f21b0490ec573d73a2c1a92861140d8d6ea65f4edb0eed448c849687f97',
                     'a7e3f1b3772181ebc9dd1f14028659b654087640535343159281822e5644994f',
                     'a80102d0f0d2c04c19e13ff408076c8b8057795029609eeb31a1744436932565',
                     'a815408dd658c55f3e345691960afdd13ee58707815666d3fbd8f1e32f3d5cff',
                     'a899cb41a8bab3fd9f422610392f65f2f0102419294af7724cf72ea417c4e9ae',
                     'a8ee1c3079e2cd9b194cd1070856e202a5e86e4070f2190566098b8645cb7bf3',
                     'a8f21e01fe3828636c0a50e126554627ca10371815e24d99d2b167ac6f896cdf',
                     'a938cef46b1ad6a97ace5276cac89a6ebee184f77ede4f300508b9cdf5d3ff62',
                     'a976e717049353a5df7197df29909477b5f31d37d9890a43f066280588e6acdb',
                     'a990fc5b32d6958f31feeecf0357c7e79ec463f418f1a341ece481314f7d1f55',
                     'aa0b920e1fbabc76a620c87fbd5218f9de182d66e2e9a2f07fd3fdd3e85a25c4',
                     'aa5563a85b650cd318402efed4f90bdaaf1d040a6603f1af4fecd17bb1a3d2c3',
                     'aa709feaa1e3d2960c85e36455aba82e5cf383cdd997f83a81e0d79c8fc096b6',
                     'aa9fede7e598760298a43ddc1ea9aee4b4b4e6efd6c716f84196cb293abe8c09',
                     'ab6453bfbb3fff8ba6aac2da2cfcc891d733bec706d8eff493fb4879dc4d9131',
                     'ac52365e2ff8a12563648794f6cda371e1815d371f05cb65ccd3ccbf49a4b2f9',
                     'acaac93c49422cca18b7bd47f6d02f7a4086a7925bba9dad3895b205d0a0ae5c',
                     'acffc003128794b2f892293c555a34cdaa0ff31b351158e2918ac9ebedbccba6',
                     'ad9ee9207c21df2aa11f8b0ecd612f39f9216d8659949d4bfeda2f3d62c76880',
                     'adaf8b8b6dc4d12aefb4a5e30cf3515f7cff4ad89358d2d56768d7c02230b4e0',
                     'ae2015c7a2ca470b32af4321f30eb839dfc3ef015fcbc19e59092fa11c1f13a8',
                     'ae21b433cce10924ab235527fd783b53a603cf09cd3668ce414262a72696c815',
                     'ae6fc32d44c547003328f6d51bbdaebe4decd35953aeb8f4f7c87ab44d6398bc',
                     'aea678149050e99ba73153691b0dbdf0710098c8d944aefc0b72b2b32700b698',
                     'af2bbfaa1cdd9aec1d5eef54f195b70122c3039bcad509ecc78233ffd5e52f6a',
                     'af6d5f0780a7136025bba4d48b7e3fd8489e9051dc0323fe131e346eb28844d3',
                     'af7c7ca13c38ef62398372003b12e59c8e4cbb2a2bda8bad39afcf4efa0d2e20',
                     'b038b4c09b549bf2059829904902303c86f5d13f37edbdb4d5f0b27111ef9668',
                     'b038bb40e413a4e6fc00eafda15c191b31b1f2c125b59957ffd89fad51e184c9',
                     'b15d370de3c38dee39f1c718d58c1aad17538d5cfc41aec76cd57dd74762f3af',
                     'b21922a3d4de5f21a041a9756c92a69bcace3d7fdb2f12e696b708047f9e1edd',
                     'b26a3ead9dfa4626bf6afeafb5ba7e23d17f0979408f61b12023b388d132bece',
                     'b310a4a603a4b7b49c4f73fd15cf1a8500625a65fc9509547ef0cfd5b13a3678',
                     'b313fe9a426189bdbf0fdf969a5f6f2a127a0ab756c8eb4b76b4a1b944d3c29b',
                     'b368f01114bbe6ff3fd1279127e41a853ac52887b3ced034024d07d6f82f34c9',
                     'b3824000098fe2a3c9a9841314ff4980370c38b12f71f760003b66a61818d9f7',
                     'b391002be2d84e14954792a154edf59b87e7f0e3ead76d2eecfbe775f833d371',
                     'b484b13816596cd872e0d5125e0c8728cb428be2a1127446f5ae7823da26cab9',
                     'b4c61ef4d26dcee7928ed6fee2f904cdc98138de00b2673fa2eebf06cb334fc9',
                     'b4ebac3adf1ea2f7036580c7fc75985e13ca189a8d4c48aad11c0652679ebe17',
                     'b4fe52fb4349d23096a5f30437abfdf6361e281115f68bb5bdcb8eba56e4843d',
                     'b545a7301eeda10758632efc66ff54cd5cadb987cdda65878c98a4d40d40d382',
                     'b559c50b6ea318bd89bfa111e302cebfd84550633082ce995cc0e5fd3504c01e',
                     'b5c5e3230703e8fb02da5d7a7f5283b68c489585fdcfe672c5dfe12ebb3ae2aa',
                     'b5ccc17c1dc51e0266ecccf365ef2eff576cf0b4b781e48cd571996c190bae06',
                     'b5fd1f7d3fee17d1659fe5f93380db2a71514aeeef88d874add9725d39c7923e',
                     'b62b27361add7e88063dd3d41046eac188e6b29074f4e426ff4f0938a6090615',
                     'b630a2132166a89323db96d7791f8553ef92f42d09e771c252a88b0772851576',
                     'b660783e297bf955a11c1927e3ae18ecd28e348136937d220a2ae2cc8773f4fb',
                     'b69f30c601a3d0c67cb5fa3c0ecbed038797e08ec028940abbf4574ccc7e7361',
                     'b6dae4a3da1577c54fe51fd607f131fd05b273457b9499ae5441a8d2ff800f89',
                     'b73baa26e681c985f3b9798957e03bde2b4e573896b3904287a88c80a2b93b3e',
                     'b73f28d24703239c6af8bb16b80c7afba5ae6a6ef35f719799fd4a56774a9a3e',
                     'b75ef8e8cd44faf109ddc477d87262566dbfb69287947f859ae0c6344903eafa',
                     'b7694a0d8cc90a321d708c46804f1185bab74a1070aac880453dfe932e9e0ccb',
                     'b7e3823573fce41e1c8a714e8c04dafeb1875e7b31dbc509d045d9343458294b',
                     'b84981587fcd77f74bb02fce0f6bd4369b00f9582d089f272324d8eea13b94b9',
                     'b88923957715de2135cce6e9b26478c3c43940bb296d3cecdf2b95902d837ff7',
                     'b924bbb286ab652117d29e05849c1e13e9542867030de7b050ff1fa2e444c25b',
                     'b98e94478d3a1a2dc968b22b8514b183b7c73f69203366248338e98f11a5f8b1',
                     'ba26fb73f55eb60ea55bdd0c5b8b57fac3f708a6e65708411ad01c9d74ee07a2',
                     'ba31ae258a567cd94531ef9009dcaa08622bed1722bc458457659f9dbcbeb408',
                     'bac1d1dd9d14912cd89ae645f71a16f899da27a8572a9af755efeacd323bd200',
                     'bba85602ac4e5649950c167c080f4fbf06e8fe4faac7caa1a3c1aaa1f27b8dbc',
                     'bbb4d6e5741265bc87dd1898ba80c49989f97854f4c489fff088cb624166fd54',
                     'bbcf644b5adf140abfc5dba6a7b536a3af260258dbff6237bb1cb0bf8e6d7605',
                     'bc63ed066c67c28bb0d812fdfddb8eeb4989677760561b2c23398b8409141054',
                     'bc7069da33611ec30ff0a67d51895a7136093053d6ada339bc5426bb331bd3e5',
                     'bc731129b257d06d9127d9a1f96384f33c7320e1a8c4df30bad7a4da987e8d22',
                     'bc7eb8de8925187e8cec597e27caa7eccc6464027220e47cfaeea9c88c477c5c',
                     'bca6d7a387adbe4d6265334459c69cccebac5e0e1b4fc2aa8fc3b39787bd216f',
                     'bcff4ce05719705fbc30bf7af94ea1b2867134ba610f2b7b5b2cc6e97eb1b898',
                     'bd0262ecadd5cc20ef2409c09246847e571256f21bdc55186bf42979851f553e',
                     'bd2eb87777ce359dc2c616e117de7fcca2b7535b51741695864c36187bbd8931',
                     'bd3887ce77f616184eda2fd55c1363c99768a15df3ecc1fb9a1352d0140c3da0',
                     'bd756121532e8766d1f2dc4b5cf8eecf591c0b77fa34adea6ad8b0cc915aae7c',
                     'bd87844fd3a0223e45940f8f0c24481c1e819ae0b3421ecde3fe0a2bff9e5de4',
                     'be0a617f1090affc229d0c10ef0c46cd983a73f5d42f3cdbf166e95313d8b2ae',
                     'be11dc86e3b3469143ced4e0df76de5e4cec79ad02182fd3f78a66c7ca80120f',
                     'be71cabede2b75f5f4225fb6f102eb667f889d4c5501d4b268155e4b46b05b13',
                     'be7fbfce2e5921bfffee3a00c06ad7c5879cfaf0fbfb8f7a6f3ca3f5900f7b51',
                     'beb0bbe1093c7f232cbe1f1cb8da467265c671cee47f51f114697706b5f9261f',
                     'bf025830ab71f7afeb66ca9a28e27e456a5928119ea2c18d00bcc6656170cd0f',
                     'bf3a56b486e0a3b54c875a4f7071b8c72026de84fba0370eee6d73f8b6083d0b',
                     'bf4819ae63911316798b91d7663f6a48c686930e1517305b307da44a68f8da26',
                     'bf72e50e77b84cccb3ee457ba4b78f0bd2350a15a6ecd7cafb6d4c2f3681e86e',
                     'c01881c04b7f32256bfb77bcfcd6fe882b9d1150ca1d5a50d8bb65163b4c04e6',
                     'c038d74af6c548476bf889955b5fe53e678ef03ae1d3822901579d15c2a38b2f',
                     'c0f77f115e4502dd098e37f9bbeb7cc2020665e14c5ee461e2ce4b515eeec76d',
                     'c1031d72142bb2a637870b199a826740a1a219249aa1f913ec28080cf3fc7f32',
                     'c11abd196a83a787b797ab1b2c4923bc8c3cd36cdc905cb69bfe670a4fe683d6',
                     'c11e050ffc6028715b1b944742b976cca52912f4c8e9e4e9e2b2b83e7e710a03',
                     'c1687df1cafab395dc4667d979a03d93ece90a53af90e000ed55918c1bf6ea82',
                     'c21e4d7fa110e18f14f911efd7181fa2363dba1032adcad2492796d7d988995e',
                     'c24cb1b2a747e2b34fb0ee2d492898363a771dbaa738ce1dea97dff66823bb89',
                     'c36125e5ff9bd3a281dfa03c22d11538231fc373668feafdbbd63c1d451cba58',
                     'c3cfd6f1aecc0d7d66cbb2ba31ef1dfc4d36379e314052b18f1a0802d50be45c',
                     'c3df380a22f968ee266e1f9f652372b4af4d69d5aea7411472baf0322902b73a',
                     'c3eaf509b339c27bc416b1306da0a573458e24b00cb03c81174719495f8ee0a6',
                     'c42ab1a1bdcad4016ff71a6cd9911448ede8004bf79a66c8b0e4e9ea2d929ee0',
                     'c492444dc30340239f974cf8cc238b4c56e20c53c7d2e74e82a2440f5ddf629f',
                     'c4a2a1e053ad027e180ce34af277d236ed4ec5015c64cf30bf1fc9e7731ba666',
                     'c4da157f7c39aafae8c0661eb062391d442442202a3f8b740dc0e92cd6940299',
                     'c4e80e2436893859aeaa571736d0a5ba894d94ff08948128d8f0c5e7e2fa7164',
                     'c4ee2dceb58aaddd65a5b81f70497eb5b9d1b8728d210cbe3cbbb51f79f2e3e6',
                     'c5b481f52db15d2c57f790f18ac4da2280b0ed43c292b92b7e6e9a6359748d62',
                     'c64050d262fb76a726894c61f9e236eb0f210ff89afb4a466f6957ed6aba8311',
                     'c68065abf9be32ce7861f996cd150366c479e14e991aac58b4f9c77909b17c54',
                     'c6fa3622ed69437a80d1f23a0ba24d079d70cb17990dd72d39e35ca8eb714083',
                     'c7114b8b0457f5d66da3f5cfdaefc188b300cc66e270768a48d0df9fca9ddae4',
                     'c7a79fc1f02af283f8c3d41ccb94084ad4634a247694fe8a333a596ebe5ba49b',
                     'c7cdc1001c97e745aca47c5fd87bbed4f1304810c3dada34703ebb74d3fa4c2f',
                     'c7d83dcdfdb8d4d70ccd4d7d8f37e46ac5eacf3858e1d6073826ee3125c1a0c9',
                     'c7e035122c3cd95105d57dab4e8dc8a784b451f610e7e27ae868167dc57b33f5',
                     'c7e618454af33dd23340e8ec92d344dd3427cba435323ecd7e853c14022297c8',
                     'c7e7d5b4bd057a2d920a23ed6d9afb27e658179936767a73cdf72eae83798ce0',
                     'c8a7e9d6f07edeb91b951762b6a785d021fb412383de193dde325e2cbd00aa6d',
                     'c8ceaa52532b8a203d78a98f3075d7ea83ddac86d83c238d68730dc7ad2243ab',
                     'c8edcc3f7184a8c812ec255b2fdce5450699632b4afc44ffa3632eaba984bde1',
                     'c9bdb1f1f6498132c84cf3ed98fd05678472043d35834aa0d144337a8dda4f77',
                     'ca04b5d93626b9c802db94d584ea872d2d4d8292ec5f082c90818ac6fbab5c8b',
                     'ca23ec30d8348ce16a99baf9e360dba6440c67f05de44e7ab7019ef8169516a9',
                     'ca3cfb8173eb54060de39f931ac3d407825b65260c1b461424cc032efa849552',
                     'ca81fc43af7c64f7154c6ac29db881a181d61da42cf82d2617f6d70204af512d',
                     'ca86df99a10963102a44ce193d4d7a280965e46474166f1517d6cff55ad6141e',
                     'ca973d22482a4c8a7abeb6b7cdffae5ab02d2b25e7ef0ba9cb309bbe923b4059',
                     'cae555cb609253ed508be00d4aa6b6b4b0a19d765f48c4c516950737e527ecf2',
                     'cb19a5529faa2463a7bbe73270eb7d2a80853cfa4bf13b3bf1b83c0f67e6f57f',
                     'cb403476bfc828da561f22d80fa6c10956012eb16e69e0c9d1f076737a19a589',
                     'cb6dc46bb6cef40661228bddffa9866b16f8afa10e3fb9c132e720cac9b212ca',
                     'cc03caae1aa3dd14afadaff694dc1f600845a8f230f464bf4e36f81d1eb3f0d7',
                     'cc359600371abde4485f388b87616f589f5228e0cca128ad427a32c13189bb6b',
                     'cc3bd6fdcee10168ec426e8680b0f458b7fe3fac71fbefe3c6276b32afe42073',
                     'cc4da25beebcee8043a745438aca9d5cb7db726eedb373206f0c44b2ae2de1e6',
                     'cc4ddb7541bd48fde370d6c3f7497ac0058ad4aebe7f4c9254ca64b4e35496e9',
                     'cc54a9a090a2fded909e70c0526911dd47c0cb60458175dc36f6b206da4ca06c',
                     'cc86679651f5593decafab9d26150a955a7fdb402fbcf5bd91d31f0af1c9368b',
                     'cca03dda23bbe014aaa17a9e51a66c34bd481cbfd73b942d74e644936e136022',
                     'cd0e8ec20009a6ea5d0ad3c9f4fb13d8c7f63d041210414ced946a198f077761',
                     'cd1b1f48ec5c57c9ec6140c91458cdc65b8b698c4f81f2547d6856167af6962d',
                     'cd66b977fb9dad411fbf3b2830e3ad4e73f89b7c3aabc481d8c76ee85d964e2a',
                     'cd80785da155141ea207b6f66deaf78d185ed5e8f41ac6dd58055f0ee1c54ebc',
                     'cdb74e18bcb651c95e4a9fcc4e5e6f585442bc0083a25e24793a99e907358334',
                     'cdd46c341d141dc0a9156546726bdfe90581eb5ad0ac6d2f5b5b1a04ca3dbf67',
                     'cde315e713ded0863544545a4f50bcae520ee4af5aa89bd0297d654ed0f8d68b',
                     'ce77c42d9cfd2148934b6bba5adbd879deb6933e378fa39943f145527e582bd6',
                     'ceae795083b4bd31c52fc6c6915c6f71f8840b57dd285ede753d21757bd84211',
                     'cebfc6fb7a013acce43261abd4f0b07a3d74444c037e474ebb4cd7a53d4cfaff',
                     'cf0311467347d93d724f8a782c72a9907ca9af718d13075661e471540e30194b',
                     'cf3e1804573c00dc978084f5d4ca9aaa56879a280fe2631f3c853c074bdf65cf',
                     'cf513ce58259ab0ba6d6d363410cb90a78a40a33dc0086200a36fa3c4dc9324d',
                     'cfc65174175f84fbcd577a4c7c194d898f55ec6547502c3281f89fb35c4c9a95',
                     'cfddfb7feb0220159a6a026254eb4049f1eaee8563532998e27a4b41280fdded',
                     'd0143d7aa4de6887b1464913d08707509bb39d1cbf9c81a5d128619852e63d1b',
                     'd093be4212c184efbb87daead69bb0b55d0da6e84325eac9f33e65658a1d7f8b',
                     'd0eb40a24d2147cae5b08cc0bb8774994c2cce989266d686507352ed601e7f79',
                     'd233430dff625ac4747727605a31a30892ba8fdbc2a69c831177d1cba1e80eb3',
                     'd236b796c8d65f93d3df173d26ea568d7bc794d45f97a75933729dcb0163b004',
                     'd23cbd926904691315e39a5d935b7dc3b256764bae2547784b3c4d631ee11947',
                     'd27372a7c7e6ca823331bedee20131e5cb477e6ed886217bb00bbb690e6caa62',
                     'd277aea55028d6ecf6362b67a7c62d75ddeebcb6fb19c700d0ddf01f58be84a3',
                     'd2b247b474ae9ffc26634789ee1478154e3d816629e4571bf77280d09a1e1508',
                     'd2bafcbead55ac01b0678608926f787942d8c356fdae79b2a3930cff44890b70',
                     'd2ccef9b3bd71cbe5ef6ed83c740730ceba96d16cde0fd060b34c59c74b347c9',
                     'd2f6845706eb555b10713575e7b852454bc3b5620e869a4bf7a198ce56879777',
                     'd33826f37c63c459ca40b9e5c50d65ae298a46472923135a10cd5397e6a4dbab',
                     'd34727afb4ffd8d7c3f812156cfcb52b7dfb0498b7e5ffa3247a59fdaab40e44',
                     'd366fab0cd8aa3ee39534b74c5c03be31ec6b385f26bfbe7a75fe28bea870d3f',
                     'd3ff9c97574f8866edc51cd1023c49a0d5e614380650d7a04c4df5d4f99b3d72',
                     'd49a682bcaedc5bcb442b784967a4ac496dc470f6d3c2945f976314f71ab6858',
                     'd4dc0e275bfbadf5bf612b69fb6c9fc661fb338c7cc6ddd5a47d49b05ea85916',
                     'd50f39bb907d192d963ee4f7308e890ea35a55412647b6a515f491c5cf251142',
                     'd5d3911f303bad74e013b8b87950f8b3d9717d4bed508ed5bd8a21d9dbbf9b5b',
                     'd6e4b7cf5a1c0121456ea0836c6f10a86ea90df1b72f592b645d82ae3064f056',
                     'd72c616c02ed0095d22bcf3eb891edb90a1949ecd5f9691d35b6cb7ca8e194f5',
                     'd7c4e92cd959583908e7f3b211572c3caa69be2bfdf770ecad9dcc4f7b2ed71f',
                     'd7cca1a9ed20b8edf8b50e8882cf0a1ae705349957bff78211dbd08f4bf5e192',
                     'd8179d722bb94be1465ec1c6521a5f901ed7d42cc0f14790237936f603b30056',
                     'd8221b6593140465a6c000b21a9103811d97242f23349b130bc637065daa625b',
                     'd840d6a5d2a5ad8bcbe53354a243978687f89f8f8e2e32974a2ff3742aa6d7c1',
                     'd88af9872162ac4aa114a352e8437afbd24d238fb639b8e73f2abd8a85428fc4',
                     'd8e603fd5523d01ae05821e07851f469cb64142c8f3e192836e75a89f6c03dd8',
                     'd8f11414f5edbc58a8375e336d8d54693a1a72ce99b4752494d29ae0971de174',
                     'd902618e5fd70fafa264b9403794e9b8b8c94f54611422caa937bd5d1f2ce08e',
                     'd92fc51c6146d93e6046933a8a129b6f7d48c3ee002f438dd453d121b2ed597e',
                     'd9341c759c0898d91307cf9b3ae5efcb25bd5384b498e2720db8253ca7e8188f',
                     'd935f8de53e51d2c65282043779e213bbb30bc57c454eeef36dde7194dc72511',
                     'd969db429415ebe59b6d8d77857262ef3a03a1746e0df78b22d08160ff6c1e89',
                     'd974955e725729cb2fd395cce764af1f957747dca7c515c416c6c450c63bd758',
                     'd9a04f6fb43b1d111c3f2d7c607582b85df021297e74aca4c10dd2379c668d7f',
                     'd9a08d251e3a48b0a8565b3503f5a6356859b19f03a7c556f15510af762a1ea2',
                     'd9e16bba25112e60241dc6610d0ecf9a5cbad1bf36cb818059e9c0cf53a280bb',
                     'd9ef933ffd908fb9676fde4f74f2ddad1fa37cd86aefb0511ea543190a1c6ab2',
                     'da036e121151a78ab1fe4a4dca346f897200d65598f8370ac943ddb8805ec0c1',
                     'dabdb3ad5aab1e2c7606e065f9ec0b5ccd222634563149176046ec6a8ea7faa4',
                     'dadf3d07ecb656edfca3fff9ee957ef5ec1e3c14d3d2c1f823e0e09db570be5f',
                     'db13a623339e7eb113b514e526b8dd4c48ea0bf4fe5e6b1ab67e46e0ae8755dd',
                     'db6de543c42b65a7565657a3c4e77caedf2511b0c811127bf869b31411c86cad',
                     'db76760e12ebdbb1902cec332f7bdcf24dafe33fe8eba0bf7ab0a2b05006d434',
                     'dbd9516dd9f7342bbe1a2508066e59241b26149fa63d2ef2ddf6679d0f889c27',
                     'dc2d61864f7af14f4a8eebf7898483f1acc9e5b2f7e4ca2da3caaad7679e61a8',
                     'dc2dfc2cd6c6ad7a84d41976d2785c76558136266719687a319cda903c5fb043',
                     'dc73c114a23d9a276a30159b29644b60fc23d7e8b993dc60033a4b19897cf786',
                     'dc862e6a19b75191e5e06461b52f356a0f6fd7d7c11c8715271fa2109fb1689c',
                     'dcfaf7d2bb65afd16465bbffdd36a52c415e9dca45dcb8e0cbe0ba7d640ccb33',
                     'dd57973bed8793da8745cbbda0c478556fd132b0e5793b8ddc0d8676da89ca5c',
                     'dde736197952b12f8c239f70f050a129e2afc81d37fa389547f3470dd790bd2e',
                     'de35b8d3a9e2097661607d71f0cea4a81494e857a66c798d9d0fab7e7c877cc0',
                     'de61449801dc066b44fcf0853318ce58af2c8d984a0126742d097380c49ee751',
                     'de929ec280ef96561b8735c5ae9494f057cb9de0720c5f0972708c7cc1fc70cc',
                     'defd624650aff73657630419ac2b51de85a2234d6ef2f9e479f38180346220fe',
                     'df1d175b14bb0ae259ff6298354fedbb71afa25daf4e8c524795ac40c10ae23c',
                     'dfa53fa00d7f6c4b7783367e4be0a16ea069b4e89eacbccd612f261ed079817f',
                     'dfb3fdf3af1d7e368f14fea574b096c53e89d5cc1d6c477b5a63022300086430',
                     'e01644a008693a71a7f2448684091f0536062c402476d16309bc418fce52f18f',
                     'e080309ce951fef55a1352366e8afb4b420c3d6871c3ad673ce90b103f4e93ce',
                     'e12febef05ccfa43359e560f630b571263f677b2aa42038626d8ceb05b7e1e3a',
                     'e14457b5749ec675fcc92e2ee2dee0898f73fea0aa30562934ce397428435a1f',
                     'e14b2a14b3bbac0f69e14e44c4d090c53b2ef144db2c101bcc0b270c4e2c6d14',
                     'e1566d858f9a68cdaf9896631e6d6ab106ff599b5f8db532138af0b42825b7f2',
                     'e19e9e9153cf33a6769c5b4cc56f7591144dd973591136899f83cc9b5cc852db',
                     'e1e99ea08226b72a67793ca36c187695ad647d021677edc3f27166384df63c8e',
                     'e2f65ef69904f43aa56eba11f5b03451df30d446cc9e0e1132c74fafe4d9032f',
                     'e2fa79999a98f5b3cabb16ebb588174ddc4d05bf08863394efa723b46dca8486',
                     'e3164b923baab1fbca4ab356b80fb9e1a9daa944bfd78d9268b11ae8434020ce',
                     'e32f5f34729d144d028482e251ee067bfe6535a534dca4354e6bd5cad54b7c83',
                     'e345ec6f23f6de8abc8194790b24b86bf456626076e1deebb013f4bb9ea863e0',
                     'e37c886c5c2caa7041bca9e1d9608d3c04e509bf7d48e4ff4f9e73c4687e5f5c',
                     'e415eee54ef7c2e9c9a7cc535aee24aac6693b5434dd5f06b0622341561e9457',
                     'e4ef9d3b81bdc0e92111503a6c60f0f25bcb1130e707e815cfaf8b82b0fa2993',
                     'e5534e43cdc30b89af72785768e9decb0a60ef74f3408cb8f8bd2ebec44589ef',
                     'e574bc98610b37e4c4b4422efdfb3666db6f72bb1b67388ee655b163cb962ca1',
                     'e5f04d86078362171936c15e342bf08ca88df1353a48502a6f1e7acf131fbb83',
                     'e6357c6de6c2352032c9e3ca16e149efc5e7e2bcd5b5ec4a7807dd99ea7b5034',
                     'e6586e2d84d8ced4cf02469af1e39686e6a3aecfe97df9f1c373d6f4b8d3a293',
                     'e69d0876a300db1e031d0cb0a01f3bb0e63c01276176fcd5a2cdd279c07af807',
                     'e6b0566c4b4dc0e89ed0cc5d4f622cd4a54b0a9213c232b93897d011b6ee8b94',
                     'e6b79cb88cf921708e332d1b88ada87026c0b04483bcef7b9f200aa5f973a996',
                     'e6b92e3fc05a80b270909b215ecb8fe6236c4da2df3d93ce225675115c19ea84',
                     'e6e4d9466de21843659473842cf8adb8588b097e6b065e3878a77576830bcdc6',
                     'e77a6e19909d4ec2bbfce3357fd7518db1042b0d5fc947db9e5275db886b545c',
                     'e7ab25d666f074fc03fc3f77490bd3808e3fd77d7d8f081c87e5c239e4948753',
                     'e7f46b76749b6c4ee329ec71bdec6a14b0c4f2e1489d4dc417ba63cdc699eb4a',
                     'e8272dfb2f764870274fd990d606a6403450a302ee0206e224afcc5102b9a003',
                     'e84b504de234cce8d89145cad2e9450606332704ea3b1b477ef68835cfb67937',
                     'e8e1d414971b5b6d696f83ddd2603a7021c07a3557c226684628ddb312569fef',
                     'e8f57178c9dda1ad3b181f6486f55c5564a20f1e86a4d56655537522cd01f21b',
                     'e97d01c10a20c598100f1217c569803de9c62a2b64a4e7a17e05a474aaacd97d',
                     'ea1680874df1971470cb0a4155aa99d92d501c8eaf27010f2ca17c2904fd2045',
                     'ea428fb6437b8ac6679d093becbfab21602e4ae2df58a26d44448fe2abefef96',
                     'eaa9d7a487710aeb6548e16c2e0dd3efc7daa03231ffa735fca26ae91d3bd158',
                     'eab35ad7c9758306b0ba00d707c82fcb235c75d96a1cdd5dc1f82481ca9458dc',
                     'ead0074e4e8cab8deb3642e4d79a3906706375233c3404d351c2e94b9a461355',
                     'eb024b1f260cad8cc06aa53080d3c878d461a76e712237d83e234b46a072d00d',
                     'eb1ca8d5f10b62163c9ed65701d2bd1e67bff9125c8b7f0bcaf5c31f1f02ac3e',
                     'eba55db0c939c9dfc0c4b393858f5f871c1531d3189731257c22a23c0037179d',
                     'ec0c2972c0b16806d604aadc9317cbadb056f7d6c5045396629d20783c1392c5',
                     'ec1168b261b9ef82895992f313604bf97e081c14ec81a9e3f940aa22ad97b950',
                     'ec95b54ef913d7937e0137da18fe873a8fdbc145e4dfbe3b9834859600f91e40',
                     'ed55f699efb2f09689d7b8ea4075a8a610a7d0f7860a833eec9c5d060439f5db',
                     'ed66f190ab90f84295ade5cedb41acff6a251a8c5ca738c5a7b7c41522801dd7',
                     'ed96c947fa5895d0ab7d38332e4019d5aecadf0253ff0753c8047e0e25306761',
                     'edbd398e13fc51376fca8effc35b797d5dbf0cb77cf3a12eff6169bf27e48d49',
                     'edfbe86f13ceffa5042815ba102513ba8d19f4ad9e17207503abd7e831de69d6',
                     'ee4cec3bda0a56d6226781d4402b8f91886d09f18e107e3c893b6a6e499e4c69',
                     'ee7bf71097a5622c351f08abad14c11a9adec0b358c858c83302ee7d4b2fef03',
                     'eec49884333a7884063392202575e3f3342aa6a4ae4c0d6a9b93db5db0a64b9a',
                     'eed09b78d2b484d666a5fa6e084df72e2b60ee7122deb52a52fd05e6bdd2ffd1',
                     'eeeeb302b91f620621b0af659c763796161ad1af8c389cdd0cee8b79a0b34c3b',
                     'eef1bc0bb45b1979e5f69920c3c7cf6d030a4ca6accfd93c0a6d36a64ec93391',
                     'ef0cad2e9d77116b8fd0882e5eb722827358172564f4c8982a214a692d26eaae',
                     'efa84ef539854106d893867c732a9253f0dd32b89e3170363ee0c9977d7afd0b',
                     'efc76c8399762286029c3f42f33dccb6e4d61bfb25a558ed7c091572a2d63265',
                     'f004393b40719704ee099a14cdae647900ab407b220946d5273feeb301e5d08b',
                     'f0318103d62cdc10f51419aa5fd71ce6d7ebac8b283a78f09a7e11b13072be12',
                     'f06533715e8df31c80ea9db678f2da295132190f46f7c80eaf288b5fe9bbde2e',
                     'f06805289345f5e8b24c25507edf4d45316d9775aaddd18c34a0febf3277ca1a',
                     'f0b363599559a9bbf9ff2b973f655f010c1478c84d74b6fe70d388224710df09',
                     'f11f56c76490305d1dff4ad0f39fd03cbb4208ad618fee401c3b25a85ecc8929',
                     'f1527fec7f84aa65818f46672a38a39617ae498e62ec741c96f2a16d6a9d9d2f',
                     'f1c35a4bd9fb0eb3f9bb3e97d80846c5432561acb6d38688311d597f7d6c7c4b',
                     'f239b953914ed24b32b8b0986cdb15de521f70fd05bdd13964edbb30ae30bc3e',
                     'f466b59899686c51c42b17f4105047d0370cddad50a64031e09a9a548f5da029',
                     'f4810de1c010ff776db227074cf01b562d5adc6a25e68cf35ae4e9a7defad789',
                     'f51b99c78324c461e23f3c78216b5bf26e6467fe4aba1a350a5fe17494e9c479',
                     'f51d6d9efded51863601fa04eadd76433b8cc96e9328798d1045365937e53c21',
                     'f58e07602cb7a38f1c9d5ec35863924a4ebfeaaed4265c95388570f0863e744b',
                     'f58ec063bab0102aa2f7d27e3b13d34b0e31e5a60af64d31776e23e1fb29e376',
                     'f59b75a7b0b397a618686b749b51232ba37dadfa465451c66c413177a2ac29de',
                     'f61f6eaacfc4b770f5e69eaebf931af158305ca1fee7aa2454b038245c20b8fd',
                     'f6dc04342b5e3813cc18175c06c18391fef1563e15849d5c14e9e598ab194ad8',
                     'f78440d114a63869ee60b76aa3fd52069d74fcc5556dae9f7e362266a8cf029a',
                     'f8126fb31bd1ecea2b84152f6674f5b437e561102e63d6f4ceb03100434d7772',
                     'f82a254bd1e8a7f922feb52bc2ba3f8ced6ee800e5bfec15cd1e9f7ce24dd9be',
                     'f897e5a9a258dcfbe1874c2d011be02e0cdf9773fc8ace2c509ffc5b1d56b732',
                     'f8d639fb05aede8c77161e4a506de726c75b024f39c5a54ca16751c72acb16a8',
                     'f90f20464e3533fca466bf0a2a181fa929b9720814297632bb539e363e3763f1',
                     'f927560f9717e9bfc3db93530abc7e8654444162c84677bdf547d4018c0d3670',
                     'f9f9901487f4c64e36c4441e00362a9f012a945b6db5cfe2030fe800215c0bb7',
                     'fa0521049456a3323e3e3c5cd39274a4e6e3a9622f3bf4e9131f3245c58b61ed',
                     'fa6446ab014661dfd372f375987f0c5cd83701b7666a31911b7eb79b86a92e3b',
                     'fa7da05a1023ec96754bcb456a7bb4893f96cc7bb3f62c08c8c84a9d0c88881c',
                     'fae6ebd585ac5b0f558231478ad79fe19dd4c58faa94beb2cb1ebfa002dce89d',
                     'fb162a0c965408d952024767012c39958bb7fb6a0fe555ce107f7141c1f906f5',
                     'fb200baa05919dbed625ba042f147faf717b52701211b491f9204ff696994feb',
                     'fb2e3ba905df1ae445ec49e8177940a17407e04bd2059a64715740039fc92d4c',
                     'fb48aec8cacb40a3851e8157d5ee9a8110888306f73c931290b897ba8607a55f',
                     'fb4aedce8f218b1e0bca705da987cbec8072744ba1b1ffa055b0f34bb0565b68',
                     'fb4dda03341891307ff3f1f0e8f5cb73ab6c9b0a44d16e77dd4ea273ae56f641',
                     'fb52cd526ff3768381eda8772955dd139630b1c92448d3ecedd800ecf14eec90',
                     'fb66870b3d4622b4393f5523bd9ee8463da677a02a08d4c3f46169d4c509abcb',
                     'fb731d322a6128b06ed703e346e861fcf0c2d9ab01211bf0dad4967c486ae3f5',
                     'fc93a2eb65773a02a1711593d82795c65528f9212f025e48f79e6f9eddfdfa54',
                     'fcb3808d9e02251e3e77933b60473c7e286a26c0d435c52468e019e418bcd245',
                     'fcb6a5ba8ffa2df8138bac17f50d72fdb9acd7d0a3a31f0f420803f81fc69264',
                     'fcd4618ac716df616eba2510c6afd4e11c2f67237836c0c97d6ba71fec1621f1',
                     'fd91f378d07178bc4ab2ffdae4702a328f8d3e026d9d7f802109abf05df933b5',
                     'fe39d216b6307c6809c5d4acea9d4f5e40e79778ec05e78022f3c33dbcc6c4ff',
                     'fe76d9c7b46f5c29f6db376129e54ce177326809ef784ed3339be7309603bdfa',
                     'fe8a75ed93a09e5384ceb4783f6baac43949d1a9f2b543a35a7681da6d053480',
                     'ff29150152186fcee2398603ada81055195bdbb1721361d4074d2581ffcee0eb',
                     'ffcf3eee073fa2bc0daa9316c1f848e79a65ec27e8f63b29f7b7f2d1e80b7898',
                     'ffdf6d08f16d933fe13a421a52df348c6f90fc54bbe10e4a76c9e5b2cb21cc9a',
                     'ffeaf0b81a670d518edcd623dde67ed4a34d40754377a0bb6b3c56a4b382d664'],
 'count': 200,
 'data_lock_sha256': '448d5ff6f97c828213877830d73f7f629b6e0973ad109a92145c9cdd1639bce6',
 'eligible_images': 500,
 'entirely_new_external_dataset_claimed': False,
 'excluded_roles': {'prior_development_200': ['0066845163f38a25828bfa1e29516b0a472661a09893d66fdf611694ad8f8284',
                                              '0127d7350d9710721527acfacfed0492dc9bacb0cee58108f994bee8cf03fec3',
                                              '0371dd8e6d255c728c45103c8adc87fbf93770559b1b2534f4fc848d1a58b9ea',
                                              '03b71f226ea44462a1ff05875a802b60fe89c41a952f879979956d32b8cc57c6',
                                              '0428ba35ff505fa6cedc57e046a455ade950dcd487cdfee7a94743c494a68d2d',
                                              '04dff21d7d781ff8ea2ed80f6da1a165e7797f9a398b4fd930bb935d84fae8d8',
                                              '05fa9cbc11ca5d438f4a7fd8f2f263f85f0f014f034dcc099a711dee58022289',
                                              '0679c8b0537a146003135e865121b10b489c18b661562795074a4c46987f2edd',
                                              '075cf56c692f03d94602f7174152164e29f6fcc3e213fe20b86394d29c044af2',
                                              '0e2ddf287e7584f51552daf7d41f4d85e9cd406da782015f5031a95cd95871a0',
                                              '0fa3fae44c1eb015fe9adc1c7dd2da907acce30ef080449e5b1d53b71a9d843b',
                                              '1269e5f1b5b64c8442881cf6aae6380d387eced7312ec5e89d677466967ee3d2',
                                              '13cf06d91aecfbbf8ce84553c0cee40fb0a2f8a428aa78434b5371780686a56d',
                                              '141fe37ce0253cb31db8fd9b604c4c9673f5b7de9aea16d7c00c4758aa6f35e6',
                                              '1628c2c04b97b49be78116becb52a0c881654de00604da82928aa7b4dae90013',
                                              '176260b73073aae1dcb99f05af28555b4682c100e063d4d4c45aebc6bb209a48',
                                              '18d63182f56c67e61fc78dc3179ad89ab7e443d6c95f431721202fe0129de547',
                                              '18e95729761855bd29115435ea4aa3720418e954fab04dc1996a423317d5ba75',
                                              '1932b91f7bbe4cde3fe574328ab25ee7c7c92f8a1c8889ade9a5bcce8bb4a6fa',
                                              '19c4310b6e09bc4b22082dfb82e88d33c8fd53daea76c08bbf32c099c93e22e2',
                                              '1b8930c4e710f3484d1b6bbde884a4c24f83cd8b77a694a59716e849ee506841',
                                              '1bf9feae32fe5ce0a380a3d37b4754ce1b18b63d05d6bb931854519ea1a661b6',
                                              '1d36b5bb3a75f95e0c2d624ba5acf329117b2a331f7f83624b0a819966a5e605',
                                              '1ed1c79090c93e3f3ba4503b11b19d8f4a027c9ddcb4394fc3b6859584ea80f8',
                                              '20f7ff1137e25b4238efe36281032f8e219e48ff2bec837c02d6941c83b2c46c',
                                              '2295f88b3afc6eb315197029301c0611399638894e36d518de076c7c95112cf2',
                                              '241f921d4cd85481e077b18605fb2d2b1ae4d52f954c86a8a743e603af0b75ec',
                                              '24cd687a287375d73d5cf90864e7c0314a180fe81af899cce5fe86ddb7f26d64',
                                              '2591258305dd07328766dcffb7232b23ae78a88a61d9a878ed913105cd21c3e8',
                                              '26b9c2917759aa9d90241b26b7fc1de4e64ad52b1a6307a5ba33095d054e19c2',
                                              '279cc38806a4e37f44a799845db932d4839b34e7b14b933146bcff349661d78f',
                                              '27eb5e3c07fcc322a18a8a5a63fd03877e16b86c0516e6fe4cd656da5e280c05',
                                              '29f63c30a424c41864cf024bff7badecc8a25f141f50236c7eaca44c851d148f',
                                              '2a40eac16287efd2e163305ea3a4b517ab642c23b229cc77ffdeb6f600e4e544',
                                              '2e63938b2fcd265175ae05748f4e266ac8dc01bd2a38294bd42d515f3a21ac85',
                                              '2e8151a6bb8aeec62a83a29bdb93303bb7265cb28ef5deb9fc2bb6f933e4801e',
                                              '2fad80f5c24928cfd53dce72ff07573f693291059baf3d78a37b139b195b8796',
                                              '3084c85b7fa890e77560bc72e757fadae863d9d32bfa30b3cb28cd74188ebcd9',
                                              '30c131076a387144a12eea1f98f9a4fca2f5af812c0e4a57d7325adff5fb0c49',
                                              '3220a60a4764025feb88510610f61f0ba228041e66a56f3dc4a332ce1b9a884b',
                                              '32aaab923cda5b9aa647da5818096ed7c6f2c9f857bf5c04f6238428fcd9bf4d',
                                              '32b668e9e325e12ef684de614ce85d30fd2b2dd3496fa6a196f3f70270a65728',
                                              '332c392379f7a2307826bc57368d861dae2c596475f3193d861ca586cf3ad873',
                                              '339b927f038ba1c46dff3a7497ba07609578b8a85ee910819aebe439770f57dd',
                                              '3558079ab6c1f10699922b3dae791721a79dea43128312de14bcc85f46d9165b',
                                              '36559d2545c16bb7eca9ee449b893c48596d2114b7b000a1819e16d4035e826e',
                                              '367126122831aa7424a9498c641885faf8461456fd6e86b1d8b915d920401d81',
                                              '36e5a076cf2ea05749a06cb2bcc26e4fdf39cc250f8139d40b8182d03c7148f4',
                                              '37e7e20d527ee6051d49773937c09beb8d178c20de96786ec44494927e9e5190',
                                              '38285e3424177b69ec49d8db865a0c83452012dd4a00b46e9dda6f406836110c',
                                              '383f1af4bb2eaa49811f0027d7ec0f89db9b340aef36616367c692b5c81d5f24',
                                              '384e0cda59102c2e30428d0327ed6d9de19a1ed8a5634423fc8d1456d950f920',
                                              '38ef6a56173fd38bd4129130a9632bc3767d5369a2d486900f8568e43eb8d9f4',
                                              '38f327e85514eb45fef5ae3eb06117ab2e16df93ec4a8b85e475d3e04265c69b',
                                              '39e1dba8068d2af620de27421472afbe76be97fdad46683b25745fc6deb883a8',
                                              '3a4cde911a3bec8da83cd0b644ee297788d5b85a0af9f2eff2a5cfcd779c333e',
                                              '3b8b929104cf13efab270b7e56ba0f38183857955095ec84191495eac11c740f',
                                              '3cbfaa6d4563c235b0907c2375dd44f3b96d591099def6946e870d66a2a86547',
                                              '3dcd17e02aae26c95a49837a391966b0b650d44d575c751e3a47bb1c73d3c23e',
                                              '40d9ebc0983e831d0f8780cac97d2dd788a681bc11f56f85d429311dceee5902',
                                              '413f07cfa1eb10f6413ea8d977b392d05de767bf4bda512038954b9a004a13a0',
                                              '41ff58423114103524a85024eb29549a416af37b154de6966992f43847ca0564',
                                              '446896b6489600ee23701379b84232fca38edc64e8c4cc90efeb579006be6662',
                                              '4477fcdd39a8d7496347489d19943f5e3034a19fdb1fbe7efc4ac80f967a71f8',
                                              '45b78077660897b40a0f02bd1166111058fc7a0c885f090a64562b5c95af7791',
                                              '46533741c06ed67856bbaf606dcd67357979e06c9434991d8cbc88aa5ede851d',
                                              '47d78f81622fab2b16724c81c6cf0c578e2fb2796f232bd4eb24e594373559ca',
                                              '499c8b5c58b4cce859361df6854f179566072c4426c8d9ed020749ffe7f9b50f',
                                              '4c722fa5afb35046c62ddb85504502d535cbed2fecdd9fbc1ee6b464a11a8d0a',
                                              '4d8dcb6103b5cde63d643b86d5232a092c0adb5ac64d5805cecacfa4301867a9',
                                              '4da39f2289e89580e18ece044b3df3ea9694b8cccf0583aa6f8d2f7384b71f10',
                                              '4e88410983856bf3a09e91b936db2ced9d720d09259f120d6f10011917b94fbd',
                                              '51272a4151c4e994491cdc6c2ec65d64bf3dd2d70e4c223b578531635eb11042',
                                              '513a05cf9fffd22ef77fff7686a9bf0054c06ed46ea506fde62763b867415e20',
                                              '51d5b11aca2a022fceeb7f40f853cca843acb0927dbcd1451525a6d048c6a52d',
                                              '5229daa45017552fd32669dcf4a96e45dff97671f5adcff6aa2e46f7d0140112',
                                              '531c76caf1ac8c69f1d4d07a8e17a86c20f4e45977dd676d9c267f3a1b448e74',
                                              '5416f31fef1d2f4b52d001216e07fbe78ba73b88251ae698689f9a6b218f34b9',
                                              '54679d0adb8a82eac4afde1ab7fdcde2308e1156254d8fa040d6e5236cfb6e3c',
                                              '553338da2701b87521a32ba805028a46d66ee0c6c6ad4e85c50df9cf50e167b6',
                                              '5715abe0581dba527e32f73cc3a6d92c67ba69448b31636135501012da89841b',
                                              '57c7f9893706b98cb1b263db5da76f6b7336380684efcfe9a167b77384cf5c6b',
                                              '58008f06339e96301472f4146cec2558e25e79df4d0e0aa8511d08a9fe5d409e',
                                              '5958f91d8bb273bbf211d7ab1511d885afee981e219b4943eae1ecfab3370b17',
                                              '5ae768884ba8c293738e481645b30e939930a6494ed00f2df6c1b2a7140148de',
                                              '5bb51e4ddda8bf153b6623460c5029d355d7ccd835a9ce1b63e46f29bc27dae0',
                                              '5c120f923a95c00b5a9c1f6328bbc9cda0850ab7cd9fe68c35b60bbe51ece0f3',
                                              '5cd3b15957e86d85be27e14c05c35b8b1d3a0aa9646995404117f630b18a758e',
                                              '5d542cf81e85c032a23824fb06a5954b8471f77bb7ea92fdef962002e4413a37',
                                              '5debd9bd1bcf027bd3df503b3fb73a68924fbd5b3d7c55852f379c06cb61f238',
                                              '5f09033528e4da13c2ebefd8dafc8d749293d6a12e1021a6c2ff78aa0e54d157',
                                              '6055a941f23c0adfabdf76e54581c320ad8f2315d69bee5521d5416ecaef7a6c',
                                              '6212fdd2df4135dff2a4f1e8952041ef98143dd4dbb0974f0fb172a21ca4adc5',
                                              '625023052b7d4ec21f9449e9e2983590976dca8ad4040b42961775bdd94b7e13',
                                              '634ee8ab28a8b1ddfa650054230cdcc97c970f8c51c45c719c34b0605964200e',
                                              '63ebb29d693bb899b4f8271559b08766c525fd555519f0d20d70df319dee630a',
                                              '6546982643a9a818a288b099c5f87e0c7f16a18597b104da060239b405a0313a',
                                              '6875b7f3b4696412011b80ccac0ae7b1afafab19e70a195bce1186d49e021cda',
                                              '6a68787b742f537eb20b088a5c9834bb03c28d0eb03ff671f9b8ff3f07395388',
                                              '6b379aecea6514f6a76fd867ed5812830569fd228f9779b6726999cc3d3f3ecd',
                                              '6d1f0862f5e340d61854b140619859d5941cf9835788f3e71711c839ead556b4',
                                              '749a2c0ae14ffac85a74ae6ad8f518175790c4ca1d997895ffa9b863ccb62dbb',
                                              '74e2ae59d0c07c953be1be0036dc681324bb7b350dd43db9ae09049e358003d9',
                                              '75b884dd33d394d344c21ee8e7b66421b43ea8cfa0982cbb76f78685428cd9f2',
                                              '762bee91ab3300e45c3e67c791694f5e9e40dd48676458697ecdc45d09a14569',
                                              '76ebba41843d5e5007eca1207ba93950938ac0d2c82163bcd6bbee0bdb55e861',
                                              '77cde4cee01ec4f412069aa2ae6ca3d50154e6a95587596652f4d002d83325d7',
                                              '792506db3de2185698771bcba1ab498bc7ec04e1d6d40406167cadccdd378770',
                                              '79d2ee709fe90a80392c3ea347b245853a2df6885b5a08e12695e95a5b1fa5ae',
                                              '7d1c26659b5abc30b4461304e0d55a45938a0202fb4dda162642ff6cdbc7583b',
                                              '7d33d0329a2b9d2c2784176df8e36680b9ccd16896c9677e9cc96307c4e7c5b0',
                                              '7dcf4a149ee83236fc6467851c1a992b4107d833c99d7977324bc24ed989c927',
                                              '7e74b61f3e6334fb3c0b359f71c9cd53e1982b50cd5427a94f36940db9de8fb2',
                                              '801ed1c5358365b330eebb20cb4f7480fe83a96cb240544509db85b37410deaf',
                                              '82aa3ccdf29aeaa6f8b3a1255532f8f58c550f0f343615d0ba8d0cd6b762267a',
                                              '82ba018e345e09ec588090eaa6d5030da344d6393ecbc0b091d7d6c01027403d',
                                              '832d52d7219035a23a80c986cd41c2961c1be6aa7f83300e928d9a7435d5f893',
                                              '83f8436a8ad19dc5f504b38407177ce9a761b8550b2dd78f6394676221c2bca8',
                                              '867884ca840bb4cffc94cb14784a59146ec17d21445fcbd912a76dcd56a288d9',
                                              '881e5969b0cc5ae398f845580772401efe205756f3a17d63549f23ff5c6a0a6d',
                                              '8d7c37d7b3c2ae9536a79dc73c978546bdae97f5050531989354ec29f2a7dd59',
                                              '8ec11e81552d3696e99982476a898b20ef63825e77e3cf2a7c5cc0d8411a381f',
                                              '92e0727f52430f0f457dcd108ca3ae6df86e38f153d3281f2f9d7c3c0d635805',
                                              '948c52fac43e824a898f1b322ef815d1c85ae5d2e84001d078585ad0ce53347c',
                                              '94d75d14c434f35a4cd69c76973a093a02287c2ad01ce8a01f0d6052a04e19a3',
                                              '95dcf2b60941b5910ab09a008eddc61fad56dfb03a59b3a7ab1b6f845cd0fe03',
                                              '9792b8709c6ac19d37bcb83aaca038f7c34b171f0cc3c0ad907179d70eab5fd2',
                                              '98f7731a4426d8c50ad3f9203f1e1d3bbf643aed9564e29139f29f9f0dcab26e',
                                              '9a4aee6a95b54619f7c8541a6497e69133bde39d782f18aa6b119ecdd6da0ffb',
                                              '9d7258d88bad246c31aae1fe27311016399fed9409d188ef8ec6b5971d130e8f',
                                              '9df229ed3b72c6f9ca679843b977eedc9fd4a0c7d9cba087242950483e98266f',
                                              '9ef8d947eb2796927fb9fa8926b44df99a1888cb92b40060f78ce6f532bac088',
                                              '9f74a9537358bd01beedb7ae65a5abeb9ba74c06d1cd036aa70256e3b7793029',
                                              '9f879f6d8aa9e5eb62cb3af349cfe8fc15e8eb9fc6262cb8a377688c597dd242',
                                              'a27e8db63340cb177a85d996eba4e9fe23764fb05fb9687133ee99910a0ea34a',
                                              'a31bcc6af1b7dc739114ef1d8037be760b99a6fec1c93efd3fb594bc06f99358',
                                              'a35875fa300436200c4fe9b56687c01feb06f1ce504fc6d3977013643a925f37',
                                              'a399dc2f02c8d4432dc14604458e3ab591b6205a8bad07c965efc84ffb7384a6',
                                              'a3bd77ef4e109265d159ac450fd96b2b2bd3630fafdb57fe57fa57a0519637bf',
                                              'a409ed9ff99ba4e15aeaa32db6bfb186a72204a376899033f62a1ba270355940',
                                              'a55621a66ebff5ef62a96a300b1167c7331c2a9fad28396cdba67b9cf9d73455',
                                              'a6b632bcdc8e1b77a219c6464867b4fb3ae55d453cd0fa6244e29d9a54bb80a3',
                                              'a7b5aab179ed65fc4b0ebe2ebb28a31a8535f93fa3e1f899dd3c6f81053b8deb',
                                              'a7e3f1b3772181ebc9dd1f14028659b654087640535343159281822e5644994f',
                                              'a8f21e01fe3828636c0a50e126554627ca10371815e24d99d2b167ac6f896cdf',
                                              'a990fc5b32d6958f31feeecf0357c7e79ec463f418f1a341ece481314f7d1f55',
                                              'aa5563a85b650cd318402efed4f90bdaaf1d040a6603f1af4fecd17bb1a3d2c3',
                                              'aa9fede7e598760298a43ddc1ea9aee4b4b4e6efd6c716f84196cb293abe8c09',
                                              'ae2015c7a2ca470b32af4321f30eb839dfc3ef015fcbc19e59092fa11c1f13a8',
                                              'af2bbfaa1cdd9aec1d5eef54f195b70122c3039bcad509ecc78233ffd5e52f6a',
                                              'b310a4a603a4b7b49c4f73fd15cf1a8500625a65fc9509547ef0cfd5b13a3678',
                                              'b73baa26e681c985f3b9798957e03bde2b4e573896b3904287a88c80a2b93b3e',
                                              'ba31ae258a567cd94531ef9009dcaa08622bed1722bc458457659f9dbcbeb408',
                                              'bc7eb8de8925187e8cec597e27caa7eccc6464027220e47cfaeea9c88c477c5c',
                                              'bf025830ab71f7afeb66ca9a28e27e456a5928119ea2c18d00bcc6656170cd0f',
                                              'bf4819ae63911316798b91d7663f6a48c686930e1517305b307da44a68f8da26',
                                              'c0f77f115e4502dd098e37f9bbeb7cc2020665e14c5ee461e2ce4b515eeec76d',
                                              'c1031d72142bb2a637870b199a826740a1a219249aa1f913ec28080cf3fc7f32',
                                              'c1687df1cafab395dc4667d979a03d93ece90a53af90e000ed55918c1bf6ea82',
                                              'c4da157f7c39aafae8c0661eb062391d442442202a3f8b740dc0e92cd6940299',
                                              'c7114b8b0457f5d66da3f5cfdaefc188b300cc66e270768a48d0df9fca9ddae4',
                                              'c7e618454af33dd23340e8ec92d344dd3427cba435323ecd7e853c14022297c8',
                                              'ca04b5d93626b9c802db94d584ea872d2d4d8292ec5f082c90818ac6fbab5c8b',
                                              'd093be4212c184efbb87daead69bb0b55d0da6e84325eac9f33e65658a1d7f8b',
                                              'd236b796c8d65f93d3df173d26ea568d7bc794d45f97a75933729dcb0163b004',
                                              'd277aea55028d6ecf6362b67a7c62d75ddeebcb6fb19c700d0ddf01f58be84a3',
                                              'd2ccef9b3bd71cbe5ef6ed83c740730ceba96d16cde0fd060b34c59c74b347c9',
                                              'd34727afb4ffd8d7c3f812156cfcb52b7dfb0498b7e5ffa3247a59fdaab40e44',
                                              'd4dc0e275bfbadf5bf612b69fb6c9fc661fb338c7cc6ddd5a47d49b05ea85916',
                                              'd50f39bb907d192d963ee4f7308e890ea35a55412647b6a515f491c5cf251142',
                                              'd7cca1a9ed20b8edf8b50e8882cf0a1ae705349957bff78211dbd08f4bf5e192',
                                              'd969db429415ebe59b6d8d77857262ef3a03a1746e0df78b22d08160ff6c1e89',
                                              'db76760e12ebdbb1902cec332f7bdcf24dafe33fe8eba0bf7ab0a2b05006d434',
                                              'dc2d61864f7af14f4a8eebf7898483f1acc9e5b2f7e4ca2da3caaad7679e61a8',
                                              'dc73c114a23d9a276a30159b29644b60fc23d7e8b993dc60033a4b19897cf786',
                                              'de929ec280ef96561b8735c5ae9494f057cb9de0720c5f0972708c7cc1fc70cc',
                                              'e14457b5749ec675fcc92e2ee2dee0898f73fea0aa30562934ce397428435a1f',
                                              'e2f65ef69904f43aa56eba11f5b03451df30d446cc9e0e1132c74fafe4d9032f',
                                              'e3164b923baab1fbca4ab356b80fb9e1a9daa944bfd78d9268b11ae8434020ce',
                                              'e32f5f34729d144d028482e251ee067bfe6535a534dca4354e6bd5cad54b7c83',
                                              'e345ec6f23f6de8abc8194790b24b86bf456626076e1deebb013f4bb9ea863e0',
                                              'e574bc98610b37e4c4b4422efdfb3666db6f72bb1b67388ee655b163cb962ca1',
                                              'e69d0876a300db1e031d0cb0a01f3bb0e63c01276176fcd5a2cdd279c07af807',
                                              'e6b92e3fc05a80b270909b215ecb8fe6236c4da2df3d93ce225675115c19ea84',
                                              'e8e1d414971b5b6d696f83ddd2603a7021c07a3557c226684628ddb312569fef',
                                              'eb024b1f260cad8cc06aa53080d3c878d461a76e712237d83e234b46a072d00d',
                                              'eb1ca8d5f10b62163c9ed65701d2bd1e67bff9125c8b7f0bcaf5c31f1f02ac3e',
                                              'ee7bf71097a5622c351f08abad14c11a9adec0b358c858c83302ee7d4b2fef03',
                                              'eec49884333a7884063392202575e3f3342aa6a4ae4c0d6a9b93db5db0a64b9a',
                                              'f06805289345f5e8b24c25507edf4d45316d9775aaddd18c34a0febf3277ca1a',
                                              'f0b363599559a9bbf9ff2b973f655f010c1478c84d74b6fe70d388224710df09',
                                              'f1527fec7f84aa65818f46672a38a39617ae498e62ec741c96f2a16d6a9d9d2f',
                                              'f58ec063bab0102aa2f7d27e3b13d34b0e31e5a60af64d31776e23e1fb29e376',
                                              'f8126fb31bd1ecea2b84152f6674f5b437e561102e63d6f4ceb03100434d7772',
                                              'f927560f9717e9bfc3db93530abc7e8654444162c84677bdf547d4018c0d3670',
                                              'fae6ebd585ac5b0f558231478ad79fe19dd4c58faa94beb2cb1ebfa002dce89d',
                                              'fb2e3ba905df1ae445ec49e8177940a17407e04bd2059a64715740039fc92d4c',
                                              'fb4aedce8f218b1e0bca705da987cbec8072744ba1b1ffa055b0f34bb0565b68',
                                              'fb52cd526ff3768381eda8772955dd139630b1c92448d3ecedd800ecf14eec90',
                                              'fc93a2eb65773a02a1711593d82795c65528f9212f025e48f79e6f9eddfdfa54'],
                    'prior_holdout_200': ['0084f8c888def7ca8e5150edd21a7248b1feb0ba3ef9e1d0b3c988f46501f818',
                                          '02cfc775aafdb1fdd0382189d3cb813f2c890a1fb6e176d907fda86ab905a684',
                                          '031dba86bd4ffa1a1d51960b39da70c5ce50b3edf183b70dda9d3218f905187b',
                                          '05c79664ff9424c0b3c30f9ec2ab084687d490fd6d0f4948b2e51176f923acab',
                                          '05e6ba245ddd2e59a3e021823ef935ebf49f6a273dcc3f2679902f74fa31940b',
                                          '09143054231e9df0f34551bc20ea40775fed5b60e64f6f9c7292ec04e772ca59',
                                          '0999f9393820e8917812ddd25413ba91f6f152fbd1220acb379a7a5474810d24',
                                          '0a91dcc15c839e62816387cb8262b74a81f9143cc5baec723099d777c9469768',
                                          '0b7dffb63da06365e02956ee0c0520dc5ff8812e339270410d5b9364f1a9eb6b',
                                          '0df6c0ecf3cb3f92cfdd8940843309292f5f8fd9d60dec27cf56b196d43827ec',
                                          '134b0136fbc951cc55e75a08a9335a524835aecbed8acf91332a655fc1af8c97',
                                          '136d362b7f3df7ad18a07b0778c9478e0efee3157baabcae0afea5da412fc9d6',
                                          '15747090059d549da3c39e1f08c4928adca0378b0a00559922556cd3d0e64b45',
                                          '167b3d90ebbffc9837a5cee597b72b178c524652090642a58c0286621737dbb9',
                                          '191224e97d30f7ea88221396753fe3b179640d179f1248addbbd5e1ecafcdd75',
                                          '1a4b3599f15372469347e6a3ba262d663076626d90890c31417d50e36a856149',
                                          '1b902ece7dd95344dc5b01077ecce2550ab8b0535d5a0b885ae4d2db86964f3d',
                                          '1d65cbf7f778fc51185fd3d4a883e44d60ae3df2d0333b55e31d90664e9889fa',
                                          '1f2b852bde28f71a1e9334e223dc81d717addd893e7870a5295fc78d33df212b',
                                          '20aaedee6d4486a09ff247ad24d80a562f0be2a338e35bdf8f82740d1255b0cf',
                                          '20ff7ce26d56fdbb1f57239cc7f13d9c7187a2fe76d8d41fc4040cc67b793228',
                                          '21a1ca36770cac71ae26295310db18513925786dbf6fdc9c8e4ed8f99dd50154',
                                          '22790101b93cb7b9e5f4136c6ac0ad559271f5d3130e1e488e863ce1ad5ec16e',
                                          '228c1a35ffdf65cb9a368379ed028cbe98e0968d4a827522c6d7ccf09f574eb0',
                                          '22b050d12e7582bd0b6b6a39bb5cdfc3b5b23ce3b376b88d5871877ec6b4b5d0',
                                          '2420281d08d9fd14717298d71717302f35a9299601607e41b548b37011e86c9f',
                                          '242f9140b8650ed6065382d89b4df19bbc281fb8388f7e76b096daa17c6efe01',
                                          '24f79f0a5e3b82171a9702b32f698668549ec806f5df4eae2f2eec402caed843',
                                          '25f4bdb171cb66d609c3b41e5af6d9fe6c4b43a20478fa8817e4204ab33634e6',
                                          '260c6068d56fddfecdaa367fbf870801776df37aaa2d954a6a0ddab092769a0c',
                                          '263a7ec3ee646b32ca436cae9a4baeddeeeb3cff8b98c67aac39a8b099a6f7b9',
                                          '27fd2d7e1e17bef7a17b67a751f6e48dd6d58c870df93fe72935a67224a9cf8f',
                                          '2d90ad29da4cc1144a890a98c668f275336330d6a7b6ec3aebe8ea0b71b0c26c',
                                          '2ef5e210eadc3638aecb896f3c3e906c284585ff0a1d1476b94ad3333feaf314',
                                          '2fb3107b17ebb27324d93eaea065b85dc2f4538c305e8f1045df56fb583f7909',
                                          '31d7041f2573fe087dd00a9faa0f6c372fb958bea8b15a956af218977e933f4d',
                                          '31f704c359ea51a9f5f599c7f331d0656792ba4f3fd6417cd35bd970d19c608b',
                                          '321eabc5858dd5f95463c798407ba16a9a26459943269cdf0228ba025f20825f',
                                          '322bb75b4f5a05b0eba7ba80b796474b81e5f47951d18342e2b9223b137a2c46',
                                          '32955221a7888fae9aa1ba5a7bab8d6ff03e326ae972c3594cf4e49a1988334b',
                                          '34e30aa6fed35f75372c32706537f0c8053da8706567a99639742d36e23b297c',
                                          '35eff67e26cac12bc12c75cdaee75d76b3dddb27ea3b4a0eb259b300addf3633',
                                          '3684e33c3883dab27bbab61eaec02a028c332b10416c3668810ff80593273227',
                                          '36984c7cebec9af59bdd0ced39e22c29d5208c02346001388601a989adfbd098',
                                          '36df2fa4a13411acf35dbfae062a15ca58c58331e52a20019fffb17c3ebbe8a7',
                                          '3983a97cb60d76efcdbb1b037e8b03a770cba61d413d2dfc58fd1017207d8635',
                                          '3a52e5716470b7ee0a695cb522e1d38858bba011de37e93a3f31bac90874f3e3',
                                          '3b2c492954627e8bc310bc32893d825da555b631c34c5acaf633ca62e8d5a401',
                                          '3fca60957977d0994804c0158199755aff91032f2c9c91db55154255ba36eb8a',
                                          '3fdcbc48a62c2fcff67e436a6bdf7aae0a40d277694895c7e80144ba41a8ebe4',
                                          '41bb5562b8edca1028ff34db563dd1d43445aeed5b9e1d080bbe63b0c4e657e9',
                                          '445ecfb409bc55ea2edbf185a3301d54bb48978c552d66da81ab5f654ab3ba26',
                                          '455a8e3207deaab6af6caa2a85c4577aaebd1b494f6e5d2fb47ce1f7471a0347',
                                          '455fbff1af91c14371e3ae707558819cf97c55f50e08f1273ff17c3e8352bb55',
                                          '45fb36f975631ab002fef0cf6cbb878ecc1ee949eba5519c8dd53f70aec6cbdf',
                                          '47b2eb51a1f560d0be0833c6d040a97d523054c16c052d0c6d0d8500f7be5507',
                                          '481d4d1b796c64e8d9447a128154774a0e68d3168d06751c942704ef18e2b9e7',
                                          '48644c96046d9bfb85294d1df49ac65c615515b59c6ea7af721b2f0b2a88933a',
                                          '486b18e3bd5b423384fe51027e0cdb1384e827ae1045c8a5b33626ab98d4e25c',
                                          '4909caa91644f73e0c459679143722235a1aef1fc58621cce427861966c6a28d',
                                          '494177b028681bf52c409f064078f954db33534a72e96370c1a7bfa11429cc39',
                                          '49b38edf9a960824ab73270077ed5e9d52aea300ffe454336d35daf0dd90f812',
                                          '49d7aed962c2ed37ab2c75bc32a946b70f07ac1d8fe56a91c15e1d4822dadf34',
                                          '4b656b4f43b3de43565b38e1f5a013193ae8fbceb330505aa30f46e0c7d266aa',
                                          '4c0dc526b6db4afeace727eba01436f121fa40d20c366e48efbf98f46f50fda0',
                                          '4d135c5e1622a00434e847ceda99c2a093a2127712ad3ab51594e18616bdc798',
                                          '50c36b1dc6fe948923c1653b715c4806ebc86f061b2ab48f0e95c42eec544b94',
                                          '521642452f15e3c142bd07fbcd015e9cb141fd29f947b584fcff64f9987e07fc',
                                          '536ea5cfa9c5e967fc3134241720847df974adf0d33ba3a7607fa577f2fe7ad3',
                                          '54792b4b4041509792612a46a3a9e3c61cf7db058cfd3ce54df9bfba958658b1',
                                          '54bebe36f250ef417216594b1cef00bb5594aee3ea36f3df54c0f18e540a8ada',
                                          '55dcb7e2220bfa7fc6500252b8db0113bd2ef50b15d43a583f032d87825dd791',
                                          '5647ff3c18cadc44baa80528bf6aaa790f05a1b0d67d96ac4fd99112c5c42238',
                                          '583a382cc04cc2c8b0db2a06c108994f8b5d0f14162db5b9630da75f6764fbf7',
                                          '589ea0e14d751b5ecefc3ed53d7685f72fcbf1caa3ad6b616129d3797702235e',
                                          '58d4d35e699fe59f3d5e0d147fb2702d37c4c02a817c3d5c99d5fb54986c0504',
                                          '5a6a952153dd90401ec40eceda54ab9c37709676eaa21f8ad83503a7b24b1951',
                                          '5ae385ed224de1f7172cd56c30ab55188becbe0a9bb7dcbaa8b0a247dd73845c',
                                          '5d83c7a55d1684799a32354258b06da03937744a04ef843d47c73fae116151ca',
                                          '5e30e31cd36bfe003c71b8d0e7675c5dc82c4f70f637963a4540c744f8466c64',
                                          '5ebce6409062d0d09c88ecd8b8873a68e3bf252d4d43f896a8bb2ff06b767a80',
                                          '5f77be7c9b614908c147be524eb802b464eefb1fd8b08bca03f286e1842d057b',
                                          '600043039e4262ce95e81c82a970c426f132bc413e97c0c6b73afc992e08c52c',
                                          '602b7b7180d02c143d7d1364110a2e96113a6dfeb70d6c2300e114afa004f47a',
                                          '605195b4df6d5c9b8694a313337bd3b8a2ef4759c1813a638bf967c728f7f3d7',
                                          '61357eca6b6068fb3a47597031319d933d97893076c86d3c9ab346d913e1b600',
                                          '6197f56fd48512c651dc9d17532a2f08afb691c1b0902e02212680d496f73683',
                                          '630da7ce1bfde039240eb9b37b0f3546ac9d2a24eb724f33dd7cad0eacd0fab3',
                                          '648b75001ebf872b35bbee5d498a94ec5d35c0086b8e03981ecb45f3b1f0c257',
                                          '6494be55906ce443273f21d6829cb2edbf38a73933f0b5299c1d1e0c5a43f887',
                                          '66886428bcd081bb7db43e6220dcbdd0c02cd99015c8a102cdceaaf7b8bc2236',
                                          '66a658465049aa95d0105690b8d79d9eb5f556bb97aa58e4fb0aab3c264b094a',
                                          '671226cea6820bb36a38267e9eea5591ae1a7f1c351b57a59d8cb326af7701c5',
                                          '678ef7d21f94b80531d7e67eac5bc0c63f7387cfc8fd0f32f4aa3bf0a060e854',
                                          '67bd27e0e0202fc0f7c911f71233c63c2191df086d0c5fe99cec8cea9e1ae46f',
                                          '680a0673b02d0e1157d321f8597d09419f1d4ddcae44d515eb310b718621fdb5',
                                          '6878cbee5f711d1b6b67cba9f3258f126e269d62e795e6a43e7641e0683f903b',
                                          '68fea67ec0719386b07199fa625761fde0926e75a6b84fe1cc84455c2d03a5f9',
                                          '6b3b65b93269d46972a31f9a62d88575f7215c594fbb51c9701bfcd3f9781b28',
                                          '6c6b71c394797b942e87cd7037a729547e0baa1a83bf9bed818f70200f1d78b9',
                                          '6ce7d2bad658bb48845625e87136ad74422072322416f430039151fc498f2b43',
                                          '71284f9ad2bf5619ed99d70f89839a85092eb67bb45c9e8da52eebf958ee49e3',
                                          '7524c3ae8661a7c7d361b8149056d31acb09e9e31ec5fc2f5a4e62889552e3ef',
                                          '76cd5408e575953f7162c493f1c2b30773be0d4ac843bedeb60267391603d23b',
                                          '79f7031a4e43a6c097ac52e3cd3c2f2e0902c7f4992259092c855dc591712ea1',
                                          '7a2391d2a2a38dbef609e990a824b07332bff012adb280dce711c019b0636fa9',
                                          '7a935559ec33c860f6fc4eb3d7fb5f1488cefc60d5412f377562826633a7bd00',
                                          '7aefcb954cfe40eb083fb2f35e0b334ef45b43c2ffffc25c825a0b6a3df1d839',
                                          '7bcdd0f5faf5011f4a12267f0968b33edd53a8a6b496eb9896cf41e73155c450',
                                          '83e5a8ed400d291c17f48e8b432b8633aaa0de3c3b482808552a7928ac3c6a62',
                                          '855bbb0ca119676f9c9135fa70d90547f606e5ae78faa5b0d8a463097f4b9ce6',
                                          '85d25a5325a781e3e0d276e47a4cfbced0e428e63ffcda50f2654ab2be699c3d',
                                          '87c9fa95545393b913efee2667d28f4fe46e23a08315602be8a4f3052dd1a729',
                                          '8865258e4cb393d1cd57ea265936df75da7fc46aa91e6ba16bb21e0d9d16a6a4',
                                          '8a090e223e0ea8b594b339816bdb6205b4c46c401be9a5b937f72ae965652299',
                                          '8ba078f1f9220437c84953f3491b2fa59b7caf3bb6f4fff2861e67669ed00bc3',
                                          '8d47b475097e840bf895f5ecaf19c05df56d90d47b21cd858f090e1c677ff29e',
                                          '8e9d5716cdb8b34a4e88bbc0bfd89a7239b26dc28a5677cf1bc892ae513b9a85',
                                          '8fa574972578872d1b818a67c4dfa1fd585e66f8c605023fb0d81075abd81339',
                                          '900deae74c7de1a6635f99bc11278ddd09d98078b7df8dd0839103d51ab0ad97',
                                          '90f28dcd3b0705cde9bf71db1eb36d1471be78d7b5c64a61c77db2c9c006b5cf',
                                          '923016553382e157b61c598820d60ea4e1f86839523968396be794ac69231567',
                                          '96c1db9596316e1c7a4f0d7000782a35c749292993a6eda9c180eb0ba65ee73a',
                                          '9746e86839c203f87344ae97c67204004eb99c35a04074a2b3a76e1c62600798',
                                          '976a330772d02cedab2f0d183f7f793f93117ab62e2f6efd7c25eee1634a4265',
                                          '97a2b95067c1c51e4bfe50d4a19eaaa18758f3e5b92080b19eac2d429dfd4d51',
                                          '982bbfff8cdc53dd939dd43265f4c5c8787ba2dc52b5e6ec37ad757c69684554',
                                          '986f5977972ec1fc0879556c5cbe75e1b871c7f5aff819996003d6c8552d53bc',
                                          '9cbabc345948633cb689c7409dea95d1416fd82141725b327d701d2e5e2fdf4d',
                                          '9e13fc095ac630e20c2148357a9042d615fbd3c8d9a986a70dcf85d193f95825',
                                          'a08b6a901fca192459f24e64173f78247a955a404f6b54d706ff8c71a4bb0660',
                                          'a20efed7426e1a5e2ac2337fce00bcbdd8259d7b3444df7c7f14368f8579c69a',
                                          'a232c63616c5d104835a81c72c440af4a185f1929da16c473bd7c4f07be9bb12',
                                          'a3db4522f6a2f4ab697c3d108848d708113fc8ef5114ae564d5fb214dc304d5f',
                                          'a4faa72f0286e0f7e7ebc4d7c43cf6e553eb0414eb83ac5618d0c1b22a0037e0',
                                          'a52c26a9be9cd50a57d2516e4c157704a2a80ccd21337649d4b638870c9f09bb',
                                          'a54662435c1d76fa7e591b18593a38fb653a4f807158a3fcf72625a4e4679f7a',
                                          'a6914c89e4c89af82929f0126a26d08e2297f650a8ceec97e0cf000a9fb65d4a',
                                          'a7a207334af3649a356e6a9b2a05c05300eef762aa73627e9773d329704051f4',
                                          'a976e717049353a5df7197df29909477b5f31d37d9890a43f066280588e6acdb',
                                          'aa0b920e1fbabc76a620c87fbd5218f9de182d66e2e9a2f07fd3fdd3e85a25c4',
                                          'ac52365e2ff8a12563648794f6cda371e1815d371f05cb65ccd3ccbf49a4b2f9',
                                          'ae21b433cce10924ab235527fd783b53a603cf09cd3668ce414262a72696c815',
                                          'b4ebac3adf1ea2f7036580c7fc75985e13ca189a8d4c48aad11c0652679ebe17',
                                          'b4fe52fb4349d23096a5f30437abfdf6361e281115f68bb5bdcb8eba56e4843d',
                                          'b559c50b6ea318bd89bfa111e302cebfd84550633082ce995cc0e5fd3504c01e',
                                          'b5c5e3230703e8fb02da5d7a7f5283b68c489585fdcfe672c5dfe12ebb3ae2aa',
                                          'b630a2132166a89323db96d7791f8553ef92f42d09e771c252a88b0772851576',
                                          'b75ef8e8cd44faf109ddc477d87262566dbfb69287947f859ae0c6344903eafa',
                                          'b7694a0d8cc90a321d708c46804f1185bab74a1070aac880453dfe932e9e0ccb',
                                          'b924bbb286ab652117d29e05849c1e13e9542867030de7b050ff1fa2e444c25b',
                                          'b98e94478d3a1a2dc968b22b8514b183b7c73f69203366248338e98f11a5f8b1',
                                          'bbb4d6e5741265bc87dd1898ba80c49989f97854f4c489fff088cb624166fd54',
                                          'bc63ed066c67c28bb0d812fdfddb8eeb4989677760561b2c23398b8409141054',
                                          'bc7069da33611ec30ff0a67d51895a7136093053d6ada339bc5426bb331bd3e5',
                                          'bcff4ce05719705fbc30bf7af94ea1b2867134ba610f2b7b5b2cc6e97eb1b898',
                                          'bf3a56b486e0a3b54c875a4f7071b8c72026de84fba0370eee6d73f8b6083d0b',
                                          'bf72e50e77b84cccb3ee457ba4b78f0bd2350a15a6ecd7cafb6d4c2f3681e86e',
                                          'c038d74af6c548476bf889955b5fe53e678ef03ae1d3822901579d15c2a38b2f',
                                          'c24cb1b2a747e2b34fb0ee2d492898363a771dbaa738ce1dea97dff66823bb89',
                                          'c36125e5ff9bd3a281dfa03c22d11538231fc373668feafdbbd63c1d451cba58',
                                          'c6fa3622ed69437a80d1f23a0ba24d079d70cb17990dd72d39e35ca8eb714083',
                                          'c8ceaa52532b8a203d78a98f3075d7ea83ddac86d83c238d68730dc7ad2243ab',
                                          'c8edcc3f7184a8c812ec255b2fdce5450699632b4afc44ffa3632eaba984bde1',
                                          'cc359600371abde4485f388b87616f589f5228e0cca128ad427a32c13189bb6b',
                                          'cc4da25beebcee8043a745438aca9d5cb7db726eedb373206f0c44b2ae2de1e6',
                                          'cd66b977fb9dad411fbf3b2830e3ad4e73f89b7c3aabc481d8c76ee85d964e2a',
                                          'ceae795083b4bd31c52fc6c6915c6f71f8840b57dd285ede753d21757bd84211',
                                          'cf0311467347d93d724f8a782c72a9907ca9af718d13075661e471540e30194b',
                                          'cf513ce58259ab0ba6d6d363410cb90a78a40a33dc0086200a36fa3c4dc9324d',
                                          'd27372a7c7e6ca823331bedee20131e5cb477e6ed886217bb00bbb690e6caa62',
                                          'd2bafcbead55ac01b0678608926f787942d8c356fdae79b2a3930cff44890b70',
                                          'd7c4e92cd959583908e7f3b211572c3caa69be2bfdf770ecad9dcc4f7b2ed71f',
                                          'd8f11414f5edbc58a8375e336d8d54693a1a72ce99b4752494d29ae0971de174',
                                          'd9e16bba25112e60241dc6610d0ecf9a5cbad1bf36cb818059e9c0cf53a280bb',
                                          'da036e121151a78ab1fe4a4dca346f897200d65598f8370ac943ddb8805ec0c1',
                                          'dc2dfc2cd6c6ad7a84d41976d2785c76558136266719687a319cda903c5fb043',
                                          'dcfaf7d2bb65afd16465bbffdd36a52c415e9dca45dcb8e0cbe0ba7d640ccb33',
                                          'dde736197952b12f8c239f70f050a129e2afc81d37fa389547f3470dd790bd2e',
                                          'e14b2a14b3bbac0f69e14e44c4d090c53b2ef144db2c101bcc0b270c4e2c6d14',
                                          'e415eee54ef7c2e9c9a7cc535aee24aac6693b5434dd5f06b0622341561e9457',
                                          'e77a6e19909d4ec2bbfce3357fd7518db1042b0d5fc947db9e5275db886b545c',
                                          'e8272dfb2f764870274fd990d606a6403450a302ee0206e224afcc5102b9a003',
                                          'ea1680874df1971470cb0a4155aa99d92d501c8eaf27010f2ca17c2904fd2045',
                                          'ec0c2972c0b16806d604aadc9317cbadb056f7d6c5045396629d20783c1392c5',
                                          'ee4cec3bda0a56d6226781d4402b8f91886d09f18e107e3c893b6a6e499e4c69',
                                          'efa84ef539854106d893867c732a9253f0dd32b89e3170363ee0c9977d7afd0b',
                                          'f06533715e8df31c80ea9db678f2da295132190f46f7c80eaf288b5fe9bbde2e',
                                          'f1c35a4bd9fb0eb3f9bb3e97d80846c5432561acb6d38688311d597f7d6c7c4b',
                                          'f239b953914ed24b32b8b0986cdb15de521f70fd05bdd13964edbb30ae30bc3e',
                                          'f51b99c78324c461e23f3c78216b5bf26e6467fe4aba1a350a5fe17494e9c479',
                                          'f51d6d9efded51863601fa04eadd76433b8cc96e9328798d1045365937e53c21',
                                          'f59b75a7b0b397a618686b749b51232ba37dadfa465451c66c413177a2ac29de',
                                          'f78440d114a63869ee60b76aa3fd52069d74fcc5556dae9f7e362266a8cf029a',
                                          'f897e5a9a258dcfbe1874c2d011be02e0cdf9773fc8ace2c509ffc5b1d56b732',
                                          'fa0521049456a3323e3e3c5cd39274a4e6e3a9622f3bf4e9131f3245c58b61ed',
                                          'fa7da05a1023ec96754bcb456a7bb4893f96cc7bb3f62c08c8c84a9d0c88881c',
                                          'fb48aec8cacb40a3851e8157d5ee9a8110888306f73c931290b897ba8607a55f',
                                          'fd91f378d07178bc4ab2ffdae4702a328f8d3e026d9d7f802109abf05df933b5',
                                          'ffeaf0b81a670d518edcd623dde67ed4a34d40754377a0bb6b3c56a4b382d664'],
                    'wire_confirmation_100': ['2783792192d71135c704a7d48f4335e8c3b43673df6f77673edb3561afec80e7',
                                              '14b62856e863f705d95d59c8fc30cab9e9a2a3e07be3c5ea248fe841e0088444',
                                              '929fdc642069bea33341b3c2460e3ceda1622a031bb9b02a3f1570bb1a5a6c1c',
                                              '11cd45d44877e9aaa9c41984f0d1651c7321b5d7049ee0d2c21cb7d3582729cb',
                                              'a488e5780fb8bcd2676617c5fe2a78df84cbed7fe997c3ae13a81baf0b29bb65',
                                              '49551daa261549f2f64482ccfb65b32fbb79e139be7a7f19cdcd12bb579df5d8',
                                              'f82a254bd1e8a7f922feb52bc2ba3f8ced6ee800e5bfec15cd1e9f7ce24dd9be',
                                              '53ae5f55519bab8de2d01e6979fbe0db650be9cfc718bca921091d316cb0f941',
                                              '694a9d3ef39ee9a9a8c5e37d3ce3721f435e8265547752b713bb4e03fd23eb65',
                                              '657965471d6d3d059a76badf497c3d0ee3613564b9698c4326b25b7e8536f4ac',
                                              '748a12289c1e5c39d6ddf297aad473c94c5471d52bfcc26b925b86431e4d9f50',
                                              '4e6784537fbf5e264f16ddd041cecc2bb9bd1a14af2717010d0823015b6e0a6f',
                                              '575549bba4ada4a16aa82123728211408b480b8fde752bd071af9eec053eb75e',
                                              'aa709feaa1e3d2960c85e36455aba82e5cf383cdd997f83a81e0d79c8fc096b6',
                                              '697c90155f4b75a5236cb78246c65d7a39a4e7fc13080dbc0d44919b99551857',
                                              '6bee3eb2eb0c7ffacfb67b4d18df51f272c59285c3866e9c14eaf518a7dd8040',
                                              '087a8df98839fd9ccf3cd63df3817899c260d6429424e093386cef1c4c65cb3a',
                                              '5aa2d1269adab9c2c36d2787c1f357f5d8161b8ed87a0698847ccce315b09664',
                                              'ab6453bfbb3fff8ba6aac2da2cfcc891d733bec706d8eff493fb4879dc4d9131',
                                              '031288bf4dc636a13973e5523dc48f03df93d0d8ae0da94a4101cb226bf23c42',
                                              'b391002be2d84e14954792a154edf59b87e7f0e3ead76d2eecfbe775f833d371',
                                              '9505898d4d8813394c9b6cb7c0b240b5ef0bf45436a4ee83c89743b990031805',
                                              '3c68b109aa59c24b0bee0a32a25f67e6764731d6db59e604a29401e5f67076ce',
                                              'ec95b54ef913d7937e0137da18fe873a8fdbc145e4dfbe3b9834859600f91e40',
                                              '040b69056ff6a04a13d1d6bb91c0c479efe2ac4f8fcb60efb0a83975496e750c',
                                              'bac1d1dd9d14912cd89ae645f71a16f899da27a8572a9af755efeacd323bd200',
                                              'a8ee1c3079e2cd9b194cd1070856e202a5e86e4070f2190566098b8645cb7bf3',
                                              'b88923957715de2135cce6e9b26478c3c43940bb296d3cecdf2b95902d837ff7',
                                              'edfbe86f13ceffa5042815ba102513ba8d19f4ad9e17207503abd7e831de69d6',
                                              '093a0130893752bf4785ac38e567c9d799a365483a376c158f3a2561fe6170c6',
                                              '37604d0b16ed45d50ad73e92f69292b29ab552895940e6344fd0c27848841bde',
                                              '71c504fc93d8b39a24c79f5ca6c1792f38dd34145cc075bd7d2a3c1bc6196ad2',
                                              'd366fab0cd8aa3ee39534b74c5c03be31ec6b385f26bfbe7a75fe28bea870d3f',
                                              '07b5f1ea62e221c7dd148a47a9c62d11ac460bef909ca3a737b5c395221fb146',
                                              '358bc297fcb7bc33ac812029f1b6917c9e25618300ed46b7b93c68eab5ac2b34',
                                              '8462f8dba78b9dd70cb78df9732b695992863d00f429897644f6af1d8228de27',
                                              '71a53233501116dec0b7a2306a681dcaf83343ad24f418a056fb594006cfa173',
                                              'c7a79fc1f02af283f8c3d41ccb94084ad4634a247694fe8a333a596ebe5ba49b',
                                              'efc76c8399762286029c3f42f33dccb6e4d61bfb25a558ed7c091572a2d63265',
                                              'be71cabede2b75f5f4225fb6f102eb667f889d4c5501d4b268155e4b46b05b13',
                                              'fe76d9c7b46f5c29f6db376129e54ce177326809ef784ed3339be7309603bdfa',
                                              'e1566d858f9a68cdaf9896631e6d6ab106ff599b5f8db532138af0b42825b7f2',
                                              'defd624650aff73657630419ac2b51de85a2234d6ef2f9e479f38180346220fe',
                                              'beb0bbe1093c7f232cbe1f1cb8da467265c671cee47f51f114697706b5f9261f',
                                              '284fab2961066535c65cb962f88aa86b463048395b305e1793c4849dd5603156',
                                              '840db84c75cbeca1f2147634e8f410d96a758627824cf5a9ea4570135cd6d1e4',
                                              '036bca3d81f26090a944f59a858a923d5975c5c5176d5e010ab25c5a7715d3f0',
                                              'fe8a75ed93a09e5384ceb4783f6baac43949d1a9f2b543a35a7681da6d053480',
                                              'cc86679651f5593decafab9d26150a955a7fdb402fbcf5bd91d31f0af1c9368b',
                                              'b6dae4a3da1577c54fe51fd607f131fd05b273457b9499ae5441a8d2ff800f89',
                                              'cc4ddb7541bd48fde370d6c3f7497ac0058ad4aebe7f4c9254ca64b4e35496e9',
                                              'acaac93c49422cca18b7bd47f6d02f7a4086a7925bba9dad3895b205d0a0ae5c',
                                              '48097204f0e1e608f4050b14ee5ed523e1fbd7f42e64144f3733b27634b2928c',
                                              '669b967662ebb763669d3d3d1486b36b269a8bf9b8805dc21481138eb4ec5505',
                                              'e97d01c10a20c598100f1217c569803de9c62a2b64a4e7a17e05a474aaacd97d',
                                              '88aafa92a657bbf38e233d76a6bbbf312c43dfc68f8db36b6fab1a9516cd9c42',
                                              '02f2301e2a66ba274c929367147510d2429dab0907f78acb13ced9e28ef2f1bb',
                                              '49fef414fb763998c606ba88cb83cb653a769244fcf0d4bd2026076045c14ecb',
                                              '181bd43cadd95ab231952fc078cacfe2384fe1125f877b4370147dd2685813df',
                                              '83ce0bb096a862358a45d465ecddeebc1249cf4a97fa29072123cc94f9bcc59a',
                                              'e2fa79999a98f5b3cabb16ebb588174ddc4d05bf08863394efa723b46dca8486',
                                              'f004393b40719704ee099a14cdae647900ab407b220946d5273feeb301e5d08b',
                                              '0318784a8740a6f2c4539e77cdcc938168574309e179b3a3ea56c77336802023',
                                              '6ba25a25b0fc8f2e0b8e61004ff7d594801a845e78ea3733024d85ee8bc9fb41',
                                              'bba85602ac4e5649950c167c080f4fbf06e8fe4faac7caa1a3c1aaa1f27b8dbc',
                                              '2da7cc2b46d1630701efddaafeb0d17618649382f2e2adc6ceca325d208dab98',
                                              '68d030b51a5c51f42c8914531a75672882ac1efe2628123eb262086f7b0445fd',
                                              'a124ded59b97ffe042097a735b80b1a92fc568753bcb1d09226ae34a306234ed',
                                              '6e21f864e12c8895ab11f41e37cd2da110e47f550373ac99c17cf75045cbecd3',
                                              '8bd404d2fc31b447ce91a7fe116316806b05e729f6150b04243d221c5a136588',
                                              'fb200baa05919dbed625ba042f147faf717b52701211b491f9204ff696994feb',
                                              '4bf6c554fa4928d8da6dc7a784cb7e6f275ebec4de98b15ddf02fa5d0a5dfb2d',
                                              'e37c886c5c2caa7041bca9e1d9608d3c04e509bf7d48e4ff4f9e73c4687e5f5c',
                                              'aea678149050e99ba73153691b0dbdf0710098c8d944aefc0b72b2b32700b698',
                                              '5cb541536ef41d7d011429679a0628423e27df7b4c2f1d921506333e354bb463',
                                              '21977530aa79b9e0e499784b1827e6b75877f1f30b3debd2419377a9fccbeefb',
                                              '71372c599c85aafa4e9714e33f875aef48f00da63640b1bc387c77ce30b09a9a',
                                              'cf3e1804573c00dc978084f5d4ca9aaa56879a280fe2631f3c853c074bdf65cf',
                                              '7e65c5ab2140d1a28103fba10c2bc53e9da2cf7b43afb0a61e0c651992bd5e25',
                                              'c7e7d5b4bd057a2d920a23ed6d9afb27e658179936767a73cdf72eae83798ce0',
                                              'ff29150152186fcee2398603ada81055195bdbb1721361d4074d2581ffcee0eb',
                                              'cb403476bfc828da561f22d80fa6c10956012eb16e69e0c9d1f076737a19a589',
                                              'f8d639fb05aede8c77161e4a506de726c75b024f39c5a54ca16751c72acb16a8',
                                              '2d28a2897efa0b74c752b4287e2facee710509c8e840aa849620a1a2b3b37104',
                                              'eeeeb302b91f620621b0af659c763796161ad1af8c389cdd0cee8b79a0b34c3b',
                                              '63f7c384b9e293cb12ce7740866bdcaae6c710930fbd81a521e7b9512ae30717',
                                              '80ac13bb6a2f846365ca4d0359de753ba0946695565c8e980cd675f0c07d89fd',
                                              '8faadf42f61d007e7fd5bf2305426a5499c94ab3ec5b9ddd08c23e8c7baff408',
                                              '7c5c847563d9e772388777c232d50716a3685e50a702138f9244def7a90635eb',
                                              'a815408dd658c55f3e345691960afdd13ee58707815666d3fbd8f1e32f3d5cff',
                                              'cc3bd6fdcee10168ec426e8680b0f458b7fe3fac71fbefe3c6276b32afe42073',
                                              'a294a4e5c0be525728ed82c78d07e4433f4fbfb03f01a520154bc4025da14be6',
                                              'cc54a9a090a2fded909e70c0526911dd47c0cb60458175dc36f6b206da4ca06c',
                                              '2ce23fdc68c43b25b9169ab09fa124885e5e4e0dc2c90cf875032a5950b5d8e8',
                                              'cfddfb7feb0220159a6a026254eb4049f1eaee8563532998e27a4b41280fdded',
                                              '26c32b6b61f258765c735ccaa8f3377fd4aeffd8ab6c4d9eb1ba72ac34b7b73d',
                                              '711429f02f197c6aa9d1e405152cd99a5c2bc2942d231ddda6fdc11239643ae7',
                                              'a2d63e41d1c12c1cf46f2bd3aa5efde29710e69a512122e7c9f7c25ddebd4cd8',
                                              '4e77140028023d0c72512da6344a5a434a7123659aca077267d9f361fcd0d253',
                                              'b038bb40e413a4e6fc00eafda15c191b31b1f2c125b59957ffd89fad51e184c9']},
 'excluded_unique_images': 500,
 'expected_total_images': 1000,
 'geographic_or_near_duplicate_independence_established': False,
 'known_prior_membership_scope': 'Archived compact evaluation, channel/BPG/final checks, v7-v25 '
                                 'development subsets and v14 confirmation; file identities only',
 'preprocessing': 'center-square/PIL-bicubic256/reducing_gap1/RGB[-1,1]',
 'prior_membership_sources': [{'file': 'config/holdout_manifest.json',
                               'roles': ['prior_development_200', 'prior_holdout_200'],
                               'sha256': '02a3044407b6296e0e8611843d11c6bc596ed5008322155052cab52340932752'},
                              {'file': 'config/wire_label_confirmation_lock_v14.json',
                               'roles': ['wire_confirmation_100'],
                               'sha256': 'cba8fb9fd75df0a87cee28cc68701a062e2998ece30f87e38a33d5931204f1de'}],
 'selected_file_sha256': ['fb66870b3d4622b4393f5523bd9ee8463da677a02a08d4c3f46169d4c509abcb',
                          '878c11cfb20e4561c89e6ce8d973d17d7b02857f0e3481b1cff39bd954579a11',
                          'd974955e725729cb2fd395cce764af1f957747dca7c515c416c6c450c63bd758',
                          '7a66bf7c0560ddc5d003b0869c6b7e8ef4bb06029a3cb0c258820a2390522a41',
                          '94d6437634b1311bec99174fad65520de1a385543eed9feb7dd67016f18d035a',
                          'f4810de1c010ff776db227074cf01b562d5adc6a25e68cf35ae4e9a7defad789',
                          '9d25f09c5077033f0c33abcef3b585c5c54d25c86251ffaeb36ca377a7d9350f',
                          '820006f4ed92e36119f0469ea3311e2739f34c88500eb583a9c37737c9f4fb45',
                          'd5d3911f303bad74e013b8b87950f8b3d9717d4bed508ed5bd8a21d9dbbf9b5b',
                          '017f1ca9cf29c2ba58943ff61f089f2fbde10ca6b7327ad05283f4b2ab12f20a',
                          'c492444dc30340239f974cf8cc238b4c56e20c53c7d2e74e82a2440f5ddf629f',
                          'fe39d216b6307c6809c5d4acea9d4f5e40e79778ec05e78022f3c33dbcc6c4ff',
                          '0cb426ae32adfdc3288ea3c4a9a10b2c123ab3d4664a32b02f052e86f1bbc4c8',
                          '824072b3ce8c6abef26bbb045d3c9abca759b0a8b1e8e8cdcaeb64a4e403ef34',
                          'c4e80e2436893859aeaa571736d0a5ba894d94ff08948128d8f0c5e7e2fa7164',
                          '5fd5ea59567a4d929551fc89ff66d786cb34673d350f52207d7d0e0ce5ea4e2f',
                          '1422861477e119b080e3e93293b9f45cd0ce20067e9beceb1bac42ddb083428b',
                          '12383bd875b6a9639e103cfbe7e31d835157db1d1dd971bedf34c70546031a42',
                          '3fc5066e006c9978ee0fe225c223a63ac4958acdaefba6d763cd6d3a7284f3c7',
                          '8f0f4324cced8cdf87b4aceff242c7ae78bdaca735f1f3ae2281eb6080caa1d9',
                          'b26a3ead9dfa4626bf6afeafb5ba7e23d17f0979408f61b12023b388d132bece',
                          '9db4679cbd14bac82a2270d094c18046b9f7dabd07802b493e7c25a220aead90',
                          'b62b27361add7e88063dd3d41046eac188e6b29074f4e426ff4f0938a6090615',
                          'fb162a0c965408d952024767012c39958bb7fb6a0fe555ce107f7141c1f906f5',
                          '98aa4599a2a4076d7aa41d7659e257d6a62bbff69cce3464b0ebae45af5322da',
                          '2b19544397470829dab6f7a402fe4196b2654d8140465715e2a7e23a1b225a37',
                          '13fce8aedce5fab9133488366c17473423687df9ccd5682f038ca815b008c817',
                          '99c3b84ddf987d4ca7d4034bc82b45cdc386bb7b59b757ff8d136d71ec118509',
                          '927cc8e6a81091f01c534c53c04dac2ea17b398ff4b99d7726c996e21024d3fb',
                          '8ce56387c6c068ddbd1694f74c3c9032a83892f615f7fa94205e70246c4317c4',
                          '41bd481208205366912cd9a36cfa1cc5c0407552b88c38d846b579692f7af208',
                          '014aeb0ef7d76df016065061c6c381bcbbbd6fa8f4e287cc6fb400d6aa0cce4d',
                          '37a34ab5c0eec0ca861e91e4a042b51fea62c0b2dfc0f3081719cfea57be04cb',
                          'e1e99ea08226b72a67793ca36c187695ad647d021677edc3f27166384df63c8e',
                          'b484b13816596cd872e0d5125e0c8728cb428be2a1127446f5ae7823da26cab9',
                          '7f85d9d3836a7943bf98317d347614821d15c01a9030113f7de5e60439c0dc89',
                          'b5fd1f7d3fee17d1659fe5f93380db2a71514aeeef88d874add9725d39c7923e',
                          'c8a7e9d6f07edeb91b951762b6a785d021fb412383de193dde325e2cbd00aa6d',
                          '71bcdc5e7cf534d0de602e8052147b1a88ffc40b622904caca87102bcf8be2b0',
                          '2840069ca725acd3e097f15bd532375ddef326880f3683f2054f118e70ef98a4',
                          'c21e4d7fa110e18f14f911efd7181fa2363dba1032adcad2492796d7d988995e',
                          '6005ccb5612b23aba5df83fb1481e96647f15c6e34d11eb26225ee342ecca069',
                          '16877e5a83f8b18b70b3a9e5c01ba954cbac334efad1b5f5a6c49ff69777e96c',
                          'cebfc6fb7a013acce43261abd4f0b07a3d74444c037e474ebb4cd7a53d4cfaff',
                          '4c9cdbe77c5df3682c148b2f6d82f53d8273a815a8ee173d5ca7967b88706bc1',
                          '7e51e3fc640024c1a77502b6b8564be1d4305b67590b4f60f0f135963787a155',
                          'cde315e713ded0863544545a4f50bcae520ee4af5aa89bd0297d654ed0f8d68b',
                          '8d3be63c4da550a3913178f9b2de4f073b8391bab78aac1e5a02a8c656749814',
                          '9ea81f44498d04af4cd2b84af078e2ca6aa42bc2a0e6939d229ce5a96d21f49c',
                          '2909154fe14311910e85e24c5abfe8fd9523d819071f3f6556f2d910ca1b0f90',
                          '526845f850b649ead4487cc741e0949e96fd81ba36f4da9810e93caff4e9aceb',
                          'a11786337ff4f59ec22cb8b9292df7220c20debc689ec52ff0328b218d041939',
                          '21bff1f8c093cc63fead0b7638ffc9940ae579d8362b986fe77cf6835abf8c67',
                          'b038b4c09b549bf2059829904902303c86f5d13f37edbdb4d5f0b27111ef9668',
                          '0b3376578021a541194d1f0c26c97e16b508d706e0fdc0e53b2ef7bb989aaf5f',
                          'a090bebaa5187803f002adb0aa189ee9ce5c6c434a95b80f8b62d4eb072224fa',
                          'b660783e297bf955a11c1927e3ae18ecd28e348136937d220a2ae2cc8773f4fb',
                          '0e408be1135a55441fb76763bd9a2aa751b847f4a0f64a9959d8a7e3700eacc7',
                          '0f328a45a3cf2bfb5a4cf60e13ef3dd6324a6af2b4b15a03e68dd864136c7984',
                          '363c6a14c10015b31f753994b513daa3dde497fe485b75fb90ab2695f49d873f',
                          '058d5c353be680ab65f033a8962772325d98cd96eafb59f6e70c077576f91ed4',
                          'd23cbd926904691315e39a5d935b7dc3b256764bae2547784b3c4d631ee11947',
                          '2c353cb2f84bdc3b46be2025a09349b66886bd7a6e21764dafabf2ef73cabb36',
                          'c9bdb1f1f6498132c84cf3ed98fd05678472043d35834aa0d144337a8dda4f77',
                          'eaa9d7a487710aeb6548e16c2e0dd3efc7daa03231ffa735fca26ae91d3bd158',
                          'ca973d22482a4c8a7abeb6b7cdffae5ab02d2b25e7ef0ba9cb309bbe923b4059',
                          '47551a2489eb09ad2da883c6adec5e4fc2f6f12dc47d376d17f6dc7cc6fef285',
                          'b73f28d24703239c6af8bb16b80c7afba5ae6a6ef35f719799fd4a56774a9a3e',
                          'd72c616c02ed0095d22bcf3eb891edb90a1949ecd5f9691d35b6cb7ca8e194f5',
                          '217c563dc3fddfb462f4abbb7ccd872ded097413301f0fb166c180ffaf10680c',
                          '7fefa921a6f14bca5cc222a3036cdeedb95a1b7058f18752969289ec2648f4ed',
                          'ef0cad2e9d77116b8fd0882e5eb722827358172564f4c8982a214a692d26eaae',
                          'cdd46c341d141dc0a9156546726bdfe90581eb5ad0ac6d2f5b5b1a04ca3dbf67',
                          '4bdfe57004a8cd7dae812253b057800ef3f265e0ea92ece98514856c15bdefaf',
                          '609e504207c753ddd5e1cd970d651576b96a41ede0fc6366711a76a8bfb7fb82',
                          'd33826f37c63c459ca40b9e5c50d65ae298a46472923135a10cd5397e6a4dbab',
                          '4755d79b5742075aafe9136853b3a45c18e2c74de1a2e037d6a292604c17b154',
                          'd8221b6593140465a6c000b21a9103811d97242f23349b130bc637065daa625b',
                          'b545a7301eeda10758632efc66ff54cd5cadb987cdda65878c98a4d40d40d382',
                          'dbd9516dd9f7342bbe1a2508066e59241b26149fa63d2ef2ddf6679d0f889c27',
                          'c64050d262fb76a726894c61f9e236eb0f210ff89afb4a466f6957ed6aba8311',
                          '74f2e2b84e80926b1d1cfa12820a74b4a2734f14779890559f8966c5a9d2f2a0',
                          '2a3a2a6b2a5b321c19baf4de1a0431816c9e1299e1a67aa3bf1f2fccb447a6ad',
                          'eba55db0c939c9dfc0c4b393858f5f871c1531d3189731257c22a23c0037179d',
                          '17d71d00032256f7aa022b26da0cb06267b554a692893213b315feacc39d6ac5',
                          '2fe7c85b683d6b09700bfa1d1e2ab63e696e53621b1255dbffbb41789ad50a18',
                          'cc03caae1aa3dd14afadaff694dc1f600845a8f230f464bf4e36f81d1eb3f0d7',
                          '501a19a8f83296edd3e291d899153d4af2b8d4307219ac31a257dd6e70c9eefb',
                          '210cdd5b6d4be8d597547d69766af2e6e5d5087a3a9e4bfcaaaafc116451152b',
                          '3af527a45ed23d571fb6a8e24ab8eb5699621194310f012da20118a900e7a7f7',
                          'b15d370de3c38dee39f1c718d58c1aad17538d5cfc41aec76cd57dd74762f3af',
                          '0b9cdd6af7411c4095d9840e3ee8ed559daa6dae5beebc93c16cc51d15c63506',
                          '3238bd3fa558879d98a3512ba61046f2f34272ae9319d7ca12a2fca67de1aedd',
                          '697212be1d00f0ebd4d251fd66e9aca0b2a2e56db4db8b38de5e066deddfedc5',
                          '8020a0d1ff778a8e8ea8385188891bad0841a590bbe445d2176dc2616d44748c',
                          '0ba797f0b2beb834faf4dc70cd0cf0d872817f03b8f49b14bf536e32eeacf785',
                          '79294e97c363afb031381a69c8f86c1742c842a55ec108730efb354078f53122',
                          'a1f8788a02599524a177b2452e712c201246453fa44f49ca2697150860566234',
                          'be0a617f1090affc229d0c10ef0c46cd983a73f5d42f3cdbf166e95313d8b2ae',
                          'a6348e3bcd5456b8b69cd5e09b133653a211fe0634aacb3634a5f815d270e589',
                          '5e525649f0e748cb0c6e6f62d6d338e884cfc1baef6d5df5d228b542df2d43a7',
                          '9847193e99e3e4a8d5df9fb8dfdbb7db903630ffbf651fa6662b986f1ba4a16f',
                          '04eefd04b4ea48004fa383bcbf35aa751dedaa95efab605c77674021bc794f71',
                          '8d0575e83fe369ccd672f27ad889a9dfdc4f517dfb0ece25e4c291d5724732b9',
                          '43d5a5f4f30c04de152b912b9e50128ec2ff837ec4c0b340aee15e025a1eb905',
                          'cd0e8ec20009a6ea5d0ad3c9f4fb13d8c7f63d041210414ced946a198f077761',
                          '3b96f65812c35e52be5d10261363d221385700c1cbd9e1cae02bca2eb358b771',
                          '7adfb0914d1916012dab439ecf342839835f41a68eaaf1c4517afa9d7847b626',
                          '3c892fa76bb770337f86ac817941f6623c04663e4e7bace7f0b8c3ced70f5d66',
                          '3030f8580b36db8eecb0fbbc889a0ea673a3acad45836dd991f98e20e9e11517',
                          '66df0f9e8befe6c405699906669c03bfc1c28d3a7e74a1fc5af1431c6ab4bdaa',
                          'f0318103d62cdc10f51419aa5fd71ce6d7ebac8b283a78f09a7e11b13072be12',
                          'c7e035122c3cd95105d57dab4e8dc8a784b451f610e7e27ae868167dc57b33f5',
                          '422945d0c129ed7803b523c91cde540f976da7618f4e2bc59aec187d0989dc53',
                          'a539eb0160f433664ffb9ac52dc0f9ff61b21518dce0fe4d5d4b20c495a8f586',
                          'ffcf3eee073fa2bc0daa9316c1f848e79a65ec27e8f63b29f7b7f2d1e80b7898',
                          '3c9c672675554754ee7385dfb80303f8fcf0915e80aafe56495bd3505ecfbe5c',
                          '84f7da83fd4a19d839b38b02fe9cfc5e4e7a232e8799794b3013cf0bde1a7066',
                          '9fa4dad55e35af304fdde434747cb718a18ceb3617257d1d936f0d09c4df5f41',
                          'dabdb3ad5aab1e2c7606e065f9ec0b5ccd222634563149176046ec6a8ea7faa4',
                          'f9f9901487f4c64e36c4441e00362a9f012a945b6db5cfe2030fe800215c0bb7',
                          '675954a180056d28e080a78fcbf0efb49e619e17dcf56f22369051a2331470b3',
                          '5144ab1e87c86a75492c7a6495c1bfef5778980a3803ca2caf00f3d8fc410f28',
                          'ca81fc43af7c64f7154c6ac29db881a181d61da42cf82d2617f6d70204af512d',
                          'dd57973bed8793da8745cbbda0c478556fd132b0e5793b8ddc0d8676da89ca5c',
                          '39e989e179963e8cf061abc0bd32c30579dc0d77eb7ec363356284915c5d294e',
                          '5b6b164ca7c58b4ad9be13b016aa2b66a64cfb5f27ec24498a73d2927b869aa5',
                          '6f7dd32e5457545710a9da309ec1a3813c9c2d8ad955686d57c8d314cf296bf9',
                          '0b34c3f737905d058400a3b299b7853c27e4cbf302bfb4f2fcaa09047cadfac8',
                          'c3eaf509b339c27bc416b1306da0a573458e24b00cb03c81174719495f8ee0a6',
                          'e8f57178c9dda1ad3b181f6486f55c5564a20f1e86a4d56655537522cd01f21b',
                          'db13a623339e7eb113b514e526b8dd4c48ea0bf4fe5e6b1ab67e46e0ae8755dd',
                          'e84b504de234cce8d89145cad2e9450606332704ea3b1b477ef68835cfb67937',
                          'cb19a5529faa2463a7bbe73270eb7d2a80853cfa4bf13b3bf1b83c0f67e6f57f',
                          '44a5ccab459559e6c665ce80951d0e5a679d5c67950682ec8c4519836553b0df',
                          '53beecbd7f9a7eee5aef8a60144d417cf8ab350c8bf9399cc13bb61733b6cc65',
                          '64585202cc972d21ab8dd0f603b76f9833f981f29b98908bc36a3bdf5dd77d51',
                          '45de925223f0a62875e752bb4c4bdaddb13b60c7fdc7c2846c83130bfef27eb1',
                          '4a6a2e6fc197b3f61e187b6c420399f456bd99d9ec4f7e678b4098792310ca0d',
                          '1caaa04d1678b772bf5bdeee0982880657cdc8741609a1dd7beb4a67d0212e55',
                          '1bee85ca8055cdc8125538e865caa59bcece6aa90a149758d4bdca3ae855835e',
                          '6f49f0f05d2df15e70a03ca6caf35d98655beb6d7fdcf036160958749f25abc4',
                          'd2f6845706eb555b10713575e7b852454bc3b5620e869a4bf7a198ce56879777',
                          '3327f2a690c915ced7bcca0d4f81597dc24daa4a7a2c454826c46fb70695ab39',
                          '6fe56567be2ca329e75234c985ee95430839817b2030c9918025dc917983f7ef',
                          '3f103a95be2a938f16ec55e5a2a5fec46f3eb34b76fd10536e5fdefbe99e7813',
                          '105fcab86f503a97dfef2f8f9a3a2cca2641306714df060f553ada3acd96c8dd',
                          '12535b48c89b2d7a339e1fa11c4edb84ec232f5d565200a06e5dd42413a70dbd',
                          '682d8e92bc195d1b0ac55206279b85330f612a07de0cea46913ef1dcb1a0a1fd',
                          'b313fe9a426189bdbf0fdf969a5f6f2a127a0ab756c8eb4b76b4a1b944d3c29b',
                          '14d704549005045ccdef7464bad5df4f81f8e3313d92fafad26d551d1b2e02e2',
                          '72e4dcb218e48f09df4eab58412389c259a85fe6fc1b649f0b6f4d8c92faad81',
                          'e5f04d86078362171936c15e342bf08ca88df1353a48502a6f1e7acf131fbb83',
                          'bd3887ce77f616184eda2fd55c1363c99768a15df3ecc1fb9a1352d0140c3da0',
                          '11305e2605ce0d223099b48600bc36d20a6706314f69c859b893bfdc25d05431',
                          'fb4dda03341891307ff3f1f0e8f5cb73ab6c9b0a44d16e77dd4ea273ae56f641',
                          '9b6c5d785c00afe6d77ca81ad052d8d55bf034dbd734093d4b72aa2d3e4d2afa',
                          '618ad1b14bdfeb156d401375001b811948ffd21943d644b8b4fbac28801eb12f',
                          '89f8d84b36432628b137c6455f17c425d49bc5b8b274fedc2103b668f7be7a04',
                          'cca03dda23bbe014aaa17a9e51a66c34bd481cbfd73b942d74e644936e136022',
                          '5c79464863cac894c5ad13d984d6cdb6834b72c8437b9de4b9a3e4072f3fea15',
                          '8f363f31f3dff7f5311bc296aa677441c61a4f7635472aa841c1198027037e7d',
                          '7d21b69701d4c9da4cb0876448f089b3529af2c70e63c52aba7bf368e8b450cf',
                          '3938bbb08b82fe0378d1f55530f94e5645fb61ee09867340aefe5dc258f0070e',
                          '9569d188333c72f87939e4fdb401a94af7aec89a8878209ef113a17b4b9481f2',
                          '47731bdc412904e4d597b0a40365380d6a69c373dd3cc8924bd1a73ce357d635',
                          '35f5b161b03998a66c750105c9c9d01a326c076f7f41ef5ec730ba4a6e9e1404',
                          '6de599b17713fe4664ce8d3e6b861a63f8f399f87a9f69b46a187735c14f68a5',
                          '07cd670dd70670e173fd703dc00c298cdf81cd60cbe5bb1884e4c281a89df151',
                          '94b957fe02c8b2166040635a490c0571e47f45a2e664e67c7d9f6c8228d960c0',
                          'eef1bc0bb45b1979e5f69920c3c7cf6d030a4ca6accfd93c0a6d36a64ec93391',
                          '151bd74973256bf122a0260db34c96e4e102f20e3bc1fd0a6c93d049c7654d97',
                          '0e5b291376a58a933f15e5ca8adbd8987ec600c138d1308bab9c147416162e19',
                          '6f27f7aba151e830cf48e2f5df8031b3810729e419fa618b6b30ba3355fa8ff3',
                          'e6357c6de6c2352032c9e3ca16e149efc5e7e2bcd5b5ec4a7807dd99ea7b5034',
                          '04ead216e16cdb9c6176d5c581c10dbcdf3a01a98871533e0ca504ade94860c3',
                          '75fe4b476e2d596cbf623bda69ed58bc09434d0168c67776fdf2059a52098e1e',
                          'b368f01114bbe6ff3fd1279127e41a853ac52887b3ced034024d07d6f82f34c9',
                          '5b44f9500cbf14d62e713a24adf813b92bb206d1eb842943aaf90a655037d7ef',
                          'eed09b78d2b484d666a5fa6e084df72e2b60ee7122deb52a52fd05e6bdd2ffd1',
                          '27b8b43eae7602fda0940a6c698b675fdadf8d201f56146dcfcf32c5e7bcde59',
                          'd9341c759c0898d91307cf9b3ae5efcb25bd5384b498e2720db8253ca7e8188f',
                          '8a8d20199886a3ef7714620594ed1f0261d99693616842b6ce4f1d71b845be10',
                          'ca23ec30d8348ce16a99baf9e360dba6440c67f05de44e7ab7019ef8169516a9',
                          '27a780b7a4eb0ca137c56aa2261ae63b279279ca452565fa8cbced8a40636ee6',
                          '1a012e6a3fa504fbf634665a7c8294daef728a218d6d8e5b3a51da49498dc017',
                          'c42ab1a1bdcad4016ff71a6cd9911448ede8004bf79a66c8b0e4e9ea2d929ee0',
                          'c01881c04b7f32256bfb77bcfcd6fe882b9d1150ca1d5a50d8bb65163b4c04e6',
                          'a938cef46b1ad6a97ace5276cac89a6ebee184f77ede4f300508b9cdf5d3ff62',
                          '61616312b79ce0a6ebea0c1edb9d2dec7cd4727165e8f7d2529393983d401447',
                          '8ef55ac94c3d2e418592b37bbab5dbb9fa0e085988d949da289e8361334be06d',
                          'b5ccc17c1dc51e0266ecccf365ef2eff576cf0b4b781e48cd571996c190bae06',
                          '26fcc503337caf416882e5de6b7762129877c4926d72927db93c6dd3b3aecd6e',
                          'd902618e5fd70fafa264b9403794e9b8b8c94f54611422caa937bd5d1f2ce08e',
                          '4300d25abee91559787f8d70f087ffcc87e63b55c9f7f475947eef2ca7d0ec7f',
                          'ed96c947fa5895d0ab7d38332e4019d5aecadf0253ff0753c8047e0e25306761',
                          '627051b23a3628de3cc5a104f482849437d719a42bdebcd59ee548faef94d0ec',
                          '861d81fd660255594cf9a736c87ca68c82ae34f45607f925da9c6967c658c231',
                          '31bcd149f49c97e3fdac5b661e2b6b3cec17a132623ee307c36a92b3d39228db',
                          '009bc6b1df70faa550bb399c021f4f7e9fa664eeab43d7c364d281aa28421946'],
 'selection_role': 'fixed model confirmation on images outside known prior evaluation memberships',
 'selection_seed': 20261107,
 'selection_uses_image_content_or_quality_scores': False,
 'selection_version': 'exclude_known500_sha256_rank_v26_1',
 'source_data_manifest_sha256': '140d5919305f3592d144ed4a1a8f8d53dcd19e170b212416591b454c0ea4e7e2',
 'source_training_images': 11000,
 'source_training_inventory_sha256': '04768543c975e0627eceb5615024145dde6fc1f4c636cfa614587a55fed3b0b3',
 'source_zips_modified': False,
 'test_zip_file': 'GEE_test.zip',
 'test_zip_sha256': '558916f53c3a1cabf04d3bda5d061a120a0ab3c5bc597bb56956e2f0e6adf21a',
 'training_RGB_duplicate_check_this_run': False,
 'version': 'fixed_compact_final_data_v26_1'}


In [ ]:
STAGE1_PIN_V32={'bytes': 1083869610,
 'checkpoint_sha256': '5e2695b72cdfbf4564ace82d98ec54899987ec37f059ae42dbace3bd38c33a3a',
 'configuration_sha256': '08afbf43ef6caea61c1208620b16f37cea7f72ece5bbf3be884ad479778350e9',
 'file': 'rebuild_v12/rebuild_v12_run1/stage1_final.pth',
 'kind': 'stage1',
 'model_state_sha256': '19b7b61e2c836198e54bdef9dcdcafdb9ad2ec881338de8cf2b8596401b55241'}


In [ ]:
import gc,hashlib,io,json,math,os,shutil,time,zipfile
from pathlib import Path,PurePosixPath
import numpy as np
import torch
from PIL import Image
from google.colab import drive
if not torch.cuda.is_available():raise RuntimeError("Select a Colab GPU runtime")
device=torch.device("cuda")
torch.set_num_threads(min(8,os.cpu_count() or 2))
drive.mount("/content/drive")
DRIVE_DATA_DIR=Path("/content/drive/MyDrive/FSO_data")
OUTPUT_ROOT=DRIVE_DATA_DIR/"stage_effect_v32"
LOCAL_CACHE=Path("/content/fso_sc_stage_effect_v32")
LOCAL_CACHE.mkdir(parents=True,exist_ok=True)
if shutil.disk_usage(LOCAL_CACHE).free<10*1024**3:raise RuntimeError("Keep10GiB LOCAL disk free")
AUTO_DOWNLOAD_RESULTS_V32=True
print("GPU:",torch.cuda.get_device_name(),"| Torch:",torch.__version__,"| CUDA:",torch.version.cuda)
print("Inference only. No optimizer/training, native BPG/FEC, API, source edits or new checkpoints.")


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

MODEL_VERSION = "fso_sc_paper_faithful_v6"
SEMANTIC_MODEL_CONFIG = {
    "group_norm_groups": 32,
    "activation": "SiLU",
    "downsample": "asymmetric_right_bottom_pad_then_stride2_conv",
    "upsample": "nearest_neighbor_then_conv",
    "loss_reduction": "mean",
    "input_preprocessing": "largest-center-square+bicubic256+RGB[-1,1]",
    "conv_initialization": "PyTorch_default",
    "codebook_initialization": "uniform(-1/K,1/K)",
    "decoder_attention_order": "VQGAN_inferred_interleaved_Res_Attn",
    "decoder_condition": "raw_Uq; post_quant_conv_only_for_decoder_feature",
}


def group_norm(channels, affine=True):
    return nn.GroupNorm(32, channels, eps=1e-6, affine=affine)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.norm1 = group_norm(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = group_norm(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        h = self.conv1(F.silu(self.norm1(x)))
        h = self.conv2(F.silu(self.norm2(h)))
        return self.shortcut(x) + h


class AttnBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = group_norm(channels)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.proj_out = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        h = self.norm(x)
        q, k, v = self.q(h), self.k(h), self.v(h)
        batch, channels, height, width = q.shape
        q = q.reshape(batch, channels, height * width).permute(0, 2, 1)
        k = k.reshape(batch, channels, height * width)
        attention = torch.bmm(q, k) * (channels ** -0.5)
        attention = F.softmax(attention, dim=2)
        v = v.reshape(batch, channels, height * width)
        h = torch.bmm(v, attention.permute(0, 2, 1)).reshape(
            batch, channels, height, width
        )
        return x + self.proj_out(h)


class SpatialNorm(nn.Module):
    """Paper Eq. (25): gamma(U_q) * GN(F) + beta(U_q)."""

    def __init__(self, channels, latent_dim):
        super().__init__()
        # MoVQ replaces VQGAN's normalization with spatial conditioning while
        # retaining GroupNorm's learned affine parameters.
        self.norm = group_norm(channels, affine=True)
        self.conv_gamma = nn.Conv2d(latent_dim, channels, 1)
        self.conv_beta = nn.Conv2d(latent_dim, channels, 1)

    def forward(self, x, quantized_condition):
        if x.shape[2:] != quantized_condition.shape[2:]:
            quantized_condition = F.interpolate(
                quantized_condition, size=x.shape[2:], mode="nearest"
            )
        gamma = self.conv_gamma(quantized_condition)
        beta = self.conv_beta(quantized_condition)
        return gamma * self.norm(x) + beta


class SpatialResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, latent_dim=4):
        super().__init__()
        self.norm1 = SpatialNorm(in_channels, latent_dim)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = SpatialNorm(out_channels, latent_dim)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x, quantized_condition):
        h = self.conv1(F.silu(self.norm1(x, quantized_condition)))
        h = self.conv2(F.silu(self.norm2(h, quantized_condition)))
        return self.shortcut(x) + h


class SpatialAttnBlock(nn.Module):
    def __init__(self, channels, latent_dim=4):
        super().__init__()
        self.norm = SpatialNorm(channels, latent_dim)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.proj_out = nn.Conv2d(channels, channels, 1)

    def forward(self, x, quantized_condition):
        h = self.norm(x, quantized_condition)
        q, k, v = self.q(h), self.k(h), self.v(h)
        batch, channels, height, width = q.shape
        q = q.reshape(batch, channels, height * width).permute(0, 2, 1)
        k = k.reshape(batch, channels, height * width)
        attention = torch.bmm(q, k) * (channels ** -0.5)
        attention = F.softmax(attention, dim=2)
        v = v.reshape(batch, channels, height * width)
        h = torch.bmm(v, attention.permute(0, 2, 1)).reshape(
            batch, channels, height, width
        )
        return x + self.proj_out(h)


class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2.0, mode="nearest"))


class Downsample(nn.Module):
    """VQGAN downsample: asymmetric right/bottom pad then padding-free Conv."""

    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=0)

    def forward(self, x):
        return self.conv(F.pad(x, (0, 1, 0, 1)))


class Encoder(nn.Module):
    def __init__(self, in_channels=3, latent_dim=4):
        super().__init__()
        self.in_conv = nn.Conv2d(in_channels, 256, 3, padding=1)
        self.res1 = nn.Sequential(ResidualBlock(256, 256), ResidualBlock(256, 256))
        self.down1 = Downsample(256)
        self.res2 = nn.Sequential(ResidualBlock(256, 512), ResidualBlock(512, 512))
        self.down2 = Downsample(512)
        self.res3 = nn.Sequential(ResidualBlock(512, 512), ResidualBlock(512, 512))
        self.down3 = Downsample(512)
        self.res4 = nn.Sequential(ResidualBlock(512, 1024), ResidualBlock(1024, 1024))
        self.attn1 = nn.Sequential(AttnBlock(1024), AttnBlock(1024))
        self.res5 = ResidualBlock(1024, 1024)
        self.attn2 = AttnBlock(1024)
        self.res6 = ResidualBlock(1024, 1024)
        self.out_norm = group_norm(1024)
        self.out_conv = nn.Conv2d(1024, latent_dim, 3, padding=1)

    def forward(self, x):
        x = self.res1(self.in_conv(x))
        x = self.res2(self.down1(x))
        x = self.res3(self.down2(x))
        x = self.res4(self.down3(x))
        x = self.attn1(x)
        x = self.attn2(self.res5(x))
        return self.out_conv(F.silu(self.out_norm(self.res6(x))))


class Decoder(nn.Module):
    """Fig. 3 block set arranged in the dimensionally valid MoVQ/VQGAN order."""

    def __init__(self, latent_dim=4):
        super().__init__()
        self.in_conv = nn.Conv2d(latent_dim, 1024, 3, padding=1)

        # VQGAN mid block: Res -> Attn -> Res.
        self.mid_res1 = SpatialResBlock(1024, 1024, latent_dim)
        self.mid_attn = SpatialAttnBlock(1024, latent_dim)
        self.mid_res2 = SpatialResBlock(1024, 1024, latent_dim)

        # Deepest level: three Res+Attn pairs, then 32 -> 64.
        self.deep_res = nn.ModuleList([
            SpatialResBlock(1024, 1024, latent_dim) for _ in range(3)
        ])
        self.deep_attn = nn.ModuleList([
            SpatialAttnBlock(1024, latent_dim) for _ in range(3)
        ])
        self.up_1024 = Upsample(1024)

        # 64x64 level: 1024 -> 512 followed by two 512 blocks, then 64 -> 128.
        self.level_512a = nn.ModuleList([
            SpatialResBlock(1024, 512, latent_dim),
            SpatialResBlock(512, 512, latent_dim),
            SpatialResBlock(512, 512, latent_dim),
        ])
        self.up_512a = Upsample(512)

        # 128x128 level: three 512 blocks, then 128 -> 256.
        self.level_512b = nn.ModuleList([
            SpatialResBlock(512, 512, latent_dim) for _ in range(3)
        ])
        self.up_512b = Upsample(512)

        # 256x256 level: 512 -> 256 followed by two 256 blocks.
        self.level_256 = nn.ModuleList([
            SpatialResBlock(512, 256, latent_dim),
            SpatialResBlock(256, 256, latent_dim),
            SpatialResBlock(256, 256, latent_dim),
        ])
        self.out_norm = SpatialNorm(256, latent_dim)
        self.out_conv = nn.Conv2d(256, 3, 3, padding=1)

    def forward(self, decoder_input, quantized_condition):
        h = self.mid_res1(self.in_conv(decoder_input), quantized_condition)
        h = self.mid_attn(h, quantized_condition)
        h = self.mid_res2(h, quantized_condition)

        for res_block, attn_block in zip(self.deep_res, self.deep_attn):
            h = attn_block(
                res_block(h, quantized_condition), quantized_condition
            )
        h = self.up_1024(h)

        for block in self.level_512a:
            h = block(h, quantized_condition)
        h = self.up_512a(h)

        for block in self.level_512b:
            h = block(h, quantized_condition)
        h = self.up_512b(h)

        for block in self.level_256:
            h = block(h, quantized_condition)
        return self.out_conv(F.silu(self.out_norm(h, quantized_condition)))


class VectorQuantizer(nn.Module):
    def __init__(
        self,
        num_embeddings=16384,
        embedding_dim=4,
        commitment_cost=1.0,
        distance_chunk_size=2048,
    ):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost
        self.distance_chunk_size = distance_chunk_size
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(
            -1.0 / num_embeddings, 1.0 / num_embeddings
        )

    @torch.no_grad()
    def nearest_indices(self, flat_input):
        # VQ assignment remains FP32 even when the surrounding model uses AMP.
        vectors = flat_input.detach().float()
        codebook = self.embedding.weight.detach().float()
        codebook_norm = codebook.square().sum(dim=1).unsqueeze(0)
        nearest = []
        for chunk in vectors.split(self.distance_chunk_size, dim=0):
            distances = (
                chunk.square().sum(dim=1, keepdim=True)
                + codebook_norm
                - 2.0 * chunk @ codebook.t()
            )
            nearest.append(distances.argmin(dim=1))
        return torch.cat(nearest, dim=0)

    def lookup(self, encoding_indices, spatial_shape=None):
        if spatial_shape is None:
            if encoding_indices.ndim != 3:
                raise ValueError("indices must have shape [batch, height, width]")
            batch, height, width = encoding_indices.shape
        else:
            batch, height, width = spatial_shape
        flat_indices = encoding_indices.reshape(-1).long()
        quantized = F.embedding(flat_indices, self.embedding.weight)
        return quantized.view(batch, height, width, self.embedding_dim).permute(
            0, 3, 1, 2
        ).contiguous()

    def forward(self, inputs):
        channels_last = inputs.permute(0, 2, 3, 1).contiguous()
        batch, height, width, channels = channels_last.shape
        if channels != self.embedding_dim:
            raise ValueError(f"Expected embedding_dim={self.embedding_dim}, got {channels}")

        flat_input = channels_last.view(-1, self.embedding_dim)
        flat_indices = self.nearest_indices(flat_input)
        encoding_indices = flat_indices.view(batch, height, width)
        quantized = self.lookup(encoding_indices, (batch, height, width))

        encoder_loss = F.mse_loss(quantized.detach().float(), inputs.float())
        codebook_loss = F.mse_loss(quantized.float(), inputs.detach().float())
        vq_loss = codebook_loss + self.commitment_cost * encoder_loss

        quantized_st = inputs + (quantized.to(inputs.dtype) - inputs).detach()
        return quantized_st, vq_loss, encoding_indices


class FSO_VQVAE(nn.Module):
    def __init__(self, num_embeddings=16384, embedding_dim=4, commitment_cost=1.0):
        super().__init__()
        self.encoder = Encoder(in_channels=3, latent_dim=embedding_dim)
        self.pre_quant_conv = nn.Conv2d(embedding_dim, embedding_dim, 1)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.post_quant_conv = nn.Conv2d(embedding_dim, embedding_dim, 1)
        self.decoder = Decoder(latent_dim=embedding_dim)

    def encode(self, x):
        if x.ndim != 4 or x.shape[1] != 3:
            raise ValueError("Expected RGB input with shape [batch, 3, height, width]")
        if x.shape[-2] % 8 or x.shape[-1] % 8:
            raise ValueError("Input height and width must be divisible by 8")
        continuous = self.pre_quant_conv(self.encoder(x))
        quantized, vq_loss, indices = self.vq(continuous)
        return quantized, vq_loss, indices

    def decode_quantized(self, quantized):
        decoder_input = self.post_quant_conv(quantized)
        return self.decoder(decoder_input, quantized_condition=quantized)

    def decode_indices(self, indices, spatial_shape):
        return self.decode_quantized(self.vq.lookup(indices, spatial_shape))

    def forward(self, x):
        quantized, vq_loss, indices = self.encode(x)
        return self.decode_quantized(quantized), vq_loss, indices


# The architecture is intentionally not instantiated here; it is large (~hundreds of M parameters).

In [ ]:
from dataclasses import dataclass
import math

import torch
import torch.nn.functional as F


BITS_PER_INDEX = 14
QAM_MAPPING_VERSION = "gray_qpsk_rect8qam_16qam_unit_energy_v1"
INTEGER_DTYPES = {torch.uint8, torch.int8, torch.int16, torch.int32, torch.int64}


def integers_to_bits(values, width):
    values = values.long()
    shifts = torch.arange(width - 1, -1, -1, device=values.device)
    return ((values.unsqueeze(-1) >> shifts) & 1).to(torch.uint8)


def bits_to_integers(bits):
    bits = bits.long()
    width = bits.shape[-1]
    weights = (2 ** torch.arange(width - 1, -1, -1, device=bits.device)).long()
    return (bits * weights).sum(dim=-1)


def indices_to_bits(indices, bits_per_index=BITS_PER_INDEX):
    if indices.dtype not in INTEGER_DTYPES:
        raise TypeError("indices must use an integer dtype")
    if indices.numel() and (
        (indices < 0).any().item() or (indices >= 2**bits_per_index).any().item()
    ):
        raise ValueError("Index is outside the fixed-width bit representation")
    return integers_to_bits(indices, bits_per_index)


def bits_to_indices(bits, bits_per_index=BITS_PER_INDEX):
    if bits.shape[-1] != bits_per_index:
        raise ValueError(f"Last dimension must be {bits_per_index} bits")
    if bits.dtype not in INTEGER_DTYPES and bits.dtype != torch.bool:
        raise TypeError("bits must use an integer or bool dtype")
    if bits.numel() and ((bits != 0) & (bits != 1)).any().item():
        raise ValueError("bits must contain only 0 and 1")
    return bits_to_integers(bits)


def qam_constellation(modulation, device):
    modulation = modulation.upper()
    supported = {"QPSK": 2, "8QAM": 3, "16QAM": 4}
    if modulation not in supported:
        raise ValueError(f"Unsupported modulation: {modulation}")
    bits_per_symbol = supported[modulation]
    labels = integers_to_bits(
        torch.arange(2**bits_per_symbol, device=device), bits_per_symbol
    ).float()

    if modulation == "QPSK":
        i = 1.0 - 2.0 * labels[:, 0]
        q = 1.0 - 2.0 * labels[:, 1]
        normalization = math.sqrt(2.0)
    elif modulation == "8QAM":
        # Rectangular Gray-labelled 8-QAM. The paper does not specify its 8-QAM geometry.
        gray_levels = torch.tensor([-3.0, -1.0, 3.0, 1.0], device=device)
        i_code = (2 * labels[:, 0] + labels[:, 1]).long()
        i = gray_levels[i_code]
        q = 2.0 * labels[:, 2] - 1.0
        normalization = math.sqrt(6.0)
    else:
        gray_levels = torch.tensor([-3.0, -1.0, 3.0, 1.0], device=device)
        i_code = (2 * labels[:, 0] + labels[:, 1]).long()
        q_code = (2 * labels[:, 2] + labels[:, 3]).long()
        i = gray_levels[i_code]
        q = gray_levels[q_code]
        normalization = math.sqrt(10.0)

    points = torch.complex(i, q) / normalization
    return points.to(torch.complex64), labels.to(torch.uint8), bits_per_symbol


def qam_modulate(bits, modulation):
    points, _, bits_per_symbol = qam_constellation(modulation, bits.device)
    if bits.shape[-1] != bits_per_symbol:
        raise ValueError(f"{modulation} expects {bits_per_symbol} bits per symbol")
    symbol_ids = bits_to_integers(bits).long()
    return points[symbol_ids]


def qam_demodulate(symbols, modulation):
    points, labels, _ = qam_constellation(modulation, symbols.device)
    # Cartesian-product QAM: limiting observations outside the outer levels
    # preserves exact nearest-point decisions and avoids large-float cancellation.
    i_limit = 2 * points.real.abs().max()
    q_limit = 2 * points.imag.abs().max()
    stable = torch.complex(symbols.real.clamp(-i_limit, i_limit),
                           symbols.imag.clamp(-q_limit, q_limit))
    distances = (stable.reshape(-1, 1) - points.reshape(1, -1)).abs().square()
    nearest = distances.argmin(dim=1)
    return labels[nearest].view(*symbols.shape, labels.shape[-1])


@dataclass(frozen=True)
class FSOChannelOutput:
    """Receiver observation before hard QAM decision.

    Return `is_equalized=True` after coherent receiver DSP, or provide the
    complex channel gain so this link can perform one-tap equalization.
    """

    symbols: torch.Tensor
    channel_gain: object = None
    is_equalized: bool = False
    diagnostics: object = None


class IdentityFSOChannel:
    """No-impairment receiver hook used to test the digital path."""

    def __call__(self, symbols, **channel_kwargs):
        return FSOChannelOutput(symbols=symbols, is_equalized=True)


def equalize_fso_output(channel_output, transmitted):
    if isinstance(channel_output, FSOChannelOutput):
        received = channel_output.symbols
        channel_gain = channel_output.channel_gain
        is_equalized = channel_output.is_equalized
    elif isinstance(channel_output, tuple) and len(channel_output) == 2:
        received, channel_gain = channel_output
        is_equalized = False
    else:
        raise TypeError(
            "FSO receiver must return FSOChannelOutput or (symbols, channel_gain)"
        )

    if not torch.is_tensor(received):
        raise TypeError("FSO receiver must return a torch.Tensor observation")
    if received.device != transmitted.device:
        raise ValueError(
            f"FSO receiver changed device: {transmitted.device} -> {received.device}"
        )
    if not torch.is_complex(received):
        raise TypeError("FSO receiver output must be complex baseband symbols")
    if received.shape != transmitted.shape:
        raise ValueError(
            f"FSO receiver changed symbol shape: {transmitted.shape} -> {received.shape}"
        )

    if not is_equalized:
        if not torch.is_tensor(channel_gain):
            raise TypeError("Unequalized FSO output must include a tensor channel_gain")
        if channel_gain.device != received.device:
            raise ValueError("channel_gain and received symbols must share a device")
        try:
            channel_gain = torch.broadcast_to(channel_gain, received.shape)
        except RuntimeError as error:
            raise ValueError("channel_gain is not broadcastable to the symbol shape") from error
        if (channel_gain.abs() == 0).any().item():
            raise ValueError("channel_gain contains an exact zero")
        if not torch.isfinite(channel_gain).all().item():
            raise ValueError("channel_gain contains NaN or Inf")
        denominator = channel_gain.abs().square().clamp_min(
            torch.finfo(received.real.dtype).tiny
        )
        received = received * channel_gain.conj() / denominator

    if not torch.isfinite(received).all().item():
        raise ValueError("FSO receiver output contains NaN or Inf")
    return received


@dataclass
class QAMIndexLink:
    """14-bit digital link ending in equalized complex [B, S] symbols."""

    fso_channel: object
    modulation: str = "QPSK"
    bits_per_index: int = BITS_PER_INDEX
    receiver_config: object = None

    def reproducibility_config(self):
        if self.receiver_config is not None and not isinstance(
            self.receiver_config, dict
        ):
            raise TypeError("receiver_config must be a serializable dict or None")
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulation": self.modulation.upper(),
            "bits_per_index": self.bits_per_index,
            "receiver_config": self.receiver_config,
        }

    @torch.no_grad()
    def __call__(self, indices, return_statistics=False, **channel_kwargs):
        if indices.ndim < 2:
            raise ValueError("indices must retain a batch dimension")
        original_shape = indices.shape
        batch = original_shape[0]
        valid_bits_per_sample = indices[0].numel() * self.bits_per_index
        bitstream = indices_to_bits(indices, self.bits_per_index).reshape(batch, -1)
        _, _, bits_per_symbol = qam_constellation(self.modulation, indices.device)
        padding = (-valid_bits_per_sample) % bits_per_symbol
        if padding:
            bitstream = F.pad(bitstream, (0, padding))

        grouped_bits = bitstream.view(batch, -1, bits_per_symbol)
        transmitted = qam_modulate(grouped_bits, self.modulation)
        channel_output = self.fso_channel(transmitted, **channel_kwargs)
        received = equalize_fso_output(channel_output, transmitted)
        recovered_grouped = qam_demodulate(received, self.modulation)
        recovered = recovered_grouped.reshape(batch, -1)
        valid_mask = torch.arange(
            recovered.shape[1], device=indices.device
        ).view(1, -1) < valid_bits_per_sample
        transmitted_bit_error_mask = recovered != bitstream
        data_bit_error_mask = transmitted_bit_error_mask & valid_mask
        # SER is defined over every transmitted QAM symbol, including the final
        # padded symbol. Payload BER/TER continue to exclude padding bits.
        symbol_error_mask = transmitted_bit_error_mask.view(
            batch, -1, bits_per_symbol
        ).any(dim=-1)
        if padding:
            recovered = recovered[:, :-padding]
        recovered = recovered.view(*original_shape, self.bits_per_index)
        recovered_indices = bits_to_indices(
            recovered, self.bits_per_index
        ).view(original_shape)
        if not return_statistics:
            return recovered_indices

        bit_errors = int(data_bit_error_mask.sum().item())
        valid_bits = batch * valid_bits_per_sample
        transmitted_bit_errors = int(transmitted_bit_error_mask.sum().item())
        transmitted_bits = bitstream.numel()
        symbol_errors = int(symbol_error_mask.sum().item())
        valid_symbols = symbol_error_mask.numel()
        index_errors = int((recovered_indices != indices).sum().item())
        statistics = {
            "modulation": self.modulation.upper(),
            "padding_bits_per_sample": padding,
            "bit_errors": bit_errors,
            "valid_bits": valid_bits,
            "ber": bit_errors / max(valid_bits, 1),
            "transmitted_bit_errors": transmitted_bit_errors,
            "transmitted_bits": transmitted_bits,
            "transmitted_ber": transmitted_bit_errors / max(transmitted_bits, 1),
            "symbol_errors": symbol_errors,
            "valid_symbols": valid_symbols,
            "ser": symbol_errors / max(valid_symbols, 1),
            "index_errors": index_errors,
            "indices": indices.numel(),
            "ter": index_errors / max(indices.numel(), 1),
        }
        if isinstance(channel_output, FSOChannelOutput) and channel_output.diagnostics:
            statistics["channel"] = channel_output.diagnostics
        return recovered_indices, statistics


def qam_identity_roundtrip_test():
    test_indices = torch.tensor([
        [0, 1, 2, 8191, 8192, 16383, 77],
        [77, 16383, 8192, 8191, 2, 1, 0],
    ]).view(2, 1, 7)
    for modulation in ("QPSK", "8QAM", "16QAM"):
        link = QAMIndexLink(IdentityFSOChannel(), modulation=modulation)
        recovered = link(test_indices)
        assert torch.equal(recovered, test_indices), f"{modulation} round trip failed"
    print("Identity-channel index/QAM round-trip passed: QPSK, 8QAM, 16QAM")


qam_identity_roundtrip_test()

In [ ]:
"""Paper-parameter symbol-rate FSO channel with explicit heterodyne shot floor.

The vacuum/shot term is a physically motivated missing receiver detail, not an
explicit Chen et al. parameter or an LPIPS-curve calibration. One complex
heterodyne mode/symbol has E|n|^2=h*nu/eta, so the optical-field (sqrt W)
convention gives N_complex=(q_e/R)*R_s W; I and Q each have half that variance.
Reference: A. Martinez, Physical Review A 77,032116 (2008),
doi:10.1103/PhysRevA.77.032116, https://ir.cwi.nl/pub/13524/13524D.pdf .
"""

from dataclasses import asdict, dataclass, replace
import math
import warnings

import numpy as np
import torch


PAPER_FSO_CHANNEL_VERSION = "chen2025_eq3_18_heterodyne_shot_v6"
WEATHER_ATTENUATION_DB_PER_KM = {
    "clear": 0.147,
    "dust": 1.043,
    "fog": 33.979,
    "rain": 1.403,
    "snow": 2.145,
}

# The paper specifies FOV=0.1 but does not print a unit, and it does not give
# the optical filter width or the proprietary VPI receiver/DSP implementation.
# These assumptions are explicit and stored with the final model metadata.
PAPER_UNDISCLOSED_RECEIVER_ASSUMPTIONS = {
    "fov_interpretation": "0.1 rad",
    "optical_filter_bandwidth": "1 nm",
    "background_noise_conversion": (
        "Eq.-(4) optical background power is used as unit-energy complex "
        "matched-filter noise variance; proprietary coherent receiver is undisclosed"
    ),
    "ase_model": (
        "Assume post-transmitter-EDFA launch power (probe point undisclosed); standard two-polarization "
        "EDFA ASE is propagated through the optical path"
    ),
    "ase_noise_bandwidth": "28 GHz symbol-rate equivalent",
    "diversity_ase_correlation": "one common transmitter-ASE waveform across Rx branches",
    "photodiode_responsivity": (
        "Table-I R=0.85 A/W determines eta=R*h*nu/q_e; "
        "one-complex-mode/symbol heterodyne shot floor q_e*Rs/R is input-referred; "
        "strong LO and negligible receiver thermal/RIN/dark current are assumptions"
    ),
    "receiver_shot_noise": (
        "complex heterodyne mode variance h*nu/eta per symbol; "
        "N_complex=q_e*Rs/R W, with half per real quadrature; "
        "physically motivated undisclosed implementation detail, not curve-fitted"
    ),
    "receiver_shot_noise_reference": "https://doi.org/10.1103/PhysRevA.77.032116",
    "pulse_shaping": "symbol-rate matched-filter equivalent; RRC rolloff undisclosed",
    "carrier_recovery": (
        "raw channel returns true complex gain for an explicitly ideal/oracle baseline; "
        "the separate SignalDSP wrapper uses only block magnitude and estimates phase/frequency"
    ),
    "rytov_quadrature": "xi=u^2 ground-clustered trapezoid with convergence self-test",
    "malaga_phase_difference": "pi/2 chosen phase assumption; paper undisclosed",
    "malaga_beta_fit": (
        "paper omits beta and phase; nearest Eq.-(15) natural beta is used with "
        "an explicit feasibility/residual diagnostic (strict policy is available)"
    ),
    "fading_coherence": "one independent block fade per image and receiver",
    "8qam_geometry": "rectangular Gray 8-QAM; paper only states 8-QAM",
}


def paper_generator_device_matches(generator, tensor_device):
    generator_device = generator.device
    if generator_device.type != tensor_device.type:
        return False
    if tensor_device.type != "cuda":
        return generator_device == tensor_device
    generator_index = (
        torch.cuda.current_device()
        if generator_device.index is None
        else generator_device.index
    )
    tensor_index = (
        torch.cuda.current_device() if tensor_device.index is None else tensor_device.index
    )
    return generator_index == tensor_index


@dataclass(frozen=True)
class PaperFSOSystemConfig:
    # Paper Table I / Eqs. (4)-(18).
    wavelength_m: float = 1550e-9
    satellite_altitude_m: float = 620e3
    receiver_altitude_m: float = 0.0
    earth_radius_m: float = 6371e3
    earth_gravitational_parameter: float = 3.986004418e14
    light_speed_m_s: float = 299792458.0
    planck_constant_j_s: float = 6.62607015e-34
    electron_charge_c: float = 1.602176634e-19
    beam_radius_m: float = 0.05
    receiver_aperture_m: float = 0.318
    rms_wind_speed_m_s: float = 21.0
    ground_cn2_m_neg_two_thirds: float = 1.7e-14
    atmospheric_range_m: float = 20e3
    fog_range_m: float = 8e3
    field_of_view_rad: float = 0.1
    optical_filter_bandwidth_um: float = 1e-3  # 1 nm; not disclosed.
    symbol_rate_baud: float = 28e9
    amplifier_gain_db: float = 30.0
    amplifier_noise_figure_db: float = 4.0
    lo_linewidth_hz: float = 1000.0
    photodiode_responsivity_a_w: float = 0.85
    doppler_residual_ratio: float = 0.1
    malaga_omega: float = 0.5
    malaga_b0: float = 0.25
    malaga_phase_difference_rad: float = math.pi / 2
    malaga_beta: object = None
    max_derived_beta: int = 1000
    beta_fit_policy: str = "nearest_warn"
    beta_fit_relative_tolerance: float = 1e-3
    integration_points: int = 4097
    integration_grid_power: float = 2.0
    num_receivers: int = 1
    combining: str = "MRC"
    include_turbulence: bool = True
    include_background_noise: bool = True
    include_ase_noise: bool = True
    include_shot_noise: bool = True
    heterodyne_shot_noise_convention: float = 1.0
    include_doppler: bool = True
    include_phase_noise: bool = True

    # Convenience/reference condition; this combination is not a paper-wide default.
    launch_power_dbm: float = 33.0
    zenith_angle_deg: float = 30.0
    pointing_error_urad: float = 20.0
    scattered_power_ratio: float = 0.95
    background_light_radiance_w_m2_sr_um: float = 1e-4
    weather: str = "clear"

    def __post_init__(self):
        if self.integration_points < 257 or self.integration_points % 2 == 0:
            raise ValueError("integration_points must be an odd integer >= 257")
        if self.integration_grid_power < 1.0:
            raise ValueError("integration_grid_power must be at least 1")
        if self.photodiode_responsivity_a_w <= 0:
            raise ValueError("Photodiode responsivity must be positive")
        quantum_efficiency = (
            self.photodiode_responsivity_a_w * self.planck_constant_j_s
            * self.light_speed_m_s / self.wavelength_m / self.electron_charge_c
        )
        if not 0 < quantum_efficiency <= 1:
            raise ValueError("Responsivity implies unphysical photodiode quantum efficiency")
        if self.heterodyne_shot_noise_convention <= 0:
            raise ValueError("Heterodyne complex-noise convention factor must be positive")
        if self.num_receivers < 1:
            raise ValueError("num_receivers must be positive")
        if self.combining.upper() not in {"MRC", "EGC"}:
            raise ValueError("combining must be MRC or EGC")
        if not math.isclose(self.malaga_omega + 2 * self.malaga_b0, 1.0):
            raise ValueError("The paper normalizes Malaga power as Omega + 2*b0 = 1")
        if self.beta_fit_policy not in {"nearest_warn", "strict"}:
            raise ValueError("beta_fit_policy must be nearest_warn or strict")
        if self.beta_fit_relative_tolerance < 0:
            raise ValueError("beta_fit_relative_tolerance must be nonnegative")


def _trapezoid(values, grid):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(values, grid)
    return np.trapz(values, grid)


def hufnagel_valley_cn2(altitude_m, config):
    h = np.asarray(altitude_m, dtype=np.float64)
    v = config.rms_wind_speed_m_s
    a = config.ground_cn2_m_neg_two_thirds
    return (
        0.00594 * (v / 27.0) ** 2 * (1e-5 * h) ** 10 * np.exp(-h / 1000.0)
        + 2.7e-16 * np.exp(-h / 1500.0)
        + a * np.exp(-h / 100.0)
    )


def satellite_slant_range_m(zenith_angle_deg, config):
    phi = math.radians(float(zenith_angle_deg))
    if not 0.0 <= phi < math.pi / 2:
        raise ValueError("zenith_angle_deg must be in [0, 90)")
    ground_radius = config.earth_radius_m + config.receiver_altitude_m
    satellite_radius = config.earth_radius_m + config.satellite_altitude_m
    return (
        math.sqrt(
            satellite_radius**2
            - ground_radius**2 * math.sin(phi) ** 2
        )
        - ground_radius * math.cos(phi)
    )


def path_altitude_m(path_fraction, slant_range_m, zenith_angle_deg, config):
    xi = np.asarray(path_fraction, dtype=np.float64)
    phi = math.radians(float(zenith_angle_deg))
    ground_radius = config.earth_radius_m + config.receiver_altitude_m
    distance = xi * slant_range_m
    radius = np.sqrt(
        ground_radius**2
        + distance**2
        + 2.0 * ground_radius * distance * math.cos(phi)
    )
    return radius - config.earth_radius_m


def shadowed_rician_second_moment(beta, omega_prime, gamma):
    mean_y = omega_prime + gamma
    beta_independent = omega_prime**2 + 4 * omega_prime * gamma + 2 * gamma**2
    return (
        beta_independent + omega_prime**2 / float(beta)
    ) / mean_y**2


def fit_malaga_beta_eq15(target_second_moment, omega_prime, gamma, max_beta):
    """Nearest natural beta plus an honest Eq.-(15) feasibility diagnostic."""
    mean_y = omega_prime + gamma
    beta_independent = omega_prime**2 + 4 * omega_prime * gamma + 2 * gamma**2
    denominator = target_second_moment * mean_y**2 - beta_independent
    continuous_beta = (
        omega_prime**2 / denominator if denominator > 0 else float("inf")
    )
    continuum_feasible = 1.0 <= continuous_beta <= float(max_beta)
    if math.isfinite(continuous_beta):
        candidates = {
            1,
            int(max_beta),
            min(max(int(math.floor(continuous_beta)), 1), int(max_beta)),
            min(max(int(math.ceil(continuous_beta)), 1), int(max_beta)),
        }
    else:
        candidates = {int(max_beta)}
    beta = min(
        candidates,
        key=lambda value: abs(
            shadowed_rician_second_moment(value, omega_prime, gamma)
            - target_second_moment
        ),
    )
    fitted = shadowed_rician_second_moment(beta, omega_prime, gamma)
    relative_residual = abs(fitted - target_second_moment) / max(
        abs(target_second_moment), 1e-30
    )
    return {
        "beta": int(beta),
        "continuous_beta": continuous_beta,
        "continuum_feasible_within_search": continuum_feasible,
        "fitted_second_moment": fitted,
        "relative_residual": relative_residual,
        "minimum_second_moment": beta_independent / mean_y**2,
        "maximum_natural_second_moment": shadowed_rician_second_moment(
            1, omega_prime, gamma
        ),
    }


def compute_paper_link_budget(
    config,
    launch_power_dbm=None,
    zenith_angle_deg=None,
    pointing_error_urad=None,
    scattered_power_ratio=None,
    background_light_radiance_w_m2_sr_um=None,
    weather=None,
):
    """Implement paper Eqs. (4)-(18) and Table I as scalar diagnostics."""
    launch_power_dbm = (
        config.launch_power_dbm if launch_power_dbm is None else float(launch_power_dbm)
    )
    zenith_angle_deg = (
        config.zenith_angle_deg if zenith_angle_deg is None else float(zenith_angle_deg)
    )
    pointing_error_urad = (
        config.pointing_error_urad
        if pointing_error_urad is None
        else float(pointing_error_urad)
    )
    rho = (
        config.scattered_power_ratio
        if scattered_power_ratio is None
        else float(scattered_power_ratio)
    )
    radiance = (
        config.background_light_radiance_w_m2_sr_um
        if background_light_radiance_w_m2_sr_um is None
        else float(background_light_radiance_w_m2_sr_um)
    )
    weather = (config.weather if weather is None else weather).lower()
    if weather not in WEATHER_ATTENUATION_DB_PER_KM:
        raise ValueError(f"Unsupported weather: {weather}")
    if not 0.0 <= rho <= 1.0:
        raise ValueError("scattered_power_ratio must be in [0, 1]")
    if radiance < 0.0:
        raise ValueError("background light radiance cannot be negative")

    wavelength = config.wavelength_m
    wavenumber = 2.0 * math.pi / wavelength
    slant_range = satellite_slant_range_m(zenith_angle_deg, config)

    theta0 = 1.0  # Collimated input beam: F0 -> infinity.
    lambda0 = 2.0 * slant_range / (wavenumber * config.beam_radius_m**2)
    denominator = theta0**2 + lambda0**2
    theta1 = theta0 / denominator
    lambda1 = lambda0 / denominator
    diffraction_radius = config.beam_radius_m / math.sqrt(theta1**2 + lambda1**2)

    # The Hufnagel-Valley ground term decays on a ~100-m altitude scale. A
    # uniform 4097-point slant grid undersamples it, so cluster samples near xi=0.
    xi = np.linspace(
        0.0, 1.0, config.integration_points, dtype=np.float64
    ) ** config.integration_grid_power
    altitude = path_altitude_m(xi, slant_range, zenith_angle_deg, config)
    cn2 = hufnagel_valley_cn2(altitude, config)
    mu_2d = float(_trapezoid(cn2 * xi ** (5.0 / 3.0), xi))
    spreading_factor = (
        4.35
        * mu_2d
        * lambda1 ** (5.0 / 6.0)
        * wavenumber ** (7.0 / 6.0)
        * slant_range ** (11.0 / 6.0)
    )
    long_term_radius = diffraction_radius * math.sqrt(1.0 + spreading_factor)

    geometric_scintillation_db = -10.0 * math.log10(
        config.receiver_aperture_m**2 / (2.0 * long_term_radius**2)
    )
    pointing_offset_m = slant_range * pointing_error_urad * 1e-6
    pointing_db = (10.0 / math.log(10.0)) * (
        2.0 * pointing_offset_m**2 / long_term_radius**2
    )
    weather_range_m = config.fog_range_m if weather == "fog" else config.atmospheric_range_m
    atmospheric_db = WEATHER_ATTENUATION_DB_PER_KM[weather] * weather_range_m / 1000.0
    total_attenuation_db = geometric_scintillation_db + pointing_db + atmospheric_db

    complex_kernel = (
        xi ** (5.0 / 6.0)
        * (
            lambda1 * xi
            + 1j * (1.0 - (1.0 - theta1) * xi)
        ) ** (5.0 / 6.0)
        - lambda1 ** (5.0 / 6.0) * xi ** (5.0 / 3.0)
    )
    rytov_variance = float(
        8.7
        * wavenumber ** (7.0 / 6.0)
        * slant_range ** (11.0 / 6.0)
        * np.real(_trapezoid(cn2 * complex_kernel, xi))
    )
    rytov_variance = max(rytov_variance, 0.0)
    sigma_lnx2 = (
        0.49
        * rytov_variance
        / (
            1.0
            + (1.0 + theta1) * 0.56 * rytov_variance ** (6.0 / 5.0)
        ) ** (7.0 / 6.0)
    )
    sigma_lny2 = (
        0.51
        * rytov_variance
        / (1.0 + 0.69 * rytov_variance ** (6.0 / 5.0)) ** (5.0 / 6.0)
    )
    alpha = 1.0 / math.expm1(max(sigma_lnx2, 1e-12))

    omega = config.malaga_omega
    b0 = config.malaga_b0
    phase_difference = config.malaga_phase_difference_rad
    omega_prime = (
        omega
        + 2.0 * b0 * rho
        + 2.0 * math.sqrt(2.0 * b0 * omega * rho) * math.cos(phase_difference)
    )
    gamma = 2.0 * b0 * (1.0 - rho)
    target_y_second_moment = math.exp(sigma_lny2)
    if config.malaga_beta is None:
        beta_fit = fit_malaga_beta_eq15(
            target_y_second_moment,
            omega_prime,
            gamma,
            config.max_derived_beta,
        )
        beta = beta_fit["beta"]
        beta_fit_feasible = (
            beta_fit["continuum_feasible_within_search"]
            and beta_fit["relative_residual"]
            <= config.beta_fit_relative_tolerance
        )
        beta_source = "eq15_nearest_natural"
        if config.beta_fit_policy == "strict" and not beta_fit_feasible:
            raise ValueError(
                "Paper-undisclosed Malaga beta/phase cannot satisfy Eq. (15) "
                f"within tolerance: target={target_y_second_moment:.8g}, "
                f"nearest={beta_fit['fitted_second_moment']:.8g}, "
                f"relative residual={beta_fit['relative_residual']:.3%}"
            )
    else:
        beta = int(config.malaga_beta)
        if beta < 1:
            raise ValueError("malaga_beta must be a natural number")
        fitted = shadowed_rician_second_moment(beta, omega_prime, gamma)
        beta_fit = {
            "beta": beta,
            "continuous_beta": None,
            "continuum_feasible_within_search": None,
            "fitted_second_moment": fitted,
            "relative_residual": abs(fitted - target_y_second_moment)
            / max(abs(target_y_second_moment), 1e-30),
            "minimum_second_moment": None,
            "maximum_natural_second_moment": None,
        }
        beta_fit_feasible = None
        beta_source = "explicit_user_assumption"
    fitted_y_second_moment = beta_fit["fitted_second_moment"]

    earth_radius = config.earth_radius_m
    orbit_radius = earth_radius + config.satellite_altitude_m
    orbital_angular_velocity = math.sqrt(
        config.earth_gravitational_parameter / orbit_radius**3
    )
    phi = math.radians(zenith_angle_deg)
    phi_max = math.pi / 2.0
    eta = math.cos(
        math.acos(earth_radius / orbit_radius * math.cos(phi_max)) - phi_max
    )
    optical_frequency_hz = config.light_speed_m_s / wavelength
    doppler_denominator = math.sqrt(
        earth_radius**2
        + orbit_radius**2
        - 2.0 * earth_radius * orbit_radius * math.cos(phi) * eta
    )
    doppler_hz = -(
        optical_frequency_hz
        / config.light_speed_m_s
        * orbital_angular_velocity
        * earth_radius
        * orbit_radius
        * math.sin(phi)
        * eta
        / doppler_denominator
    )
    residual_doppler_hz = config.doppler_residual_ratio * doppler_hz

    background_noise_w = (
        radiance
        * math.pi**2
        * config.field_of_view_rad**2
        * config.receiver_aperture_m**2
        / 16.0
        * config.optical_filter_bandwidth_um
    )
    amplifier_gain_linear = 10.0 ** (config.amplifier_gain_db / 10.0)
    noise_factor = 10.0 ** (config.amplifier_noise_figure_db / 10.0)
    spontaneous_emission_factor = (
        noise_factor - 1.0 / amplifier_gain_linear
    ) / (2.0 * (1.0 - 1.0 / amplifier_gain_linear))
    ase_noise_w = (
        2.0
        * spontaneous_emission_factor
        * config.planck_constant_j_s
        * optical_frequency_hz
        * (amplifier_gain_linear - 1.0)
        * config.symbol_rate_baud
    )

    # One detected complex temporal mode per Nyquist symbol. Equivalent
    # input field noise is independent between receiver branches, not common
    # transmitter ASE. No unknown LO power or arbitrary thermal floor is fitted.
    quantum_efficiency = (
        config.photodiode_responsivity_a_w * config.planck_constant_j_s
        * optical_frequency_hz / config.electron_charge_c
    )
    receiver_shot_noise_w = (
        config.heterodyne_shot_noise_convention * config.electron_charge_c
        / config.photodiode_responsivity_a_w * config.symbol_rate_baud
    )

    return {
        "channel_version": PAPER_FSO_CHANNEL_VERSION,
        "launch_power_dbm": launch_power_dbm,
        "launch_power_w": 10.0 ** ((launch_power_dbm - 30.0) / 10.0),
        "zenith_angle_deg": zenith_angle_deg,
        "pointing_error_urad": pointing_error_urad,
        "scattered_power_ratio": rho,
        "background_light_radiance_w_m2_sr_um": radiance,
        "weather": weather,
        "slant_range_m": slant_range,
        "theta1": theta1,
        "lambda1": lambda1,
        "diffraction_radius_m": diffraction_radius,
        "long_term_radius_m": long_term_radius,
        "geometric_scintillation_db": geometric_scintillation_db,
        "pointing_db": pointing_db,
        "atmospheric_db": atmospheric_db,
        "total_attenuation_db": total_attenuation_db,
        "rytov_variance": rytov_variance,
        "sigma_lnx2": sigma_lnx2,
        "sigma_lny2": sigma_lny2,
        "malaga_alpha": alpha,
        "malaga_beta": beta,
        "malaga_beta_source": beta_source,
        "malaga_beta_fit_feasible": beta_fit_feasible,
        "malaga_beta_continuous": beta_fit["continuous_beta"],
        "malaga_beta_fit_relative_residual": beta_fit["relative_residual"],
        "malaga_beta_minimum_second_moment": beta_fit["minimum_second_moment"],
        "malaga_beta_maximum_natural_second_moment": beta_fit[
            "maximum_natural_second_moment"
        ],
        "malaga_omega_prime": omega_prime,
        "malaga_gamma": gamma,
        "malaga_mean_y": omega_prime + gamma,
        "target_y_second_moment": target_y_second_moment,
        "fitted_y_second_moment": fitted_y_second_moment,
        "doppler_hz": doppler_hz,
        "residual_doppler_hz": residual_doppler_hz,
        "background_noise_w": background_noise_w,
        "amplifier_gain_linear": amplifier_gain_linear,
        "ase_noise_w": ase_noise_w,
        "photodiode_quantum_efficiency": quantum_efficiency,
        "receiver_shot_noise_w": receiver_shot_noise_w,
        "complex_shot_noise_convention": config.heterodyne_shot_noise_convention,
    }


def _real_dtype(complex_dtype):
    if complex_dtype == torch.complex128:
        return torch.float64
    return torch.float32


def _complex_normal(shape, device, complex_dtype, generator=None):
    real_dtype = _real_dtype(complex_dtype)
    real = torch.randn(shape, device=device, dtype=real_dtype, generator=generator)
    imag = torch.randn(shape, device=device, dtype=real_dtype, generator=generator)
    return torch.complex(real, imag) / math.sqrt(2.0)


def _gamma_unit_mean(shape_parameter, shape, device, dtype, generator=None):
    concentration = torch.full(
        shape,
        float(shape_parameter),
        device=device,
        dtype=dtype,
    )
    # PyTorch distributions do not expose a generator argument. This native
    # primitive does and is available in the paper's PyTorch 2.0 runtime.
    return torch._standard_gamma(concentration, generator=generator) / float(
        shape_parameter
    )


class PaperMalagaFSOChannel:
    """Paper Eq. (3) channel with Eqs. (4)-(18) and ideal coherent DSP.

    This is a symbol-rate matched-filter equivalent. Block Malaga fading is
    sampled independently per image/receiver. Doppler and LO phase are applied
    before ideal CFR/CPR/equalization through the returned complex channel gain.
    """

    def __init__(self, config=None):
        self.config = config or PaperFSOSystemConfig()
        self._budget_cache = {}
        self._reproducibility_config_cache = None
        self._warned_infeasible_beta_fit = False

    def reproducibility_config(self):
        if self._reproducibility_config_cache is None:
            reference = compute_paper_link_budget(self.config)
            self._reproducibility_config_cache = {
                "channel_version": PAPER_FSO_CHANNEL_VERSION,
                "system": asdict(self.config),
                "undisclosed_assumptions": PAPER_UNDISCLOSED_RECEIVER_ASSUMPTIONS,
                "reference_beta_diagnostic": {
                    "beta": reference["malaga_beta"],
                    "source": reference["malaga_beta_source"],
                    "fit_feasible": reference["malaga_beta_fit_feasible"],
                    "target_second_moment": round(
                        reference["target_y_second_moment"], 12
                    ),
                    "fitted_second_moment": round(
                        reference["fitted_y_second_moment"], 12
                    ),
                    "relative_residual": round(
                        reference["malaga_beta_fit_relative_residual"], 12
                    ),
                },
            }
        return self._reproducibility_config_cache

    def _budget(self, **condition):
        key = tuple(sorted(condition.items()))
        if key not in self._budget_cache:
            self._budget_cache[key] = compute_paper_link_budget(
                self.config, **condition
            )
        budget = self._budget_cache[key]
        if (
            self.config.malaga_beta is None
            and self.config.beta_fit_policy == "nearest_warn"
            and not budget["malaga_beta_fit_feasible"]
            and not self._warned_infeasible_beta_fit
        ):
            warnings.warn(
                "The paper omits Malaga beta and phase, and Eq. (15) has no "
                "fit within the configured range/tolerance for this condition; "
                f"using beta={budget['malaga_beta']} with relative moment "
                f"residual={budget['malaga_beta_fit_relative_residual']:.3%}. "
                "Set an explicit malaga_beta or beta_fit_policy='strict'.",
                RuntimeWarning,
            )
            self._warned_infeasible_beta_fit = True
        return budget

    def sample_malaga_power(self, shape, budget, device, dtype, generator=None):
        if not self.config.include_turbulence:
            return torch.ones(shape, device=device, dtype=dtype)
        large_scale = _gamma_unit_mean(
            budget["malaga_alpha"], shape, device, dtype, generator
        )
        shadowing = _gamma_unit_mean(
            budget["malaga_beta"], shape, device, dtype, generator
        )
        rho = budget["scattered_power_ratio"]
        coherent_phasor = (
            math.sqrt(self.config.malaga_omega)
            + math.sqrt(2.0 * self.config.malaga_b0 * rho)
            * complex(
                math.cos(-self.config.malaga_phase_difference_rad),
                math.sin(-self.config.malaga_phase_difference_rad),
            )
        )
        diffuse = _complex_normal(
            shape,
            device,
            torch.complex128 if dtype == torch.float64 else torch.complex64,
            generator,
        ) * math.sqrt(budget["malaga_gamma"])
        small_field = shadowing.sqrt().to(diffuse.dtype) * coherent_phasor + diffuse
        small_power = small_field.abs().square().to(dtype)
        return large_scale * small_power / budget["malaga_mean_y"]

    @torch.no_grad()
    def __call__(
        self,
        symbols,
        launch_power_dbm=None,
        zenith_angle_deg=None,
        pointing_error_urad=None,
        scattered_power_ratio=None,
        background_light_radiance_w_m2_sr_um=None,
        weather=None,
        generator=None,
    ):
        if symbols.ndim != 2 or not torch.is_complex(symbols):
            raise ValueError("symbols must be complex baseband with shape [batch, symbols]")
        if generator is not None and not paper_generator_device_matches(
            generator, symbols.device
        ):
            raise ValueError("generator and symbols must use the same device")

        condition = {
            "launch_power_dbm": (
                self.config.launch_power_dbm
                if launch_power_dbm is None
                else float(launch_power_dbm)
            ),
            "zenith_angle_deg": (
                self.config.zenith_angle_deg
                if zenith_angle_deg is None
                else float(zenith_angle_deg)
            ),
            "pointing_error_urad": (
                self.config.pointing_error_urad
                if pointing_error_urad is None
                else float(pointing_error_urad)
            ),
            "scattered_power_ratio": (
                self.config.scattered_power_ratio
                if scattered_power_ratio is None
                else float(scattered_power_ratio)
            ),
            "background_light_radiance_w_m2_sr_um": (
                self.config.background_light_radiance_w_m2_sr_um
                if background_light_radiance_w_m2_sr_um is None
                else float(background_light_radiance_w_m2_sr_um)
            ),
            "weather": self.config.weather if weather is None else str(weather),
        }
        budget = self._budget(**condition)
        batch, symbol_count = symbols.shape
        receivers = self.config.num_receivers
        real_dtype = _real_dtype(symbols.dtype)
        fading_shape = (batch, receivers, 1)
        scintillation_power = self.sample_malaga_power(
            fading_shape,
            budget,
            symbols.device,
            real_dtype,
            generator,
        )

        attenuation_amplitude = 10.0 ** (-budget["total_attenuation_db"] / 20.0)
        # Assume the listed power is launched after the transmitter AMP; the paper omits its probe point.
        # Therefore its 30-dB gain must not be applied to the signal a second time.
        propagation_amplitude = attenuation_amplitude * scintillation_power.sqrt()

        time_index = torch.arange(
            symbol_count, device=symbols.device, dtype=real_dtype
        )
        phase = torch.zeros(
            (batch, receivers, symbol_count),
            device=symbols.device,
            dtype=real_dtype,
        )
        if self.config.include_doppler:
            phase = phase + (
                2.0
                * math.pi
                * budget["residual_doppler_hz"]
                / self.config.symbol_rate_baud
                * time_index
            ).view(1, 1, -1)
        if self.config.include_phase_noise and self.config.lo_linewidth_hz > 0:
            increment_std = math.sqrt(
                2.0
                * math.pi
                * self.config.lo_linewidth_hz
                / self.config.symbol_rate_baud
            )
            increments = torch.randn(
                phase.shape,
                device=symbols.device,
                dtype=real_dtype,
                generator=generator,
            ) * increment_std
            phase = phase + increments.cumsum(dim=-1)

        propagation_gain = propagation_amplitude.to(symbols.dtype) * torch.exp(
            1j * phase
        ).to(symbols.dtype)
        channel_gain = math.sqrt(budget["launch_power_w"]) * propagation_gain
        received = channel_gain * symbols[:, None, :]

        if self.config.include_background_noise and budget["background_noise_w"] > 0:
            received = received + _complex_normal(
                received.shape, symbols.device, symbols.dtype, generator
            ) * math.sqrt(budget["background_noise_w"])
        if self.config.include_shot_noise and budget["receiver_shot_noise_w"] > 0:
            received = received + _complex_normal(
                received.shape, symbols.device, symbols.dtype, generator
            ) * math.sqrt(budget["receiver_shot_noise_w"])
        if self.config.include_ase_noise and budget["ase_noise_w"] > 0:
            transmitter_ase = _complex_normal(
                (batch, 1, symbol_count),
                symbols.device,
                symbols.dtype,
                generator,
            ) * math.sqrt(budget["ase_noise_w"])
            received = received + propagation_gain * transmitter_ase

        # The transmitter-EDFA ASE is one common source waveform, so diversity
        # cannot average it away. Receiver background noise is branch-independent.
        equivalent_noise_variance = torch.zeros(
            (batch, 1), device=symbols.device, dtype=real_dtype
        )
        if self.config.include_ase_noise:
            equivalent_noise_variance = equivalent_noise_variance + (
                budget["ase_noise_w"] / budget["launch_power_w"]
            )
        receiver_independent_noise_w = (
            budget["background_noise_w"] if self.config.include_background_noise else 0.0
        ) + (
            budget["receiver_shot_noise_w"] if self.config.include_shot_noise else 0.0
        )
        if receiver_independent_noise_w > 0:
            if receivers == 1 or self.config.combining.upper() == "MRC":
                combined_signal_power = channel_gain[..., :1].abs().square().sum(dim=1)
                equivalent_noise_variance = equivalent_noise_variance + (
                    receiver_independent_noise_w
                    / combined_signal_power.clamp_min(torch.finfo(real_dtype).tiny)
                )
            else:
                combined_signal_amplitude = channel_gain[..., :1].abs().sum(dim=1)
                equivalent_noise_variance = equivalent_noise_variance + (
                    receivers
                    * receiver_independent_noise_w
                    / combined_signal_amplitude.square().clamp_min(
                        torch.finfo(real_dtype).tiny
                    )
                )
        if bool((equivalent_noise_variance > 0).any()):
            instantaneous_snr = equivalent_noise_variance.clamp_min(
                torch.finfo(real_dtype).tiny
            ).reciprocal()
            snr_linear_sum = float(instantaneous_snr.sum().item())
            snr_sample_count = int(instantaneous_snr.numel())
            mean_snr_linear = snr_linear_sum / max(snr_sample_count, 1)
            mean_snr_db = 10.0 * math.log10(max(mean_snr_linear, 1e-30))
        else:
            snr_linear_sum = float("inf")
            snr_sample_count = batch
            mean_snr_db = float("inf")
        diagnostics = {
            "channel_version": PAPER_FSO_CHANNEL_VERSION,
            "mean_snr_db": mean_snr_db,
            "snr_linear_sum": snr_linear_sum,
            "snr_sample_count": snr_sample_count,
            "num_receivers": receivers,
            "combining": self.config.combining.upper(),
            "budget": budget,
        }

        if receivers == 1:
            return FSOChannelOutput(
                symbols=received[:, 0, :],
                channel_gain=channel_gain[:, 0, :],
                is_equalized=False,
                diagnostics=diagnostics,
            )

        if self.config.combining.upper() == "MRC":
            denominator = (
                channel_gain.abs().square().sum(dim=1)
            ).clamp_min(torch.finfo(real_dtype).tiny)
            combined = (channel_gain.conj() * received).sum(dim=1) / denominator
        else:
            magnitudes = channel_gain.abs().clamp_min(1e-30)
            combined = (
                channel_gain.conj() / magnitudes * received
            ).sum(dim=1) / magnitudes.sum(dim=1).clamp_min(1e-30)
        return FSOChannelOutput(
            symbols=combined,
            is_equalized=True,
            diagnostics=diagnostics,
        )


@dataclass
class PaperFSOIndexChannel:
    receiver: object
    modulations: tuple = ("QPSK", "8QAM", "16QAM")
    bits_per_index: int = BITS_PER_INDEX

    @property
    def receiver_config(self):
        return self.receiver.reproducibility_config()

    def reproducibility_config(self):
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulations": tuple(name.upper() for name in self.modulations),
            "bits_per_index": self.bits_per_index,
            "receiver": self.receiver_config,
        }

    def __call__(
        self,
        indices,
        modulation="QPSK",
        return_statistics=False,
        **channel_kwargs,
    ):
        modulation = modulation.upper()
        if modulation not in tuple(name.upper() for name in self.modulations):
            raise ValueError(f"Unsupported paper modulation: {modulation}")
        link = QAMIndexLink(
            self.receiver,
            modulation=modulation,
            bits_per_index=self.bits_per_index,
            receiver_config=self.receiver_config,
        )
        return link(
            indices,
            return_statistics=return_statistics,
            **channel_kwargs,
        )


def paper_fig4_conditions():
    return [
        {
            "label": f"{modulation}, Pt={power} dBm, zenith={zenith} deg",
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": float(power),
                "zenith_angle_deg": float(zenith),
                "pointing_error_urad": 20.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for power in (30, 33, 36)
        for modulation in ("QPSK", "8QAM", "16QAM")
        for zenith in range(0, 81, 10)
    ]


def paper_fig6_pointing_conditions():
    return [
        {
            "label": (
                f"{modulation}, Pt=36 dBm, zenith={zenith} deg, "
                f"pointing={pointing} urad"
            ),
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": 36.0,
                "zenith_angle_deg": float(zenith),
                "pointing_error_urad": float(pointing),
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for modulation in ("QPSK", "8QAM", "16QAM")
        for zenith in (0, 30, 60)
        for pointing in range(18, 23)
    ]


def paper_fig6_background_conditions():
    return [
        {
            "label": (
                f"{modulation}, Pt={power} dBm, "
                f"BGL={radiance:g} W/m2/sr/um"
            ),
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 19.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": float(radiance),
                "weather": "clear",
            },
        }
        for modulation in ("QPSK", "8QAM", "16QAM")
        for radiance in (1e-4, 1e-2, 1.0)
        for power in range(28, 37)
    ]


def paper_fig7_weather_conditions():
    return [
        {
            "label": f"QPSK, Pt={power} dBm, weather={weather}",
            "channel_kwargs": {
                "modulation": "QPSK",
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 15.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": weather,
            },
        }
        for weather in ("clear", "dust", "rain", "snow", "fog")
        for power in range(28, 37)
    ]


def paper_fig8_turbulence_conditions():
    """Return the common FSO-SC rho/Pt sweep, not the BPG codec pipelines."""
    return [
        {
            "label": f"16QAM, Pt={power} dBm, rho={rho}",
            "channel_kwargs": {
                "modulation": "16QAM",
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 19.0,
                "scattered_power_ratio": float(rho),
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for rho in (0.95, 0.75, 0.596)
        for power in range(28, 37)
    ]


def paper_fig8_diversity_index_channels(base_config=None):
    """Expose the paper's 2/4-Rx EGC/MRC comparisons.

    The paper does not disclose inter-branch turbulence correlation, so the
    receiver implementation records its independent-block-fade assumption.
    """
    base_config = PaperFSOSystemConfig() if base_config is None else base_config
    channels = {
        "1Rx": PaperFSOIndexChannel(
            PaperMalagaFSOChannel(replace(base_config, num_receivers=1))
        )
    }
    for receivers in (2, 4):
        for combining in ("EGC", "MRC"):
            label = f"{receivers}Rx-{combining}"
            receiver_config = replace(
                base_config,
                num_receivers=receivers,
                combining=combining,
            )
            channels[label] = PaperFSOIndexChannel(
                PaperMalagaFSOChannel(receiver_config)
            )
    return channels


def paper_fso_equation_self_test():
    config = PaperFSOSystemConfig()
    reference = compute_paper_link_budget(
        config,
        launch_power_dbm=33.0,
        zenith_angle_deg=0.0,
        pointing_error_urad=20.0,
        weather="clear",
    )
    assert abs(reference["slant_range_m"] - 620e3) < 1e-6
    assert abs(reference["geometric_scintillation_db"] - 28.69) < 0.05
    assert abs(reference["pointing_db"] - 35.68) < 0.05
    assert abs(reference["atmospheric_db"] - 2.94) < 1e-9
    # The paper omits beta and the coherent phase. With the documented pi/2
    # phase, its Eq. (15) has no feasible beta here; this must stay explicit.
    assert reference["malaga_beta_fit_feasible"] is False
    assert reference["malaga_beta_fit_relative_residual"] > 0.0
    try:
        compute_paper_link_budget(
            PaperFSOSystemConfig(beta_fit_policy="strict"),
            zenith_angle_deg=0.0,
            scattered_power_ratio=0.95,
        )
    except ValueError:
        pass
    else:
        raise AssertionError("Strict beta fitting must reject the underdetermined fit")
    expected_ranges_km = {
        0: 620.000,
        20: 655.958,
        30: 705.628,
        60: 1107.733,
        80: 1977.252,
    }
    for zenith, expected_km in expected_ranges_km.items():
        actual_km = satellite_slant_range_m(zenith, config) / 1000.0
        assert abs(actual_km - expected_km) < 0.001
    expected_weather_db = {
        "clear": 2.94,
        "dust": 20.86,
        "rain": 28.06,
        "snow": 42.90,
        "fog": 271.832,
    }
    for weather, expected_db in expected_weather_db.items():
        weather_budget = compute_paper_link_budget(config, weather=weather)
        assert abs(weather_budget["atmospheric_db"] - expected_db) < 0.001
    peak = compute_paper_link_budget(config, zenith_angle_deg=20.0)
    high = compute_paper_link_budget(config, zenith_angle_deg=80.0)
    assert abs(peak["doppler_hz"]) > abs(high["doppler_hz"])

    convergence_reference = compute_paper_link_budget(
        PaperFSOSystemConfig(integration_points=32769),
        zenith_angle_deg=0.0,
        pointing_error_urad=20.0,
    )
    assert abs(
        reference["rytov_variance"] / convergence_reference["rytov_variance"] - 1.0
    ) < 5e-3
    assert abs(
        reference["malaga_alpha"] / convergence_reference["malaga_alpha"] - 1.0
    ) < 5e-3

    receiver = PaperMalagaFSOChannel(config)
    generator_a = torch.Generator().manual_seed(123)
    generator_b = torch.Generator().manual_seed(123)
    sample_a = receiver.sample_malaga_power(
        (4096, 1, 1), reference, torch.device("cpu"), torch.float32, generator_a
    )
    sample_b = receiver.sample_malaga_power(
        (4096, 1, 1), reference, torch.device("cpu"), torch.float32, generator_b
    )
    assert torch.equal(sample_a, sample_b)
    assert bool((sample_a >= 0).all()) and bool(torch.isfinite(sample_a).all())
    assert 0.9 < float(sample_a.mean()) < 1.1
    print(
        "Paper FSO Eq. (3)-(18) self-test passed: "
        f"asg={reference['geometric_scintillation_db']:.2f} dB, "
        f"ape={reference['pointing_db']:.2f} dB, "
        f"alpha={reference['malaga_alpha']:.2f}, beta={reference['malaga_beta']} "
        f"(Eq15 residual={reference['malaga_beta_fit_relative_residual']:.2%})"
    )


paper_fso_equation_self_test()


In [ ]:
"""Signal-estimated symbol-rate coherent carrier recovery for the FSO notebook.

Embed after ``qam_index_link`` and ``paper_fso_channel_v1``. This is a
plausible receiver implementation, NOT the undisclosed VPI DSP used by Chen
et al. It introduces no pilot symbols and never uses transmitted symbol values
or the simulated channel phase to recover the carrier. The default magnitude
mode retains explicitly assumed block-amplitude CSI; ``blind_rms`` is also
available, but its uniform-symbol assumption is unsuitable for strongly biased
semantic bitstreams.

Reference for feed-forward blind phase search: Pfau, Hoffmann, and Noe,
JLT 27(8), 989-999 (2009), doi:10.1109/JLT.2008.2010511. Fourth-power spectral
CFR is a conventional modulation-removal approximation, with its finite-frame
failure modes retained. Exact frame/timing/IQ calibration and an initial phase
branch near zero remain assumptions. No RRC/ADC waveform is synthesized here.
"""

from dataclasses import asdict, dataclass
import math

import torch
import torch.nn.functional as F


SIGNAL_DSP_VERSION = "fourth_power_fft_newton_bps_v1"
SIGNAL_DSP_REFERENCE = {
    "blind_phase_search": "https://doi.org/10.1109/JLT.2008.2010511",
    "scope": "symbol-rate, single receiver; no waveform/timing/IQ calibration model",
    "cfr": "fourth-power FFT peak plus continuous periodogram Newton refinement",
    "cpr": "windowed minimum-distance blind phase search and symmetry unwrapping",
    "phase_ambiguity": (
        "initial carrier phase lies in the branch nearest zero, consistent with "
        "the simulator phase origin; no transmitted data is used to select a branch"
    ),
    "no_pilots": True,
    "amplitude_default": "known block magnitude only, not the complex channel gain",
    "failure_policy": "keep estimates/errors; do not drop low-SNR frames",
}


@dataclass(frozen=True)
class SignalDSPConfig:
    amplitude_mode: str = "known_block_magnitude"
    cfo_fft_oversampling: int = 4
    cfo_newton_steps: int = 4
    bps_test_phases: int = 64
    bps_window_symbols: int = 65
    cfo_enabled: bool = True
    cpr_enabled: bool = True

    def __post_init__(self):
        if self.amplitude_mode not in {"known_block_magnitude", "blind_rms"}:
            raise ValueError("Unsupported amplitude_mode")
        if self.cfo_fft_oversampling < 1 or self.cfo_newton_steps < 0:
            raise ValueError("Invalid CFR FFT/refinement configuration")
        if self.bps_test_phases < 8:
            raise ValueError("Use at least eight BPS trial phases")
        if self.bps_window_symbols < 1 or self.bps_window_symbols % 2 != 1:
            raise ValueError("BPS window must be a positive odd integer")


def _qam_axis_levels(modulation, device, dtype):
    modulation = modulation.upper()
    if modulation == "QPSK":
        i = q = torch.tensor([-1.0, 1.0], device=device, dtype=dtype) / math.sqrt(2)
        period = math.pi / 2
    elif modulation == "16QAM":
        i = q = torch.tensor([-3.0, -1.0, 1.0, 3.0], device=device, dtype=dtype) / math.sqrt(10)
        period = math.pi / 2
    elif modulation == "8QAM":
        # Rectangular geometry in qam_constellation, NOT square-QAM symmetry.
        i = torch.tensor([-3.0, -1.0, 1.0, 3.0], device=device, dtype=dtype) / math.sqrt(6)
        q = torch.tensor([-1.0, 1.0], device=device, dtype=dtype) / math.sqrt(6)
        period = math.pi
    else:
        raise ValueError(f"Unsupported modulation: {modulation}")
    return i, q, period


def _estimate_cfo_fourth_power(observed, symbol_rate_baud, config):
    batch, count = observed.shape
    # The fourth harmonic repeats at Fs/4, so the unambiguous CFO interval is
    # [-Fs/8, Fs/8). Paper residual offsets are within that acquisition range.
    nfft = 1 << max(1, (count * config.cfo_fft_oversampling - 1).bit_length())
    # Severe fog can make gain-normalized receiver noise enormous. Remove a
    # positive block scale before the fourth power; it does not change the
    # frequency peak and prevents overflow in frames whose data is unrecoverable.
    peak_scale = observed.abs().amax(dim=-1, keepdim=True).clamp_min(
        torch.finfo(observed.real.dtype).tiny
    )
    cfr_samples = observed / peak_scale
    fourth = cfr_samples.square().square()
    spectrum = torch.fft.fft(fourth, n=nfft, dim=-1)
    power = spectrum.abs().square()
    peak = power.argmax(dim=-1)
    signed_peak = torch.where(peak > nfft // 2, peak - nfft, peak)
    real_dtype = observed.real.dtype
    omega = signed_peak.to(real_dtype) * (2.0 * math.pi / nfft)
    index = torch.arange(count, device=observed.device, dtype=real_dtype)
    centered = index - (count - 1) / 2.0
    max_step = 2.0 * math.pi / nfft
    for _ in range(config.cfo_newton_steps):
        derotated = fourth * torch.exp(-1j * omega[:, None] * centered[None, :])
        value = derotated.sum(dim=-1)
        derivative = (-1j * centered[None, :] * derotated).sum(dim=-1)
        curvature = (-centered.square()[None, :] * derotated).sum(dim=-1)
        slope = 2 * (derivative * value.conj()).real
        second = 2 * ((curvature * value.conj()).real + derivative.abs().square())
        stable = second < -torch.finfo(real_dtype).eps
        denominator = torch.where(stable, second, torch.ones_like(second))
        step = torch.where(stable, slope / denominator, torch.zeros_like(slope))
        omega = omega - step.clamp(min=-max_step, max=max_step)
    omega = torch.remainder(omega + math.pi, 2 * math.pi) - math.pi
    estimated_hz = omega * symbol_rate_baud / (8 * math.pi)
    phase = omega[:, None] * index[None, :] / 4.0
    compensated = observed * torch.exp(-1j * phase)
    peak_ratio = power.max(dim=-1).values / power.mean(dim=-1).clamp_min(
        torch.finfo(real_dtype).tiny
    )
    return compensated, estimated_hz, peak_ratio


def _unwrap_symmetry_phase(phase, period):
    differences = torch.diff(phase, dim=-1)
    wrapped = torch.remainder(differences + period / 2, period) - period / 2
    corrections = wrapped - differences
    cumulative = torch.cat(
        [torch.zeros_like(phase[:, :1]), corrections.cumsum(dim=-1)], dim=-1
    )
    return phase + cumulative, (corrections.abs() > period / 2).sum(dim=-1)


def _blind_phase_search(observed, modulation, config):
    i_levels, q_levels, period = _qam_axis_levels(
        modulation, observed.device, observed.real.dtype
    )
    trial_phases = (
        torch.arange(config.bps_test_phases, device=observed.device, dtype=observed.real.dtype)
        / config.bps_test_phases - 0.5
    ) * period
    rotated = observed[:, None, :] * torch.exp(-1j * trial_phases[None, :, None])
    i_distance = (rotated.real[..., None] - i_levels).square().amin(dim=-1)
    q_distance = (rotated.imag[..., None] - q_levels).square().amin(dim=-1)
    costs = F.avg_pool1d(
        i_distance + q_distance,
        kernel_size=config.bps_window_symbols,
        stride=1,
        padding=config.bps_window_symbols // 2,
        count_include_pad=False,
    )
    selected = trial_phases[costs.argmin(dim=1)]
    phase, unwrap_events = _unwrap_symmetry_phase(selected, period)
    # Global rotational ambiguity is not solved using the source symbols. The
    # initial branch near zero is an explicit simulator convention, not a pilot.
    phase = phase - torch.round(phase[:, :1] / period) * period
    corrected = observed * torch.exp(-1j * phase)
    return corrected, phase, unwrap_events


@torch.no_grad()
def recover_qam_carrier(
    received,
    modulation,
    symbol_rate_baud,
    config=None,
    known_block_magnitude=None,
):
    """Estimate CFR/CPR from received samples, without source data/true phase.

    ``known_block_magnitude`` may contain positive block-amplitude CSI only.
    It must not contain the true phase trajectory or the transmitted symbols.
    """
    config = SignalDSPConfig() if config is None else config
    if received.ndim != 2 or not torch.is_complex(received) or received.shape[-1] < 8:
        raise ValueError("Expected complex [batch, symbols>=8] receiver samples")
    if symbol_rate_baud <= 0 or not torch.isfinite(received).all():
        raise ValueError("Invalid receiver samples or symbol rate")
    if config.amplitude_mode == "known_block_magnitude":
        if known_block_magnitude is None or torch.is_complex(known_block_magnitude):
            raise ValueError("Positive REAL block-amplitude CSI is required")
        if not torch.isfinite(known_block_magnitude).all() or (known_block_magnitude <= 0).any():
            raise ValueError("Invalid block-amplitude CSI")
        amplitude = torch.broadcast_to(known_block_magnitude, (received.shape[0], 1))
    else:
        amplitude = received.abs().square().mean(dim=-1, keepdim=True).sqrt()
        amplitude = amplitude.clamp_min(torch.finfo(received.real.dtype).tiny)
    observed = received / amplitude
    zeros = torch.zeros(received.shape[0], device=received.device, dtype=received.real.dtype)
    if config.cfo_enabled:
        observed, cfo_hz, peak_ratio = _estimate_cfo_fourth_power(
            observed, symbol_rate_baud, config
        )
    else:
        cfo_hz, peak_ratio = zeros, zeros
    if config.cpr_enabled:
        observed, phase, unwrap_events = _blind_phase_search(observed, modulation, config)
    else:
        phase = torch.zeros_like(received.real)
        unwrap_events = zeros.to(torch.int64)
    diagnostics = {
        "receiver_version": SIGNAL_DSP_VERSION,
        "amplitude_mode": config.amplitude_mode,
        "estimated_cfo_hz": cfo_hz.detach().cpu().tolist(),
        "cfr_peak_to_mean_power": peak_ratio.detach().cpu().tolist(),
        "cfr_unambiguous_interval_hz": [-symbol_rate_baud / 8, symbol_rate_baud / 8],
        "estimated_phase_start_rad": phase[:, 0].detach().cpu().tolist(),
        "estimated_phase_end_rad": phase[:, -1].detach().cpu().tolist(),
        "phase_symmetry_unwrap_events": unwrap_events.detach().cpu().tolist(),
        "failure_policy": "estimates/errors retained, no frame rejection",
    }
    return observed, diagnostics


@dataclass
class SignalDSPFSOReceiver:
    """Replace phase-oracle equalization for one physical receiver branch."""

    physical_receiver: object
    modulation: str = "QPSK"
    dsp_config: object = None

    def __post_init__(self):
        if self.dsp_config is None:
            self.dsp_config = SignalDSPConfig()
        if self.physical_receiver.config.num_receivers != 1:
            raise ValueError("Signal DSP wrapper supports single Rx only; no oracle MRC")

    def reproducibility_config(self):
        return {
            "physical_receiver": self.physical_receiver.reproducibility_config(),
            "receiver_version": SIGNAL_DSP_VERSION,
            "receiver_config": asdict(self.dsp_config),
            "receiver_assumptions": SIGNAL_DSP_REFERENCE,
            "modulation": self.modulation.upper(),
        }

    @torch.no_grad()
    def __call__(self, symbols, **channel_kwargs):
        observation = self.physical_receiver(symbols, **channel_kwargs)
        if observation.is_equalized:
            raise ValueError("Expected un-equalized physical receiver observation")
        magnitude = None
        if self.dsp_config.amplitude_mode == "known_block_magnitude":
            # Deliberately read only real block magnitude, NOT its phase. Do
            # not pass complex channel_gain to recover_qam_carrier.
            magnitude = observation.channel_gain.abs().mean(dim=-1, keepdim=True)
        recovered, receiver_diagnostics = recover_qam_carrier(
            observation.symbols,
            self.modulation,
            self.physical_receiver.config.symbol_rate_baud,
            self.dsp_config,
            known_block_magnitude=magnitude,
        )
        diagnostics = dict(observation.diagnostics or {})
        diagnostics["signal_dsp"] = receiver_diagnostics
        return FSOChannelOutput(symbols=recovered, is_equalized=True, diagnostics=diagnostics)


@dataclass
class SignalDSPPaperIndexChannel:
    """Drop-in single-Rx alternative to PaperFSOIndexChannel in this notebook."""

    receiver: object
    dsp_config: object = None
    modulations: tuple = ("QPSK", "8QAM", "16QAM")
    bits_per_index: int = 14

    def __post_init__(self):
        if self.dsp_config is None:
            self.dsp_config = SignalDSPConfig()
        if self.receiver.config.num_receivers != 1:
            raise ValueError("Estimated carrier receiver is currently single-Rx only")

    @property
    def receiver_config(self):
        return {
            "physical_receiver": self.receiver.reproducibility_config(),
            "receiver_version": SIGNAL_DSP_VERSION,
            "receiver_config": asdict(self.dsp_config),
            "receiver_assumptions": SIGNAL_DSP_REFERENCE,
        }

    def reproducibility_config(self):
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulations": tuple(name.upper() for name in self.modulations),
            "bits_per_index": self.bits_per_index,
            "receiver": self.receiver_config,
        }

    def __call__(self, indices, modulation="QPSK", return_statistics=False, **channel_kwargs):
        modulation = modulation.upper()
        if modulation not in tuple(name.upper() for name in self.modulations):
            raise ValueError(f"Unsupported modulation: {modulation}")
        receiver = SignalDSPFSOReceiver(self.receiver, modulation, self.dsp_config)
        link = QAMIndexLink(
            receiver,
            modulation=modulation,
            bits_per_index=self.bits_per_index,
            receiver_config=receiver.reproducibility_config(),
        )
        return link(indices, return_statistics=return_statistics, **channel_kwargs)


def signal_dsp_offline_self_test(device="cpu"):
    """Synthetic local checks only; does not download weights or run training."""
    device = torch.device(device)
    generator = torch.Generator(device=device).manual_seed(71819)
    count = 4096
    sample_rate = 28e9
    time = torch.arange(count, device=device) / sample_rate
    records = {}
    for modulation in ("QPSK", "8QAM", "16QAM"):
        points, _, _ = qam_constellation(modulation, device)
        ids = torch.randint(len(points), (2, count), generator=generator, device=device)
        source = points[ids]
        true_cfo = torch.tensor([700e6, -320e6], device=device)
        phase = 0.03 + 2 * math.pi * true_cfo[:, None] * time[None, :]
        amplitude = torch.tensor([[0.002], [0.0005]], device=device)
        noise = torch.complex(
            torch.randn(source.shape, generator=generator, device=device),
            torch.randn(source.shape, generator=generator, device=device),
        ) * (0.005 / math.sqrt(2))
        received = amplitude * (source * torch.exp(1j * phase) + noise)
        recovered, stats = recover_qam_carrier(
            received, modulation, sample_rate, known_block_magnitude=amplitude
        )
        recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
        errors = int((recovered_ids != ids).sum())
        estimated_cfo = torch.tensor(stats["estimated_cfo_hz"], device=device)
        assert errors == 0, (modulation, errors, stats)
        # Finite-frame fourth-moment self-noise is nonzero for QAM. Require
        # error below one 4th-harmonic FFT bin; BPS must still give exact data.
        assert (estimated_cfo - true_cfo).abs().max() < sample_rate / (4 * count), stats
        records[modulation] = {"symbol_errors": errors, "estimated_cfo_hz": stats["estimated_cfo_hz"]}

    # Nonzero phase trajectory must be inferred and unwrapped, not divided out
    # with a hidden simulator phase. Deliberately turn CFR off for this check.
    points, _, _ = qam_constellation("QPSK", device)
    ids = torch.randint(len(points), (1, count), generator=generator, device=device)
    source = points[ids]
    phase = torch.linspace(0, 3, count, device=device)[None, :]
    recovered, stats = recover_qam_carrier(
        source * torch.exp(1j * phase), "QPSK", sample_rate,
        SignalDSPConfig(cfo_enabled=False), torch.ones((1, 1), device=device),
    )
    recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
    assert torch.equal(ids, recovered_ids), stats
    assert stats["phase_symmetry_unwrap_events"][0] >= 1

    # A global pi/2 ambiguity is intentionally NOT resolved using source data.
    received = source * torch.exp(torch.tensor(1j * math.pi / 2, device=device))
    recovered, _ = recover_qam_carrier(
        received, "QPSK", sample_rate,
        known_block_magnitude=torch.ones((1, 1), device=device),
    )
    recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
    assert not torch.equal(ids, recovered_ids), "Undisclosed source-aided phase ambiguity resolution"
    print("Signal-estimated CFR/CPR offline self-test passed:", records)
    return records


In [ ]:
"""Received-sample-only rectangular-8QAM carrier acquisition repair.

Append this source after ``fso_sc_signal_dsp.py`` in a notebook.  Standalone
imports are also supported.  QPSK and 16QAM dispatch to the unmodified v1
receiver, and the existing SignalDSPConfig dataclass is not changed.

The fourth-moment periodogram is only a coarse frequency estimator: a
nonuniform finite payload can bias its peak even without receiver noise.
Rectangular 8QAM also has *data-dependent* phase aliases on constant-amplitude
runs; these are not the constellation's pi rotational symmetry.  Independent
BPS windows can therefore jump between aliases on zero-padded coded data.

The repair refines frequency AND constant phase by the AWGN constellation
likelihood, then uses windowed decision-directed phase maximum likelihood
inside the acquired basin.  For fixed decisions that phase estimate is
arg(sum(y * conj(decision))).  No source bits, channel phase, codec state,
parity checks, or image-quality metric enters this receiver.  The initial
phase-branch-near-zero assumption and known real block magnitude are retained
from v1; fundamental blind phase ambiguity is not claimed to be solved.

Related work: Yan et al., JLT 37, 5856-5866 (2019) document nonuniform-QAM
failure of conventional fourth-power frequency estimation.  Meiyappan et al.,
JLT 31, 2055-2069 (2013) study decision-aided ML phase/frequency recovery.
These motivate the estimator family, not a claim that this simple local
likelihood search reproduces either paper's particular algorithm.
"""

import math

import torch
import torch.nn.functional as F


if "recover_qam_carrier" not in globals():
    pass  # already embedded above
_rect8_legacy_recover_qam_carrier = globals().get(
    "_rect8_legacy_recover_qam_carrier", recover_qam_carrier
)


RECT8_DSP_VERSION = "rect8_constellation_ml_cfr_dd_ml_cpr_v2"
RECT8_DSP_REFERENCE = {
    "affected_modulation": "8QAM",
    "scope": "rectangular 8QAM only; QPSK and 16QAM use unchanged v1",
    "coarse_cfr": "v1 fourth-power FFT and Newton periodogram refinement",
    "fine_cfr": "received-sample constellation-distance likelihood; frequency and constant phase jointly fitted",
    "cpr": "two windowed decision-directed ML phase updates, arg(sum(y*conj(decision)))",
    "acquisition_search": "one unoversampled fourth-harmonic CFO bin either side of the coarse estimate",
    "acquisition_samples": "at most 2048 uniformly spaced received samples; no data-dependent sampling",
    "fine_frequency_trials": 17,
    "fine_newton_steps": 5,
    "phase_trials_and_window": "unchanged SignalDSPConfig phase count and window length",
    "phase_ambiguity": "initial phase branch nearest zero; unidentifiable likelihood ties use that convention",
    "no_pilots": True,
    "uses_transmitted_bits_or_symbols": False,
    "uses_true_channel_phase": False,
    "uses_decoder_or_codec_feedback": False,
    "amplitude_default": "known positive real block magnitude only, as in v1",
    "failure_policy": "estimates and errors retained; no frame rejection",
    "legacy_bps_reference": "https://doi.org/10.1109/JLT.2008.2010511",
    "nonuniform_qam_cfr_related_work": "https://opg.optica.org/jlt/abstract.cfm?uri=jlt-37-23-5856",
    "decision_aided_ml_related_work": "https://opg.optica.org/jlt/abstract.cfm?uri=jlt-31-13-2055",
    "related_work_scope": "estimator-family motivation only; neither paper's exact algorithm is reproduced",
}


def rect8_repair_metadata():
    """JSON-safe provenance for the notebook's additional receiver stage."""
    return {"receiver_version": RECT8_DSP_VERSION, **RECT8_DSP_REFERENCE}


def _rect8_nearest_point(value):
    """Exact nearest point for I={-3,-1,1,3}/sqrt(6), Q={-1,1}/sqrt(6)."""
    root = math.sqrt(6.0)
    i = (2 * ((value.real * root + 3) / 2).round().clamp(0, 3) - 3) / root
    q = (2 * (value.imag >= 0).to(value.real.dtype) - 1) / root
    return torch.complex(i, q)


def _rect8_joint_likelihood_acquisition(observed, symbol_rate_baud, config, fit_frequency):
    """Local frequency acquisition with continuous constant-phase refinement.

    A fourth-harmonic Fourier bin corresponds to Fs/(4*N) carrier Hz, or
    pi/2 radians of end-to-end carrier phase.  This fixes the search width
    from the acquisition resolution, not from any image/BER experiment.
    Equally likely phase solutions are resolved by the declared initial
    near-zero phase convention, with a floating-point roundoff tolerance.
    """
    batch, count = observed.shape
    dtype, device = observed.real.dtype, observed.device
    sample_indices = torch.linspace(0, count - 1, min(count, 2048), device=device).round().long()
    sample_time = sample_indices.to(dtype) / (count - 1) - 0.5
    initial_phases = (torch.arange(config.bps_test_phases, device=device, dtype=dtype)
                      / config.bps_test_phases - 0.5) * math.pi
    slopes = (torch.linspace(-math.pi / 2, math.pi / 2, 17, device=device, dtype=dtype)
              if fit_frequency else torch.zeros(1, device=device, dtype=dtype))
    fitted_hz, fitted_intercept, fitted_cost = [], [], []
    for row in observed:
        sample = row[sample_indices]
        scale = sample.abs().amax().clamp_min(1.0)
        phase_candidates, slope_candidates = [], []
        # Bound temporary memory independently of frame/batch length.
        for phase_chunk in initial_phases.split(16):
            angle = phase_chunk[:, None, None] + slopes[None, :, None] * sample_time
            rotated = sample * torch.exp(-1j * angle)
            costs = ((rotated - _rect8_nearest_point(rotated)) / scale).abs().square().mean(-1)
            phase_candidates.append(phase_chunk)
            slope_candidates.append(slopes[costs.argmin(-1)])
        phase = torch.cat(phase_candidates)
        slope = torch.cat(slope_candidates)
        # Newton optimization for the local piecewise-smooth likelihood.
        # Keeping all phase seeds also retains data-dependent ambiguous basins.
        for _ in range(5):
            rotated = sample * torch.exp(-1j * (phase[:, None] + slope[:, None] * sample_time))
            decisions = _rect8_nearest_point(rotated)
            cross = (rotated / scale) * decisions.conj()
            h00 = cross.real.mean(-1)
            g0 = cross.imag.mean(-1)
            if fit_frequency:
                h01 = (cross.real * sample_time).mean(-1)
                h11 = (cross.real * sample_time.square()).mean(-1)
                g1 = (cross.imag * sample_time).mean(-1)
                determinant = h00 * h11 - h01.square()
                stable = (h00 > 0) & (determinant > torch.finfo(dtype).eps * (h00 * h11).abs())
                denominator = torch.where(stable, determinant, torch.ones_like(determinant))
                phase_step = torch.where(stable, (h11 * g0 - h01 * g1) / denominator, 0.0)
                slope_step = torch.where(stable, (h00 * g1 - h01 * g0) / denominator, 0.0)
            else:
                stable = h00 > torch.finfo(dtype).tiny
                phase_step = torch.where(stable, g0 / h00.clamp_min(torch.finfo(dtype).tiny), 0.0)
                slope_step = torch.zeros_like(slope)
            phase = phase + phase_step.clamp(-math.pi / 8, math.pi / 8)
            slope = (slope + slope_step.clamp(-math.pi / 4, math.pi / 4)).clamp(-math.pi / 2, math.pi / 2)
        rotated = sample * torch.exp(-1j * (phase[:, None] + slope[:, None] * sample_time))
        costs = ((rotated - _rect8_nearest_point(rotated)) / scale).abs().square().mean(-1)
        intercept = torch.remainder(phase - slope / 2 + math.pi / 2, math.pi) - math.pi / 2
        tolerance = 32 * torch.finfo(dtype).eps
        eligible = costs <= costs.amin() + tolerance
        selection = torch.where(eligible, intercept.abs(), float("inf")).argmin()
        fitted_hz.append(slope[selection] * symbol_rate_baud / (2 * math.pi * (count - 1)))
        fitted_intercept.append(intercept[selection])
        fitted_cost.append(costs[selection])
    frequency = torch.stack(fitted_hz)
    intercept = torch.stack(fitted_intercept)
    index = torch.arange(count, device=device, dtype=dtype)
    phase = intercept[:, None] + frequency[:, None] * (2 * math.pi / symbol_rate_baud) * index
    return observed * torch.exp(-1j * phase), frequency, intercept, torch.stack(fitted_cost)


def _rect8_decision_directed_phase(observed, config):
    """Windowed constant-phase ML, initialized inside the acquired basin."""
    corrected = observed
    scale = observed.abs().amax(-1, keepdim=True).clamp_min(1.0)
    phase = torch.zeros_like(observed.real)
    unwrap_events = torch.zeros(observed.shape[0], device=observed.device, dtype=torch.int64)
    for _ in range(2):
        decisions = _rect8_nearest_point(corrected)
        cross = (observed / scale) * decisions.conj()
        kwargs = dict(kernel_size=config.bps_window_symbols, stride=1,
                      padding=config.bps_window_symbols // 2, count_include_pad=False)
        average = torch.complex(F.avg_pool1d(cross.real[:, None], **kwargs)[:, 0],
                                F.avg_pool1d(cross.imag[:, None], **kwargs)[:, 0])
        phase, unwrap_events = _unwrap_symmetry_phase(average.angle(), 2 * math.pi)
        corrected = observed * torch.exp(-1j * phase)
    return corrected, phase, unwrap_events


@torch.no_grad()
def recover_qam_carrier(received, modulation, symbol_rate_baud, config=None, known_block_magnitude=None):
    """V1-compatible API; only rectangular-8QAM takes the repaired path."""
    if modulation.upper() != "8QAM":
        return _rect8_legacy_recover_qam_carrier(
            received, modulation, symbol_rate_baud, config, known_block_magnitude
        )
    config = SignalDSPConfig() if config is None else config
    if received.ndim != 2 or not torch.is_complex(received) or received.shape[-1] < 8:
        raise ValueError("Expected complex [batch, symbols>=8] receiver samples")
    if symbol_rate_baud <= 0 or not torch.isfinite(received).all():
        raise ValueError("Invalid receiver samples or symbol rate")
    if config.amplitude_mode == "known_block_magnitude":
        if known_block_magnitude is None or torch.is_complex(known_block_magnitude):
            raise ValueError("Positive REAL block-amplitude CSI is required")
        if not torch.isfinite(known_block_magnitude).all() or (known_block_magnitude <= 0).any():
            raise ValueError("Invalid block-amplitude CSI")
        amplitude = torch.broadcast_to(known_block_magnitude, (received.shape[0], 1))
    else:
        amplitude = received.abs().square().mean(-1, keepdim=True).sqrt()
        amplitude = amplitude.clamp_min(torch.finfo(received.real.dtype).tiny)
    observed = received / amplitude
    zeros = torch.zeros(received.shape[0], device=received.device, dtype=received.real.dtype)
    if config.cfo_enabled:
        observed, coarse_hz, peak_ratio = _estimate_cfo_fourth_power(observed, symbol_rate_baud, config)
    else:
        coarse_hz, peak_ratio = zeros, zeros
    if config.cfo_enabled or config.cpr_enabled:
        acquired, fine_hz, initial_phase, cost = _rect8_joint_likelihood_acquisition(
            observed, symbol_rate_baud, config, config.cfo_enabled
        )
        if config.cpr_enabled:
            observed = acquired
        else:
            index = torch.arange(received.shape[-1], device=received.device, dtype=received.real.dtype)
            observed = observed * torch.exp(-1j * fine_hz[:, None] * (2 * math.pi / symbol_rate_baud) * index)
    else:
        fine_hz, initial_phase, cost = zeros, zeros, zeros
    if config.cpr_enabled:
        observed, phase, unwrap_events = _rect8_decision_directed_phase(observed, config)
        phase = phase + initial_phase[:, None]
    else:
        phase = torch.zeros_like(received.real)
        unwrap_events = zeros.to(torch.int64)
    diagnostics = {
        "receiver_version": RECT8_DSP_VERSION,
        "amplitude_mode": config.amplitude_mode,
        "estimated_cfo_hz": (coarse_hz + fine_hz).detach().cpu().tolist(),
        "coarse_cfo_hz": coarse_hz.detach().cpu().tolist(),
        "fine_cfo_correction_hz": fine_hz.detach().cpu().tolist(),
        "acquisition_normalized_distance": cost.detach().cpu().tolist(),
        "cfr_peak_to_mean_power": peak_ratio.detach().cpu().tolist(),
        "cfr_unambiguous_interval_hz": [-symbol_rate_baud / 8, symbol_rate_baud / 8],
        "estimated_phase_start_rad": phase[:, 0].detach().cpu().tolist(),
        "estimated_phase_end_rad": phase[:, -1].detach().cpu().tolist(),
        "phase_symmetry_unwrap_events": unwrap_events.detach().cpu().tolist(),
        "failure_policy": "estimates/errors retained, no frame rejection",
    }
    return observed, diagnostics


def rect8_repair_self_test(device="cpu"):
    """Small synthetic preflight; no model, image, codec, or external data."""
    device = torch.device(device)
    generator = torch.Generator(device=device).manual_seed(20261028)
    count, sample_rate = 7168, 28e9
    ids = torch.randint(8, (1, count), generator=generator, device=device)
    ids[:, 5000:] = 0
    i = torch.tensor([-3, -3, -1, -1, 3, 3, 1, 1], device=device) / math.sqrt(6)
    q = torch.tensor([-1, 1, -1, 1, -1, 1, -1, 1], device=device) / math.sqrt(6)
    points = torch.complex(i, q)
    source = points[ids]
    index = torch.arange(count, device=device, dtype=source.real.dtype)
    records = []
    for cfo in (0.0, 700e6):
        received = source * torch.exp(1j * (0.03 + 2 * math.pi * cfo / sample_rate * index))
        recovered, diagnostics = recover_qam_carrier(
            received, "8QAM", sample_rate, known_block_magnitude=torch.ones((1, 1), device=device)
        )
        errors = int(((recovered[..., None] - points).abs().argmin(-1) != ids).sum())
        if errors:
            raise AssertionError(("rect8 preflight failed", cfo, errors, diagnostics))
        records.append({"cfo_hz": cfo, "symbol_errors": errors,
                        "estimated_cfo_hz": diagnostics["estimated_cfo_hz"],
                        "acquisition_normalized_distance": diagnostics["acquisition_normalized_distance"]})
    return {"receiver_version": RECT8_DSP_VERSION, "synthetic_only": True,
            "symbols_per_condition": count, "zero_tail_symbols": 2168, "conditions": records}


In [ ]:
"""Explicit noise interpretations; embeddable after the legacy channel namespace.

No training/model changes and no default profile. These are sensitivity profiles,
not settings selected to match the published LPIPS curve or a full VPI replica.
"""

import math


_PROFILE_FLAGS = {
    "paper_eq3_4_background_only": (True, False, False),
    "coherent_receiver_extension_v6": (True, True, True),
    "background_plus_ase_ablation": (True, True, False),
}
PROFILE_NAMES = tuple(_PROFILE_FLAGS)
_NOISE_FLAGS = (
    "include_background_noise", "include_ase_noise", "include_shot_noise"
)


def _profile_flags(name):
    if name not in _PROFILE_FLAGS:
        raise ValueError(f"Unknown explicit channel profile {name!r}; choose {PROFILE_NAMES}")
    return dict(zip(_NOISE_FLAGS, _PROFILE_FLAGS[name]))


def make_channel_profile(name, config_class, **overrides):
    """Construct the caller's config class; noise flags cannot contradict its label."""
    flags = _profile_flags(name)
    protected = set(overrides).intersection(_NOISE_FLAGS)
    if protected:
        raise ValueError(f"Profile noise flags cannot be overridden: {sorted(protected)}")
    return config_class(**flags, **overrides)


def profile_metadata(name):
    """Return fresh JSON-safe provenance, including what this profile cannot claim."""
    flags = _profile_flags(name)
    basis = {
        "paper_eq3_4_background_only": (
            "Chen et al. Eq. (3) defines n(t) as background-light noise; Eq. (4) "
            "gives its collected optical power. No receiver shot or ASE term is "
            "printed in those equations."
        ),
        "coherent_receiver_extension_v6": (
            "Unchanged v6 interpretation: background plus propagated transmitter "
            "EDFA ASE and input-referred single-polarization heterodyne shot noise."
        ),
        "background_plus_ase_ablation": (
            "Diagnostic ablation only: retain v6 background and transmitter ASE "
            "but omit its receiver shot term. Not an independently published model."
        ),
    }
    return {
        "profile": name,
        "noise_flags": flags,
        "basis": basis[name],
        "diagnostic_only": name == "background_plus_ase_ablation",
        "full_vpi_reproduction": False,
        "physical_superiority_claim": False,
        "curve_fitted": False,
        "limitations": [
            "Eq. (4) optical background power to complex matched-filter variance "
            "is an implementation approximation, not a disclosed receiver transfer.",
            "FOV unit, optical filter width, Malaga phase/beta, fading coherence "
            "and proprietary coherent receiver/DSP details remain assumptions.",
            "Background-only omission of quantum shot noise is an equation-level "
            "idealization, not a physically complete coherent receiver.",
            "ASE bandwidth/post-EDFA launch reference and shot bandwidth/polarization "
            "are inferred; neither establishes the undisclosed VPI configuration.",
        ],
        "provenance": {
            "paper": "https://ieeexplore.ieee.org/document/10969992/",
            "published_channel": "Chen et al., Sec. II-B, Eqs. (3)-(4); Table I",
            "v6_shot_reference": "https://doi.org/10.1103/PhysRevA.77.032116",
            "selection_rule": "Choose explicitly before examining evaluation outcomes.",
        },
    }


def _positive_power_dbm(power_w):
    return 10.0 * math.log10(power_w) + 30.0 if 0.0 < power_w < math.inf else None


def summarize_channel_budget(config, condition, budget_function):
    """Scalar no-fade, single-branch Es/N0; excludes fading and DSP errors.

    Powers are in W; the active noise terms denote E|n|^2 at the receiver.
    ASE is propagated through the same power attenuation as the signal. Raw
    formula outputs remain visible even for disabled noise terms. No Torch is
    imported; pass the existing compute_paper_link_budget function explicitly.
    """
    flags = {key: bool(getattr(config, key)) for key in _NOISE_FLAGS}
    matches = [name for name in PROFILE_NAMES if _profile_flags(name) == flags]
    if len(matches) != 1:
        raise ValueError("Config noise flags do not match an explicit channel profile")
    budget = budget_function(config, **dict(condition))
    attenuation = float(budget["total_attenuation_db"])
    launch = float(budget["launch_power_w"])
    if not math.isfinite(attenuation) or not math.isfinite(launch) or launch < 0.0:
        raise ValueError("Attenuation and nonnegative launch power must be finite")
    transmission = 10.0 ** (-attenuation / 10.0)
    signal = launch * transmission
    computed = {
        "background": float(budget["background_noise_w"]),
        "transmitter_ase": float(budget["ase_noise_w"]),
        "receiver_shot": float(budget["receiver_shot_noise_w"]),
    }
    if any(not math.isfinite(value) or value < 0.0 for value in computed.values()):
        raise ValueError("Computed noise powers must be finite and nonnegative")
    active = {
        "background": computed["background"] if flags[_NOISE_FLAGS[0]] else 0.0,
        "transmitter_ase": computed["transmitter_ase"] * transmission
        if flags[_NOISE_FLAGS[1]] else 0.0,
        "receiver_shot": computed["receiver_shot"] if flags[_NOISE_FLAGS[2]] else 0.0,
    }
    active["total"] = sum(active.values())
    snr = signal / active["total"] if active["total"] > 0.0 else None
    snr = snr if snr is None or math.isfinite(snr) else None
    snr_db = 10.0 * math.log10(snr) if snr is not None and snr > 0.0 else None
    return {
        "profile": matches[0],
        "condition": dict(condition),
        "noise_flags": flags,
        "total_attenuation_db": attenuation,
        "weather_attenuation_db": float(budget["atmospheric_db"]),
        "signal_power_no_fading_w": signal,
        "signal_power_no_fading_dbm": _positive_power_dbm(signal),
        "computed_noise_variance_w": computed,
        "computed_noise_power_dbm": {
            key: _positive_power_dbm(value) for key, value in computed.items()
        },
        "active_received_noise_variance_w": active,
        "active_received_noise_power_dbm": {
            key: _positive_power_dbm(value) for key, value in active.items()
        },
        "no_fading_esn0_linear": snr,
        "no_fading_esn0_db": snr_db,
        "snr_status": "finite" if snr_db is not None else "zero_signal_or_noise",
        "snr_scope": "single branch; unit-energy symbols; no fading or DSP errors",
    }


def qpsk_awgn_ber_from_esn0_db(esn0_db):
    """Gray QPSK coherent AWGN BER: Q(sqrt(Es/N0)); not a fading/DSP prediction."""
    esn0_db = float(esn0_db)
    if not math.isfinite(esn0_db):
        raise ValueError("Es/N0 must be finite dB")
    if esn0_db > 300.0:
        return 0.0
    return 0.5 * math.erfc(math.sqrt(10.0 ** (esn0_db / 10.0) / 2.0))


In [ ]:
"""Experimental QPSK block-likelihood acquisition, not author VPI DSP.

V1/V2 are preserved.  This candidate replaces only QPSK when called explicitly.
It uses received samples and the existing real block-amplitude CSI, never source
bits, true phase/CFO, decoder feedback or image scores.  A global residual-CFO
bound is derived from the configured orbit and wavelength, NOT the actual angle
or simulated CFO.  No frame is dropped.  The zero-frequency grid candidate is
always included, alongside received-spectrum candidates, and is never forced.

The short frames at 28 GBaud / 1 kHz linewidth motivate one affine phase per
frame instead of unwrapping independent 65-symbol noisy BPS windows.  This is
an explicit slow-phase assumption and can fail for fast phase wander.  Blind
QPSK's pi/2 ambiguity is retained; nearest-zero initial branch is an assumption.

References: Viterbi & Viterbi, IEEE Trans. IT 29(4), 543-551 (1983),
doi:10.1109/TIT.1983.1056713 (nonlinear M-PSK phase estimation); OptiCommPy
fourthPowerFOE documentation for M-th power spectral acquisition.  The exact
finite candidate search below is an independent implementation, not either
publication's released algorithm and not a confirmed FSO-SC author setting.
"""
from dataclasses import asdict, dataclass
import math

import torch
import torch.nn.functional as F

if "_rect8_legacy_recover_qam_carrier" not in globals():
    from fso_sc_signal_dsp_v2 import recover_qam_carrier as _v3_previous_recover
    from fso_sc_signal_dsp import SignalDSPConfig
else:
    _v3_previous_recover = recover_qam_carrier


QPSK_BLOCK_DSP_VERSION = "qpsk_bounded_spectral_marginal_ml_block_v3_candidate"


@dataclass(frozen=True)
class QPSKBlockMLConfig:
    wavelength_m: float = 1550e-9
    orbital_radius_m: float = 6371e3 + 620e3
    gravitational_parameter: float = 3.986004418e14
    residual_doppler_ratio: float = .1
    spectral_peaks_per_statistic: int = 8
    phase_trials: int = 16
    newton_steps: int = 6
    maximum_fit_samples: int = 8192

    def __post_init__(self):
        if not all(math.isfinite(v) and v > 0 for v in (self.wavelength_m,
                self.orbital_radius_m,self.gravitational_parameter,self.residual_doppler_ratio)):
            raise ValueError("Positive finite orbital/CFO parameters required")
        if any(type(value) is not int for value in (self.spectral_peaks_per_statistic,
                self.phase_trials,self.newton_steps,self.maximum_fit_samples)):
            raise ValueError("Integer QPSK search parameters required")
        if (self.spectral_peaks_per_statistic < 1 or self.phase_trials < 8
                or self.newton_steps < 1 or self.maximum_fit_samples < 128):
            raise ValueError("Invalid QPSK block-ML search configuration")

    @property
    def maximum_residual_cfo_hz(self):
        return self.residual_doppler_ratio * math.sqrt(
            self.gravitational_parameter/self.orbital_radius_m)/self.wavelength_m


def qpsk_block_ml_metadata(config=None):
    config = QPSKBlockMLConfig() if config is None else config
    return {"receiver_version":QPSK_BLOCK_DSP_VERSION,"experimental_candidate":True,
        "scope":"QPSK only; 8QAM/16QAM unchanged v2/v1", "config":asdict(config),
        "maximum_residual_cfo_hz":config.maximum_residual_cfo_hz,
        "frequency_prior":"global circular-orbit speed / wavelength times residual ratio; no actual CFO/angle",
        "phase_model":"one affine phase across a short frame; no local BPS unwrapping",
        "likelihood":"uniform-QPSK symbol-marginal AWGN likelihood; no transmitted decisions",
        "noise_variance":"normalized frame energy minus unit QPSK signal power, floor 1e-3",
        "acquisition_gate":"retain nominal zero residual CFO unless N*loglikelihood gain exceeds .5*log(N)+log(number of resolvable global CFO bins); conservative independent implementation choice",
        "nominal_cfo_prior":"zero after nominal carrier/precompensation, never actual simulated CFO; nonzero weak signals may fail acquisition",
        "phase_ambiguity":"initial branch nearest zero, not blind ambiguity resolution",
        "uses_true_channel_phase":False,"uses_true_cfo":False,
        "uses_transmitted_bits_or_symbols":False,"uses_codec_or_lpips_feedback":False,
        "no_pilots":True,"failure_policy":"all estimated frames and errors retained",
        "references":["https://doi.org/10.1109/TIT.1983.1056713",
        "https://opticommpy.readthedocs.io/en/stable/generated/optic.dsp.carrierRecovery.fourthPowerFOE.html"]}


def _qpsk_candidates(row, rate, config):
    count=row.numel(); nfft=1 << (4*count-1).bit_length()
    frequencies=torch.fft.fftfreq(nfft,1/rate,device=row.device)/4
    eligible=frequencies.abs() <= config.maximum_residual_cfo_hz
    indices=eligible.nonzero().flatten()
    candidates=[torch.zeros(1,device=row.device,dtype=row.real.dtype)]
    # Raw fourth moment and bounded-amplitude fourth moment.  The latter limits
    # the influence of rare AWGN amplitude outliers without selecting symbols.
    energy=row.abs().square().mean().clamp_min(torch.finfo(row.real.dtype).tiny)
    for value in (row, row/(row.abs().square()+energy).sqrt()):
        scale=value.abs().amax().clamp_min(torch.finfo(row.real.dtype).tiny)
        fourth=(value/scale).square().square()
        power=torch.fft.fft(fourth,nfft).abs().square()
        k=min(config.spectral_peaks_per_statistic,indices.numel())
        selected=indices[power[indices].topk(k).indices]
        candidates.append(frequencies[selected])
    return torch.unique(torch.cat(candidates)), rate/(4*count)


def _qpsk_fit(row, rate, config, fit_cfo=True):
    count=row.numel();dtype=row.real.dtype;device=row.device
    frequencies,bin_width=_qpsk_candidates(row,rate,config) if fit_cfo else (
        torch.zeros(1,device=device,dtype=dtype),rate/(4*count))
    ids=torch.linspace(0,count-1,min(count,config.maximum_fit_samples),device=device).round().long()
    time=ids.to(dtype)/(count-1)-.5
    sample=row[ids]
    noise=(row.abs().square().mean()-1).clamp_min(1e-3)
    c=math.sqrt(2)/noise
    phase_grid=(torch.arange(config.phase_trials,device=device,dtype=dtype)
                /config.phase_trials-.5)*(math.pi/2)
    base_frequency=frequencies.repeat_interleave(config.phase_trials)
    phase=phase_grid.repeat(frequencies.numel())
    slope=torch.zeros_like(phase)
    base_angle=base_frequency[:,None]*(2*math.pi/rate)*ids.to(dtype)
    for _ in range(config.newton_steps):
        rotated=sample*torch.exp(-1j*(base_angle+phase[:,None]+slope[:,None]*time))
        x,y=rotated.real,rotated.imag
        tx,ty=torch.tanh(c*x),torch.tanh(c*y)
        g=c*(y*tx-x*ty)
        h=c.square()*(y.square()*(1-tx.square())+x.square()*(1-ty.square()))-c*(x*tx+y*ty)
        h00=h.mean(-1);h01=(h*time).mean(-1);h11=(h*time.square()).mean(-1)
        g0=g.mean(-1);g1=(g*time).mean(-1)
        determinant=h00*h11-h01.square()
        stable=(h00 < 0)&(h11 < 0)&(determinant > 1e-8*(h00*h11).abs())
        denominator=torch.where(stable,determinant,torch.ones_like(determinant))
        phase_step=(torch.where(stable,(h11*g0-h01*g1)/denominator,0) if fit_cfo else
                    torch.where(h00<0,g0/h00.clamp_max(-torch.finfo(dtype).tiny),0))
        slope_step=torch.where(stable,(h00*g1-h01*g0)/denominator,0) if fit_cfo else torch.zeros_like(slope)
        phase=phase-phase_step.clamp(-math.pi/8,math.pi/8)
        slope=(slope-slope_step.clamp(-math.pi/4,math.pi/4)).clamp(-math.pi/2,math.pi/2)
    rotated=sample*torch.exp(-1j*(base_angle+phase[:,None]+slope[:,None]*time))
    logcosh=lambda x:F.softplus(2*x)-x-math.log(2)
    scores=(logcosh(c*rotated.real)+logcosh(c*rotated.imag)).mean(-1)
    frequency=base_frequency+slope*rate/(2*math.pi*(count-1))
    allowed=frequency.abs() <= config.maximum_residual_cfo_hz
    scores=torch.where(allowed,scores,-float("inf"))
    selected=scores.argmax()
    initial=phase[selected]-slope[selected]/2
    initial=torch.remainder(initial+math.pi/4,math.pi/2)-math.pi/4
    freq=frequency[selected]
    index=torch.arange(count,device=device,dtype=dtype)
    trajectory=initial+(2*math.pi/rate)*freq*index
    return row*torch.exp(-1j*trajectory),freq,initial,noise,scores[selected]


@torch.no_grad()
def recover_qam_carrier_v3(received,modulation,symbol_rate_baud,config=None,
                           known_block_magnitude=None,*,block_config=None):
    if modulation.upper() != "QPSK":
        return _v3_previous_recover(received,modulation,symbol_rate_baud,config,known_block_magnitude)
    config=SignalDSPConfig() if config is None else config
    block_config=QPSKBlockMLConfig() if block_config is None else block_config
    if received.ndim != 2 or not torch.is_complex(received) or received.shape[-1]<8:
        raise ValueError("Expected complex [batch,symbols>=8] receiver samples")
    if not math.isfinite(symbol_rate_baud) or symbol_rate_baud<=0 or not torch.isfinite(received).all():
        raise ValueError("Invalid receiver samples/rate")
    if block_config.maximum_residual_cfo_hz >= symbol_rate_baud/8:
        raise ValueError("Orbital CFO prior exceeds the fourth-power acquisition range")
    if config.amplitude_mode != "known_block_magnitude":
        raise ValueError("QPSK marginal likelihood requires declared real block-amplitude CSI")
    if known_block_magnitude is None or torch.is_complex(known_block_magnitude):
        raise ValueError("Positive real amplitude CSI required")
    if not torch.isfinite(known_block_magnitude).all() or (known_block_magnitude<=0).any():
        raise ValueError("Invalid amplitude CSI")
    amplitude=torch.broadcast_to(known_block_magnitude,(received.shape[0],1))
    observed=received/amplitude
    if not config.cfo_enabled and not config.cpr_enabled:
        return observed,{"receiver_version":QPSK_BLOCK_DSP_VERSION,"both_stages_disabled":True}
    rows=[];cfo=[];phase=[];noise=[];score=[];gates=[];gains=[];thresholds=[]
    for row in observed:
        recovered,f,p,n,s=_qpsk_fit(row,symbol_rate_baud,block_config,config.cfo_enabled)
        nominal_used=False; gain=0.;threshold=0.
        if config.cfo_enabled:
            nominal,nf,np,nn,ns=_qpsk_fit(row,symbol_rate_baud,block_config,False)
            fit_count=min(row.numel(),block_config.maximum_fit_samples)
            resolvable=max(1.,2*block_config.maximum_residual_cfo_hz/(symbol_rate_baud/(4*row.numel())))
            threshold=.5*math.log(fit_count)+math.log(resolvable)
            gain=float((s-ns)*fit_count)
            if gain<=threshold:
                recovered,f,p,n,s=nominal,nf,np,nn,ns
                nominal_used=True
        if not config.cpr_enabled:
            recovered=recovered*torch.exp(1j*p)
        rows.append(recovered);cfo.append(float(f));phase.append(float(p));noise.append(float(n));score.append(float(s))
        gates.append(nominal_used);gains.append(gain);thresholds.append(threshold)
    return torch.stack(rows),{"receiver_version":QPSK_BLOCK_DSP_VERSION,
        "estimated_cfo_hz":cfo,"estimated_phase_start_rad":phase,
        "estimated_normalized_noise_variance":noise,"marginal_log_likelihood":score,
        "nominal_cfo_prior_used":gates,"frequency_loglikelihood_gain":gains,
        "frequency_evidence_threshold":thresholds,
        "maximum_residual_cfo_hz":block_config.maximum_residual_cfo_hz,
        "phase_symmetry_unwrap_events":[0]*received.shape[0],
        "experimental_candidate":True,"failure_policy":"estimates/errors retained, no frame rejection"}


In [ ]:
"""Fixed, disclosed channel interpretations for the v12 rebuild.

The primary profile reproduces the public-equation idealization, not the unknown
VPI receiver. The coherent extension remains a separate stress interpretation.
No channel parameter or noise multiplier is fitted to image quality.
"""
import hashlib
import json
import math

import torch
import torch.nn.functional as F


REBUILD_CHANNEL_VERSION_V12 = "public_equations_and_coherent_stress_v12"
REBUILD_PROFILES_V12 = ("paper_eq3_4_background_only", "coherent_receiver_extension_v6")
_FIELDS_V12 = {"modulation", "launch_power_dbm", "zenith_angle_deg", "pointing_error_urad",
    "weather", "scattered_power_ratio", "background_light_radiance_w_m2_sr_um"}


def rebuild_training_conditions_v12():
    """A fixed training grid, never selected by development/holdout LPIPS."""
    rows = [None]
    for power in (30., 33., 36.):
        for zenith in (0., 30., 60.):
            for modulation in ("QPSK", "8QAM", "16QAM"):
                rows.append({"modulation": modulation, "launch_power_dbm": power,
                    "zenith_angle_deg": zenith, "pointing_error_urad": 20., "weather": "clear",
                    "scattered_power_ratio": .95, "background_light_radiance_w_m2_sr_um": 1e-4})
    for power in (28., 32., 36.):
        rows.append({"modulation": "QPSK", "launch_power_dbm": power,
            "zenith_angle_deg": 0., "pointing_error_urad": 15., "weather": "snow",
            "scattered_power_ratio": .95, "background_light_radiance_w_m2_sr_um": 1e-4})
    return rows


def _rebuild_sha_v12(tensor):
    tensor = tensor.detach().cpu().contiguous()
    digest = hashlib.sha256((str(tensor.dtype)+str(tuple(tensor.shape))).encode())
    digest.update(memoryview(tensor.numpy()).cast("B"))
    return digest.hexdigest()


def _rebuild_condition_v12(condition):
    if condition is None:return None
    if not isinstance(condition,dict) or set(condition)!=_FIELDS_V12:
        raise ValueError("Supply an explicit modulation and the six fixed channel arguments; no noise tuning")
    result = dict(condition)
    if result["modulation"] not in ("QPSK","8QAM","16QAM"):
        raise ValueError("Unsupported paper modulation")
    if result["weather"] not in ("clear","snow","fog","rain","dust"):
        raise ValueError("Unsupported published weather")
    for key in _FIELDS_V12-{"modulation","weather"}:
        if isinstance(result[key],bool) or not isinstance(result[key],(float,int)) or not math.isfinite(result[key]):
            raise ValueError("Nonfinite/non-numeric channel condition")
        result[key]=float(result[key])
    if (not 28<=result["launch_power_dbm"]<=36 or not 0<=result["zenith_angle_deg"]<=80
            or result["pointing_error_urad"]<0 or not 0<=result["scattered_power_ratio"]<=1
            or result["background_light_radiance_w_m2_sr_um"]<0):
        raise ValueError("Channel condition outside the declared paper range")
    return result


def _rebuild_observe_v12(physical,symbols,generator,kwargs):
    """Scope deterministic mode to this synchronous physical call only."""
    enabled=torch.are_deterministic_algorithms_enabled()
    warn_only=torch.is_deterministic_algorithms_warn_only_enabled()
    if symbols.device.type!="cuda":
        return physical(symbols,generator=generator,**kwargs),"cpu_native"
    initial=generator.get_state().clone()
    try:
        torch.use_deterministic_algorithms(True,warn_only=False)
        try:
            return physical(symbols,generator=generator,**kwargs),"cuda_scoped_deterministic"
        except RuntimeError as error:
            if "deterministic" not in str(error).lower() or (enabled and not warn_only):raise
            # Older Torch lacks deterministic CUDA cumsum. Do not consume a
            # second realization or claim byte-identical paired observations.
            generator.set_state(initial)
            torch.use_deterministic_algorithms(enabled,warn_only=warn_only)
            return physical(symbols,generator=generator,**kwargs),"cuda_unsupported_fallback_not_pair_verified"
    finally:torch.use_deterministic_algorithms(enabled,warn_only=warn_only)


def _rebuild_awgn_sanity_v12(namespace,device):
    rows=[];samples=65536
    for offset,snr_db in enumerate((0.,6.,12.)):
        generator=torch.Generator(device=device).manual_seed(20261113+offset)
        bits=torch.randint(0,2,(1,samples,2),device=device,generator=generator,dtype=torch.uint8)
        symbols=namespace["qam_modulate"](bits,"QPSK")
        noise=torch.complex(torch.randn(symbols.shape,device=device,generator=generator),
            torch.randn(symbols.shape,device=device,generator=generator))/math.sqrt(2)
        decoded=namespace["qam_demodulate"](symbols+noise*10**(-snr_db/20),"QPSK")
        errors=int((decoded!=bits).sum());expected=.5*math.erfc(math.sqrt(10**(snr_db/10)/2))
        measured=errors/(2*samples)
        tolerance=max(6*math.sqrt(expected*(1-expected)/(2*samples)),5/(2*samples))
        rows.append({"esn0_db":snr_db,"bit_errors":errors,"valid_bits":2*samples,
            "measured_ber":measured,"theoretical_ber":expected,"absolute_tolerance":tolerance,
            "passed":abs(measured-expected)<=tolerance})
    if not all(row["passed"] for row in rows):raise RuntimeError("QPSK AWGN theory sanity check failed")
    return rows


class RebuildIndexLinkV12:
    def __init__(self,namespace,profile="paper_eq3_4_background_only",base_seed=20261103):
        if profile not in REBUILD_PROFILES_V12:raise ValueError("Only the two preregistered interpretations are allowed")
        if type(base_seed) is not int or not 0<=base_seed<2**63:raise ValueError("Invalid channel base seed")
        self.ns=namespace;self.profile=profile;self.base_seed=base_seed
        self.config=namespace["make_channel_profile"](profile,namespace["PaperFSOSystemConfig"])
        self.physical=namespace["PaperMalagaFSOChannel"](self.config)
        expected=(True,False,False) if profile==REBUILD_PROFILES_V12[0] else (True,True,True)
        if (tuple(getattr(self.config,name) for name in ("include_background_noise","include_ase_noise","include_shot_noise"))!=expected
                or self.config.num_receivers!=1 or not all((self.config.include_turbulence,
                    self.config.include_doppler,self.config.include_phase_noise))):raise ValueError("Profile identity differs")

    def metadata(self):
        return {"version":REBUILD_CHANNEL_VERSION_V12,"profile":self.profile,"base_seed":self.base_seed,
            "frozen_physical_version":self.ns["PAPER_FSO_CHANNEL_VERSION"],
            "receiver":"unchanged v3 QPSK / prior 8QAM-16QAM DSP",
            "power_convention":"nominal launch power for unit-uniform-constellation energy; actual per-frame power is recorded, not secretly normalized",
            "receiver_csi":"existing ideal real block magnitude only; no true complex phase, CFO or transmitted symbols",
            "same_external_randomness_claimed":False,"curve_fitted":False,"full_vpi_reproduction":False}

    @torch.no_grad()
    def transmit(self,indices,conditions,seeds,*,device):
        if (not isinstance(indices,torch.Tensor) or indices.dtype!=torch.int64 or indices.ndim!=3
                or tuple(indices.shape[1:])!=(32,32) or len(indices)<1
                or bool(((indices<0)|(indices>=16384)).any())):
            raise ValueError("Expected current-model int64 [B,32,32] indices in [0,16384)")
        if len(conditions)!=len(indices) or len(seeds)!=len(indices):raise ValueError("Incomplete per-image channel plan")
        plans=[_rebuild_condition_v12(c) for c in conditions]
        if any(type(s) is not int or not 0<=s<2**63 for s in seeds):raise ValueError("Invalid per-image seed")
        device=torch.device(device);indices=indices.to(device);outputs=[];records=[]
        for i,(condition,seed) in enumerate(zip(plans,seeds)):
            source=indices[i:i+1]
            record={"profile":self.profile,"condition":condition,"seed":seed,"bit_errors":0,"valid_bits":14336,
                "index_errors":0,"indices":1024,"symbol_errors":0,"valid_symbols":0,"padding_bits":0,
                "physical_calls":0,"channel_gain_sha256":None,"channel_rng_state_sha256":None,
                "transmitted_symbols_mean_power":None,"nominal_launch_power_w":None,"actual_launch_power_w":None,
                "receiver_diagnostics":{},"deterministic_physical":"clean_identity","same_external_randomness_verified":False}
            record["condition_label"]="clean" if condition is None else (
                f"{condition['weather']}_{condition['modulation']}_P{condition['launch_power_dbm']:g}_z{condition['zenith_angle_deg']:g}_p{condition['pointing_error_urad']:g}")
            if condition is None:
                output=source.clone()
            else:
                modulation=condition["modulation"]
                _,_,bps=self.ns["qam_constellation"](modulation,device)
                bits=self.ns["indices_to_bits"](source).reshape(1,-1)
                padding=(-bits.shape[-1])%bps
                grouped=F.pad(bits,(0,padding)).reshape(1,-1,bps)
                symbols=self.ns["qam_modulate"](grouped,modulation)
                generator=torch.Generator(device=device).manual_seed(seed)
                observed,mode=_rebuild_observe_v12(self.physical,symbols,generator,
                    {k:v for k,v in condition.items() if k!="modulation"})
                if (observed.is_equalized or observed.channel_gain is None
                        or observed.symbols.shape!=symbols.shape or observed.channel_gain.shape!=symbols.shape
                        or not bool(torch.isfinite(observed.symbols).all()) or not bool(torch.isfinite(observed.channel_gain).all())):
                    raise FloatingPointError("Invalid raw physical observation; no frame discarded")
                magnitude=observed.channel_gain.abs().mean(-1,keepdim=True)
                if not bool((magnitude>0).all()):raise FloatingPointError("Nonpositive block amplitude")
                recovered,diagnostics=self.ns["recover_qam_carrier_v3"](observed.symbols,modulation,
                    self.config.symbol_rate_baud,known_block_magnitude=magnitude)
                if recovered.shape!=symbols.shape or not bool(torch.isfinite(recovered).all()):
                    raise FloatingPointError("Invalid receiver output; no frame discarded")
                hard_groups=self.ns["qam_demodulate"](recovered,modulation)
                if hard_groups.shape!=grouped.shape:raise ValueError("Receiver bit grouping changed")
                hard=hard_groups.reshape(1,-1)[:,:bits.shape[-1]]
                output=self.ns["bits_to_indices"](hard.reshape(1,32,32,14)).long()
                energy=float(symbols.abs().square().mean());launch=10**((condition["launch_power_dbm"]-30)/10)
                record.update(bit_errors=int((hard!=bits).sum()),index_errors=int((output!=source).sum()),
                    symbol_errors=int((hard_groups!=grouped).any(-1).sum()),valid_symbols=grouped.shape[1],padding_bits=padding,
                    physical_calls=1,channel_gain_sha256=_rebuild_sha_v12(observed.channel_gain),
                    channel_rng_state_sha256=_rebuild_sha_v12(generator.get_state()),deterministic_physical=mode,
                    transmitted_symbols_mean_power=energy,nominal_launch_power_w=launch,actual_launch_power_w=launch*energy,
                    receiver_diagnostics=diagnostics)
            record.update(payload_ber=record["bit_errors"]/record["valid_bits"],
                index_error_rate=record["index_errors"]/record["indices"],
                symbol_error_rate=record["symbol_errors"]/record["valid_symbols"] if record["valid_symbols"] else None)
            json.dumps(record,allow_nan=False)
            outputs.append(output);records.append(record)
        return torch.cat(outputs),records


def preflight_rebuild_channel_v12(namespace,device="cpu"):
    """Eight synthetic physical frames; no weights, image metrics or tuning."""
    device=torch.device(device);generator=torch.Generator(device=device).manual_seed(20261103)
    indices=torch.randint(0,16384,(1,32,32),device=device,generator=generator)
    rows=[];warnings=[];awgn=_rebuild_awgn_sanity_v12(namespace,device)
    for profile in REBUILD_PROFILES_V12:
        link=RebuildIndexLinkV12(namespace,profile)
        for modulation in ("QPSK","16QAM"):
            for weather in ("clear","snow"):
                condition={"modulation":modulation,"launch_power_dbm":36.,"zenith_angle_deg":0.,
                    "pointing_error_urad":15.,"weather":weather,"scattered_power_ratio":.95,
                    "background_light_radiance_w_m2_sr_um":1e-4}
                _,records=link.transmit(indices,[condition],[20261103+len(rows)],device=device)
                budget=namespace["compute_paper_link_budget"](link.config,
                    **{k:v for k,v in condition.items() if k!="modulation"})
                transmission=10**(-budget["total_attenuation_db"]/10)
                noise=(budget["background_noise_w"]
                    +(budget["ase_noise_w"]*transmission if link.config.include_ase_noise else 0.)
                    +(budget["receiver_shot_noise_w"] if link.config.include_shot_noise else 0.))
                signal=budget["launch_power_w"]*transmission
                records[0]["no_fading_scalar_budget"]={"attenuation_db":budget["total_attenuation_db"],
                    "signal_w":signal,"complex_noise_variance_w":noise,"esn0_db":10*math.log10(signal/noise),
                    "scope":"unit-uniform-constellation nominal energy; excludes random fading and DSP error"}
                rows.extend(records)
                if weather=="snow" and records[0]["payload_ber"]<1e-3:
                    warnings.append(profile+"/"+modulation+": near-error-free snow does not establish the paper's BPG failure or full curve")
                if records[0]["payload_ber"]>.1:
                    warnings.append(profile+"/"+modulation+": severe raw errors; do not promise semantic structure recovery")
    return {"version":REBUILD_CHANNEL_VERSION_V12,"synthetic_only":True,"frames":len(rows),"records":rows,
        "qpsk_awgn_theory_checks":awgn,"all_awgn_theory_checks_passed":True,
        "training_updates":0,"model_weights_loaded":False,"lpips_used_for_channel_selection":False,
        "same_external_randomness_across_profiles_verified":False,"warnings":warnings,
        "interpretation":"Two fixed interpretations, not a profile search or proof of authors' receiver settings"}


In [ ]:
import hashlib,io,json,os
from pathlib import Path
import numpy as np
from PIL import Image
import torch

def rebuild_file_sha256_v12(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda:handle.read(8*1024**2),b""):digest.update(block)
    return digest.hexdigest()

def rebuild_save_json_v12(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.is_symlink():raise ValueError("Refuse linked output")
    pending=path.with_name(path.name+".partial")
    if pending.is_symlink():raise ValueError("Refuse linked pending output")
    with pending.open("w",encoding="utf-8") as handle:
        json.dump(value,handle,ensure_ascii=False,indent=2,allow_nan=False)
        handle.flush();os.fsync(handle.fileno())
    pending.replace(path)

def _rebuild_image_v12(content):
    with Image.open(io.BytesIO(content)) as opened:
        image=opened.convert("RGB");w,h=image.size;side=min(w,h)
        image=image.crop(((w-side)/2,(h-side)/2,(w+side)/2,(h+side)/2))
        image=image.resize((256,256),resample=Image.Resampling.BICUBIC,reducing_gap=1)
        array=np.asarray(image,dtype=np.float32)/127.5-1.
    return torch.from_numpy(np.ascontiguousarray(array.transpose(2,0,1)))


In [ ]:
import hashlib,json
from pathlib import Path,PurePosixPath
import torch

def _pilot_json_sha_v12(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")).encode()).hexdigest()

def _pilot_sha_valid_v12(value):
    return isinstance(value, str) and len(value) == 64 and all(x in "0123456789abcdef" for x in value)

def _pilot_file_sha_v12(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

def _pilot_state_sha_v12(state):
    """Same model-state digest as the frozen v9/v12 exporter."""
    if not isinstance(state, dict) or not state:
        raise ValueError("Missing Stage1/model tensor state")
    digest = hashlib.sha256()
    for name, value in sorted(state.items()):
        if not isinstance(name, str) or not isinstance(value, torch.Tensor):
            raise ValueError("Model state must contain named tensors only")
        array = value.detach().cpu().contiguous()
        if not bool(torch.isfinite(array).all()):
            raise FloatingPointError("Nonfinite model tensor")
        digest.update(name.encode())
        digest.update(str(array.dtype).encode())
        digest.update(str(tuple(array.shape)).encode())
        digest.update(memoryview(array.numpy()).cast("B"))
    return digest.hexdigest()

def _pilot_child_v12(root, relative):
    root = Path(root).resolve()
    if not isinstance(relative, str):
        raise ValueError("Pilot-relative output must be a string")
    name = PurePosixPath(relative)
    if (name.is_absolute() or ".." in name.parts
            or "\\" in relative or not name.parts):
        raise ValueError("Unsafe pilot-relative output")
    target = root.joinpath(*name.parts)
    for ancestor in (target, *target.parents):
        if ancestor == root:
            break
        if ancestor.is_symlink():
            raise ValueError("Linked pilot output is not permitted")
    if root == Path(root.anchor) or root not in target.resolve().parents:
        raise ValueError("Pilot output escaped its scoped directory")
    return target

def _pilot_save_json_v12(ns, root, relative, value):
    path = _pilot_child_v12(root, relative)
    ns["rebuild_save_json_v12"](path, value)
    return path

def _pilot_write_identity_v12(ns, root, relative, value):
    path = _pilot_child_v12(root, relative)
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8")) != value:
            raise RuntimeError("Existing pilot identity differs: " + relative)
    else:
        _pilot_save_json_v12(ns, root, relative, value)

def _pilot_inventory_v12(root, paths):
    root = Path(root).resolve()
    rows = []
    for path in paths:
        relative = Path(path).resolve().relative_to(root).as_posix()
        candidate = _pilot_child_v12(root, relative)
        if not candidate.is_file():
            raise RuntimeError("Missing pilot result: " + relative)
        rows.append({"file": relative, "bytes": candidate.stat().st_size,
            "sha256": _pilot_file_sha_v12(candidate)})
    return sorted(rows, key=lambda row: row["file"])

def _pilot_verify_inventory_v12(root, rows):
    if not isinstance(rows, list) or not rows:
        return False
    if len({row.get("file") for row in rows if isinstance(row, dict)}) != len(rows):
        return False
    for row in rows:
        if (not isinstance(row, dict) or type(row.get("bytes")) is not int
                or row["bytes"] < 1 or not _pilot_sha_valid_v12(row.get("sha256"))):
            return False
        path = _pilot_child_v12(root, row.get("file"))
        if (not path.is_file() or path.stat().st_size != row["bytes"]
                or _pilot_file_sha_v12(path) != row["sha256"]):
            return False
    return True


In [ ]:
import gc,hashlib,json,math,os
from pathlib import Path
import torch
BER_RESPONSE_INPUT_VERSION_V12="read_only_development_and_weights_v12_1"

def _ber_input_valid_sha_v12(value):
    return (isinstance(value, str) and len(value) == 64
            and all(character in "0123456789abcdef" for character in value))

def _ber_input_file_sha_v12(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

def _ber_input_json_sha_v12(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),
        ensure_ascii=False, allow_nan=False).encode("utf-8")).hexdigest()

def _ber_input_state_sha_v12(state):
    """Same sorted-name/tensor digest as the frozen v9/v12 model exporter."""
    if not isinstance(state, dict) or not state:
        raise ValueError("A nonempty named-tensor model state is required")
    digest = hashlib.sha256()
    for name, value in sorted(state.items()):
        if not isinstance(name, str) or not isinstance(value, torch.Tensor):
            raise ValueError("Model state must contain named tensors only")
        array = value.detach().cpu().contiguous()
        if not bool(torch.isfinite(array).all()):
            raise FloatingPointError("Nonfinite model tensor in pinned checkpoint")
        digest.update(name.encode())
        digest.update(str(array.dtype).encode())
        digest.update(str(tuple(array.shape)).encode())
        digest.update(memoryview(array.numpy()).cast("B"))
    return digest.hexdigest()

def _ber_input_cache_path_v12(source, local_cache, basename):
    original, cache = Path(source), Path(local_cache)
    if original.is_symlink() or cache.is_symlink() or not original.is_file():
        raise ValueError("Pinned source must exist and source/cache must not be linked")
    original = original.resolve()
    cache = cache.resolve()
    parent = original.parent
    if (cache == Path(cache.anchor) or cache == parent or parent in cache.parents
            or cache in parent.parents or not isinstance(basename, str)
            or not basename or Path(basename).name != basename):
        raise ValueError("Use a scoped local cache separate from the source tree")
    target = cache / basename
    pending = cache / (basename + ".partial")
    if (target.is_symlink() or pending.is_symlink()
            or target.resolve().parent != cache or pending.resolve().parent != cache
            or target.resolve() == original or pending.resolve() == original
            or any(path.exists() and path.stat().st_nlink > 1 for path in (target, pending))):
        raise ValueError("Linked or out-of-scope local input cache")
    cache.mkdir(parents=True, exist_ok=True)
    return original, target, pending

def _ber_input_verified_copy_v12(source, local_cache, basename, expected_sha256,
                                 expected_bytes=None):
    if not _ber_input_valid_sha_v12(expected_sha256):
        raise ValueError("Pin a lowercase SHA256 before copying input data")
    if expected_bytes is not None and (type(expected_bytes) is not int or expected_bytes < 1):
        raise ValueError("Pinned input byte count must be a positive integer")
    original, target, pending = _ber_input_cache_path_v12(source, local_cache, basename)
    source_bytes = original.stat().st_size
    if source_bytes < 1 or (expected_bytes is not None and source_bytes != expected_bytes):
        raise RuntimeError("Pinned source is absent or incomplete: " + original.name)
    expected_bytes = source_bytes if expected_bytes is None else expected_bytes
    reused = (target.is_file() and target.stat().st_size == expected_bytes
        and _ber_input_file_sha_v12(target) == expected_sha256)
    if not reused:
        # One source read: hash while copying, not remote-hash then remote-copy.
        digest = hashlib.sha256()
        copied = 0
        with original.open("rb") as reader, pending.open("wb") as writer:
            for block in iter(lambda: reader.read(8 * 1024**2), b""):
                copied += len(block)
                digest.update(block)
                writer.write(block)
            writer.flush()
            os.fsync(writer.fileno())
        if (copied != expected_bytes or pending.stat().st_size != expected_bytes
                or original.stat().st_size != source_bytes
                or digest.hexdigest() != expected_sha256):
            raise RuntimeError("Pinned input copy SHA256/size verification failed; source was not modified")
        pending.replace(target)
    return target, {"source": str(original), "local_path": str(target),
        "sha256": expected_sha256, "bytes": expected_bytes,
        "verified_pinned_bytes": True, "reused_verified_local_cache": bool(reused),
        "source_read_passes_this_call": 0 if reused else 1,
        "source_sha256_rechecked_on_cache_reuse": False,
        "source_metadata_checked_on_cache_reuse": True, "source_modified": False}

def load_ber_checkpoint_v12(source, local_cache, *, expected_sha256, expected_bytes,
                           expected_state_sha256, expected_configuration_sha256, kind,
                           expected_parent_stage1_sha256=None):
    """Return pinned CPU model/config/progress only; drop any Adam/RNG payload."""
    if kind not in {"stage1", "rxgrad_on512"}:
        raise ValueError("Only complete Stage1 and the frozen ON512 pilot are supported")
    if any(not _ber_input_valid_sha_v12(value) for value in
           (expected_sha256, expected_state_sha256, expected_configuration_sha256)):
        raise ValueError("Pin checkpoint, model-state and configuration SHA256")
    if (expected_parent_stage1_sha256 is not None
            and not _ber_input_valid_sha_v12(expected_parent_stage1_sha256)):
        raise ValueError("Invalid expected parent Stage1 SHA256")
    local, copy_audit = _ber_input_verified_copy_v12(source, local_cache,
        kind + "_" + expected_sha256 + ".pth", expected_sha256, expected_bytes)
    packet = torch.load(local, map_location="cpu", weights_only=True)
    if not isinstance(packet, dict):
        raise ValueError("Pinned checkpoint is not a v12 tensor/primitives packet")
    configuration = packet.get("configuration")
    progress = packet.get("progress")
    if (packet.get("engine_version") != "rebuild_full_data_v12"
            or not isinstance(configuration, dict) or not isinstance(progress, dict)
            or _ber_input_json_sha_v12(configuration) != expected_configuration_sha256
            or packet.get("configuration_sha256", expected_configuration_sha256)
                != expected_configuration_sha256):
        raise RuntimeError("Pinned checkpoint engine/configuration differs")
    n, effective = configuration.get("training_images"), configuration.get("effective_batch_size")
    if type(n) is not int or n < 1 or type(effective) is not int or effective < 1:
        raise RuntimeError("Checkpoint training corpus/effective batch identity is missing")
    if kind == "stage1":
        if (packet.get("stage") != 1 or configuration.get("stage1_epochs") != 100
                or progress.get("epoch") != 100 or progress.get("cursor") != 0
                or progress.get("updates") != 100 * math.ceil(n / effective)
                or progress.get("training_images") != n
                or not isinstance(progress.get("history"), list) or len(progress["history"]) != 100):
            raise RuntimeError("Source Stage1 is not a completed compatible100-epoch checkpoint")
    else:
        spec = configuration.get("rxgrad_ablation", {})
        parent = spec.get("stage1_checkpoint_sha256")
        if (packet.get("stage") != 2
                or packet.get("ablation_version") != "received_codebook_gradient_only_v12_1"
                or spec.get("version") != "received_codebook_gradient_only_v12_1"
                or spec.get("arm") != "rxgrad_on" or spec.get("rxgrad_enabled") is not True
                or spec.get("budget_updates") != 512 or not _ber_input_valid_sha_v12(parent)
                or (expected_parent_stage1_sha256 is not None and parent != expected_parent_stage1_sha256)
                or n != 11000 or effective != 8
                or progress.get("phase") != "rxgrad_ablation_pilot"
                or progress.get("status") != "pilot_budget_reached"
                or progress.get("rxgrad_enabled") is not True
                or progress.get("budget_updates") != 512 or progress.get("updates") != 512
                or progress.get("epoch") != 0 or progress.get("cursor") != 4096
                or progress.get("training_images") != 11000 or progress.get("history") != []
                or progress.get("stage1_checkpoint_sha256") != parent):
            raise RuntimeError("Source is not the completed same-parent ON512 pilot checkpoint")
    state = packet.get("state_dict")
    if _ber_input_state_sha_v12(state) != expected_state_sha256:
        raise RuntimeError("Pinned model tensor SHA256 differs")
    state = {name: value.detach().cpu() for name, value in state.items()}
    identity = {"version": BER_RESPONSE_INPUT_VERSION_V12, "kind": kind,
        "checkpoint": copy_audit, "state_sha256": expected_state_sha256,
        "configuration_sha256": expected_configuration_sha256,
        "source_stage": packet["stage"], "source_epoch": progress["epoch"],
        "source_updates": progress["updates"], "source_cursor": progress["cursor"],
        "optimizer_created_or_restored": False, "source_modified": False,
        "retained_payload_fields": ["state_dict", "configuration", "progress"]}
    # No optimizer/RNG tensor survives this input helper. Do not move the full
    # resume packet onto the GPU merely to recover a fixed model's weights.
    del packet
    gc.collect()
    print("Verified inference-only weights:", kind, "| no Adam state retained or restored.", flush=True)
    return {"state_dict": state, "configuration": configuration,
        "progress": progress, "identity": identity}


In [ ]:
"""Predeclared image-disjoint confirmation without reading training ZIPs.

Only GEE_test.zip is copied and decoded. The selected 200 images exclude
the archived development200, old holdout200, and wire-label confirmation100.
Raw train-file overlap is checked against the pinned checkpoint's 11000-entry
training inventory. RGB checks here cover the test pool, not unseen train RGB
or geographical/near-duplicate independence.
"""
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import stat
import zipfile

import torch



COMPACT_FINAL_DATA_VERSION_V26 = "compact_final_data_predeclared_v26_1"
_COMPACT_DATA_LOCK_VERSION_V26 = "fixed_compact_final_data_v26_1"
_COMPACT_DATA_SELECTION_VERSION_V26 = "exclude_known500_sha256_rank_v26_1"
_COMPACT_DATA_PREPROCESSING_V26 = "center-square/PIL-bicubic256/reducing_gap1/RGB[-1,1]"
_COMPACT_EXCLUSION_ROLES_V26 = ("prior_development_200", "prior_holdout_200", "wire_confirmation_100")


def _compact_data_json_sha_v26(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")).encode()).hexdigest()


def _compact_data_valid_sha_v26(value):
    return isinstance(value, str) and len(value) == 64 and all(x in "0123456789abcdef" for x in value)


def _compact_data_hash_list_v26(values, name, count=None):
    if (not isinstance(values, list) or (count is not None and len(values) != count)
            or not values or len(set(values)) != len(values)
            or any(not _compact_data_valid_sha_v26(value) for value in values)):
        raise ValueError("Invalid distinct file SHA256 list: " + name)
    return set(values)


def _compact_data_rank_v26(digest, seed):
    return hashlib.sha256(("compact-final-v26\0" + str(seed) + "\0" + digest).encode()).hexdigest()


def validate_compact_final_data_lock_v26(lock):
    """Check membership from identities only; never resample after seeing pixels."""
    if (not isinstance(lock, dict) or lock.get("version") != _COMPACT_DATA_LOCK_VERSION_V26
            or lock.get("selection_version") != _COMPACT_DATA_SELECTION_VERSION_V26
            or lock.get("selection_uses_image_content_or_quality_scores") is not False
            or lock.get("test_zip_file") != "GEE_test.zip"
            or lock.get("preprocessing") != _COMPACT_DATA_PREPROCESSING_V26
            or lock.get("source_zips_modified") is not False
            or lock.get("geographic_or_near_duplicate_independence_established") is not False
            or lock.get("entirely_new_external_dataset_claimed") is not False
            or lock.get("training_RGB_duplicate_check_this_run") is not False):
        raise ValueError("Use the predeclared confirmation data lock and disclosed scope")
    unsigned = {key: value for key, value in lock.items() if key != "data_lock_sha256"}
    if _compact_data_json_sha_v26(unsigned) != lock.get("data_lock_sha256"):
        raise ValueError("The confirmation data-lock SHA256 differs")
    for key in ("selection_seed", "expected_total_images", "count", "eligible_images", "excluded_unique_images", "source_training_images"):
        if type(lock.get(key)) is not int or lock[key] < (0 if key == "selection_seed" else 1):
            raise ValueError("Invalid fixed data count/seed: " + key)
    for key in ("test_zip_sha256", "source_training_inventory_sha256", "source_data_manifest_sha256"):
        if not _compact_data_valid_sha_v26(lock.get(key)):
            raise ValueError("Pin the reviewed source corpus digest: " + key)
    all_hashes = _compact_data_hash_list_v26(lock.get("all_file_sha256"), "test pool", lock["expected_total_images"])
    if lock["all_file_sha256"] != sorted(all_hashes):
        raise ValueError("Retain the sorted complete test inventory")
    roles = lock.get("excluded_roles")
    if not isinstance(roles, dict) or set(roles) != set(_COMPACT_EXCLUSION_ROLES_V26):
        raise ValueError("Retain all three known previous-evaluation roles")
    excluded = set()
    for role in _COMPACT_EXCLUSION_ROLES_V26:
        hashes = _compact_data_hash_list_v26(roles[role], role)
        if not hashes.issubset(all_hashes) or excluded & hashes:
            raise ValueError("Previous-evaluation memberships are invalid or overlap")
        excluded.update(hashes)
    eligible = all_hashes - excluded
    selected = _compact_data_hash_list_v26(lock.get("selected_file_sha256"), "selected confirmation", lock["count"])
    if (len(excluded) != lock["excluded_unique_images"] or len(eligible) != lock["eligible_images"]
            or lock["count"] > len(eligible) or selected & excluded):
        raise ValueError("Confirmation eligibility or exclusion counts differ")
    expected = sorted(eligible, key=lambda value: _compact_data_rank_v26(value, lock["selection_seed"]))[:lock["count"]]
    if lock["selected_file_sha256"] != expected:
        raise ValueError("Confirmation membership/order must be frozen before pixels and metrics")
    return {"all": all_hashes, "excluded": excluded, "selected": selected, "eligible": eligible}


def _compact_source_training_inventory_v26(configuration, lock, memberships):
    data = configuration.get("data") if isinstance(configuration, dict) else None
    if not isinstance(data, dict) or _compact_data_json_sha_v26(data) != lock["source_data_manifest_sha256"]:
        raise RuntimeError("The fixed primary model's complete data manifest differs")
    count = lock["source_training_images"]
    train_hashes = data.get("training_file_sha256")
    train = _compact_data_hash_list_v26(train_hashes, "source training", count)
    inventory_sha = hashlib.sha256("\n".join(train_hashes).encode()).hexdigest()
    if (configuration.get("training_images") != count or data.get("training_images") != count
            or data.get("training_inventory_sha256") != inventory_sha
            or inventory_sha != lock["source_training_inventory_sha256"]
            or data.get("zip_sha256", {}).get("gee_test") != lock["test_zip_sha256"]
            or data.get("preprocessing") != _COMPACT_DATA_PREPROCESSING_V26):
        raise RuntimeError("The fixed model training inventory/preprocessing identity differs")
    if train & memberships["all"]:
        raise RuntimeError("Exact training-file content overlaps the test corpus")
    return inventory_sha


def _compact_test_zip_copy_v26(drive_root, local_root, expected_sha):
    drive_arg, local_arg = Path(drive_root), Path(local_root)
    if drive_arg.is_symlink() or local_arg.is_symlink():
        raise ValueError("Use unlinked scoped source/cache directories")
    drive, local = drive_arg.resolve(), local_arg.resolve()
    if (drive == Path(drive.anchor) or local == Path(local.anchor)
            or drive == local or drive in local.parents or local in drive.parents):
        raise ValueError("Keep local test staging separate from Drive")
    source = drive / "GEE_test.zip"
    if source.is_symlink() or not source.is_file():
        raise RuntimeError("Restore the existing GEE_test.zip; no dataset replacement fallback")
    if rebuild_file_sha256_v12(source) != expected_sha:
        raise RuntimeError("GEE_test.zip differs from the frozen test corpus")
    local.mkdir(parents=True, exist_ok=True)
    target = local / ("GEE_test_" + expected_sha + ".zip")
    for path in (target, target.with_name(target.name + ".partial")):
        if path.exists() and (path.is_symlink() or not path.is_file() or path.stat().st_nlink > 1):
            raise ValueError("Unsafe own test staging file")
    copied = False
    if not target.is_file() or rebuild_file_sha256_v12(target) != expected_sha:
        pending = target.with_name(target.name + ".partial")
        with source.open("rb") as reader, pending.open("wb") as writer:
            shutil.copyfileobj(reader, writer, 8*1024**2); writer.flush(); os.fsync(writer.fileno())
        if rebuild_file_sha256_v12(pending) != expected_sha:
            raise RuntimeError("Local test ZIP copy SHA256 differs")
        pending.replace(target); copied = True
    return target, {"source_zip_verified": True, "local_zip_verified": True,
        "local_copy_performed_this_call": copied, "zip_sha256": expected_sha}


def _compact_rgb_sha_v26(image):
    if (not isinstance(image, torch.Tensor) or image.dtype != torch.float32
            or tuple(image.shape) != (3, 256, 256) or not bool(torch.isfinite(image).all())):
        raise ValueError("Keep the frozen finite FP32 RGB256 preprocessing")
    raster = ((image.permute(1,2,0)+1)*127.5).round().to(torch.uint8).contiguous().numpy()
    return hashlib.sha256(raster.tobytes()).hexdigest()


def prepare_compact_final_data_v26(drive_root, local_root, lock, source_configuration):
    """Return fixed RGB200, ordered file hashes and an audit-only manifest.

    Decode all test members to detect selected RGB duplicates against the entire
    test pool. Keep only the selected tensor images. Training ZIPs, models and
    quality metrics are never loaded here.
    """
    memberships = validate_compact_final_data_lock_v26(lock)
    training_sha = _compact_source_training_inventory_v26(source_configuration, lock, memberships)
    path, copy_audit = _compact_test_zip_copy_v26(drive_root, local_root, lock["test_zip_sha256"])
    rows, selected_images, rgb_members = [], {}, {}
    with zipfile.ZipFile(path) as archive:
        members = []
        for item in archive.infolist():
            source_names = (item.filename, item.orig_filename)
            if (any(PurePosixPath(name).is_absolute() or ".." in PurePosixPath(name).parts
                    or "\\" in name or any(":" in part for part in PurePosixPath(name).parts)
                    for name in source_names) or item.file_size > 128*1024**2
                    or stat.S_ISLNK(item.external_attr >> 16)):
                raise ValueError("Unsafe test ZIP member; no extraction performed")
            name = PurePosixPath(item.filename)
            if item.is_dir() or name.name.startswith(".") or "__MACOSX" in name.parts:continue
            if name.suffix.lower() in {".png", ".jpg", ".jpeg"}:members.append(item)
        if (len(members) != lock["expected_total_images"]
                or len({item.filename for item in members}) != len(members)):
            raise RuntimeError("The frozen test ZIP image count/names differ")
        for i, item in enumerate(sorted(members, key=lambda value: value.filename)):
            content = archive.read(item); digest = hashlib.sha256(content).hexdigest()
            image = _rebuild_image_v12(content); rgb_sha = _compact_rgb_sha_v26(image)
            rows.append({"file_sha256": digest, "bytes": len(content), "rgb256_sha256": rgb_sha})
            rgb_members.setdefault(rgb_sha, []).append(digest)
            if digest in memberships["selected"]:
                if digest in selected_images:raise RuntimeError("Duplicate selected file content in test ZIP")
                selected_images[digest] = image
            if (i+1) % 250 == 0 or i+1 == len(members):
                print(f"Frozen test-only RGB audit: {i+1}/{len(members)}", flush=True)
    observed = [row["file_sha256"] for row in rows]
    if len(set(observed)) != len(observed) or set(observed) != memberships["all"]:
        raise RuntimeError("The complete test-file SHA256 inventory differs")
    if set(selected_images) != memberships["selected"]:
        raise RuntimeError("A predeclared confirmation image is missing")
    collisions = [digests for digests in rgb_members.values()
                  if len(digests) > 1 and memberships["selected"].intersection(digests)]
    if collisions:
        raise RuntimeError("A selected image duplicates preprocessed RGB in the test pool; stop without resampling")
    selected_hashes = list(lock["selected_file_sha256"])
    images = torch.stack([selected_images[digest] for digest in selected_hashes])
    by_hash = {row["file_sha256"]: row for row in rows}
    canonical_inventory = sorted(rows, key=lambda row: row["file_sha256"])
    manifest = {"version": COMPACT_FINAL_DATA_VERSION_V26, "data_lock_sha256": lock["data_lock_sha256"],
        "images": len(images), "test_images": len(rows), "test_zip_sha256": lock["test_zip_sha256"],
        "selected_file_sha256": selected_hashes,
        "selected_rgb256_sha256": [by_hash[digest]["rgb256_sha256"] for digest in selected_hashes],
        "all_test_file_inventory_sha256": hashlib.sha256("\n".join(sorted(observed)).encode()).hexdigest(),
        "all_test_rgb_inventory_sha256": _compact_data_json_sha_v26(canonical_inventory),
        "source_training_inventory_sha256": training_sha,
        "preprocessing": _COMPACT_DATA_PREPROCESSING_V26,
        "known_prior_evaluation_excluded": len(memberships["excluded"]),
        "prior_role_counts": {key: len(value) for key, value in lock["excluded_roles"].items()},
        "selected_prior_file_overlap": 0, "selected_train_file_overlap": 0,
        "all_test_train_file_overlap": 0, "selected_test_rgb_duplicate_count": 0,
        "nonselected_test_rgb_duplicate_groups": sum(len(values)>1 for values in rgb_members.values()),
        "membership_used_pixels_or_quality_scores": False, "selection_role": lock["selection_role"],
        "training_RGB_duplicate_check_this_run": False,
        "geographic_or_near_duplicate_independence_established": False,
        "entirely_new_external_dataset_claimed": False, "source_zips_modified": False,
        "training_zips_loaded": False, "quality_metrics_called": False, "earth_engine_calls": 0,
        # Transfer work is per-call operational state, not experiment identity.
        # A verified local-cache hit must reproduce exactly the same manifest.
        "copy_audit": {key: value for key, value in copy_audit.items()
                       if key != "local_copy_performed_this_call"}}
    return images, selected_hashes, manifest


In [ ]:
import json
from pathlib import Path

def _compact_atomic_bytes_v26(root, relative, content):
    path = _pilot_child_v12(root, relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_name(path.name+".partial")
    if pending.is_symlink():raise ValueError("Linked compact-result pending file")
    with pending.open("wb") as handle:
        handle.write(content);handle.flush()
    pending.replace(path)
    return path


In [ ]:
import torch

def _extension_runtime_v22(device):
    return {"torch": str(torch.__version__), "cuda": torch.version.cuda, "device": str(device),
        "gpu": torch.cuda.get_device_name(device) if torch.device(device).type == "cuda" else None}


In [ ]:
import hashlib,json,math
import numpy as np
import torch
SOURCE_FACTOR_VERSION_V28='frozen_source_codec_rate_chroma_controls_v28_1'

def _source_json_sha_v28(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def _source_hashes_v28(hashes):
    if (not isinstance(hashes,list) or not hashes or len(set(hashes))!=len(hashes)
            or any(not isinstance(h,str) or len(h)!=64 or any(c not in '0123456789abcdef' for c in h) for h in hashes)):
        raise ValueError('Use ordered distinct lowercase file SHA256 identities')

def select_source_diagnostic_members_v28(all_hashes,count=32,seed=20261108):
    """Result-blind diagnostic subset of an already-used image pool, NOT holdout."""
    _source_hashes_v28(all_hashes)
    if type(count) is not int or not 1<=count<=len(all_hashes) or type(seed) is not int or seed<0:
        raise ValueError('Invalid explicit diagnostic subset size or seed')
    ranked=sorted(all_hashes,key=lambda digest:hashlib.sha256(f'factor-source-v28:{seed}:{digest}'.encode()).digest())
    selected=ranked[:count];positions={h:i for i,h in enumerate(all_hashes)}
    manifest={'version':SOURCE_FACTOR_VERSION_V28,'selection':'SHA256 rank without pixels or quality metrics',
        'selected_file_sha256':selected,'source_membership_sha256':_source_json_sha_v28(all_hashes),
        'source_pool_count':len(all_hashes),'count':count,'seed':seed,
        'already_used_v26_v27_pool':True,'untouched_holdout_claimed':False,'selection_used_quality':False}
    manifest['manifest_sha256']=_source_json_sha_v28(manifest)
    return {'selected_hashes':selected,'positions':[positions[h] for h in selected],'manifest':manifest}

def source_scores_v28(prediction,target,metrics,device='cpu'):
    """LPIPS consumes[-1,1]; PSNR is separately computed with RGB[0,1] peak1."""
    target=target.to(device);prediction=prediction.to(device)
    if (tuple(prediction.shape)!=tuple(target.shape) or not bool(torch.isfinite(prediction).all())
            or bool((prediction.abs()>1.000001).any())):raise ValueError('Invalid source reconstruction')
    mse=float((prediction-target).square().mean())
    result={'mse_minus1_1':mse,'mse_0_1':mse/4,'psnr_db':None if mse==0 else 10*math.log10(4/mse),
        'pixel_exact':bool(torch.equal(prediction,target)),
        'psnr_infinite_for_exact_pixels':mse==0}
    if set(metrics)!={'alex','vgg'}:raise ValueError('Keep declared Alex and VGG LPIPS backbones')
    for name,metric in metrics.items():
        score=metric(prediction,target).reshape(-1)
        if score.numel()!=1 or not bool(torch.isfinite(score).all()):raise FloatingPointError('Nonfinite source LPIPS')
        result['lpips_'+name]=float(score[0])
    return result


In [ ]:
import hashlib,json
POWER_EXTENSION_VERSION_V31 = "frozen_launch_power_extension_v31_1"
POWER_EXTENSION_FAMILIES_V31 = ("clear16", "snowqpsk")
POWER_EXTENSION_METHODS_V31 = ("fso_sc", "bpg_ldpc")
POWER_EXTENSION_POWERS_V31 = {
    "clear16": tuple(sorted(set(range(14, 43, 2)) | {33})),
    "snowqpsk": tuple(sorted(set(range(24, 55, 2)) | set(range(28, 37)))),
}
_POWER_PUBLISHED_V31 = {"clear16": (30, 33, 36), "snowqpsk": tuple(range(28, 37))}

def _power_json_sha_v31(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()

def power_extension_protocol_v31():
    """Use exact geometry, without any power adjustment fitted to LPIPS."""
    conditions, cases, anchors = [], [], {}
    for group in POWER_EXTENSION_FAMILIES_V31:
        clear = group == "clear16"
        for power in POWER_EXTENSION_POWERS_V31[group]:
            kwargs = dict(modulation="16QAM" if clear else "QPSK", launch_power_dbm=float(power),
                          zenith_angle_deg=30. if clear else 0., pointing_error_urad=20. if clear else 15.,
                          scattered_power_ratio=.95, background_light_radiance_w_m2_sr_um=1e-4,
                          weather="clear" if clear else "snow")
            condition = {"id": _power_json_sha_v31(kwargs)[:16], "label": str(kwargs),
                         "group": group, "channel_kwargs": kwargs}
            conditions.append(condition)
            for method in POWER_EXTENSION_METHODS_V31:
                case = {"method": method, "condition": condition, "receivers": 1, "combining": "MRC"}
                cases.append({"id": _power_json_sha_v31(case)[:20], **case})
                if power in _POWER_PUBLISHED_V31[group]:
                    v27 = {**case, "condition": {**condition, "group": "paper_figures_v27"}}
                    anchors.setdefault(condition["id"], {})[method] = _power_json_sha_v31(v27)[:20]
    result = {
        "version": POWER_EXTENSION_VERSION_V31,
        "conditions": conditions, "cases": cases, "v27_anchor_case_ids": anchors,
        "groups": list(POWER_EXTENSION_FAMILIES_V31), "methods": list(POWER_EXTENSION_METHODS_V31),
        "powers_dbm": {g: list(p) for g, p in POWER_EXTENSION_POWERS_V31.items()},
        "published_comparison_powers_dbm": {g: list(p) for g, p in _POWER_PUBLISHED_V31.items()},
        "primary_profile": "coherent_receiver_extension_v6",
        "images": 32, "physical_condition_count": len(conditions), "physical_image_method_evaluations": len(cases)*32,
        "base_seed": 20261107, "channel_realizations_per_image": 1,
        "power_seed_policy": "Same v27 private seed and unchanged source payload per group/method/image across power.",
        "over_36_dbm": "Numerical extension of the simulator only; not an operational or hardware assertion.",
        "max_launch_power_dbm": 54, "max_nominal_launch_power_w": 10**((54-30)/10),
        "training_updates": 0, "power_shift_fitted": False, "monotone_fit_applied": False,
        "paper_extrapolated": False, "no_cross_method_identical_waveform_claim": True,
    }
    result["protocol_sha256"] = _power_json_sha_v31(result)
    return result


In [ ]:
import hashlib,json

def paper_sweep_seed_v27(condition,image_sha256,base_seed=20261107):
    """Common random numbers across power, not common method-dependent payloads."""
    family=None if condition is None else {k:v for k,v in condition['channel_kwargs'].items() if k!='launch_power_dbm'}
    return int.from_bytes(hashlib.sha256(json.dumps([base_seed,family,image_sha256],sort_keys=True,separators=(',',':')).encode()).digest()[:8],'big')%(2**63-1)


In [ ]:
import math
import torch

def _factor_bits_v28(bits,device):
    if (not isinstance(bits,torch.Tensor) or bits.dtype!=torch.uint8 or bits.ndim!=2
            or bits.shape[0]!=1 or bits.shape[1]<32 or not bool(((bits==0)|(bits==1)).all())):
        raise ValueError('Expected one nonempty uint8 bit frame[1,L>=32]')
    return bits.detach().to(device).clone()

def _factor_seed_v28(seed):
    if type(seed) is not int or not 0<=seed<2**63:raise ValueError('Use an explicit nonnegative63-bit seed')
    return seed

def _factor_device_v28(device):
    device=torch.device(device)
    if device.type=='cuda' and device.index is None:device=torch.device('cuda',torch.cuda.current_device())
    if device.type not in ('cpu','cuda'):raise ValueError('Use the existing CPU or CUDA diagnostic runtime')
    return device

def _factor_ldpc_v28(ldpc,device,include_100_iterations,ldpc100=None):
    if type(include_100_iterations) is not bool:raise ValueError('Declare decoder sensitivity explicitly')
    if ldpc is None:
        if include_100_iterations or ldpc100 is not None:raise ValueError('100-iteration control requires the fixed50-iteration code')
        return None
    if (ldpc.n!=2048 or ldpc.k!=1536 or ldpc.max_iterations!=50
            or ldpc.normalization!=.8 or ldpc.seed!=20261028 or ldpc.llr_clip!=30.):
        raise ValueError('Keep the existing frozen2048/1536 seededRA50-iteration settings')
    if ldpc.check_variables.device!=device:
        raise ValueError('LDPC and diagnostic input must share a device')
    if not include_100_iterations:
        if ldpc100 is not None:raise ValueError('Do not supply an unused100-iteration decoder')
        return None
    more=(type(ldpc)(max_iterations=100,normalization=ldpc.normalization,seed=ldpc.seed,llr_clip=ldpc.llr_clip).to(device)
        if ldpc100 is None else ldpc100)
    if (more.matrix_sha256!=ldpc.matrix_sha256 or more.max_iterations!=100 or more.normalization!=ldpc.normalization
            or more.seed!=ldpc.seed or more.llr_clip!=ldpc.llr_clip or more.check_variables.device!=device):
        raise RuntimeError('Decoder sensitivity changed device, matrix or a setting other than iteration limit')
    return more

def _factor_encode_v28(namespace,source,modulation,ldpc):
    length=source.numel();blocks=math.ceil(length/ldpc.k) if ldpc is not None else 0
    information_padding=blocks*ldpc.k-length if ldpc is not None else 0
    if ldpc is None:information=None;coded=source
    else:
        information=torch.cat((source.reshape(-1),torch.zeros(information_padding,dtype=torch.uint8,device=source.device))).reshape(blocks,ldpc.k)
        coded=ldpc.encode_bits(information).reshape(1,-1)
    _,_,bps=namespace['qam_constellation'](modulation,source.device)
    qam_padding=(-coded.numel())%bps
    padded=torch.cat((coded,torch.zeros((1,qam_padding),dtype=torch.uint8,device=source.device)),dim=-1)
    grouped=padded.reshape(1,-1,bps);symbols=namespace['qam_modulate'](grouped,modulation)
    if symbols.shape[-1]<8:raise ValueError('Receiver needs at least8QAM symbols')
    return {'coded':coded,'information':information,'groups':grouped,'symbols':symbols,
        'blocks':blocks,'information_padding':information_padding,'qam_padding':qam_padding,'bps':bps}

def _factor_n0_v28(observation,configuration):
    gain=observation.channel_gain
    if (observation.is_equalized or gain is None or gain.shape!=observation.symbols.shape
            or not bool(torch.isfinite(gain).all()) or not bool(torch.isfinite(observation.symbols).all())):
        raise FloatingPointError('Require finite raw single-Rx physical observation and aligned gain')
    magnitude=gain.abs().mean(-1,keepdim=True)
    if not bool((magnitude>0).all()):raise FloatingPointError('Nonpositive raw channel gain')
    budget=observation.diagnostics['budget']
    noise=(budget['background_noise_w'] if configuration.include_background_noise else 0.)
    noise+=budget['receiver_shot_noise_w'] if configuration.include_shot_noise else 0.
    n0=noise/magnitude.double().square().clamp_min(1e-300)
    if configuration.include_ase_noise:n0+=budget['ase_noise_w']/budget['launch_power_w']
    if not bool(torch.isfinite(n0).all()) or bool((n0<0).any()):raise FloatingPointError('Invalid equalized noise variance')
    return magnitude,n0

def _factor_decoded_v28(namespace,recovered,source,wave,modulation,n0,ldpc,more):
    if recovered.shape!=wave['symbols'].shape or not bool(torch.isfinite(recovered).all()):
        raise FloatingPointError('Receiver output is nonfinite or wrong-shaped')
    hard_groups=namespace['qam_demodulate'](recovered,modulation)
    if hard_groups.shape!=wave['groups'].shape:raise ValueError('Receiver QAM bit grouping changed')
    hard=hard_groups.reshape(1,-1)[:,:wave['coded'].numel()]
    pre_errors=int((hard!=wave['coded']).sum())
    llr=None;statistics=None;results=[]
    if ldpc is None:
        choices=[('uncoded',hard[:,:source.numel()],None,None)]
    else:
        llr=soft_qam_llr(recovered,modulation,n0,namespace,clip=ldpc.llr_clip).reshape(-1)[:wave['coded'].numel()].reshape(wave['blocks'],ldpc.n)
        choices=[]
        for label,code in (('ldpc50',ldpc),('ldpc100_DIAGNOSTIC_ONLY',more)):
            if code is None:continue
            information,statistics=code.decode_llr(llr)
            choices.append((label,information.reshape(1,-1)[:,:source.numel()],statistics,information))
    for label,decoded,statistics,information in choices:
        errors=int((decoded!=source).sum())
        systematic=(hard.reshape(wave['blocks'],ldpc.n)[:,:ldpc.k].reshape(1,-1)[:,:source.numel()]
            if ldpc is not None else hard[:,:source.numel()])
        systematic_errors=int((systematic!=source).sum())
        row={'fec':label,'source_bits':source.numel(),'coded_bits':wave['coded'].numel(),
            'transmitted_bits':wave['coded'].numel()+wave['qam_padding'],'channel_symbols':wave['symbols'].numel(),
            'bits_per_symbol':wave['bps'],'information_padding_bits':wave['information_padding'],'qam_padding_bits':wave['qam_padding'],
            'pre_fec_bit_errors':pre_errors,'pre_fec_valid_bits':wave['coded'].numel(),'pre_fec_ber':pre_errors/wave['coded'].numel(),
            'systematic_predecoder_information_bit_errors':systematic_errors,
            'systematic_predecoder_valid_information_bits':source.numel(),
            'systematic_predecoder_information_ber':systematic_errors/source.numel(),
            'bit_errors':errors,'valid_bits':source.numel(),'payload_ber':errors/source.numel(),'frame_error':bool(errors),
            'symbol_errors':int((hard_groups!=wave['groups']).any(-1).sum()),'valid_symbols':wave['groups'].shape[-2],
            'evm_squared_truth_diagnostic_only':float((recovered-wave['symbols']).abs().square().mean()/wave['symbols'].abs().square().mean()),
            'soft_llr_sha256':None if llr is None else _rebuild_sha_v12(llr),'ldpc_blocks':wave['blocks'],
            'ldpc_nonconverged_blocks':0,'ldpc_iterations_mean':0.,'ldpc_iterations_max':0,
            'ldpc_initial_syndrome_weight_sum':0,'ldpc_final_syndrome_weight_sum':0,
            'ldpc_information_block_errors':None,'ldpc_undetected_wrong_converged_blocks':None,
            'actual_payload_rate_information_per_coded_bit':source.numel()/wave['coded'].numel(),
            'post_error_statistics_use_truth_after_decoding_only':True,'parity_convergence_is_CRC':False}
        if statistics is not None:
            block_wrong=(information!=wave['information']).any(-1)
            row.update(ldpc_nonconverged_blocks=int((~statistics['converged']).sum()),
                ldpc_iterations_mean=float(statistics['iterations'].float().mean()),ldpc_iterations_max=int(statistics['iterations'].max()),
                ldpc_initial_syndrome_weight_sum=int(statistics['initial_syndrome_weight'].sum()),
                ldpc_final_syndrome_weight_sum=int(statistics['syndrome_weight'].sum()),
                ldpc_information_block_errors=int(block_wrong.sum()),
                ldpc_undetected_wrong_converged_blocks=int((block_wrong&statistics['converged']).sum()))
        results.append((row,decoded.detach().cpu().clone()))
    return results


In [ ]:
"""Read-only legacy v6/v3 physical transport on a fixed expanded power grid.

Unlike v29 noise-budget or v30 noise-interpretation checks, this changes only
the nominal optical launch power passed to the original physical function.
No monkeypatching of the original paper grid, no SNR re-labeling, no model
selection and no receiver/FEC changes. Out-of-paper conditions are diagnostic.
"""

from dataclasses import asdict
from functools import lru_cache
import hashlib
import json
import math

import torch



POWER_EXTENSION_LINK_VERSION_V31 = "fixed_grid_native_v6_power_extension_v31_1"
POWER_EXTENSION_PROFILE_V31 = "coherent_receiver_extension_v6"
_POWER_EXTENSION_CHANNEL_VERSION_V31 = "chen2025_eq3_18_heterodyne_shot_v6"
_POWER_EXTENSION_FIELDS_V31 = (
    "modulation", "launch_power_dbm", "zenith_angle_deg", "pointing_error_urad",
    "weather", "scattered_power_ratio", "background_light_radiance_w_m2_sr_um",
)
_POWER_EXTENSION_NUMERIC_V31 = tuple(key for key in _POWER_EXTENSION_FIELDS_V31
                                     if key not in ("modulation", "weather"))


def _extension_link_condition_key_v31(raw):
    if not isinstance(raw, dict) or set(raw) != set(_POWER_EXTENSION_FIELDS_V31):
        raise ValueError("Use an explicit v31 condition with the seven physical parameters")
    normalized = {}
    for name in _POWER_EXTENSION_FIELDS_V31:
        value = raw[name]
        if name in _POWER_EXTENSION_NUMERIC_V31:
            if type(value) not in (int, float) or not math.isfinite(value):
                raise ValueError(f"Nonfinite or nonnumeric v31 channel parameter: {name}")
            value = float(value)
        elif type(value) is not str:
            raise ValueError(f"Invalid v31 channel string: {name}")
        normalized[name] = value
    return tuple(normalized[k] for k in _POWER_EXTENSION_FIELDS_V31)


@lru_cache(maxsize=1)
def _extension_link_whitelist_v31():
    conditions = power_extension_protocol_v31()["conditions"]
    table = {}
    for condition in conditions:
        key = _extension_link_condition_key_v31(condition["channel_kwargs"])
        if key in table:
            raise ValueError("Duplicate condition in fixed v31 protocol")
        table[key] = condition["id"]
    if len(table) != 36:
        raise ValueError("Keep the declared 36 distinct v31 channel conditions")
    return table


def _extension_link_condition_v31(condition):
    raw = condition.get("channel_kwargs", condition) if isinstance(condition, dict) else condition
    key = _extension_link_condition_key_v31(raw)
    cond_id = _extension_link_whitelist_v31().get(key)
    if cond_id is None or (raw is not condition and condition.get("id") != cond_id):
        raise ValueError("Condition outside fixed v31 extension grid or inconsistent condition ID")
    return dict(zip(_POWER_EXTENSION_FIELDS_V31, key)), cond_id


def _extension_physical_v31(namespace, cache):
    config_type = namespace["PaperFSOSystemConfig"]
    config = namespace["make_channel_profile"](POWER_EXTENSION_PROFILE_V31, config_type)
    if (namespace.get("PAPER_FSO_CHANNEL_VERSION") != _POWER_EXTENSION_CHANNEL_VERSION_V31
            or config.num_receivers != 1 or not all((
                config.include_background_noise, config.include_shot_noise, config.include_ase_noise,
                config.include_turbulence, config.include_doppler, config.include_phase_noise))):
        raise ValueError("Use the frozen single-Rx full coherent v6 channel")
    config_values = asdict(config)
    if type(config) is not config_type or config_values != asdict(config_type()):
        raise ValueError("The v31 launch-power experiment does not override any base channel parameter")
    key = hashlib.sha256(json.dumps(config_values, sort_keys=True, separators=(",", ":"),
                                    allow_nan=False).encode("utf-8")).hexdigest()
    physical_type = namespace["PaperMalagaFSOChannel"]
    if cache is None:
        physical = physical_type(config)
    else:
        if not isinstance(cache, dict):
            raise TypeError("Provide a caller-owned physical cache dict")
        if key not in cache:
            cache[key] = physical_type(config)
        physical = cache[key]
    if type(physical) is not physical_type or asdict(physical.config) != config_values:
        raise ValueError("Physical cache does not match the frozen class and configuration")
    return physical, key


@torch.inference_mode()
def power_extension_transport_v31(namespace, bits, condition, seed, *, device="cpu", ldpc=None,
                                  physical_cache=None):
    """Transmit real uint8 bits, changing only launch power on the fixed grid.

    Frozen v6 call, v3 demodulator, real block magnitude/PSD side information,
    and fixed optional LDPC50. No complex-gain oracle or FEC success filter.
    Returns one record and CPU decoded bits, leaving image reconstruction and
    metrics to the caller. Each frame uses exactly one physical invocation.
    """
    d = _factor_device_v28(device)
    source = _factor_bits_v28(bits, d)
    seed = _factor_seed_v28(seed)
    condition, condition_id = _extension_link_condition_v31(condition)
    _factor_ldpc_v28(ldpc, d, False)
    receiver = namespace.get("recover_qam_carrier_v3")
    if not callable(receiver):
        raise ValueError("Use the frozen shared v3 DSP")
    physical, config_sha = _extension_physical_v31(namespace, physical_cache)
    c = physical.config
    modulation = condition["modulation"]
    wave = _factor_encode_v28(namespace, source, modulation, ldpc)
    kwargs = {k: v for k, v in condition.items() if k != "modulation"}
    generator = torch.Generator(device=d).manual_seed(seed)
    obs, mode = _rebuild_observe_v12(physical, wave["symbols"], generator, kwargs)
    magnitude, n0 = _factor_n0_v28(obs, c)
    recovery, diagnostics = receiver(obs.symbols, modulation, c.symbol_rate_baud,
                                     known_block_magnitude=magnitude)
    [(row, decoded)] = _factor_decoded_v28(namespace, recovery, source, wave, modulation, n0, ldpc, None)
    energy = float(wave["symbols"].abs().square().mean())
    variance = float(n0.mean())
    budget = obs.diagnostics["budget"]
    launch, loss = budget["launch_power_w"], budget["total_attenuation_db"]
    v27_unit_snr = obs.diagnostics["mean_snr_db"]
    if not all(math.isfinite(v) and v > 0 for v in (energy, variance, launch)):
        raise FloatingPointError("Expected positive finite launch, symbol energy and noise variance")
    if not math.isfinite(v27_unit_snr):
        raise FloatingPointError("Original physical link must report finite SNR")
    signal_at_receiver = float(obs.channel_gain.abs().square().mean())*energy
    nominal_path = 10**(-loss/10.)
    if ldpc is None and source.numel() == 14336:
        token_errors = int((source.detach().cpu().reshape(-1,14) != decoded.reshape(-1,14)).any(-1).sum())
        tokens = 1024
    else:
        token_errors, tokens = 0, 0
    row.update(
        profile=POWER_EXTENSION_PROFILE_V31, seed=seed, condition=condition,
        condition_id=condition_id, group="clear16" if modulation=="16QAM" else "snowqpsk",
        launch_power_dbm=condition["launch_power_dbm"], receiver="current_v3",
        source_bits_sha256=_rebuild_sha_v12(source),
        transmitted_symbols_sha256=_rebuild_sha_v12(wave["symbols"]),
        observation_sha256=_rebuild_sha_v12(obs.symbols),
        received_bits_sha256=_rebuild_sha_v12(decoded),
        channel_gain_sha256=_rebuild_sha_v12(obs.channel_gain),
        channel_rng_state_sha256=_rebuild_sha_v12(generator.get_state()),
        deterministic_physical=mode, physical_config_sha256=config_sha,
        physical_calls=1, symbol_rate_baud=c.symbol_rate_baud,
        source_bpp=source.numel()/65536., transmitted_bpp=row["transmitted_bits"]/65536.,
        index_errors=token_errors, indices=tokens, token_errors=token_errors, tokens=tokens,
        token_error_rate=token_errors/tokens if tokens else None,
        nominal_launch_power_w=launch, actual_launch_power_w=launch*energy,
        transmitted_symbols_mean_power=energy, deterministic_loss_db=loss,
        nominal_received_power_no_random_fade_w=launch*nominal_path,
        actual_received_signal_power_with_fade_w=signal_at_receiver,
        random_fade_power_diagnostic_only=signal_at_receiver/(launch*nominal_path*energy),
        unit_symbol_esn0_db=v27_unit_snr, actual_symbol_esn0_db=v27_unit_snr+10.*math.log10(energy),
        noise_variance_equalized=variance,
        noise_budget_w={name: budget[name] for name in (
            "background_noise_w", "receiver_shot_noise_w", "ase_noise_w")},
        receiver_diagnostics=diagnostics,
    )
    json.dumps(row, allow_nan=False)
    return {"version": POWER_EXTENSION_LINK_VERSION_V31, "record": row,
            "decoded_bits": decoded, "physical_waveform_calls": 1, "training_updates": 0}


In [ ]:
"""Image-paired Stage1/Stage2 inference reports, not a noise-training causal claim.

The reported difference-in-differences subtracts each checkpoint's own clean
reconstruction.  The whole trained model can change, and there is no matched
clean-only continuation arm.  Thus an improvement here is a checkpoint-level
robustness association, not proof that noisy training alone caused it.
"""
import math
from pathlib import Path
import re

import numpy as np


STAGE_EFFECT_REPORT_VERSION_V32 = "paired_checkpoint_clean_adjusted_effect_v32_1"
STAGE_EFFECT_MODELS_V32 = ("stage1", "stage2")
STAGE_EFFECT_BOOTSTRAP_SEED_V32 = 20261132
_STAGE_EFFECT_METRICS_V32 = ("lpips_alex", "lpips_vgg", "mse_minus1_1")
_STAGE_EFFECT_PAIR_FIELDS_V32 = (
    "seed", "valid_bits", "tokens", "physical_config_sha256",
    "channel_gain_sha256", "channel_rng_state_sha256", "deterministic_physical",
)
_STAGE_EFFECT_LABELS_V32 = {
    "clear16": "Clear / 16QAM / zenith 30 deg",
    "snowqpsk": "Snow / QPSK / zenith 0 deg",
}
_STAGE_EFFECT_STYLE_V32 = {
    "stage1": ("Stage 1", "#446587", "o"),
    "stage2": ("Stage 2 (v25)", "#bc562d", "s"),
}


def _stage_effect_require_v32(condition, message):
    if not condition:
        raise ValueError(message)


def _stage_effect_sha_v32(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def _stage_effect_number_v32(value, *, nonnegative=False):
    return (type(value) in (int, float) and math.isfinite(value)
            and (not nonnegative or value >= 0))


def _stage_effect_metric_validation_v32(row):
    for key in _STAGE_EFFECT_METRICS_V32:
        _stage_effect_require_v32(_stage_effect_number_v32(row.get(key), nonnegative=True),
                                  "Nonfinite or invalid reconstruction metric: "+key)
    psnr = row.get("psnr_db")
    _stage_effect_require_v32(_stage_effect_number_v32(psnr) or (
        psnr is None and row["mse_minus1_1"] == 0), "Invalid PSNR; exact pixels use JSON null")


def _stage_effect_physical_validation_v32(row, condition):
    _stage_effect_metric_validation_v32(row)
    _stage_effect_require_v32(row.get("group") == condition["group"]
        and row.get("launch_power_dbm") == condition["channel_kwargs"]["launch_power_dbm"],
        "Physical result does not belong to the declared condition")
    if "condition" in row:
        _stage_effect_require_v32(row["condition"] == condition["channel_kwargs"],
                                  "Physical result channel parameters differ")
    _stage_effect_require_v32(type(row.get("seed")) is int and 0 <= row["seed"] < 2**63,
                              "Invalid channel seed")
    for field in ("physical_config_sha256", "channel_gain_sha256", "channel_rng_state_sha256"):
        _stage_effect_require_v32(_stage_effect_sha_v32(row.get(field)), "Missing or invalid external-channel hash: "+field)
    _stage_effect_require_v32(row.get("deterministic_physical") in (
        "cpu_native", "cuda_scoped_deterministic"),
        "External-channel pairing is unverified on this deterministic fallback")
    for errors, total, expected in (("bit_errors", "valid_bits", 14336), ("token_errors", "tokens", 1024)):
        _stage_effect_require_v32(type(row.get(errors)) is int and type(row.get(total)) is int
            and row[total] == expected and 0 <= row[errors] <= expected,
            "Uncoded semantic payload must retain 14336 bits and 1024 tokens")
    for field, numerator, denominator in (("payload_ber", "bit_errors", "valid_bits"),
                                          ("token_error_rate", "token_errors", "tokens")):
        _stage_effect_require_v32(_stage_effect_number_v32(row.get(field), nonnegative=True)
            and math.isclose(row[field], row[numerator]/row[denominator], rel_tol=0, abs_tol=1e-12),
            "Saved error rate differs from its integer counters: "+field)
    for field in ("unit_symbol_esn0_db", "actual_symbol_esn0_db", "transmitted_symbols_mean_power"):
        _stage_effect_require_v32(_stage_effect_number_v32(row.get(field)), "Invalid physical diagnostic: "+field)
    energy = row["transmitted_symbols_mean_power"]
    _stage_effect_require_v32(energy > 0 and math.isclose(row["actual_symbol_esn0_db"],
        row["unit_symbol_esn0_db"]+10*math.log10(energy), abs_tol=1e-5, rel_tol=0),
        "Actual symbol SNR must disclose the model-dependent constellation energy")


def _stage_effect_summary_v32(values, resamples):
    array = np.asarray(values, dtype=np.float64)
    _stage_effect_require_v32(array.ndim == 1 and np.isfinite(array).all(), "Expected finite aligned image values")
    return {"mean": float(array.mean()), "ci95": np.quantile(array[resamples].mean(1), (.025, .975)).tolist()}


def summarize_stage_effect_v32(rows, clean_rows, protocol, bootstrap_samples=2000):
    """Summarize one complete grid; negative paired excess means less degradation.

    Exactly the same image resample indices are used for both checkpoints,
    both baselines, every power point and both LPIPS backbones.  CIs are
    pointwise image-bootstrap intervals, not independent-channel confidence
    limits or multiplicity-adjusted inference over the whole curve.
    """
    _stage_effect_require_v32(type(bootstrap_samples) is int and bootstrap_samples >= 2,
                              "Use at least two image bootstrap replicates")
    _stage_effect_require_v32(isinstance(protocol, dict) and isinstance(protocol.get("conditions"), list),
                              "Missing fixed physical protocol")
    conditions = protocol["conditions"]
    images = protocol.get("images")
    _stage_effect_require_v32(type(images) is int and images >= 2 and bool(conditions), "Invalid fixed image/condition count")
    conds = {c["id"]: c for c in conditions}
    _stage_effect_require_v32(len(conds) == len(conditions), "Duplicate declared physical condition")
    clean = {model: {} for model in STAGE_EFFECT_MODELS_V32}
    for row in clean_rows:
        model, digest = row.get("model"), row.get("image_sha256")
        _stage_effect_require_v32(model in clean and _stage_effect_sha_v32(digest), "Invalid clean checkpoint/image identity")
        _stage_effect_metric_validation_v32(row)
        _stage_effect_require_v32(digest not in clean[model], "Duplicate clean checkpoint/image")
        clean[model][digest] = row
    hashes = list(clean["stage1"])
    _stage_effect_require_v32(len(hashes) == images and list(clean["stage2"]) == hashes,
                              "Both clean checkpoints must use the declared identical ordered image membership")
    slots = {(c["id"], model): {} for c in conditions for model in STAGE_EFFECT_MODELS_V32}
    within_power_identity = {}
    for row in rows:
        cid, model, digest = row.get("condition_id"), row.get("model"), row.get("image_sha256")
        _stage_effect_require_v32((cid, model) in slots and digest in clean["stage1"],
                                  "Result identity is outside the declared checkpoint/condition/image grid")
        _stage_effect_physical_validation_v32(row, conds[cid])
        _stage_effect_require_v32(digest not in slots[cid, model], "Duplicate physical checkpoint/condition/image")
        slots[cid, model][digest] = row
        identity_key = (row["group"], model, digest)
        constants = (row["seed"], row["valid_bits"], row["tokens"], row["physical_config_sha256"])
        if "source_bits_sha256" in row:
            _stage_effect_require_v32(_stage_effect_sha_v32(row["source_bits_sha256"]), "Invalid source bit hash")
            constants += (row["source_bits_sha256"],)
        _stage_effect_require_v32(within_power_identity.setdefault(identity_key, constants) == constants,
                                  "Model payload, external seed or channel configuration changed across power")
    _stage_effect_require_v32(all(list(slot) == hashes for slot in slots.values()),
                              "Every checkpoint/condition must contain the same complete ordered image membership")
    max_unit_snr_delta = 0.
    for c in conditions:
        for digest in hashes:
            before, after = slots[c["id"], "stage1"][digest], slots[c["id"], "stage2"][digest]
            for field in _STAGE_EFFECT_PAIR_FIELDS_V32:
                _stage_effect_require_v32(before[field] == after[field], "External-channel pairing differs: "+field)
            snr_delta = abs(before["unit_symbol_esn0_db"]-after["unit_symbol_esn0_db"])
            max_unit_snr_delta = max(max_unit_snr_delta, snr_delta)
            _stage_effect_require_v32(snr_delta <= 1e-5, "Paired unit-symbol SNR differs")
    resamples = np.random.default_rng(STAGE_EFFECT_BOOTSTRAP_SEED_V32).integers(
        0, images, size=(bootstrap_samples, images))
    clean_arrays = {model: {key: np.asarray([clean[model][h][key] for h in hashes], dtype=float)
        for key in _STAGE_EFFECT_METRICS_V32} for model in STAGE_EFFECT_MODELS_V32}
    clean_models = {model: {key: _stage_effect_summary_v32(value, resamples)
        for key, value in arrays.items()} for model, arrays in clean_arrays.items()}
    clean_delta = {key: _stage_effect_summary_v32(clean_arrays["stage2"][key]-clean_arrays["stage1"][key], resamples)
                   for key in _STAGE_EFFECT_METRICS_V32}
    points, paired_points = [], []
    for c in conditions:
        cid, group, power = c["id"], c["group"], c["channel_kwargs"]["launch_power_dbm"]
        arrays = {}
        for model in STAGE_EFFECT_MODELS_V32:
            aligned = [slots[cid, model][h] for h in hashes]
            arrays[model] = {key: np.asarray([r[key] for r in aligned], dtype=float) for key in (
                *_STAGE_EFFECT_METRICS_V32, "bit_errors", "token_errors", "unit_symbol_esn0_db",
                "actual_symbol_esn0_db", "transmitted_symbols_mean_power")}
            point = {"condition_id": cid, "group": group, "launch_power_dbm": power,
                     "model": model, "images": images, "above_paper_maximum_power": power > 36}
            for key in _STAGE_EFFECT_METRICS_V32:
                raw = _stage_effect_summary_v32(arrays[model][key], resamples)
                excess = _stage_effect_summary_v32(arrays[model][key]-clean_arrays[model][key], resamples)
                point[key], point[key+"_ci95"] = raw["mean"], raw["ci95"]
                point[key+"_minus_clean"], point[key+"_minus_clean_ci95"] = excess["mean"], excess["ci95"]
            for key, counter, denominator in (("payload_ber", "bit_errors", 14336), ("token_error_rate", "token_errors", 1024)):
                ratio = _stage_effect_summary_v32(arrays[model][counter]/denominator, resamples)
                point[key], point[key+"_ci95"] = ratio["mean"], ratio["ci95"]
                point[counter] = int(arrays[model][counter].sum())
            point.update(valid_bits=images*14336, tokens=images*1024)
            for key in ("unit_symbol_esn0_db", "actual_symbol_esn0_db", "transmitted_symbols_mean_power"):
                diagnostic = _stage_effect_summary_v32(arrays[model][key], resamples)
                point[key], point[key+"_ci95"] = diagnostic["mean"], diagnostic["ci95"]
            points.append(point)
        paired = {"condition_id": cid, "group": group, "launch_power_dbm": power, "images": images,
                  "above_paper_maximum_power": power > 36}
        for key in _STAGE_EFFECT_METRICS_V32:
            absolute = arrays["stage2"][key]-arrays["stage1"][key]
            clean_difference = clean_arrays["stage2"][key]-clean_arrays["stage1"][key]
            paired[key+"_stage2_minus_stage1"] = _stage_effect_summary_v32(absolute, resamples)
            paired[key+"_excess_stage2_minus_stage1"] = _stage_effect_summary_v32(absolute-clean_difference, resamples)
        paired_points.append(paired)
    return {"version": STAGE_EFFECT_REPORT_VERSION_V32, "protocol": protocol,
        "images": images, "image_sha256": hashes, "models": list(STAGE_EFFECT_MODELS_V32),
        "physical_conditions": len(conditions), "physical_image_model_evaluations": len(rows),
        "clean_image_model_evaluations": len(clean_rows), "points": points, "paired_points": paired_points,
        "clean_models": clean_models, "clean_stage2_minus_stage1": clean_delta,
        "primary_metric": "lpips_alex", "secondary_metrics": ["lpips_vgg", "mse_minus1_1"],
        "primary_effect": "(Stage2 noisy - Stage2 clean) - (Stage1 noisy - Stage1 clean)",
        "negative_primary_effect_means": "Stage2 has less added channel degradation relative to its own clean reference",
        "clean_subtraction_is_hard_bound": False, "negative_excess_values_clipped": False,
        "bootstrap_samples": bootstrap_samples, "bootstrap_seed": STAGE_EFFECT_BOOTSTRAP_SEED_V32,
        "bootstrap_convention": "Pointwise percentile 95% image bootstrap, identical resample indices across all models/baselines/powers; not simultaneous curve intervals",
        "pairing": {"verified_image_condition_pairs": len(conditions)*images,
            "exact_equal_fields": list(_STAGE_EFFECT_PAIR_FIELDS_V32),
            "max_abs_unit_symbol_snr_delta_db": max_unit_snr_delta,
            "identical_payload_waveform_observation_or_BER_claimed": False,
            "model_dependent_constellation_energy_and_actual_snr_recorded": True},
        "limitations": [
            "Reused diagnostic32, not a new independent or untouched holdout.",
            "One physical realization per image/condition; image bootstrap does not independently resample channel fades.",
            "Stage2 changes the whole encoder/codebook/decoder and adds updates; no equal-budget clean-only continuation control.",
            "A checkpoint-associated robustness difference does not isolate noisy training as its cause.",
            "Powers above36 dBm are idealized simulator diagnostics, not hardware operating recommendations.",
            "No model selection, curve fitting, monotonic smoothing, payload-power normalization or post-treatment BER/TER matching.",
        ], "training_updates": 0, "causal_noise_training_effect_claimed": False,
        "paper_numerical_reproduction_claimed": False}


def _stage_effect_axis_v32(axis, group, powers):
    axis.set_xlim(min(powers)-.7, max(powers)+.7)
    if max(powers) > 36:
        axis.axvspan(36, max(powers)+.7, color="#d9dde2", alpha=.35, zorder=-5)
    axis.grid(alpha=.22, linewidth=.65)
    axis.spines[["top", "right"]].set_visible(False)
    axis.set_xlabel("Launch optical power (dBm)")


def plot_stage_effect_v32(result, out_dir):
    """Plot measured scores and paired clean-adjusted effects, without fitting."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    groups = list(dict.fromkeys(c["group"] for c in result["protocol"]["conditions"]))
    _stage_effect_require_v32(groups == ["clear16", "snowqpsk"], "Keep the clear/snow diagnostic panel order")
    fig, axes = plt.subplots(3, 2, figsize=(13, 10.6), layout="constrained")
    for column, group in enumerate(groups):
        paired = sorted((p for p in result["paired_points"] if p["group"] == group), key=lambda p:p["launch_power_dbm"])
        powers = [p["launch_power_dbm"] for p in paired]
        for model, (label, color, marker) in _STAGE_EFFECT_STYLE_V32.items():
            points = sorted((p for p in result["points"] if p["group"] == group and p["model"] == model),
                            key=lambda p:p["launch_power_dbm"])
            for row, field in ((0, "lpips_alex"), (1, "lpips_alex_minus_clean")):
                ax = axes[row, column]
                value = np.asarray([p[field] for p in points])
                ci = np.asarray([p[field+"_ci95"] for p in points])
                ax.plot(powers, value, color=color, marker=marker, ms=3.8, lw=1.5, label=label)
                ax.fill_between(powers, ci[:, 0], ci[:, 1], color=color, alpha=.13, linewidth=0)
            axes[0, column].axhline(result["clean_models"][model]["lpips_alex"]["mean"],
                color=color, linestyle=":", lw=1.1, label=label+" clean")
        ax = axes[2, column]
        field = "lpips_alex_excess_stage2_minus_stage1"
        value = np.asarray([p[field]["mean"] for p in paired])
        ci = np.asarray([p[field]["ci95"] for p in paired])
        ax.axhline(0, color="#4a4a4a", lw=1, linestyle="--")
        ax.plot(powers, value, color="#3b806b", marker="o", ms=3.8, lw=1.5,
                label="Stage2 excess - Stage1 excess")
        ax.fill_between(powers, ci[:, 0], ci[:, 1], color="#3b806b", alpha=.18, linewidth=0)
        axes[1, column].axhline(0, color="#4a4a4a", lw=1, linestyle="--")
        axes[0, column].set_title(_STAGE_EFFECT_LABELS_V32[group]+"\nRaw reconstruction quality")
        axes[1, column].set_title("Added degradation relative to each model's clean score")
        axes[2, column].set_title("Paired difference of added degradation\nNegative = less degradation after Stage2")
        for ax in axes[:, column]:
            _stage_effect_axis_v32(ax, group, powers)
            ax.legend(fontsize=8, loc="best")
    for row, label in enumerate(("Alex LPIPS", "Alex LPIPS noisy - clean", "Paired clean-adjusted Alex LPIPS difference")):
        axes[row, 0].set_ylabel(label)
    fig.suptitle(f"Frozen Stage1 vs v25 Stage2 | same {result['images']} images, verified external channel draws", fontsize=14)
    fig.supxlabel("Bands: pointwise 95% image bootstrap; no clipping or fitting. Gray: >36 dBm simulator-only. Not noise-only causal attribution.", fontsize=9)
    path = out/"stage_effect_quality_v32.png"
    fig.savefig(path, dpi=180, bbox_inches="tight");plt.close(fig)
    fig, axes = plt.subplots(2, 2, figsize=(13, 7.2), layout="constrained")
    for column, group in enumerate(groups):
        powers = None
        for model, (label, color, marker) in _STAGE_EFFECT_STYLE_V32.items():
            points = sorted((p for p in result["points"] if p["group"] == group and p["model"] == model),
                            key=lambda p:p["launch_power_dbm"])
            powers = [p["launch_power_dbm"] for p in points]
            for row, field in enumerate(("payload_ber", "token_error_rate")):
                value = np.asarray([p[field] for p in points])
                ci = np.asarray([p[field+"_ci95"] for p in points])
                axes[row, column].plot(powers, value, color=color, marker=marker, ms=3.8, lw=1.5, label=label)
                axes[row, column].fill_between(powers, ci[:, 0], ci[:, 1], color=color, alpha=.13, linewidth=0)
        axes[0, column].set_title(_STAGE_EFFECT_LABELS_V32[group])
        for row in range(2):
            _stage_effect_axis_v32(axes[row, column], group, powers)
            axes[row, column].legend(fontsize=9)
            axes[row, column].set_ylim(bottom=0)
    axes[0, 0].set_ylabel("Payload bit error rate")
    axes[1, 0].set_ylabel("Token error rate (14 bits/token)")
    fig.suptitle("Transport context | same exogenous draws does not mean identical model payloads or errors", fontsize=13)
    fig.supxlabel(f"No error-rate matching or post-treatment image selection. Same fixed {result['images']} images at every power; gray: >36 dBm simulator-only.", fontsize=9)
    transport = out/"stage_effect_transport_v32.png"
    fig.savefig(transport, dpi=180, bbox_inches="tight");plt.close(fig)
    return [str(path.resolve()), str(transport.resolve())]


In [ ]:
"""Inference-only, same-ancestry checkpoint contrast on the fixed v31 grid.

No optimizer, source checkpoint writes, native codecs, FEC, Drive API, or model
selection. A frozen power-grid contrast is not a noise-only causal ablation.
"""
import gc
import hashlib
import json
from pathlib import Path
import time
import zipfile

import numpy as np
import torch



STAGE_EFFECT_VERSION_V32 = 'same_parent_fixed_power_checkpoint_contrast_v32_1'
STAGE_EFFECT_MODELS_V32 = ('stage1', 'stage2')


def _stage_require_v32(ok, message):
    if not ok:
        raise ValueError(message)


def stage_effect_protocol_v32():
    protocol = power_extension_protocol_v31()
    grid_sha = protocol.pop('protocol_sha256')
    for key in ('methods', 'cases', 'v27_anchor_case_ids', 'physical_image_method_evaluations'):
        protocol.pop(key, None)
    protocol.update(stage_effect_version=STAGE_EFFECT_VERSION_V32,
        inherited_v31_grid_protocol_sha256=grid_sha,
        power_seed_policy='Same v27 family/image private seed across powers and both checkpoint models; no payload-dependent seed.',
        compared_models=list(STAGE_EFFECT_MODELS_V32), training_updates=0,
        physical_evaluations=2304, clean_evaluations=64,
        primary_estimand='(Stage2 noisy - Stage2 clean) - (Stage1 noisy - Stage1 clean)',
        negative_estimand='Stage2 has less additional degradation on the same images/condition',
        extra_updates_controlled=False, noise_only_causal_claim=False,
        checkpoint_selection_from_this_evaluation=False,
        inference_only=True, same_source_bitrate=True, matched_waveforms_claimed=False)
    # v31's embedded digest describes its inherited grid, not the new contrast.
    protocol['v32_protocol_sha256'] = _pilot_json_sha_v12(protocol)
    return protocol


def discover_stage_effect_source_v32(ns):
    """Validate immutable v25 report/config, not transient resume pointers."""
    pin = ns['COMPACT_FINAL_INPUT_LOCK_V26']
    parent = Path(ns['DRIVE_DATA_DIR']) / pin['source_root_name']
    _stage_require_v32(not parent.is_symlink() and parent.is_dir(),
        'Restore FSO_data/stage2_error_range_to1_v25 from the NAS; no other model is substituted')
    candidates = []
    for folder in parent.iterdir():
        if folder.is_symlink() or not folder.is_dir() or not folder.name.startswith('extension_'):
            continue
        path = _pilot_child_v12(folder, pin['source_summary_file'])
        if (path.is_file() and path.stat().st_size == pin['source_summary_bytes']
                and _pilot_file_sha_v12(path) == pin['source_summary_file_sha256']):
            candidates.append(folder)
    _stage_require_v32(len(candidates) == 1,
        'Need one original v25 extension_* folder with the pinned summary; restore it rather than retrain')
    folder = candidates[0]
    summary = json.loads(_pilot_child_v12(folder, pin['source_summary_file']).read_text(encoding='utf-8'))
    cp = pin['source_checkpoint']
    common = json.loads(_pilot_child_v12(folder, cp['configuration_file']).read_text(encoding='utf-8'))
    configuration = common['configuration']
    endpoint = summary['endpoint']
    reference = endpoint['checkpoint']
    progress = endpoint['progress']
    ancestry = configuration['stage2_controlled_errors']
    stage1_pin = ns['STAGE1_PIN_V32']
    _stage_require_v32(summary.get('status') == 'one_total_epoch_completed'
        and summary['binding']['code_sha256'] == pin['source_code_sha256']
        and folder.name == 'extension_' + _pilot_json_sha_v12(common)[:16]
        and common == summary['common_configuration']
        and _pilot_json_sha_v12(configuration) == cp['configuration_sha256']
        and configuration['semantic_model'] == ns['SEMANTIC_MODEL_CONFIG']
        and _pilot_json_sha_v12(configuration['data']) == pin['data_manifest_json_sha256']
        and configuration['channel']['profile'] == pin['profile']
        and ancestry['stage1_checkpoint_sha256'] == stage1_pin['checkpoint_sha256']
        and ancestry['parent_configuration_sha256'] == stage1_pin['configuration_sha256']
        and endpoint['model_state_sha256'] == cp['model_state_sha256']
        and reference['configuration_sha256'] == cp['configuration_sha256']
        and reference['checkpoint_file'] == cp['file']
        and all(reference['generation'][k] == cp[k] for k in ('sha256', 'bytes', 'updates'))
        and progress['updates'] == 1375 and progress['epoch'] == 1 and progress['cursor'] == 0
        and progress['status'] == 'range_extension_budget_reached' and progress['arm'] == 'wide'
        and progress['counts'] == [10, 102, 307, 512]
        and summary['best_reference']['case'] == 'wide1375',
        'Pinned v25 report, configuration or endpoint differs; no alternate checkpoint is selected')
    return folder, summary, configuration


def load_stage_effect_model_v32(ns, label, source, summary, configuration):
    """Return a CPU named-tensor state; never restore optimizer/RNG state."""
    local = _pilot_child_v12(ns['LOCAL_CACHE'], 'checkpoints')
    if label == 'stage1':
        cp = ns['STAGE1_PIN_V32']
        result = load_ber_checkpoint_v12(_pilot_child_v12(ns['DRIVE_DATA_DIR'], cp['file']), local,
            expected_sha256=cp['checkpoint_sha256'], expected_bytes=cp['bytes'],
            expected_state_sha256=cp['model_state_sha256'],
            expected_configuration_sha256=cp['configuration_sha256'], kind='stage1')
        old = result['configuration']
        _stage_require_v32(old['semantic_model'] == configuration['semantic_model']
            and old['data'] == configuration['data'], 'Stage1 parent architecture or training data differs')
        return result['state_dict'], result['identity']
    _stage_require_v32(label == 'stage2', 'Use only the two pinned ancestral models')
    cp = ns['COMPACT_FINAL_INPUT_LOCK_V26']['source_checkpoint']
    path, audit = _ber_input_verified_copy_v12(_pilot_child_v12(source, cp['file']), local,
        'stage2_' + cp['sha256'] + '.pth', cp['sha256'], cp['bytes'])
    packet = torch.load(path, map_location='cpu', weights_only=True)
    _stage_require_v32(packet.get('stage') == 2
        and packet.get('configuration') == configuration
        and packet.get('configuration_sha256') == cp['configuration_sha256']
        and packet.get('progress') == summary['endpoint']['progress']
        and _pilot_state_sha_v12(packet['state_dict']) == cp['model_state_sha256'],
        'Pinned Stage2 checkpoint model/configuration/progress differs')
    state = {key: value.detach().cpu() for key, value in packet['state_dict'].items()}
    del packet
    gc.collect()
    return state, dict(checkpoint=audit, state_sha256=cp['model_state_sha256'],
        configuration_sha256=cp['configuration_sha256'], source_updates=1375,
        optimizer_created_or_restored=False, source_modified=False)


@torch.inference_mode()
def evaluate_stage_effect_frame_v32(ns, condition, label, digest, image, *, indices,
                                    model, metrics, physical_cache):
    _stage_require_v32(label in STAGE_EFFECT_MODELS_V32, 'Unknown checkpoint label')
    source = ns['indices_to_bits'](indices).reshape(1, -1).to(torch.uint8)
    _stage_require_v32(tuple(indices.shape) == (1, 32, 32) and tuple(source.shape) == (1, 14336),
        'Keep the common 32x32 latent with 14-bit indices')
    seed = paper_sweep_seed_v27(condition, digest)
    packet = power_extension_transport_v31(ns, source, condition, seed,
        device=ns['device'], ldpc=None, physical_cache=physical_cache)
    row = dict(packet['record'])
    _stage_require_v32(row['deterministic_physical'] in ('cpu_native', 'cuda_scoped_deterministic'),
        'This Torch cannot provide deterministic paired channel calls; use a supported Colab runtime')
    received = ns['bits_to_indices'](packet['decoded_bits'].reshape(1, 32, 32, 14)).long().to(ns['device'])
    prediction = model.decode_indices(received, received.shape).clamp(-1, 1)
    errors = int((received.cpu() != indices.cpu()).sum())
    _stage_require_v32(errors == row['token_errors'], 'Token-error accounting differs')
    row.update(model=label, image_sha256=digest, token_error_rate=errors/1024,
        **source_scores_v28(prediction, image, metrics, ns['device']))
    return row


def _stage_cache_v32(path, binding_sha, tag, hashes, runtime):
    if not path.is_file():
        return None
    p = json.loads(path.read_text(encoding='utf-8'))
    _stage_require_v32(p.get('binding_sha256') == binding_sha and p.get('case') == tag
        and p.get('runtime') == runtime and p.get('rows_sha256') == _pilot_json_sha_v12(p['rows'])
        and [r['image_sha256'] for r in p['rows']] == hashes
        and all(r['model'] == tag['model'] for r in p['rows']),
        'Cached case identity or checksum differs; do not mix conditions/runtimes')
    if tag['condition'] is not None:
        c = tag['condition']
        _stage_require_v32(all(r['condition_id'] == c['id'] and r['group'] == c['group']
            and r['condition'] == c['channel_kwargs'] for r in p['rows']), 'Cached condition differs')
    return p


def _stage_save_v32(branch, relative, binding_sha, tag, rows, hashes, runtime):
    value = dict(binding_sha256=binding_sha, case=tag, rows=rows,
        rows_sha256=_pilot_json_sha_v12(rows), runtime=runtime)
    path = _compact_atomic_bytes_v26(branch, relative,
        (json.dumps(value, allow_nan=False, separators=(',', ':')) + '\n').encode())
    return _stage_cache_v32(path, binding_sha, tag, hashes, runtime)


def publish_stage_effect_v32(branch, result):
    path = _compact_atomic_bytes_v26(branch, 'stage_effect_v32_summary.json',
        (json.dumps(result, ensure_ascii=False, allow_nan=False, indent=2) + '\n').encode())
    plots = plot_stage_effect_v32(result, branch)
    inventory = _pilot_inventory_v12(branch, [path, *[Path(x) for x in plots]])
    archive = _pilot_child_v12(branch, 'stage_effect_v32_results.zip')
    partial = _pilot_child_v12(branch, archive.name + '.partial')
    with zipfile.ZipFile(partial, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for entry in inventory:
            z.write(_pilot_child_v12(branch, entry['file']), entry['file'])
        z.writestr('manifest.json', json.dumps(dict(files=inventory), indent=2))
    with zipfile.ZipFile(partial) as z:
        _stage_require_v32(z.testzip() is None and all(
            hashlib.sha256(z.read(entry['file'])).hexdigest() == entry['sha256']
            for entry in inventory), 'Output archive integrity check failed')
    partial.replace(archive)
    return archive


def run_stage_effect_v32(ns, *, code_sha256):
    started = time.monotonic()
    protocol = stage_effect_protocol_v32()
    subset = select_source_diagnostic_members_v28(ns['COMPACT_FINAL_DATA_LOCK_V26']['selected_file_sha256'], count=32)
    hashes = subset['selected_hashes']
    runtime = _extension_runtime_v22(ns['device'])
    source, source_summary, configuration = discover_stage_effect_source_v32(ns)
    root, local = Path(ns['OUTPUT_ROOT']), Path(ns['LOCAL_CACHE'])
    _stage_require_v32(root.name == 'stage_effect_v32' and not root.is_symlink() and not local.is_symlink(),
        'Use only the scoped new v32 result directory and local cache')
    for a, b in ((root, source), (local, source), (root, local), (root, Path(ns['DRIVE_DATA_DIR'])/'rebuild_v12')):
        a, b = a.resolve(), b.resolve()
        _stage_require_v32(a != b and a not in b.parents and b not in a.parents, 'Source/output paths overlap')
    binding = dict(version=STAGE_EFFECT_VERSION_V32, code_sha256=code_sha256, protocol=protocol,
        stage1_pin=ns['STAGE1_PIN_V32'], stage2_pin=ns['COMPACT_FINAL_INPUT_LOCK_V26']['source_checkpoint'],
        v25_report_sha256=ns['COMPACT_FINAL_INPUT_LOCK_V26']['source_summary_file_sha256'],
        data_lock_sha256=ns['COMPACT_FINAL_DATA_LOCK_V26']['data_lock_sha256'],
        membership=subset['manifest'], runtime=runtime, training_updates=0)
    binding_sha = _pilot_json_sha_v12(binding)
    branch = _pilot_child_v12(root, 'paired_' + binding_sha[:16]); branch.mkdir(parents=True, exist_ok=True)
    _pilot_write_identity_v12(ns, branch, 'protocol.json', binding)
    conditions = protocol['conditions']
    packets, clean_packets = {}, {}
    for label in STAGE_EFFECT_MODELS_V32:
        clean_packets[label] = _stage_cache_v32(_pilot_child_v12(branch, label+'/clean.json'), binding_sha,
            dict(model=label, condition=None), hashes, runtime)
        for c in conditions:
            packets[label, c['id']] = _stage_cache_v32(_pilot_child_v12(branch, label+'/conditions/'+c['id']+'.json'),
                binding_sha, dict(model=label, condition=c), hashes, runtime)
    reused = sum(p is not None for p in packets.values())
    print(f'Stage1 vs Stage2 | 32 images x 36 powers x 2 models | cached {reused}/72 cases', flush=True)
    data_manifest = None
    model_identities = {}
    if reused < 72 or any(p is None for p in clean_packets.values()):
        for path, expected_bytes in (
            (_pilot_child_v12(ns['DRIVE_DATA_DIR'], ns['STAGE1_PIN_V32']['file']), ns['STAGE1_PIN_V32']['bytes']),
            (_pilot_child_v12(source, ns['COMPACT_FINAL_INPUT_LOCK_V26']['source_checkpoint']['file']),
             ns['COMPACT_FINAL_INPUT_LOCK_V26']['source_checkpoint']['bytes'])):
            _stage_require_v32(not path.is_symlink() and path.is_file() and path.stat().st_size == expected_bytes,
                f'Restore the complete original checkpoint from NAS to: {path} (expected {expected_bytes} bytes)')
        print('Preparing frozen test data only; no training ZIPs, native codec or FEC', flush=True)
        all_images, all_hashes, data_manifest = ns['prepare_compact_final_data_v26'](
            ns['DRIVE_DATA_DIR'], local/'test_zip', ns['COMPACT_FINAL_DATA_LOCK_V26'], configuration)
        _stage_require_v32(all_hashes == ns['COMPACT_FINAL_DATA_LOCK_V26']['selected_file_sha256'], 'Test membership differs')
        images = all_images[subset['positions']].clone(); del all_images
        _pilot_write_identity_v12(ns, branch, 'data_manifest.json', data_manifest)
        link = ns['RebuildIndexLinkV12'](ns, ns['COMPACT_FINAL_INPUT_LOCK_V26']['profile'],
            base_seed=configuration['channel']['base_seed'])
        _stage_require_v32(link.metadata() == configuration['channel'], 'Frozen channel or v3 receiver differs')
        metrics = ns['create_stage_metrics_v32'](ns['device'])
        physical_cache = {}
        timings = []
        for label in STAGE_EFFECT_MODELS_V32:
            needed = clean_packets[label] is None or any(packets[label, c['id']] is None for c in conditions)
            if not needed:
                continue
            print(f'Loading pinned {label} for inference; optimizer is not restored', flush=True)
            state, identity = load_stage_effect_model_v32(ns, label, source, source_summary, configuration)
            model_identities[label] = identity
            model = ns['FSO_VQVAE']().to(ns['device']).eval().requires_grad_(False)
            model.load_state_dict(state, strict=True); del state; gc.collect()
            versions = tuple(p._version for p in model.parameters())
            with torch.inference_mode():
                encoded = [model.encode(image.to(ns['device']))[2].cpu() for image in images.split(1)]
                indices = torch.cat(encoded); del encoded
                if clean_packets[label] is None:
                    rows = []
                    for i, h in enumerate(hashes):
                        idx = indices[i:i+1].to(ns['device'])
                        pred = model.decode_indices(idx, idx.shape).clamp(-1, 1)
                        rows.append(dict(model=label, image_sha256=h,
                            **source_scores_v28(pred, images[i:i+1], metrics, ns['device'])))
                    clean_packets[label] = _stage_save_v32(branch, label+'/clean.json', binding_sha,
                        dict(model=label, condition=None), rows, hashes, runtime)
                for c in conditions:
                    if packets[label, c['id']] is not None:
                        continue
                    t0 = time.monotonic()
                    rows = [evaluate_stage_effect_frame_v32(ns, c, label, h, images[i:i+1],
                        indices=indices[i:i+1], model=model, metrics=metrics, physical_cache=physical_cache)
                        for i, h in enumerate(hashes)]
                    packets[label, c['id']] = _stage_save_v32(branch, label+'/conditions/'+c['id']+'.json',
                        binding_sha, dict(model=label, condition=c), rows, hashes, runtime)
                    timings.append(time.monotonic()-t0)
                    done = sum(p is not None for p in packets.values())
                    eta = (72-done)*float(np.mean(timings))/60
                    print(f"{done}/72 | {label} | {c['group']} {c['channel_kwargs']['launch_power_dbm']:g} dBm"
                          f" | {timings[-1]:.1f}s | ~{eta:.1f} min left", flush=True)
            _stage_require_v32(versions == tuple(p._version for p in model.parameters())
                and not any(p.grad is not None or p.requires_grad for p in model.parameters()),
                'Inference unexpectedly modified parameters or gradients')
            del model, indices; gc.collect()
            if torch.device(ns['device']).type == 'cuda':
                torch.cuda.empty_cache()
    else:
        data_manifest = json.loads(_pilot_child_v12(branch, 'data_manifest.json').read_text(encoding='utf-8'))
    data_lock = ns['COMPACT_FINAL_DATA_LOCK_V26']
    _stage_require_v32(data_manifest.get('data_lock_sha256') == data_lock['data_lock_sha256']
        and data_manifest.get('selected_file_sha256') == data_lock['selected_file_sha256']
        and data_manifest.get('images') == 200
        and data_manifest.get('test_zip_sha256') == data_lock['test_zip_sha256'],
        'Cached data-manifest identity differs; preserve the original inputs and results')
    rows = [r for label in STAGE_EFFECT_MODELS_V32 for c in conditions for r in packets[label, c['id']]['rows']]
    clean = [r for label in STAGE_EFFECT_MODELS_V32 for r in clean_packets[label]['rows']]
    _stage_require_v32(len(rows) == 2304 and len(clean) == 64, 'Incomplete fixed-grid experiment')
    result = summarize_stage_effect_v32(rows, clean, protocol)
    result.update(binding=binding, status='completed', raw_records=rows, clean_records=clean,
        data_manifest=data_manifest, training_updates=0, new_checkpoint_writes=0,
        model_load_identities_this_execution=model_identities,
        elapsed_this_execution_seconds=time.monotonic()-started)
    archive = publish_stage_effect_v32(branch, result)
    print('완료. 보내줄 파일:', archive, flush=True)
    return result, archive


In [ ]:
def create_stage_metrics_v32(device):
    import lpips
    return {name:lpips.LPIPS(net=name,version="0.1").to(device).eval().requires_grad_(False)
            for name in ("alex","vgg")}


In [ ]:
STAGE_EFFECT_CODE_SHA256_V32='b72d17ce414b3b739376d34f3248056a0c94031b60752f310710aeedfe9e55db'
STAGE_EFFECT_RESULT_V32,STAGE_EFFECT_ARCHIVE_V32=run_stage_effect_v32(globals(),code_sha256=STAGE_EFFECT_CODE_SHA256_V32)
if AUTO_DOWNLOAD_RESULTS_V32:
    from google.colab import files
    files.download(str(STAGE_EFFECT_ARCHIVE_V32))
